# MTAP deletion prediction: manuscript modeling workflow

Complete supplied L2, 44-gene Elastic Net, and deep-learning code, organized
into nine scientific sections. Historical regional and feature-reduction
analyses are retained as supporting development stages.

Code blocks declare analysis stages. The final execution cell can run them
in their original dependency order, which preserves shared variables and
artifact relationships while allowing the manuscript layout requested here.
Set `MTAP_RUN_WORKFLOW=1` before launch to execute the research pipeline.
Otherwise, the notebook registers all stages and summarizes available cached
predictions. No training or data download is started by default.

The GSE327167 input setup reads local reference labels. No patient data,
model weights, or execution outputs are embedded. Scientific execution has
not been repeated in this preparation environment.


## 1. Environment Setup and Data Ingestion

Configure the project root, inspect the environment, and read the audited source inputs.


In [ ]:
import os
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

PROJECT_ROOT = (
    Path(os.environ.get("MTAP_PROJECT_ROOT", ".")).expanduser().resolve()
)


def require(condition, message):
    """Stop when a recorded scientific input or artifact invariant is violated."""
    if not bool(condition):
        raise RuntimeError(message)


def _public_summary(namespace, stage_name):
    """Display only small aggregate metric tables without participant columns."""
    candidates = (
        "performance",
        "metrics",
        "summary",
        "results",
        "comparison",
        "results_df",
        "performance_df",
        "all_results",
        "metrics_df",
        "split_summary",
        "label_by_cancer",
        "counts",
        "coefficient_table",
        "coefficient_df",
        "threshold_results",
        "result_table",
    )
    metric_pattern = re.compile(
        r"auroc|auprc|brier|sensitivity|specificity|balanced_accuracy|"
        r"ci95|n_patients|n_positive|n_negative|coefficient|(^|_)ap($|_)",
        re.IGNORECASE,
    )
    identity_pattern = re.compile(
        r"patient_id|participant|sample_id|barcode|cell_id|case_id|gsm|"
        r"matrix_cell|source_cell|aliquot",
        re.IGNORECASE,
    )
    print(f"Completed: {stage_name}")
    for name in candidates:
        table = namespace.get(name)
        if (
            not isinstance(table, pd.DataFrame)
            or table.empty
            or len(table) > 40
        ):
            continue
        previous = namespace.get("_publication_previous_tables", {})
        if previous.get(name) == id(table):
            continue
        if identity_pattern.search(" ".join(map(str, table.columns))):
            continue
        if not any(metric_pattern.search(str(col)) for col in table.columns):
            continue
        values = table.astype(str).to_numpy().ravel()
        if any(
            re.search(r"TCGA-|C3[NL]-|BNO[0-9]|GSM[0-9]|[/\\]home[/\\]", v)
            for v in values
        ):
            continue
        display(table.reset_index(drop=True).round(4))
        return


In [ ]:
import ast as _nb9_ast
import functools as _nb9_functools
import inspect as _nb9_inspect
import textwrap as _nb9_textwrap


class _NotebookStageRegistry:
    """Compile declared source blocks as module code to preserve model scope."""

    def __init__(self, namespace):
        self.namespace = namespace
        self.stages = {}
        self.order = []
        self.completed = []

    def register(self, identifier):
        def decorate(function):
            source = _nb9_textwrap.dedent(_nb9_inspect.getsource(function))
            declaration = _nb9_ast.parse(source).body[0]
            module = _nb9_ast.Module(body=declaration.body, type_ignores=[])
            _nb9_ast.fix_missing_locations(module)
            code = compile(module, function.__code__.co_filename, "exec")
            self.stages[identifier] = code

            @_nb9_functools.wraps(function)
            def execute():
                return self.run(identifier)

            return execute

        return decorate

    def run(self, identifier):
        if identifier in self.completed:
            return
        position = self.order.index(identifier)
        missing = [
            key for key in self.order[:position] if key not in self.completed
        ]
        if missing:
            raise RuntimeError(
                "Earlier source stages are incomplete. Run the final workflow "
                "execution cell to preserve dependency order."
            )
        print(f"Running source stage: {identifier}")
        exec(self.stages[identifier], self.namespace)
        self.completed.append(identifier)

    def run_all(self):
        for identifier in self.order:
            self.run(identifier)


_nb9_registry = _NotebookStageRegistry(globals())
RUN_FULL_WORKFLOW = os.environ.get("MTAP_RUN_WORKFLOW", "0") == "1"
DEPLOYMENT_THRESHOLD = 0.75


### Data ingestion


#### TCGA patient, RNA, and copy-number input audit


In [ ]:
@_nb9_registry.register("s02_0")
def _nb9_stage_s02_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from importlib import metadata as package_metadata
    import sys
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    PATIENT_FILE = (
        PROJECT / "data_processed" / "TCGA_primary_solid_MTAP_DNA_RNA.tsv"
    )
    RNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA"
        / "EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv"
    )
    PURITY_FILE = (
        PROJECT
        / "data_processed"
        / "TCGA_primary_solid_MTAP_DNA_RNA_purity.tsv"
    )
    OUTPUT_DIR = (
        PROJECT / "data_processed" / "MTAP_RNA_prediction" / "01_input_audit"
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for package in [
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "matplotlib",
        "joblib",
    ]:
        try:
            version = package_metadata.version(package)
        except package_metadata.PackageNotFoundError:
            version = "Not installed"
    for label, path in [
        ("Patient-level DNA/RNA table", PATIENT_FILE),
        ("Unified RNA matrix", RNA_FILE),
        ("Purity-linked table", PURITY_FILE),
    ]:
        pass
    for path in [PATIENT_FILE, RNA_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "TCGA patient, RNA, and copy-number input audit: integrity check failed; condition: input or artifact integrity check."
            )
    patients = pd.read_csv(
        PATIENT_FILE,
        sep="\t",
        dtype={"patient_id": "string", "acronym": "string"},
    )
    required = {"patient_id", "acronym", "MTAP_GISTIC_call"}
    missing = required.difference(patients.columns)
    if missing:
        raise ValueError(
            "TCGA patient, RNA, and copy-number input audit: integrity check failed; condition: input or artifact integrity check."
        )
    patients["patient_id"] = patients["patient_id"].str.strip().str.upper()
    patients["acronym"] = patients["acronym"].str.strip().str.upper()
    patients["MTAP_GISTIC_call"] = pd.to_numeric(
        patients["MTAP_GISTIC_call"], errors="coerce"
    )
    invalid_identity = (
        patients["patient_id"].isna()
        | patients["patient_id"].eq("")
        | patients["acronym"].isna()
        | patients["acronym"].eq("")
    )
    invalid_call = ~patients["MTAP_GISTIC_call"].isin([-2, -1, 0, 1, 2])
    if invalid_identity.any() or invalid_call.any():
        invalid = patients.loc[invalid_identity | invalid_call].copy()
        invalid.to_csv(
            OUTPUT_DIR / "invalid_patient_records.tsv", sep="\t", index=False
        )
        raise ValueError(
            "TCGA patient, RNA, and copy-number input audit: integrity check failed; condition: input or artifact integrity check."
        )
    duplicated = patients["patient_id"].duplicated(keep=False)
    if duplicated.any():
        patients.loc[duplicated].to_csv(
            OUTPUT_DIR / "duplicated_patients.tsv", sep="\t", index=False
        )
        raise ValueError(
            "TCGA patient, RNA, and copy-number input audit: integrity check failed; condition: input or artifact integrity check."
        )
    patients["MTAP_GISTIC_call"] = patients["MTAP_GISTIC_call"].astype(int)
    patients["MTAP_deep_label"] = (
        patients["MTAP_GISTIC_call"].eq(-2).astype(int)
    )
    counts = (
        patients.groupby("acronym", observed=True)
        .agg(
            n_patients=("patient_id", "size"),
            n_positive=("MTAP_deep_label", "sum"),
        )
        .reset_index()
    )
    counts["n_negative"] = counts["n_patients"] - counts["n_positive"]
    counts["positive_pct"] = 100 * counts["n_positive"] / counts["n_patients"]
    counts["has_both_classes"] = counts["n_positive"].gt(0) & counts[
        "n_negative"
    ].gt(0)
    counts = counts.sort_values(
        ["n_positive", "acronym"], ascending=[False, True]
    ).reset_index(drop=True)
    rna_columns = pd.read_csv(RNA_FILE, sep="\t", nrows=0).columns.tolist()
    gene_id_column = rna_columns[0]
    sample_columns = rna_columns[1:]
    records = []
    for sample in sample_columns:
        barcode = str(sample).strip()
        fields = barcode.split("-")
        if len(fields) >= 4 and fields[0] == "TCGA" and (len(fields[3]) >= 2):
            sample_type = fields[3][:2]
            if sample_type == "01":
                records.append(
                    {
                        "patient_id": "-".join(fields[:3]),
                        "RNA_column": sample,
                        "sample_type": sample_type,
                    }
                )
    if not records:
        raise ValueError(
            "TCGA patient, RNA, and copy-number input audit: integrity check failed; condition: input or artifact integrity check."
        )
    rna_map = pd.DataFrame(records)
    rna_per_patient = rna_map.groupby("patient_id", as_index=False).agg(
        n_primary_RNA_columns=("RNA_column", "size"),
        primary_RNA_columns=("RNA_column", lambda values: ";".join(values)),
    )
    audited = patients.merge(
        rna_per_patient,
        on="patient_id",
        how="left",
        validate="one_to_one",
        indicator="RNA_header_match",
    )
    unmatched = audited.loc[audited["RNA_header_match"].ne("both")].copy()
    if not unmatched.empty:
        unmatched.to_csv(
            OUTPUT_DIR / "patients_without_primary_RNA.tsv",
            sep="\t",
            index=False,
        )
        raise ValueError(
            "TCGA patient, RNA, and copy-number input audit: integrity check failed; condition: input or artifact integrity check."
        )
    candidate_paths = set()
    known_processed = (
        PROJECT / "data_processed" / "TCGA_primary_solid_9p21_calls.tsv"
    )
    if known_processed.is_file():
        candidate_paths.add(known_processed)
    dna_search_root = PROJECT / "data_raw/PANCAN/TCGA"
    if dna_search_root.is_dir():
        for path in dna_search_root.rglob("*"):
            if not path.is_file():
                continue
            name = path.name.lower()
            is_candidate = (
                "thresholded" in name
                or "by_genes" in name
                or (
                    "9p21" in name
                    and (".tsv" in name or ".txt" in name or ".csv" in name)
                )
            )
            if is_candidate:
                candidate_paths.add(path)
    candidate_records = []
    if not candidate_paths:
        pass
    for index, path in enumerate(sorted(candidate_paths, key=str), start=1):
        size_mb = path.stat().st_size / 1024**2
        candidate_records.append({"path": str(path), "size_MB": size_mb})
        name = path.name.lower()
        if name.endswith(
            (".tsv", ".txt", ".tsv.gz", ".txt.gz", ".csv", ".csv.gz")
        ):
            separator = "," if ".csv" in name else "\t"
            try:
                preview = pd.read_csv(path, sep=separator, nrows=3)
            except Exception as error:
                pass
    audited.to_csv(
        OUTPUT_DIR / "TCGA_prediction_patient_audit.tsv", sep="\t", index=False
    )
    counts.to_csv(
        OUTPUT_DIR / "TCGA_prediction_label_counts_by_cancer.tsv",
        sep="\t",
        index=False,
    )
    rna_map.to_csv(
        OUTPUT_DIR / "TCGA_primary_RNA_column_map.tsv", sep="\t", index=False
    )
    pd.DataFrame(candidate_records, columns=["path", "size_MB"]).to_csv(
        OUTPUT_DIR / "existing_DNA_candidate_files.tsv", sep="\t", index=False
    )
    _public_summary(
        globals(), "TCGA patient, RNA, and copy-number input audit"
    )


## 2. Exploratory Data Analysis and Cohort Definition

Link DNA and RNA records, define MTAP deep deletion, and summarize the eligible cohort.


### Cohort definition


#### TCGA cohort linkage and DNA target definitions


In [ ]:
@_nb9_registry.register("s03a_0")
def _nb9_stage_s03a_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split

    PROJECT = Path(str(PROJECT_ROOT))
    PATIENT_FILE = (
        PROJECT / "data_processed/TCGA_primary_solid_MTAP_DNA_RNA.tsv"
    )
    DNA_FILE = PROJECT / "data_processed/TCGA_primary_solid_9p21_calls.tsv"
    OUTPUT_DIR = (
        PROJECT / "data_processed/MTAP_RNA_prediction/02_labels_and_split"
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    RANDOM_SEED = 20261016
    TEST_FRACTION = 0.2
    SPLIT_FILE = OUTPUT_DIR / "fixed_patient_split.tsv"
    VALID_CALLS = [-2, -1, 0, 1, 2]
    for path in [PATIENT_FILE, DNA_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
    patients = pd.read_csv(
        PATIENT_FILE,
        sep="\t",
        dtype={"patient_id": "string", "acronym": "string"},
    )
    dna = pd.read_csv(
        DNA_FILE, sep="\t", dtype={"patient_id": "string", "acronym": "string"}
    )
    required_patient = {
        "patient_id",
        "acronym",
        "MTAP_GISTIC_call",
        "MTAP_RNA",
        "MTAP_log2_RNA",
    }
    required_dna = {
        "patient_id",
        "acronym",
        "MTAP_GISTIC_call",
        "CDKN2A_GISTIC_call",
        "CDKN2B_GISTIC_call",
    }
    for name, frame, required in [
        ("DNA/RNA table", patients, required_patient),
        ("9p21 table", dna, required_dna),
    ]:
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
        for column in ["patient_id", "acronym"]:
            frame[column] = frame[column].str.strip().str.upper()
            if frame[column].isna().any() or frame[column].eq("").any():
                raise ValueError(
                    "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
                )
        if frame["patient_id"].duplicated().any():
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
    dna_columns = [
        "patient_id",
        "acronym",
        "MTAP_GISTIC_call",
        "CDKN2A_GISTIC_call",
        "CDKN2B_GISTIC_call",
    ]
    cohort = patients.merge(
        dna[dna_columns].rename(
            columns={
                "acronym": "DNA_acronym",
                "MTAP_GISTIC_call": "MTAP_GISTIC_call_DNA",
            }
        ),
        on="patient_id",
        how="left",
        validate="one_to_one",
        indicator="DNA_match",
    )
    if cohort["DNA_match"].ne("both").any():
        raise ValueError(
            "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
        )
    call_columns = [
        "MTAP_GISTIC_call",
        "MTAP_GISTIC_call_DNA",
        "CDKN2A_GISTIC_call",
        "CDKN2B_GISTIC_call",
    ]
    for column in call_columns:
        cohort[column] = pd.to_numeric(cohort[column], errors="coerce")
        if not cohort[column].isin(VALID_CALLS).all():
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
        cohort[column] = cohort[column].astype(int)
    if not cohort["acronym"].eq(cohort["DNA_acronym"]).all():
        raise ValueError(
            "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
        )
    if not cohort["MTAP_GISTIC_call"].eq(cohort["MTAP_GISTIC_call_DNA"]).all():
        raise ValueError(
            "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
        )
    cohort = cohort.drop(
        columns=["DNA_acronym", "MTAP_GISTIC_call_DNA", "DNA_match"]
    )
    cohort["label_MTAP_deep"] = cohort["MTAP_GISTIC_call"].eq(-2).astype(int)
    cohort["label_MTAP_CDKN2B_codeep"] = (
        cohort["MTAP_GISTIC_call"].eq(-2) & cohort["CDKN2B_GISTIC_call"].eq(-2)
    ).astype(int)
    cohort["DNA_subgroup"] = np.select(
        [
            cohort["label_MTAP_CDKN2B_codeep"].eq(1),
            cohort["MTAP_GISTIC_call"].eq(-2),
            cohort["CDKN2B_GISTIC_call"].eq(-2),
            cohort["MTAP_GISTIC_call"].eq(-1),
            cohort["MTAP_GISTIC_call"].eq(0),
        ],
        [
            "MTAP_CDKN2B_codeep",
            "MTAP_deep_without_CDKN2B_deep",
            "CDKN2B_deep_without_MTAP_deep",
            "MTAP_shallow_without_CDKN2B_deep",
            "MTAP_neutral_without_CDKN2B_deep",
        ],
        default="MTAP_gain_without_CDKN2B_deep",
    )
    cross_table = pd.crosstab(
        cohort["MTAP_GISTIC_call"], cohort["CDKN2B_GISTIC_call"], dropna=False
    )
    n_mt = int(cohort["label_MTAP_deep"].sum())
    n_joint = int(cohort["label_MTAP_CDKN2B_codeep"].sum())
    if n_mt > 0:
        pass
    for column in ["MTAP_RNA", "MTAP_log2_RNA"]:
        cohort[column] = pd.to_numeric(cohort[column], errors="coerce")
    expression = cohort[["MTAP_RNA", "MTAP_log2_RNA"]].to_numpy(dtype=float)
    if not np.isfinite(expression).all():
        raise ValueError(
            "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
        )
    if cohort["MTAP_RNA"].lt(0).any():
        raise ValueError(
            "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
        )
    expected_log = np.log2(cohort["MTAP_RNA"].to_numpy(dtype=float) + 1.0)
    observed_log = cohort["MTAP_log2_RNA"].to_numpy(dtype=float)
    absolute_error = np.abs(expected_log - observed_log)
    if not np.allclose(expected_log, observed_log, rtol=1e-05, atol=1e-05):
        raise ValueError(
            "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
        )
    cohort.to_csv(
        OUTPUT_DIR / "TCGA_prediction_labeled_cohort.tsv",
        sep="\t",
        index=False,
    )
    cross_table.to_csv(
        OUTPUT_DIR / "MTAP_CDKN2B_GISTIC_crosstab.tsv", sep="\t"
    )
    label_by_cancer = cohort.groupby("acronym", as_index=False).agg(
        n_patients=("patient_id", "size"),
        n_MTAP_deep=("label_MTAP_deep", "sum"),
        n_codeep=("label_MTAP_CDKN2B_codeep", "sum"),
    )
    label_by_cancer.to_csv(
        OUTPUT_DIR / "DNA_labels_by_cancer.tsv", sep="\t", index=False
    )
    _public_summary(
        globals(), "TCGA cohort linkage and DNA target definitions"
    )


## 3. Data Preprocessing and Patient Splitting

Preserve the original 80/20 patient split and model-specific feature definitions. Input-preparation stages are executed when their upstream artifacts are available.


### Patient split


#### Fixed 80/20 development and holdout split


In [ ]:
@_nb9_registry.register("s03b_0")
def _nb9_stage_s03b_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    cohort = cohort.sort_values("patient_id").reset_index(drop=True)
    if SPLIT_FILE.is_file():
        split = pd.read_csv(
            SPLIT_FILE, sep="\t", dtype={"patient_id": "string"}
        )
        if not {"patient_id", "dataset_split", "label_MTAP_deep"}.issubset(
            split.columns
        ):
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
        if split["patient_id"].duplicated().any():
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
        if set(split["patient_id"]) != set(cohort["patient_id"]):
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
        check = cohort[["patient_id", "label_MTAP_deep"]].merge(
            split[["patient_id", "label_MTAP_deep"]],
            on="patient_id",
            suffixes=("_current", "_saved"),
            validate="one_to_one",
        )
        if (
            not check["label_MTAP_deep_current"]
            .eq(check["label_MTAP_deep_saved"])
            .all()
        ):
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
        if not split["dataset_split"].isin(["development", "test"]).all():
            raise ValueError(
                "TCGA labels, co-deletion checks, and fixed patient split: integrity check failed; condition: input or artifact integrity check."
            )
    else:
        development_ids, test_ids = train_test_split(
            cohort["patient_id"].to_numpy(),
            test_size=TEST_FRACTION,
            random_state=RANDOM_SEED,
            stratify=cohort["label_MTAP_deep"].to_numpy(),
        )
        test_ids = set(test_ids)
        split = cohort[["patient_id", "acronym", "label_MTAP_deep"]].copy()
        split["dataset_split"] = np.where(
            split["patient_id"].isin(test_ids), "test", "development"
        )
        split.to_csv(SPLIT_FILE, sep="\t", index=False)
    cohort = cohort.merge(
        split[["patient_id", "dataset_split"]],
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
    cohort.to_csv(
        OUTPUT_DIR / "TCGA_prediction_cohort_with_split.tsv",
        sep="\t",
        index=False,
    )
    split_summary = cohort.groupby("dataset_split", as_index=False).agg(
        n_patients=("patient_id", "size"),
        n_positive=("label_MTAP_deep", "sum"),
    )
    split_summary["n_negative"] = (
        split_summary["n_patients"] - split_summary["n_positive"]
    )
    split_summary["positive_pct"] = (
        100 * split_summary["n_positive"] / split_summary["n_patients"]
    )
    split_cancer = cohort.groupby(
        ["acronym", "dataset_split"], as_index=False
    ).agg(
        n_patients=("patient_id", "size"),
        n_positive=("label_MTAP_deep", "sum"),
    )
    split_cancer["n_negative"] = (
        split_cancer["n_patients"] - split_cancer["n_positive"]
    )
    split_summary.to_csv(
        OUTPUT_DIR / "split_summary.tsv", sep="\t", index=False
    )
    split_cancer.to_csv(
        OUTPUT_DIR / "split_counts_by_cancer.tsv", sep="\t", index=False
    )
    config = {
        "main_target": "MTAP_GISTIC_call == -2",
        "secondary_target": "MTAP_GISTIC_call == -2 AND CDKN2B_GISTIC_call == -2",
        "random_seed": RANDOM_SEED,
        "test_fraction_on_initial_creation": TEST_FRACTION,
        "initial_split_stratification": "main DNA label",
        "n_patients": int(len(cohort)),
        "training_performed": False,
    }
    with open(
        OUTPUT_DIR / "preparation_config.json", "w", encoding="utf-8"
    ) as handle:
        json.dump(config, handle, ensure_ascii=False, indent=2)
    _public_summary(globals(), "Fixed 80/20 development and holdout split")


### Input and feature preparation


#### Reference pan-cancer source-code preparation


In [ ]:
@_nb9_registry.register("s09_0")
def _nb9_stage_s09_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path, PurePosixPath
    import hashlib
    import stat
    import zipfile

    PROJECT = Path(str(PROJECT_ROOT))
    ZIP_FILE = None
    DESTINATION = PROJECT / "code_external" / "greenelab_pancancer"
    if ZIP_FILE is None:
        candidates = sorted(
            (
                path
                for path in PROJECT.rglob("*")
                if path.is_file()
                and path.suffix.lower() == ".zip"
                and ("pancancer" in path.name.lower())
            )
        )
        for index, path in enumerate(candidates, start=1):
            pass
        if len(candidates) == 0:
            raise FileNotFoundError(
                "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
            )
        if len(candidates) > 1:
            raise ValueError(
                "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
            )
        ZIP_FILE = candidates[0]
    else:
        ZIP_FILE = Path(ZIP_FILE)
    if not ZIP_FILE.is_file():
        raise FileNotFoundError(
            "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if not zipfile.is_zipfile(ZIP_FILE):
        raise ValueError(
            "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
        )
    with zipfile.ZipFile(ZIP_FILE) as archive:
        main_script_suffix = "scripts/pancancer_classifier.py"
        matches = [
            info.filename
            for info in archive.infolist()
            if not info.is_dir()
            and (
                info.filename == main_script_suffix
                or info.filename.endswith("/" + main_script_suffix)
            )
        ]
        if len(matches) != 1:
            raise ValueError(
                "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
            )
        archive_prefix = matches[0][: -len(main_script_suffix)]
        planned_files = []
        for info in archive.infolist():
            if info.is_dir():
                continue
            if not info.filename.startswith(archive_prefix):
                continue
            relative_name = info.filename[len(archive_prefix) :]
            if not relative_name:
                continue
            if "\\" in relative_name:
                raise ValueError(
                    "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                )
            relative_path = PurePosixPath(relative_name)
            if (
                relative_path.is_absolute()
                or ".." in relative_path.parts
                or ":" in relative_name
            ):
                raise ValueError(
                    "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                )
            file_mode = info.external_attr >> 16
            if stat.S_ISLNK(file_mode):
                raise ValueError(
                    "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                )
            target = DESTINATION.joinpath(*relative_path.parts)
            planned_files.append((info, target))
        if not planned_files:
            raise ValueError(
                "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
            )
        targets = [str(target) for _, target in planned_files]
        if len(targets) != len(set(targets)):
            raise ValueError(
                "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
            )
        for info, target in planned_files:
            if target.is_symlink():
                raise ValueError(
                    "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                )
            if target.exists():
                if not target.is_file():
                    raise ValueError(
                        "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                    )
                incoming_hash = hashlib.sha256(archive.read(info)).hexdigest()
                existing_hash = hashlib.sha256(target.read_bytes()).hexdigest()
                if incoming_hash != existing_hash:
                    raise ValueError(
                        "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                    )
        n_written = 0
        n_reused = 0
        for info, target in planned_files:
            if target.is_file():
                n_reused += 1
                continue
            destination_root = DESTINATION.resolve()
            if not target.resolve().is_relative_to(destination_root):
                raise ValueError(
                    "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
                )
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source:
                with target.open("xb") as output:
                    while True:
                        block = source.read(1024 * 1024)
                        if not block:
                            break
                        output.write(block)
            n_written += 1
    required_files = [
        "README.md",
        "environment.yml",
        "tp53_analysis.sh",
        "scripts/pancancer_classifier.py",
        "scripts/util/tcga_util.py",
    ]
    missing_files = []
    for relative_name in required_files:
        path = DESTINATION / relative_name
        if not path.is_file():
            missing_files.append(relative_name)
    if missing_files:
        raise FileNotFoundError(
            "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
        )
    lfs_pointers = []
    for relative_name in required_files:
        path = DESTINATION / relative_name
        with path.open("rb") as handle:
            start = handle.read(150)
        if start.startswith(b"version https://git-lfs.github.com/spec/v1"):
            lfs_pointers.append(str(path))
    if lfs_pointers:
        raise RuntimeError(
            "Reference pan-cancer source-code preparation: integrity check failed; condition: input or artifact integrity check."
        )
    utility_file = DESTINATION / "scripts/util/tcga_util.py"
    _public_summary(globals(), "Reference pan-cancer source-code preparation")


#### 9p21.3 candidate features and Entrez identity audit


In [ ]:
@_nb9_registry.register("s10_0")
def _nb9_stage_s10_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import re
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    DNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA/all_thresholded.by_genes_whitelisted.tsv"
    )
    RNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv"
    )
    OUTPUT_DIR = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction/08_9p21_region_candidate_audit"
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    TARGET_CYTOBAND = "9p21.3"
    EXPECTED_CORE_GENES = {"MTAP": "4507", "CDKN2A": "1029", "CDKN2B": "1030"}
    for path in [DNA_FILE, RNA_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "9p21.3 candidate features and Entrez identity audit: integrity check failed; condition: input or artifact integrity check."
            )

    def clean_entrez(value):
        if pd.isna(value):
            return ""
        value = str(value).strip()
        if re.fullmatch("\\d+(?:\\.0+)?", value):
            return value.split(".")[0]
        return ""

    def parse_rna_gene_id(value):
        """Parse symbol/Entrez identifiers without replacing ambiguous members."""
        raw = str(value).strip()
        if "|" in raw:
            symbol, identifier = raw.rsplit("|", 1)
            return (raw, symbol.strip(), clean_entrez(identifier))
        return (raw, raw, "")

    dna_columns = ["Gene Symbol", "Locus ID", "Cytoband"]
    dna_annotation = pd.read_csv(
        DNA_FILE, sep="\t", usecols=dna_columns, dtype="string"
    )
    dna_annotation = dna_annotation.rename(
        columns={
            "Gene Symbol": "DNA_symbol",
            "Locus ID": "Entrez_ID",
            "Cytoband": "Cytoband",
        }
    )
    for column in ["DNA_symbol", "Cytoband"]:
        dna_annotation[column] = dna_annotation[column].fillna("").str.strip()
    dna_annotation["Entrez_ID"] = dna_annotation["Entrez_ID"].map(clean_entrez)
    candidates = dna_annotation.loc[
        dna_annotation["Cytoband"].eq(TARGET_CYTOBAND)
    ].copy()
    candidates = candidates.drop_duplicates().reset_index(drop=True)
    if candidates.empty:
        raise ValueError(
            "9p21.3 candidate features and Entrez identity audit: integrity check failed; condition: input or artifact integrity check."
        )
    for symbol, entrez in EXPECTED_CORE_GENES.items():
        matched = candidates.loc[candidates["Entrez_ID"].eq(entrez)]
        if matched.empty:
            raise ValueError(
                "9p21.3 candidate features and Entrez identity audit: integrity check failed; condition: input or artifact integrity check."
            )
    rna_header = pd.read_csv(RNA_FILE, sep="\t", nrows=0)
    gene_column = rna_header.columns[0]
    candidate_ids = set(
        candidates.loc[candidates["Entrez_ID"].ne(""), "Entrez_ID"]
    )
    rna_records = []
    for chunk in pd.read_csv(
        RNA_FILE,
        sep="\t",
        usecols=[gene_column],
        dtype="string",
        chunksize=4096,
    ):
        for value in chunk[gene_column].dropna():
            raw, symbol, entrez = parse_rna_gene_id(value)
            if entrez in candidate_ids:
                rna_records.append(
                    {
                        "Entrez_ID": entrez,
                        "RNA_gene_id": raw,
                        "RNA_symbol": symbol,
                    }
                )
    rna_matches = pd.DataFrame(
        rna_records, columns=["Entrez_ID", "RNA_gene_id", "RNA_symbol"]
    )
    rna_summary_rows = []
    for entrez, subset in rna_matches.groupby("Entrez_ID", sort=False):
        rna_summary_rows.append(
            {
                "Entrez_ID": entrez,
                "n_RNA_rows": len(subset),
                "RNA_gene_ids": "; ".join(sorted(set(subset["RNA_gene_id"]))),
                "RNA_symbols": "; ".join(sorted(set(subset["RNA_symbol"]))),
            }
        )
    rna_summary = pd.DataFrame(
        rna_summary_rows,
        columns=["Entrez_ID", "n_RNA_rows", "RNA_gene_ids", "RNA_symbols"],
    )
    audit = candidates.merge(
        rna_summary, on="Entrez_ID", how="left", validate="many_to_one"
    )
    audit["n_RNA_rows"] = audit["n_RNA_rows"].fillna(0).astype(int)
    for column in ["RNA_gene_ids", "RNA_symbols"]:
        audit[column] = audit[column].fillna("")
    audit["mapping_status"] = "Not_found_by_Entrez"
    audit.loc[audit["n_RNA_rows"].eq(1), "mapping_status"] = (
        "Unique_Entrez_match"
    )
    audit.loc[audit["n_RNA_rows"].gt(1), "mapping_status"] = (
        "Multiple_RNA_rows_review_required"
    )
    audit.loc[audit["Entrez_ID"].eq(""), "mapping_status"] = (
        "Missing_numeric_Entrez"
    )
    audit["is_core_three_gene"] = audit["Entrez_ID"].isin(
        EXPECTED_CORE_GENES.values()
    )
    audit = audit.sort_values(
        ["is_core_three_gene", "DNA_symbol"], ascending=[False, True]
    ).reset_index(drop=True)
    audit.to_csv(
        OUTPUT_DIR / "9p21_3_candidate_gene_RNA_mapping.tsv",
        sep="\t",
        index=False,
    )
    rna_matches.to_csv(
        OUTPUT_DIR / "9p21_3_matched_RNA_gene_rows.tsv", sep="\t", index=False
    )
    dna_annotation.groupby("Cytoband", dropna=False).size().rename(
        "n_annotation_rows"
    ).to_csv(OUTPUT_DIR / "DNA_cytoband_annotation_counts.tsv", sep="\t")
    _public_summary(
        globals(), "9p21.3 candidate features and Entrez identity audit"
    )


#### RCANE source, architecture, and environment audit


In [ ]:
@_nb9_registry.register("s21_0")
def _nb9_stage_s21_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from importlib import metadata
    from collections import deque
    import csv
    import hashlib
    import re
    import sys

    PROJECT = Path(str(PROJECT_ROOT))
    SOURCE = PROJECT / "code_external/RCANE"
    OUT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "13_RCANE_source_audit"
    )
    if not SOURCE.is_dir():
        raise FileNotFoundError(
            "RCANE source, architecture, and environment audit: integrity check failed; condition: input or artifact integrity check."
        )
    required = [
        "README.md",
        "requirements.txt",
        "preprocess.py",
        "train.py",
        "predict.py",
        "postprocess.py",
    ]
    missing = [name for name in required if not (SOURCE / name).is_file()]
    if missing:
        raise FileNotFoundError(
            "RCANE source, architecture, and environment audit: integrity check failed; condition: input or artifact integrity check."
        )
    OUT.mkdir(parents=True, exist_ok=True)
    report = []

    def record(text=""):
        report.append(str(text))

    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    record("===== Environment =====")
    record(f"Python：{sys.executable}")
    record(f"Version: {sys.version}")
    record(f"Source path: {SOURCE}")
    excluded_parts = {
        ".git",
        "__pycache__",
        ".ipynb_checkpoints",
        ".venv",
        "venv",
    }
    files = sorted(
        (
            path
            for path in SOURCE.rglob("*")
            if path.is_file()
            and (
                not excluded_parts.intersection(path.relative_to(SOURCE).parts)
            )
        )
    )
    inventory = []
    for path in files:
        inventory.append(
            {
                "relative_path": str(path.relative_to(SOURCE)),
                "size_bytes": path.stat().st_size,
                "suffix": path.suffix.lower(),
            }
        )
    inventory_file = OUT / "RCANE_file_inventory.tsv"
    with inventory_file.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=["relative_path", "size_bytes", "suffix"],
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(inventory)
    record("\n===== Source files =====")
    for item in inventory:
        record(f"{item['relative_path']}\t{item['size_bytes']} bytes")
        if item["suffix"] in {".py", ".yaml", ".yml"} or Path(
            item["relative_path"]
        ).name in {"README.md", "requirements.txt"}:
            pass
    record("\n===== Installed package versions =====")
    for package in [
        "numpy",
        "pandas",
        "scipy",
        "scikit-learn",
        "torch",
        "torch-geometric",
        "PyYAML",
    ]:
        try:
            version = metadata.version(package)
        except metadata.PackageNotFoundError:
            version = "Not installed"
        record(f"{package}: {version}")
    source_paths = {SOURCE / name for name in required}
    for name in ["train_config.yaml", "predict_config.yaml"]:
        path = SOURCE / name
        if path.is_file():
            source_paths.add(path)
    model_dir = SOURCE / "model"
    if model_dir.is_dir():
        source_paths.update(model_dir.rglob("*.py"))
        source_paths.update(model_dir.rglob("*.yaml"))
        source_paths.update(model_dir.rglob("*.yml"))
    MAX_SOURCE_BYTES = 2 * 1024 * 1024
    requirements_text = (SOURCE / "requirements.txt").read_text(
        encoding="utf-8-sig", errors="replace"
    )
    for path in sorted(source_paths):
        if not path.is_file():
            continue
        relative = path.relative_to(SOURCE)
        record("\n" + "=" * 72)
        record(f"File: {relative}")
        record(f"SHA256：{sha256_file(path)}")
        if path.stat().st_size > MAX_SOURCE_BYTES:
            record("File exceeds 2 MB; contents were not collected.")
            continue
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        record(text)
    table_suffixes = {".csv", ".tsv", ".txt", ".bed"}
    MAX_TABLE_BYTES = 25 * 1024 * 1024
    mtap_pattern = re.compile(
        "(?<![A-Za-z0-9])MTAP(?![A-Za-z0-9])", flags=re.IGNORECASE
    )
    table_paths = [
        path
        for path in files
        if path.suffix.lower() in table_suffixes
        and path.name != "requirements.txt"
    ]
    total_hits = 0
    for path in table_paths:
        relative = path.relative_to(SOURCE)
        size = path.stat().st_size
        record("\n" + "-" * 72)
        record(f"Table: {relative}")
        record(f"Size: {size:,} bytes")
        if size > MAX_TABLE_BYTES:
            record("File exceeds 25 MB; contents were not scanned.")
            continue
        first_lines = []
        hit_contexts = []
        previous = deque(maxlen=1)
        pending_next = False
        n_hits = 0
        with path.open("r", encoding="utf-8-sig", errors="replace") as handle:
            for line_number, line in enumerate(handle, start=1):
                line = line.rstrip("\n\r")
                if line_number <= 3:
                    first_lines.append(f"Row {line_number}: {line[:2500]}")
                if pending_next:
                    hit_contexts.append(
                        f"Next row{line_number}：{line[:6000]}"
                    )
                    pending_next = False
                if mtap_pattern.search(line):
                    n_hits += 1
                    if n_hits <= 20:
                        for prev_number, prev_line in previous:
                            hit_contexts.append(
                                f"Previous row{prev_number}：{prev_line[:6000]}"
                            )
                        hit_contexts.append(
                            f"MTAP matching row{line_number}：{line[:12000]}"
                        )
                        pending_next = True
                previous.append((line_number, line))
        for line in first_lines:
            record(line)
        record(f"Rows containing MTAP: {n_hits}")
        for line in hit_contexts:
            record(line)
        total_hits += n_hits
        if n_hits:
            for line in hit_contexts:
                if line.startswith("MTAP matching row"):
                    pass
    weight_files = [
        path
        for path in files
        if path.suffix.lower() in {".pth", ".pt", ".ckpt", ".h5", ".hdf5"}
    ]
    record("\n===== Pretrained weights =====")
    if weight_files:
        for path in weight_files:
            description = (
                f"{path.relative_to(SOURCE)} ({path.stat().st_size:,} bytes)"
            )
            record(description)
    else:
        record("No common weight format was found.")
    record("\n===== Audit scope =====")
    record("No packages were installed or updated.")
    record("Reference source code was not executed or modified.")
    record("Pretrained models were not loaded.")
    record("TCGA expression was not read.")
    record("No model training or test evaluation was performed.")
    record(
        "MTAP text matches require additional regional-resolution verification."
    )
    report_file = OUT / "RCANE_source_review.txt"
    report_file.write_text("\n".join(report), encoding="utf-8")
    _public_summary(
        globals(), "RCANE source, architecture, and environment audit"
    )


#### RCANE-inspired regional feature and target definition


In [ ]:
@_nb9_registry.register("s22_0")
def _nb9_stage_s22_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from collections import defaultdict
    import csv
    import json
    import re
    import time
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    SOURCE = PROJECT / "code_external/RCANE"
    PREDICTION_DIR = PROJECT / "data_processed/MTAP_RNA_prediction"
    SPLIT_FILE = PREDICTION_DIR / "02_labels_and_split/fixed_patient_split.tsv"
    RNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA"
        / "EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv"
    )
    ORIGINAL_PAIR_FILE = (
        PROJECT / "data_processed/TCGA_primary_solid_MTAP_DNA_RNA.tsv"
    )
    GENE_FILE = SOURCE / "data/gene_names_in_segments.csv"
    SEGMENT_FILE = SOURCE / "data/start_end_chr_in_segs.csv"
    ANNOTATION_FILE = SOURCE / "data/ensembl-gene-info.hg38.tsv"
    OUT = PREDICTION_DIR / "14_RCANE_inspired_region_inputs"
    for path in [
        SPLIT_FILE,
        RNA_FILE,
        ORIGINAL_PAIR_FILE,
        GENE_FILE,
        SEGMENT_FILE,
        ANNOTATION_FILE,
    ]:
        if not path.is_file():
            raise FileNotFoundError(
                "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
            )
    OUT.mkdir(parents=True, exist_ok=True)
    split = pd.read_csv(
        SPLIT_FILE,
        sep="\t",
        dtype={"patient_id": str, "acronym": str, "dataset_split": str},
    )
    required_columns = {
        "patient_id",
        "acronym",
        "label_MTAP_deep",
        "dataset_split",
    }
    missing = required_columns.difference(split.columns)
    if missing:
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    if split["patient_id"].isna().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    if split["patient_id"].duplicated().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    development = (
        split.loc[split["dataset_split"].eq("development")]
        .copy()
        .set_index("patient_id")
    )
    labels = pd.to_numeric(development["label_MTAP_deep"], errors="raise")
    if labels.isna().any() or not labels.isin([0, 1]).all():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    development["label_MTAP_deep"] = labels.astype(int)
    development["MTAP_deep_label"] = development["label_MTAP_deep"]
    n_positive = int(development["MTAP_deep_label"].sum())
    if len(development) != 7431 or n_positive != 558:
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    if development["acronym"].isna().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    with GENE_FILE.open("r", encoding="utf-8-sig", newline="") as handle:
        gene_blocks = [
            [value.strip() for value in row] for row in csv.reader(handle)
        ]
    segments = pd.read_csv(SEGMENT_FILE)
    required_segment_columns = {"chrom", "loc.start", "loc.end"}
    if not required_segment_columns.issubset(segments.columns):
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    if len(gene_blocks) != len(segments):
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )

    def locate_gene(symbol):
        hits = [i for i, genes in enumerate(gene_blocks) if symbol in genes]
        if len(hits) != 1:
            raise ValueError(
                "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
            )
        return hits[0]

    mtap_segment = locate_gene("MTAP")
    cdkn2a_segment = locate_gene("CDKN2A")
    cdkn2b_segment = locate_gene("CDKN2B")
    if cdkn2a_segment != cdkn2b_segment:
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    central_segments = sorted({mtap_segment, cdkn2a_segment})
    if (
        len(central_segments) != 2
        or central_segments[1] - central_segments[0] != 1
    ):
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    selected_segments = list(
        range(central_segments[0] - 1, central_segments[-1] + 2)
    )
    if min(selected_segments) < 0 or max(selected_segments) >= len(segments):
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    region_records = []
    gene_records = []
    for segment_index in selected_segments:
        coordinate = segments.iloc[segment_index]
        chromosome = str(coordinate["chrom"]).removeprefix("chr")
        if chromosome != "9":
            raise ValueError(
                "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
            )
        if segment_index == mtap_segment:
            role = "MTAP_region"
        elif segment_index == cdkn2a_segment:
            role = "CDKN2A_B_region"
        elif segment_index < min(central_segments):
            role = "Upstream_flank"
        else:
            role = "Downstream_flank"
        genes = [
            gene
            for gene in gene_blocks[segment_index]
            if gene and gene != "None"
        ]
        region_records.append(
            {
                "segment_index_0based": segment_index,
                "region_role": role,
                "chrom": chromosome,
                "author_loc_start": coordinate["loc.start"],
                "author_loc_end": coordinate["loc.end"],
                "n_author_members": len(genes),
                "author_members": ";".join(genes),
            }
        )
        for position, gene in enumerate(genes):
            gene_records.append(
                {
                    "segment_index_0based": segment_index,
                    "region_role": role,
                    "position_in_segment_0based": position,
                    "author_symbol": gene,
                }
            )
    regions = pd.DataFrame(region_records)
    members = pd.DataFrame(gene_records)
    if members["author_symbol"].duplicated().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    regions.to_csv(OUT / "fixed_author_regions.tsv", sep="\t", index=False)
    annotation = pd.read_csv(
        ANNOTATION_FILE, sep="\t", dtype=str, keep_default_na=False
    )
    required_annotation_columns = {"Gene name", "NCBI gene ID"}
    if not required_annotation_columns.issubset(annotation.columns):
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )

    def clean_entrez(value):
        value = str(value).strip()
        if re.fullmatch("\\d+(?:\\.0+)?", value):
            return str(int(value.split(".")[0]))
        return ""

    annotation["clean_Entrez"] = annotation["NCBI gene ID"].map(clean_entrez)
    annotation["Gene name"] = annotation["Gene name"].str.strip()
    symbol_to_entrez = {}
    for symbol, group in annotation.groupby("Gene name"):
        identifiers = sorted(set(group["clean_Entrez"]) - {""})
        if len(identifiers) == 1:
            symbol_to_entrez[symbol] = identifiers[0]
    header = pd.read_csv(RNA_FILE, sep="\t", nrows=0).columns.tolist()
    gene_column = header[0]
    rna_gene_ids = pd.read_csv(
        RNA_FILE, sep="\t", usecols=[gene_column], dtype=str
    )[gene_column]
    if rna_gene_ids.isna().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    if rna_gene_ids.duplicated().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    entrez_to_rows = defaultdict(list)
    symbol_to_rows = defaultdict(list)
    for gene_id in rna_gene_ids:
        parts = gene_id.split("|")
        symbol = parts[0].strip()
        symbol_to_rows[symbol].append(gene_id)
        if len(parts) >= 2:
            identifier = clean_entrez(parts[-1])
            if identifier:
                entrez_to_rows[identifier].append(gene_id)
    mapping_records = []
    for row in members.to_dict("records"):
        symbol = row["author_symbol"]
        identifier = symbol_to_entrez.get(symbol, "")
        matched_gene_id = ""
        status = "Not_found"
        if identifier:
            hits = entrez_to_rows.get(identifier, [])
            if len(hits) == 1:
                matched_gene_id = hits[0]
                status = "Unique_Entrez_match"
            elif len(hits) > 1:
                status = "Ambiguous_Entrez_match"
        if status == "Not_found":
            hits = symbol_to_rows.get(symbol, [])
            if len(hits) == 1:
                matched_gene_id = hits[0]
                status = "Unique_exact_symbol_match"
            elif len(hits) > 1:
                status = "Ambiguous_symbol_match"
        mapping_records.append(
            {
                **row,
                "Entrez_ID": identifier,
                "RNA_gene_id": matched_gene_id,
                "mapping_status": status,
            }
        )
    mapping = pd.DataFrame(mapping_records)
    mapping.to_csv(
        OUT / "author_member_RNA_mapping.tsv", sep="\t", index=False
    )
    matched = mapping.loc[mapping["RNA_gene_id"].ne("")].copy()
    if matched["RNA_gene_id"].duplicated().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    for gene in ["MTAP", "CDKN2A", "CDKN2B"]:
        if gene not in set(matched["author_symbol"]):
            raise ValueError(
                "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
            )
    patient_set = set(development.index)
    sample_to_patient = {}
    for column in header[1:]:
        if not column.startswith("TCGA-"):
            continue
        parts = column.split("-")
        if len(parts) < 4:
            continue
        patient_id = "-".join(parts[:3])
        sample_type = parts[3][:2]
        if patient_id in patient_set and sample_type == "01":
            sample_to_patient[column] = patient_id
    missing_patients = patient_set - set(sample_to_patient.values())
    if missing_patients:
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    sample_columns = list(sample_to_patient)
    target_gene_ids = set(matched["RNA_gene_id"])
    found = set()
    pieces = []
    started = time.monotonic()
    reader = pd.read_csv(
        RNA_FILE,
        sep="\t",
        usecols=[gene_column] + sample_columns,
        dtype={gene_column: str},
        chunksize=512,
    )
    try:
        for chunk_index, chunk in enumerate(reader, start=1):
            selected = chunk.loc[chunk[gene_column].isin(target_gene_ids)]
            if not selected.empty:
                pieces.append(selected.copy())
                found.update(selected[gene_column])
            if chunk_index % 10 == 0:
                pass
            if found == target_gene_ids:
                break
    finally:
        reader.close()
    if found != target_gene_ids:
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    sample_expression = (
        pd.concat(pieces, ignore_index=True)
        .set_index(gene_column)
        .reindex(matched["RNA_gene_id"])
    )
    sample_expression = sample_expression.apply(pd.to_numeric, errors="raise")
    if np.isinf(sample_expression.to_numpy(dtype=float)).any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    patient_expression = sample_expression.T.copy()
    patient_expression.index = pd.Index(
        [sample_to_patient[sample] for sample in patient_expression.index],
        name="patient_id",
    )
    raw = (
        patient_expression.groupby(level=0, sort=False)
        .mean()
        .reindex(development.index)
    )
    raw.columns = matched["author_symbol"].tolist()
    raw.index.name = "patient_id"
    original_pair = pd.read_csv(
        ORIGINAL_PAIR_FILE,
        sep="\t",
        usecols=["patient_id", "MTAP_RNA"],
        dtype={"patient_id": str},
    )
    if original_pair["patient_id"].duplicated().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    reference = original_pair.set_index("patient_id")["MTAP_RNA"].reindex(
        raw.index
    )
    reference = pd.to_numeric(reference, errors="raise")
    if reference.isna().any():
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.allclose(
        raw["MTAP"].to_numpy(),
        reference.to_numpy(),
        rtol=1e-06,
        atol=1e-06,
        equal_nan=False,
    ):
        raise ValueError(
            "RCANE-inspired regional feature and target definition: integrity check failed; condition: input or artifact integrity check."
        )
    signed_log = np.sign(raw) * np.log2(1 + np.abs(raw))
    quality = pd.DataFrame(
        {
            "author_symbol": raw.columns,
            "n_patients_missing": raw.isna().sum().to_numpy(),
            "n_negative_raw": raw.lt(0).sum().to_numpy(),
            "n_observed": raw.notna().sum().to_numpy(),
        }
    )
    quality = mapping.merge(
        quality, on="author_symbol", how="left", validate="one_to_one"
    )
    raw.to_csv(
        OUT / "development_region_raw_expression.tsv", sep="\t", index=True
    )
    signed_log.to_csv(
        OUT / "development_region_signedlog_expression.tsv",
        sep="\t",
        index=True,
    )
    development.to_csv(
        OUT / "fixed_development_metadata.tsv", sep="\t", index=True
    )
    quality.to_csv(
        OUT / "member_expression_coverage.tsv", sep="\t", index=False
    )
    summary = (
        quality.assign(
            RNA_matched=lambda table: table["RNA_gene_id"].ne(""),
            complete_expression=lambda table: table["n_patients_missing"].eq(
                0
            ),
        )
        .groupby(["segment_index_0based", "region_role"], sort=False)
        .agg(
            n_author_members=("author_symbol", "size"),
            n_RNA_matched=("RNA_matched", "sum"),
            n_complete_members=("complete_expression", "sum"),
        )
        .reset_index()
    )
    summary.to_csv(OUT / "region_coverage_summary.tsv", sep="\t", index=False)
    manifest = {
        "stage": "exploratory_new_candidate_input_preparation",
        "source_code_directory": str(SOURCE),
        "label_source": str(SPLIT_FILE),
        "label_column_original": "label_MTAP_deep",
        "n_development_patients": len(development),
        "n_positive": n_positive,
        "selected_segment_indices_0based": selected_segments,
        "n_author_members": len(mapping),
        "n_matched_members": len(matched),
        "expression_transform": "sign(x)*log2(1+abs(x))",
        "expression_is_TPM": False,
        "missing_imputed": False,
        "test_patient_expression_read": False,
        "model_trained": False,
    }
    (OUT / "input_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _public_summary(
        globals(), "RCANE-inspired regional feature and target definition"
    )


#### Regional DL training-data preparation


In [ ]:
@_nb9_registry.register("s23_0")
def _nb9_stage_s23_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import subprocess
    import sys
    import time
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    ROOT = PROJECT / "data_processed/MTAP_RNA_prediction"
    INPUT_DIR = ROOT / "14_RCANE_inspired_region_inputs"
    OUT = ROOT / "15_region_multitask_DL" / "01_training_inputs"
    EXPRESSION_FILE = INPUT_DIR / "development_region_signedlog_expression.tsv"
    METADATA_FILE = INPUT_DIR / "fixed_development_metadata.tsv"
    MEMBER_FILE = INPUT_DIR / "author_member_RNA_mapping.tsv"
    DNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA"
        / "all_thresholded.by_genes_whitelisted.tsv"
    )
    DNA_SAMPLE_FILE = (
        PROJECT / "data_processed" / "TCGA_primary_solid_9p21_calls.tsv"
    )
    for path in [
        EXPRESSION_FILE,
        METADATA_FILE,
        MEMBER_FILE,
        DNA_FILE,
        DNA_SAMPLE_FILE,
    ]:
        if not path.is_file():
            raise FileNotFoundError(
                "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
            )
    OUT.mkdir(parents=True, exist_ok=True)
    metadata = pd.read_csv(
        METADATA_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    expression = pd.read_csv(EXPRESSION_FILE, sep="\t", index_col=0)
    expression.index = expression.index.astype(str)
    expression.index.name = "patient_id"
    required_metadata = {"acronym", "dataset_split", "label_MTAP_deep"}
    if not required_metadata.issubset(metadata.columns):
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if metadata.index.duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if expression.index.duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if expression.columns.duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if not metadata["dataset_split"].eq("development").all():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if set(expression.index) != set(metadata.index):
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    expression = expression.reindex(metadata.index)
    expression = expression.apply(pd.to_numeric, errors="raise")
    fixed_label = pd.to_numeric(metadata["label_MTAP_deep"], errors="raise")
    if fixed_label.isna().any() or not fixed_label.isin([0, 1]).all():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    metadata["label_MTAP_deep"] = fixed_label.astype(int)
    if len(metadata) != 7431 or metadata["label_MTAP_deep"].sum() != 558:
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if np.isinf(expression.to_numpy(dtype=float)).any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    dna_header = pd.read_csv(DNA_FILE, sep="\t", nrows=0).columns.tolist()
    annotation_columns = ["Gene Symbol", "Locus ID", "Cytoband"]
    if not set(annotation_columns).issubset(dna_header):
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    dna_annotation = pd.read_csv(
        DNA_FILE, sep="\t", usecols=annotation_columns, dtype=str
    )
    targets = dna_annotation.loc[
        dna_annotation["Cytoband"].fillna("").str.strip().eq("9p21.3")
    ].copy()
    targets["Entrez_ID"] = (
        pd.to_numeric(targets["Locus ID"], errors="raise")
        .astype("int64")
        .astype(str)
    )
    targets = targets.rename(columns={"Gene Symbol": "DNA_symbol"})
    targets = targets[["DNA_symbol", "Entrez_ID", "Cytoband"]].reset_index(
        drop=True
    )
    if targets.empty:
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if targets["Entrez_ID"].duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    for gene in ["MTAP", "CDKN2A", "CDKN2B"]:
        if not targets["DNA_symbol"].eq(gene).any():
            raise ValueError(
                "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
            )
    targets["target_index"] = np.arange(len(targets))
    dna_samples = pd.read_csv(
        DNA_SAMPLE_FILE,
        sep="\t",
        dtype={"patient_id": str, "sample_barcode": str},
    )
    required_sample_columns = {
        "patient_id",
        "sample_barcode",
        "MTAP_GISTIC_call",
    }
    if not required_sample_columns.issubset(dna_samples.columns):
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    dna_samples = dna_samples.loc[
        dna_samples["patient_id"].isin(metadata.index)
    ].copy()
    if dna_samples["patient_id"].duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    dna_samples = dna_samples.set_index("patient_id").reindex(metadata.index)
    if dna_samples["sample_barcode"].isna().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    if dna_samples["sample_barcode"].duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    sample_columns = dna_samples["sample_barcode"].tolist()
    missing_dna_columns = set(sample_columns) - set(dna_header)
    if missing_dna_columns:
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    sample_label = (
        pd.to_numeric(dna_samples["MTAP_GISTIC_call"], errors="raise")
        .eq(-2)
        .astype(int)
    )
    if not np.array_equal(
        sample_label.to_numpy(), metadata["label_MTAP_deep"].to_numpy()
    ):
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    target_ids = set(targets["Entrez_ID"])
    pieces = []
    found_ids = set()
    started = time.monotonic()
    reader = pd.read_csv(
        DNA_FILE,
        sep="\t",
        usecols=annotation_columns + sample_columns,
        dtype={"Locus ID": str},
        chunksize=256,
    )
    try:
        for chunk_index, chunk in enumerate(reader, start=1):
            clean_ids = (
                pd.to_numeric(chunk["Locus ID"], errors="raise")
                .astype("int64")
                .astype(str)
            )
            selected_mask = clean_ids.isin(target_ids)
            if selected_mask.any():
                selected = chunk.loc[selected_mask].copy()
                selected["Entrez_ID"] = clean_ids.loc[selected_mask]
                pieces.append(selected)
                found_ids.update(selected["Entrez_ID"])
            if chunk_index % 20 == 0:
                pass
            if found_ids == target_ids:
                break
    finally:
        reader.close()
    if found_ids != target_ids:
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    dna_selected = pd.concat(pieces, ignore_index=True)
    if dna_selected["Entrez_ID"].duplicated().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    dna_selected = dna_selected.set_index("Entrez_ID").reindex(
        targets["Entrez_ID"]
    )
    dna_calls = dna_selected[sample_columns].T.copy()
    dna_calls.index = metadata.index
    dna_calls.columns = targets["Entrez_ID"].tolist()
    dna_calls.index.name = "patient_id"
    dna_calls = dna_calls.apply(pd.to_numeric, errors="raise")
    valid_values = dna_calls.isna() | dna_calls.isin([-2, -1, 0, 1, 2])
    if not valid_values.to_numpy().all():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    mtap_id = targets.loc[targets["DNA_symbol"].eq("MTAP"), "Entrez_ID"].iloc[
        0
    ]
    if dna_calls[mtap_id].isna().any():
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    extracted_label = dna_calls[mtap_id].eq(-2).astype(int)
    if not np.array_equal(
        extracted_label.to_numpy(), metadata["label_MTAP_deep"].to_numpy()
    ):
        raise ValueError(
            "Regional DL training-data preparation: integrity check failed; condition: input or artifact integrity check."
        )
    class_labels = pd.DataFrame(
        -100, index=dna_calls.index, columns=dna_calls.columns, dtype="int64"
    )
    class_labels = class_labels.mask(dna_calls.eq(-2), 0)
    class_labels = class_labels.mask(dna_calls.eq(-1), 1)
    class_labels = class_labels.mask(dna_calls.isin([0, 1, 2]), 2)
    class_labels = class_labels.astype("int64")
    statistics = []
    for row in targets.itertuples(index=False):
        gene_calls = dna_calls[row.Entrez_ID]
        counts = {
            value: int(gene_calls.eq(value).sum())
            for value in [-2, -1, 0, 1, 2]
        }
        statistics.append(
            {
                "target_index": row.target_index,
                "DNA_symbol": row.DNA_symbol,
                "Entrez_ID": row.Entrez_ID,
                "n_deep": counts[-2],
                "n_shallow": counts[-1],
                "n_neutral": counts[0],
                "n_gain": counts[1],
                "n_amplification": counts[2],
                "n_no_deletion": counts[0] + counts[1] + counts[2],
                "n_missing": int(gene_calls.isna().sum()),
                "n_observed_classes": int(
                    class_labels[row.Entrez_ID]
                    .loc[lambda values: values.ne(-100)]
                    .nunique()
                ),
            }
        )
    target_statistics = pd.DataFrame(statistics)
    expression.to_csv(
        OUT / "X_development_signedlog.tsv", sep="\t", index=True
    )
    metadata.to_csv(OUT / "development_metadata.tsv", sep="\t", index=True)
    dna_calls.to_csv(OUT / "Y_development_GISTIC.tsv", sep="\t", index=True)
    class_labels.to_csv(
        OUT / "Y_development_three_class.tsv", sep="\t", index=True
    )
    targets.to_csv(OUT / "prediction_targets.tsv", sep="\t", index=False)
    target_statistics.to_csv(
        OUT / "target_class_counts.tsv", sep="\t", index=False
    )
    member_mapping = pd.read_csv(
        MEMBER_FILE, sep="\t", dtype=str, keep_default_na=False
    )
    member_mapping.to_csv(
        OUT / "expression_member_mapping.tsv", sep="\t", index=False
    )
    torch_check = '\nimport json\nimport torch\n\nresult = {\n    "torch_version": torch.__version__,\n    "cuda_available": torch.cuda.is_available(),\n    "cuda_version": torch.version.cuda,\n}\n\nif torch.cuda.is_available():\n    device = torch.device("cuda:0")\n    result["device"] = torch.cuda.get_device_name(0)\nelse:\n    device = torch.device("cpu")\n    result["device"] = "CPU"\n\nx = torch.randn(4, 8, device=device)\nmodel = torch.nn.Linear(8, 3).to(device)\nloss = model(x).square().mean()\nloss.backward()\n\nif device.type == "cuda":\n    torch.cuda.synchronize()\n\nresult["forward_backward_test"] = "passed"\nprint(json.dumps(result, ensure_ascii=False))\n'
    try:
        check = subprocess.run(
            [sys.executable, "-c", torch_check],
            capture_output=True,
            text=True,
            timeout=60,
        )
        torch_log = "STDOUT:\n" + check.stdout + "\nSTDERR:\n" + check.stderr
        (OUT / "torch_runtime_check.txt").write_text(
            torch_log, encoding="utf-8"
        )
        if check.returncode != 0:
            pass
    except subprocess.TimeoutExpired:
        pass
    manifest = {
        "purpose": "MTAP-focused regional multitask classification",
        "n_patients": int(len(metadata)),
        "n_MTAP_deep": int(metadata["label_MTAP_deep"].sum()),
        "n_expression_features": int(expression.shape[1]),
        "n_DNA_targets": int(len(targets)),
        "MTAP_target_Entrez": str(mtap_id),
        "MTAP_target_index": int(
            targets.loc[targets["Entrez_ID"].eq(mtap_id), "target_index"].iloc[
                0
            ]
        ),
        "class_definition": {
            "0": "GISTIC -2: deep deletion",
            "1": "GISTIC -1: shallow deletion",
            "2": "GISTIC 0/+1/+2: no deletion",
            "-100": "missing DNA label; ignore in loss",
        },
        "expression_transform": "sign(x)*log2(1+abs(x))",
        "expression_imputed": False,
        "class_weights_fitted": False,
        "test_evaluated": False,
        "model_trained": False,
    }
    (OUT / "training_input_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _public_summary(globals(), "Regional DL training-data preparation")


#### CPTAC input-file and expression-scale audit


In [ ]:
@_nb9_registry.register("s29_0")
def _nb9_stage_s29_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from collections import defaultdict
    import csv
    import gzip
    import io
    import json
    import re
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    RAW = PROJECT / "data_raw/CPTAC/LinkedOmicsKB"
    MODEL_BASE = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL"
    )
    LOCKED = MODEL_BASE / "04_locked_models_and_thresholds"
    OUT = MODEL_BASE / "05_CPTAC_input_audit"
    SPEC_FILE = LOCKED / "records/feature_and_prediction_specification.json"
    MAPPING_FILE = LOCKED / "records/expression_member_mapping.tsv"
    AUTHOR_ANNOTATION = (
        PROJECT / "code_external/RCANE/data" / "ensembl-gene-info.hg38.tsv"
    )
    if not RAW.is_dir():
        raise FileNotFoundError(
            "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
        )
    for path in [SPEC_FILE, MAPPING_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
    OUT.mkdir(parents=True, exist_ok=True)
    CANCERS = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    specification = json.loads(SPEC_FILE.read_text(encoding="utf-8"))
    features = specification["expression_feature_order"]
    if len(features) != 45:
        raise ValueError(
            "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
        )
    mapping = pd.read_csv(
        MAPPING_FILE, sep="\t", dtype=str, keep_default_na=False
    )
    mapping = mapping.loc[mapping["author_symbol"].isin(features)].copy()
    if mapping["author_symbol"].duplicated().any():
        raise ValueError(
            "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
        )
    if set(mapping["author_symbol"]) != set(features):
        raise ValueError(
            "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
        )
    report = []
    inventory_records = []
    summary_records = []
    coverage_records = []
    pair_records = []
    all_files = sorted(
        (
            path
            for path in RAW.rglob("*")
            if path.is_file() and (not path.name.startswith("."))
        )
    )

    def note(text=""):
        report.append(str(text))

    def clean_entrez(value):
        value = str(value).strip()
        if re.fullmatch("\\d+(?:\\.0+)?", value):
            return value.split(".")[0]
        return ""

    identifier_to_features = defaultdict(set)
    for row in mapping.to_dict("records"):
        feature = row["author_symbol"]
        identifier_to_features[feature].add(feature)
        entrez = clean_entrez(row.get("Entrez_ID", ""))
        if entrez:
            identifier_to_features[entrez].add(feature)
        old_gene_id = row.get("RNA_gene_id", "")
        if old_gene_id:
            identifier_to_features[old_gene_id].add(feature)
            identifier_to_features[old_gene_id.split("|")[0]].add(feature)
    if AUTHOR_ANNOTATION.is_file():
        annotation = pd.read_csv(
            AUTHOR_ANNOTATION,
            sep="\t",
            dtype=str,
            keep_default_na=False,
            usecols=["Gene stable ID", "Gene name", "NCBI gene ID"],
        )
        annotation["clean_Entrez"] = annotation["NCBI gene ID"].map(
            clean_entrez
        )
        for row in mapping.to_dict("records"):
            feature = row["author_symbol"]
            entrez = clean_entrez(row.get("Entrez_ID", ""))
            if entrez:
                selected = annotation.loc[
                    annotation["clean_Entrez"].eq(entrez)
                ]
            else:
                selected = annotation.loc[annotation["Gene name"].eq(feature)]
            for identifier in selected["Gene stable ID"].unique():
                if identifier:
                    identifier_to_features[identifier].add(feature)

    def match_identifier(value):
        if pd.isna(value):
            return set()
        value = str(value).strip()
        candidates = {value}
        candidates.update(
            (token.strip() for token in value.split("|") if token.strip())
        )
        for candidate in list(candidates):
            if re.fullmatch("ENSG\\d+\\.\\d+", candidate):
                candidates.add(candidate.split(".")[0])
            entrez = clean_entrez(candidate)
            if entrez:
                candidates.add(entrez)
        hits = set()
        for candidate in candidates:
            hits.update(identifier_to_features.get(candidate, set()))
        return hits

    def belongs_to_cancer(path, cancer):
        name_match = re.match(
            f"^{re.escape(cancer)}(?:[-_.]|$)", path.name, flags=re.IGNORECASE
        )
        folder_match = cancer.lower() in {
            part.lower() for part in path.relative_to(RAW).parts[:-1]
        }
        return bool(name_match or folder_match)

    def classify_file(path):
        name = path.name.lower()
        normalized = re.sub("[^a-z0-9]", "", name)
        if "gistic" in name:
            return "GISTIC"
        if "tpm" in name and "gene" in name:
            return "RNA_TPM"
        if "rsem" in name and "gene" in name:
            return "RNA_RSEM"
        if "tumorcaselist" in normalized:
            return "Tumor_case_list"
        if "phenotype" in name:
            return "Phenotype"
        if re.search("(^|[-_])meta([._-]|$)", name):
            return "Meta"
        return "Other"

    def unique_file(files, kind):
        matches = [path for path in files if classify_file(path) == kind]
        if len(matches) == 1:
            return matches[0]
        note(f"{kind}Candidates: {len(matches)}")
        for path in matches:
            note(f"  {path}")
        return None

    def read_table(path, header=0):
        with path.open("rb") as handle:
            magic = handle.read(8)
        if magic.startswith(b"\x1f\x8b"):
            raw_bytes = gzip.decompress(path.read_bytes())
        else:
            raw_bytes = path.read_bytes()
        if raw_bytes.startswith(b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"):
            engine = "xlrd"
        elif raw_bytes.startswith(b"PK\x03\x04"):
            engine = "openpyxl"
        else:
            engine = None
        if engine is not None:
            with pd.ExcelFile(
                io.BytesIO(raw_bytes), engine=engine
            ) as workbook:
                if len(workbook.sheet_names) != 1:
                    raise ValueError(
                        "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
                    )
                return (
                    pd.read_excel(
                        workbook,
                        sheet_name=workbook.sheet_names[0],
                        header=header,
                    ),
                    f"Excel/{engine}",
                )
        text = None
        encoding_used = None
        for encoding in ["utf-8-sig", "gb18030", "cp1252"]:
            try:
                text = raw_bytes.decode(encoding)
                encoding_used = encoding
                break
            except UnicodeDecodeError:
                continue
        if text is None:
            raise ValueError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
        first_nonempty = next(
            (line for line in text.splitlines() if line.strip()), ""
        )
        if (
            first_nonempty.lstrip()
            .lower()
            .startswith(("<!doctype html", "<html"))
        ):
            raise ValueError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
        skiprows = 0
        if text.startswith(("#1.2", "#1.3")):
            skiprows = 2
        lines = text.splitlines()
        sample = "\n".join(lines[skiprows : skiprows + 10])
        if "\t" in sample:
            separator = "\t"
        else:
            try:
                separator = (
                    csv.Sniffer().sniff(sample, delimiters=",;").delimiter
                )
            except csv.Error:
                separator = "\t"
        table = pd.read_csv(
            io.StringIO(text),
            sep=separator,
            header=header,
            skiprows=skiprows,
            low_memory=False,
        )
        table.columns = [str(column).strip() for column in table.columns]
        return (table, f"text/{encoding_used}/separator={repr(separator)}")

    def describe_table(name, table, format_name):
        note(f"\nFile type: {name}")
        note(f"Detected format: {format_name}")
        note(f"Shape: {table.shape}")
        note(f"Columns: {table.columns.tolist()}")
        note("First three rows and eight columns: ")
        note(table.iloc[:3, :8].to_string(index=False))

    annotation_header_pattern = re.compile(
        "gene|symbol|entrez|ensembl|description|cytoband|chrom|locus|^id$|^name$|^start$|^end$|^loc\\.",
        flags=re.IGNORECASE,
    )

    def inspect_matrix(table, cancer, kind):
        if table.empty:
            raise ValueError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
        candidate_identifier_columns = [
            column
            for column in table.columns
            if annotation_header_pattern.search(str(column))
        ]
        if table.columns[0] not in candidate_identifier_columns:
            candidate_identifier_columns.insert(0, table.columns[0])
        row_hits = [set() for _ in range(len(table))]
        identifier_columns_with_hits = []
        for column in candidate_identifier_columns:
            column_has_hit = False
            for position, value in enumerate(table[column]):
                hits = match_identifier(value)
                if hits:
                    row_hits[position].update(hits)
                    column_has_hit = True
            if column_has_hit:
                identifier_columns_with_hits.append(column)
        if not identifier_columns_with_hits:
            raise ValueError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
        feature_rows = defaultdict(list)
        ambiguous_rows = []
        for position, hits in enumerate(row_hits):
            if len(hits) == 1:
                feature_rows[next(iter(hits))].append(position)
            elif len(hits) > 1:
                ambiguous_rows.append(
                    {
                        "row_position": position,
                        "matched_features": ";".join(sorted(hits)),
                    }
                )
        excluded_columns = set(candidate_identifier_columns)
        sample_candidates = [
            column
            for column in table.columns
            if column not in excluded_columns
        ]
        numeric = table[sample_candidates].apply(
            pd.to_numeric, errors="coerce"
        )
        sample_columns = []
        rejected_columns = []
        for column in sample_candidates:
            original_nonmissing = table[column].notna().sum()
            numeric_nonmissing = numeric[column].notna().sum()
            if (
                original_nonmissing > 0
                and numeric_nonmissing / original_nonmissing >= 0.95
            ):
                sample_columns.append(column)
            else:
                rejected_columns.append(column)
        if not sample_columns:
            raise ValueError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
        numeric = numeric[sample_columns]
        finite_values = numeric.to_numpy(dtype=float).ravel()
        n_infinite = int(np.isinf(finite_values).sum())
        finite_values = finite_values[np.isfinite(finite_values)]
        if finite_values.size == 0:
            raise ValueError(
                "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
            )
        quantiles = np.quantile(
            finite_values, [0, 0.01, 0.25, 0.5, 0.75, 0.99, 1]
        )
        note(f"Gene identifier field: {identifier_columns_with_hits}")
        note(f"Candidate samples: {len(sample_columns)}")
        note(f"First twelve samples: {sample_columns[:12]}")
        note(f"Excluded nonsample columns: {rejected_columns}")
        note(f"Infinite values: {n_infinite}")
        note(
            "Value quantiles [min,P1,P25,P50,P75,P99,max]: "
            + str(quantiles.tolist())
        )
        note(f"Negative values: {int((finite_values < 0).sum())}")
        note(f"Missing values: {int(numeric.isna().sum().sum())}")
        for feature in features:
            rows = feature_rows.get(feature, [])
            if len(rows) == 0:
                status = "Not_found"
                n_missing = np.nan
            elif len(rows) == 1:
                status = "Unique_match"
                n_missing = int(numeric.iloc[rows[0]].isna().sum())
            else:
                status = "Multiple_rows_review_required"
                n_missing = np.nan
            coverage_records.append(
                {
                    "database_cancer": cancer,
                    "matrix_type": kind,
                    "feature": feature,
                    "n_matching_rows": len(rows),
                    "status": status,
                    "n_samples_missing_expression": n_missing,
                }
            )
        if ambiguous_rows:
            note(
                "Rows with conflicting field matches: "
                + str(ambiguous_rows[:20])
            )
        n_unique_features = sum(
            (len(feature_rows.get(feature, [])) == 1 for feature in features)
        )
        mtap = None
        if len(feature_rows.get("MTAP", [])) == 1:
            mtap = numeric.iloc[feature_rows["MTAP"][0]].copy()
            mtap.index = mtap.index.astype(str)
            note("MTAP value summary: ")
            note(mtap.describe().to_string())
            if kind == "GISTIC":
                note("MTAP call distribution: ")
                note(mtap.value_counts(dropna=False).sort_index().to_string())
        if kind == "GISTIC":
            invalid = ~np.isin(finite_values, [-2, -1, 0, 1, 2])
            note(
                "Nonmissing values outside valid GISTIC calls: "
                + str(int(invalid.sum()))
            )
            if invalid.any():
                raise ValueError(
                    "CPTAC input-file and expression-scale audit: integrity check failed; condition: input or artifact integrity check."
                )
        return {
            "sample_columns": [str(column) for column in sample_columns],
            "MTAP": mtap,
            "n_unique_features": n_unique_features,
            "quantiles": quantiles.tolist(),
            "n_negative": int((finite_values < 0).sum()),
            "n_missing": int(numeric.isna().sum().sum()),
        }

    for database_cancer, model_cancer in CANCERS.items():
        note("\n" + "=" * 80)
        note(f"Cancer: {database_cancer} → {model_cancer}")
        cancer_files = [
            path
            for path in all_files
            if belongs_to_cancer(path, database_cancer)
        ]
        summary = {
            "database_cancer": database_cancer,
            "model_cancer": model_cancer,
            "n_files": len(cancer_files),
            "status": "Pending",
        }
        for path in cancer_files:
            kind = classify_file(path)
            inventory_records.append(
                {
                    "database_cancer": database_cancer,
                    "type": kind,
                    "path": str(path),
                    "size_MB": path.stat().st_size / 1024**2,
                }
            )
            note(f"{kind}：{path}；{path.stat().st_size / 1024 ** 2:.2f} MB")
        if not cancer_files:
            summary["status"] = "No_files_found"
            summary_records.append(summary)
            continue
        matrix_results = {}
        tumor_tokens = set()
        for kind in [
            "Meta",
            "Tumor_case_list",
            "Phenotype",
            "GISTIC",
            "RNA_TPM",
            "RNA_RSEM",
        ]:
            path = unique_file(cancer_files, kind)
            if path is None:
                summary[f"{kind}_file_status"] = (
                    "Missing_or_multiple_candidates"
                )
                continue
            try:
                header = None if kind == "Tumor_case_list" else 0
                table, format_name = read_table(path, header=header)
                describe_table(kind, table, format_name)
                summary[f"{kind}_file_status"] = "Read_success"
                if kind == "Tumor_case_list":
                    for value in table.to_numpy().ravel():
                        if pd.isna(value):
                            continue
                        tumor_tokens.update(
                            (
                                token
                                for token in re.split(
                                    "[;,\\s]+", str(value).strip()
                                )
                                if token
                            )
                        )
                    note(
                        "First twenty nonempty case-list tokens: "
                        + str(sorted(tumor_tokens)[:20])
                    )
                elif kind in ["GISTIC", "RNA_TPM", "RNA_RSEM"]:
                    result = inspect_matrix(table, database_cancer, kind)
                    matrix_results[kind] = result
                    summary[f"{kind}_n_samples"] = len(
                        result["sample_columns"]
                    )
                    summary[f"{kind}_n_unique_model_members"] = result[
                        "n_unique_features"
                    ]
                    if kind.startswith("RNA"):
                        summary[f"{kind}_median"] = result["quantiles"][3]
                        summary[f"{kind}_P99"] = result["quantiles"][5]
                        summary[f"{kind}_n_negative"] = result["n_negative"]
                del table
            except Exception as error:
                summary[f"{kind}_file_status"] = (
                    f"Review_required: {type(error).__name__}"
                )
                note(f"Read or audit failed: {repr(error)}")
        if "GISTIC" in matrix_results:
            dna_result = matrix_results["GISTIC"]
            dna_samples = set(dna_result["sample_columns"])
            for rna_kind in ["RNA_TPM", "RNA_RSEM"]:
                if rna_kind not in matrix_results:
                    continue
                rna_samples = set(matrix_results[rna_kind]["sample_columns"])
                common = sorted(dna_samples & rna_samples)
                tumor_common = sorted(set(common) & tumor_tokens)
                summary[f"{rna_kind}_DNA_exact_ID_overlap"] = len(common)
                summary[f"{rna_kind}_DNA_CaseList_exact_overlap"] = len(
                    tumor_common
                )
                note(
                    f"\n{rna_kind}Exact DNA identifier overlap: {len(common)}"
                )
                note(f"Exact tumor-case overlap: {len(tumor_common)}")
                mtap = dna_result["MTAP"]
                if mtap is not None and tumor_common:
                    calls = mtap.reindex(tumor_common)
                    note("MTAP calls in the provisional intersection: ")
                    note(
                        calls.value_counts(dropna=False)
                        .sort_index()
                        .to_string()
                    )
                    for sample in tumor_common:
                        pair_records.append(
                            {
                                "database_cancer": database_cancer,
                                "model_cancer": model_cancer,
                                "RNA_type": rna_kind,
                                "sample_id_exact": sample,
                                "MTAP_GISTIC_call": mtap.loc[sample],
                                "pairing_status": "Exact_ID_candidate_not_final_patient_pairing",
                            }
                        )
                if not tumor_common:
                    note(
                        "An empty exact intersection requires review of the sample-to-patient mapping."
                    )
        summary["status"] = "Audit_complete_prediction_not_run"
        summary_records.append(summary)
    inventory = pd.DataFrame(inventory_records)
    summary_table = pd.DataFrame(summary_records)
    coverage_table = pd.DataFrame(coverage_records)
    inventory.to_csv(OUT / "file_inventory.tsv", sep="\t", index=False)
    summary_table.to_csv(
        OUT / "cohort_input_summary.tsv", sep="\t", index=False
    )
    coverage_table.to_csv(
        OUT / "model_feature_coverage.tsv", sep="\t", index=False
    )
    pair_columns = [
        "database_cancer",
        "model_cancer",
        "RNA_type",
        "sample_id_exact",
        "MTAP_GISTIC_call",
        "pairing_status",
    ]
    pd.DataFrame(pair_records, columns=pair_columns).to_csv(
        OUT / "exact_ID_pairing_candidates.tsv", sep="\t", index=False
    )
    note("\n" + "=" * 80)
    note("This stage audits files, identifiers, and values only.")
    note("RNA was not transformed, standardized, or batch corrected.")
    note("DL and Elastic Net inference were not performed.")
    note("Locked models and thresholds were not changed.")
    note(
        "Value distributions and filenames alone do not establish input-scale compatibility."
    )
    note(
        "Exact identifier overlap does not establish patient-level equivalence."
    )
    report_file = OUT / "LinkedOmicsKB_input_review.txt"
    report_file.write_text("\n".join(report), encoding="utf-8")
    display_columns = [
        "database_cancer",
        "model_cancer",
        "GISTIC_n_samples",
        "RNA_TPM_n_samples",
        "RNA_TPM_n_unique_model_members",
        "RNA_TPM_DNA_exact_ID_overlap",
        "RNA_TPM_DNA_CaseList_exact_overlap",
        "RNA_TPM_median",
        "RNA_TPM_P99",
        "RNA_TPM_n_negative",
    ]
    display_columns = [
        column for column in display_columns if column in summary_table.columns
    ]
    _public_summary(globals(), "CPTAC input-file and expression-scale audit")


#### CPTAC metadata and expression matching


In [ ]:
@_nb9_registry.register("s30_0")
def _nb9_stage_s30_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import numpy as np
    import pandas as pd

    required_objects = [
        "RAW",
        "MODEL_BASE",
        "CANCER_MAP",
        "feature_table",
        "features",
        "MTAP_ENSEMBL",
        "read_gene_matrix",
        "unique_file",
    ]
    missing_objects = [
        name for name in required_objects if name not in globals()
    ]
    if missing_objects:
        raise RuntimeError(
            "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
        )
    OUT_V2 = MODEL_BASE / "06_CPTAC_RNA_pairing_v2"
    OUT_V2.mkdir(parents=True, exist_ok=True)
    summaries = []
    all_metadata = []
    for database_cancer, model_cancer in CANCER_MAP.items():
        cohort_out = OUT_V2 / database_cancer
        cohort_out.mkdir(exist_ok=True)
        try:
            rna_file = unique_file(
                f"{database_cancer}-gene_TPM*log2(x+1)*.txt"
            )
            dna_file = unique_file(
                f"{database_cancer}_WES_CNV_gene_gistic_level.txt"
            )
            case_file = unique_file(f"{database_cancer}_Tumor_CaseList.txt")
            meta_file = unique_file(f"{database_cancer}_meta.txt")
            rna, layout = read_gene_matrix(rna_file)
            dna, _ = read_gene_matrix(dna_file)
            n_rna_original = rna.shape[1]
            original_columns = list(rna.columns)
            has_tumor_suffix = any(
                (name.endswith("_T") for name in original_columns)
            )
            if has_tumor_suffix:
                selected_columns = [
                    name for name in original_columns if name.endswith("_T")
                ]
                renamed_columns = [name[:-2] for name in selected_columns]
                sample_rule = "Keep_terminal_T_then_remove_suffix"
            else:
                if any((name.endswith("_A") for name in original_columns)):
                    raise ValueError(
                        "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                    )
                selected_columns = original_columns
                renamed_columns = original_columns
                sample_rule = "Original_ID_filtered_by_Tumor_CaseList"
            if len(renamed_columns) != len(set(renamed_columns)):
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            sample_mapping = pd.DataFrame(
                {
                    "RNA_original_sample_id": selected_columns,
                    "patient_id": renamed_columns,
                }
            )
            rna = rna.loc[:, selected_columns].copy()
            rna.columns = renamed_columns
            sample_mapping.to_csv(
                cohort_out / "RNA_sample_to_patient_mapping.tsv",
                sep="\t",
                index=False,
            )
            cases = pd.read_csv(
                case_file,
                sep="\t",
                header=None,
                dtype=str,
                keep_default_na=False,
            )
            if cases.shape[1] != 1:
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            case_ids = cases.iloc[:, 0].str.strip()
            case_ids = case_ids.loc[case_ids.ne("")]
            if case_ids.duplicated().any():
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            meta = pd.read_csv(
                meta_file, sep="\t", dtype=str, keep_default_na=False
            )
            meta["case_id"] = meta["case_id"].str.strip()
            meta = meta.loc[
                meta["case_id"].ne("")
                & meta["case_id"].str.lower().ne("data_type")
            ].copy()
            if meta["case_id"].duplicated().any():
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            pairing = pd.DataFrame({"patient_id": sorted(case_ids.tolist())})
            pairing["has_RNA"] = pairing["patient_id"].isin(rna.columns)
            pairing["has_DNA"] = pairing["patient_id"].isin(dna.columns)
            pairing["has_Meta"] = pairing["patient_id"].isin(meta["case_id"])
            pairing["paired"] = pairing["has_RNA"] & pairing["has_DNA"]
            pairing.to_csv(
                cohort_out / "tumor_patient_pairing.tsv", sep="\t", index=False
            )
            patients = pairing.loc[pairing["paired"], "patient_id"].tolist()
            if not patients:
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            ensembl_order = feature_table["Ensembl_ID"].tolist()
            absent = feature_table.loc[
                ~feature_table["Ensembl_ID"].isin(rna.index), "model_feature"
            ].tolist()
            if absent:
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            X = rna.loc[ensembl_order, patients].T.copy()
            X.columns = features
            X.index.name = "patient_id"
            calls = dna.loc[MTAP_ENSEMBL, patients]
            invalid = calls.notna() & ~calls.isin([-2, -1, 0, 1, 2])
            if invalid.any():
                raise ValueError(
                    "CPTAC metadata and expression matching: integrity check failed; condition: input or artifact integrity check."
                )
            patient_meta = pd.DataFrame(
                {
                    "patient_id": patients,
                    "database_cancer": database_cancer,
                    "acronym": model_cancer,
                    "MTAP_GISTIC_call": calls.to_numpy(),
                }
            )
            label = pd.Series(pd.NA, index=patient_meta.index, dtype="Int64")
            observed = patient_meta["MTAP_GISTIC_call"].notna()
            label.loc[observed] = (
                patient_meta.loc[observed, "MTAP_GISTIC_call"]
                .eq(-2)
                .astype(int)
            )
            patient_meta["label_MTAP_deep"] = label
            patient_meta = patient_meta.merge(
                sample_mapping,
                on="patient_id",
                how="left",
                validate="one_to_one",
            )
            patient_meta = patient_meta.merge(
                meta.rename(columns={"case_id": "patient_id"}),
                on="patient_id",
                how="left",
                validate="one_to_one",
            )
            X.to_csv(cohort_out / "X_45_log2TPM_as_downloaded.tsv", sep="\t")
            patient_meta.to_csv(
                cohort_out / "paired_patient_metadata.tsv",
                sep="\t",
                index=False,
            )
            numeric_summary = X.describe(
                percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
            ).T
            numeric_summary["n_missing"] = X.isna().sum()
            numeric_summary["n_negative"] = X.lt(0).sum()
            numeric_summary.to_csv(
                cohort_out / "RNA_45_feature_numeric_summary.tsv",
                sep="\t",
                index_label="model_feature",
            )
            all_metadata.append(patient_meta)
            summaries.append(
                {
                    "database_cancer": database_cancer,
                    "model_cancer": model_cancer,
                    "n_RNA_original": n_rna_original,
                    "n_RNA_after_tissue_filter": rna.shape[1],
                    "n_DNA": dna.shape[1],
                    "n_paired": len(patients),
                    "n_MTAP_deep": int(calls.eq(-2).sum()),
                    "n_MTAP_shallow": int(calls.eq(-1).sum()),
                    "n_MTAP_neutral": int(calls.eq(0).sum()),
                    "n_MTAP_gain": int(calls.isin([1, 2]).sum()),
                    "n_MTAP_missing": int(calls.isna().sum()),
                    "n_patients_with_RNA_missing": int(
                        X.isna().any(axis=1).sum()
                    ),
                    "status": "ok",
                }
            )
            del rna, dna
        except Exception as error:
            message = f"{type(error).__name__}: {error}"
            summaries.append(
                {
                    "database_cancer": database_cancer,
                    "model_cancer": model_cancer,
                    "status": message,
                }
            )
    summary_table = pd.DataFrame(summaries)
    summary_table.to_csv(
        OUT_V2 / "CPTAC_pairing_summary.tsv", sep="\t", index=False
    )
    if all_metadata:
        pd.concat(all_metadata, ignore_index=True).to_csv(
            OUT_V2 / "ALL_paired_patient_metadata.tsv", sep="\t", index=False
        )
    _public_summary(globals(), "CPTAC metadata and expression matching")


#### CPTAC compatibility with the TCGA training input


In [ ]:
@_nb9_registry.register("s31_0")
def _nb9_stage_s31_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL"
        )
    )
    TRAIN_X_FILE = BASE / "01_training_inputs/X_development_signedlog.tsv"
    TRAIN_META_FILE = BASE / "01_training_inputs/development_metadata.tsv"
    SPEC_FILE = (
        BASE
        / "04_locked_models_and_thresholds/records/feature_and_prediction_specification.json"
    )
    EXTERNAL = BASE / "06_CPTAC_RNA_pairing_v2"
    OUT = BASE / "07_CPTAC_expression_scale_review"
    OUT.mkdir(parents=True, exist_ok=True)
    CANCER_MAP = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    for path in [TRAIN_X_FILE, TRAIN_META_FILE, SPEC_FILE, EXTERNAL]:
        if not path.exists():
            raise FileNotFoundError(path)
    with SPEC_FILE.open("r", encoding="utf-8") as handle:
        specification = json.load(handle)
    features = specification["expression_feature_order"]
    if len(features) != 45 or len(set(features)) != 45:
        raise ValueError(
            "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
        )

    def read_expression(path):
        frame = pd.read_csv(path, sep="\t", index_col=0)
        frame.index = frame.index.astype(str).str.strip()
        frame.columns = frame.columns.astype(str)
        if frame.index.duplicated().any():
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        if frame.columns.duplicated().any():
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        missing = sorted(set(features) - set(frame.columns))
        if missing:
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        frame = frame.loc[:, features].apply(pd.to_numeric, errors="raise")
        if np.isinf(frame.to_numpy(dtype=float)).any():
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        return frame

    train_X = read_expression(TRAIN_X_FILE)
    train_meta = pd.read_csv(
        TRAIN_META_FILE, sep="\t", usecols=["patient_id", "acronym"], dtype=str
    )
    if train_meta["patient_id"].duplicated().any():
        raise ValueError(
            "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
        )
    train_meta = train_meta.set_index("patient_id")
    if set(train_X.index) != set(train_meta.index):
        raise ValueError(
            "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
        )
    train_meta = train_meta.loc[train_X.index]
    train_mean = train_X.mean()
    train_sd = train_X.std(ddof=0).replace(0, np.nan)
    train_p01 = train_X.quantile(0.01)
    train_p99 = train_X.quantile(0.99)
    feature_results = []
    cohort_results = []
    patient_results = []
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    for database_cancer, model_cancer in CANCER_MAP.items():
        folder = EXTERNAL / database_cancer
        external_X = read_expression(folder / "X_45_log2TPM_as_downloaded.tsv")
        external_meta = pd.read_csv(
            folder / "paired_patient_metadata.tsv",
            sep="\t",
            usecols=["patient_id", "acronym"],
            dtype=str,
        )
        if external_meta["patient_id"].duplicated().any():
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        if set(external_X.index) != set(external_meta["patient_id"]):
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        if set(external_meta["acronym"]) != {model_cancer}:
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        same_cancer_ids = train_meta.index[
            train_meta["acronym"].eq(model_cancer)
        ]
        if len(same_cancer_ids) == 0:
            raise ValueError(
                "CPTAC compatibility with the TCGA training input: integrity check failed; condition: input or artifact integrity check."
            )
        same_cancer_X = train_X.loc[same_cancer_ids]
        for gene in features:
            source = same_cancer_X[gene].dropna()
            target = external_X[gene].dropna()
            source_sd = source.std(ddof=0)
            median_difference = target.median() - source.median()
            median_shift_sd = (
                median_difference / source_sd
                if np.isfinite(source_sd) and source_sd > 0
                else np.nan
            )
            feature_results.append(
                {
                    "database_cancer": database_cancer,
                    "model_cancer": model_cancer,
                    "feature": gene,
                    "n_TCGA": len(source),
                    "n_CPTAC": len(target),
                    "TCGA_P01": source.quantile(0.01),
                    "TCGA_P25": source.quantile(0.25),
                    "TCGA_median": source.median(),
                    "TCGA_P75": source.quantile(0.75),
                    "TCGA_P99": source.quantile(0.99),
                    "CPTAC_P01": target.quantile(0.01),
                    "CPTAC_P25": target.quantile(0.25),
                    "CPTAC_median": target.median(),
                    "CPTAC_P75": target.quantile(0.75),
                    "CPTAC_P99": target.quantile(0.99),
                    "median_difference_CPTAC_minus_TCGA": median_difference,
                    "median_shift_in_same_cancer_TCGA_SD": median_shift_sd,
                    "CPTAC_n_missing": int(external_X[gene].isna().sum()),
                }
            )
        diagnostic_z = (external_X - train_mean) / train_sd
        outside_train_range = external_X.lt(train_p01) | external_X.gt(
            train_p99
        )
        patient_qc = pd.DataFrame(index=external_X.index)
        patient_qc.index.name = "patient_id"
        patient_qc["database_cancer"] = database_cancer
        patient_qc["acronym"] = model_cancer
        patient_qc["n_features_outside_TCGA_P01_P99"] = (
            outside_train_range.sum(axis=1)
        )
        patient_qc["n_features_abs_diagnostic_z_gt_3"] = (
            diagnostic_z.abs().gt(3).sum(axis=1)
        )
        patient_qc["n_missing_features"] = external_X.isna().sum(axis=1)
        patient_results.append(patient_qc.reset_index())
        cohort_results.append(
            {
                "database_cancer": database_cancer,
                "model_cancer": model_cancer,
                "n_TCGA_same_cancer": len(same_cancer_X),
                "n_CPTAC": len(external_X),
                "median_n_features_outside_TCGA_P01_P99": patient_qc[
                    "n_features_outside_TCGA_P01_P99"
                ].median(),
                "median_n_features_abs_z_gt_3": patient_qc[
                    "n_features_abs_diagnostic_z_gt_3"
                ].median(),
                "TCGA_MTAP_median": same_cancer_X["MTAP"].median(),
                "CPTAC_MTAP_median": external_X["MTAP"].median(),
                "TCGA_CDKN2A_median": same_cancer_X["CDKN2A"].median(),
                "CPTAC_CDKN2A_median": external_X["CDKN2A"].median(),
                "TCGA_CDKN2B_median": same_cancer_X["CDKN2B"].median(),
                "CPTAC_CDKN2B_median": external_X["CDKN2B"].median(),
            }
        )
    feature_table = pd.DataFrame(feature_results)
    cohort_table = pd.DataFrame(cohort_results)
    patient_table = pd.concat(patient_results, ignore_index=True)
    core_table = feature_table.loc[feature_table["feature"].isin(CORE)].copy()
    feature_table.to_csv(
        OUT / "all_45_features_scale_comparison.tsv", sep="\t", index=False
    )
    core_table.to_csv(
        OUT / "core_three_genes_scale_comparison.tsv", sep="\t", index=False
    )
    cohort_table.to_csv(
        OUT / "cohort_scale_summary.tsv", sep="\t", index=False
    )
    patient_table.to_csv(
        OUT / "patient_input_range_diagnostics.tsv", sep="\t", index=False
    )
    display_columns = [
        "database_cancer",
        "feature",
        "TCGA_median",
        "CPTAC_median",
        "median_shift_in_same_cancer_TCGA_SD",
    ]
    _public_summary(
        globals(), "CPTAC compatibility with the TCGA training input"
    )


#### TPM-compatible 44-member feature construction


In [ ]:
@_nb9_registry.register("s32_0")
def _nb9_stage_s32_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd

    required_objects = ["found", "sample_map"]
    missing_objects = [
        name for name in required_objects if name not in globals()
    ]
    if missing_objects:
        raise RuntimeError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    PROJECT = Path(str(PROJECT_ROOT))
    BASE = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL"
    )
    BRANCH = BASE / "08_TPM_retraining"
    OUT = BRANCH / "01_expression_inputs_44"
    OUT.mkdir(parents=True, exist_ok=True)
    EXTERNAL_SOURCE = BASE / "06_CPTAC_RNA_pairing_v2"
    EXTERNAL_OUT = BRANCH / "CPTAC_inputs_44"
    EXTERNAL_OUT.mkdir(parents=True, exist_ok=True)
    META_FILE = BASE / "01_training_inputs/development_metadata.tsv"
    SPEC_FILE = (
        BASE
        / "04_locked_models_and_thresholds/records/feature_and_prediction_specification.json"
    )
    CANCER_MAP = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    with SPEC_FILE.open("r", encoding="utf-8") as handle:
        original_spec = json.load(handle)
    original_features = original_spec["expression_feature_order"]
    unmatched = [gene for gene in original_features if gene not in found]
    if unmatched != ["LINC00032"]:
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    features44 = [gene for gene in original_features if gene != "LINC00032"]
    if len(features44) != 44:
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    meta = pd.read_csv(META_FILE, sep="\t", dtype=str)
    if meta["patient_id"].duplicated().any():
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    if "label_MTAP_deep" in meta.columns:
        label_column = "label_MTAP_deep"
    elif "MTAP_deep_label" in meta.columns:
        label_column = "MTAP_deep_label"
    else:
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    labels = pd.to_numeric(meta[label_column], errors="raise")
    if labels.isna().any() or not labels.isin([0, 1]).all():
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    meta["label_MTAP_deep"] = labels.astype(int)
    if sample_map["sample_id"].duplicated().any():
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    if not set(sample_map["patient_id"]).issubset(set(meta["patient_id"])):
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    for gene in features44:
        if len(found[gene]) != len(sample_map):
            raise ValueError(
                "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
            )
    source_log = pd.DataFrame(
        {gene: found[gene] for gene in features44},
        index=sample_map["sample_id"].tolist(),
    )
    source_log.index.name = "sample_id"
    if np.isinf(source_log.to_numpy(dtype=float)).any():
        raise ValueError(
            "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
        )
    source_log.to_csv(
        OUT / "development_sample_log2TPM_plus_0001.tsv", sep="\t"
    )
    sample_map.to_csv(
        OUT / "development_Toil_sample_mapping.tsv", sep="\t", index=False
    )
    sample_to_patient = sample_map.set_index("sample_id")["patient_id"]
    patient_rows = {}
    for patient, group in sample_to_patient.groupby(
        sample_to_patient, sort=False
    ):
        sample_ids = group.index.tolist()
        values = source_log.loc[sample_ids, features44].to_numpy(dtype=float)
        mean_source_log = np.logaddexp.reduce(
            values * np.log(2), axis=0
        ) / np.log(2) - np.log2(len(sample_ids))
        patient_rows[patient] = np.logaddexp2(mean_source_log, np.log2(0.999))
    patient_X = pd.DataFrame.from_dict(
        patient_rows, orient="index", columns=features44
    )
    patient_X.index.name = "patient_id"
    available_ids = [
        patient for patient in meta["patient_id"] if patient in patient_X.index
    ]
    patient_X = patient_X.loc[available_ids]
    n_source_samples = sample_map.groupby("patient_id").size()
    availability = meta[["patient_id", "acronym", "label_MTAP_deep"]].copy()
    availability["n_Toil_primary_samples"] = (
        availability["patient_id"].map(n_source_samples).fillna(0).astype(int)
    )
    availability["has_Toil_RNA"] = availability["n_Toil_primary_samples"].gt(0)
    missing_counts = patient_X.isna().sum(axis=1)
    availability["n_missing_features"] = (
        availability["patient_id"].map(missing_counts).astype("Int64")
    )
    availability["RNA_status"] = np.where(
        availability["has_Toil_RNA"],
        "RNA_available",
        "No_matching_Toil_primary_RNA",
    )
    availability.to_csv(
        OUT / "all_development_patient_availability.tsv", sep="\t", index=False
    )
    training_meta = availability.loc[availability["has_Toil_RNA"]].copy()
    patient_X.to_csv(OUT / "X_development_log2TPM_plus_1.tsv", sep="\t")
    training_meta.to_csv(
        OUT / "development_metadata.tsv", sep="\t", index=False
    )
    availability["retained_positive"] = (
        availability["has_Toil_RNA"] & availability["label_MTAP_deep"].eq(1)
    ).astype(int)
    availability["unmatched_positive"] = (
        ~availability["has_Toil_RNA"] & availability["label_MTAP_deep"].eq(1)
    ).astype(int)
    coverage_summary = availability.groupby("acronym").agg(
        n_original=("patient_id", "size"),
        n_retained=("has_Toil_RNA", "sum"),
        n_original_positive=("label_MTAP_deep", "sum"),
        n_retained_positive=("retained_positive", "sum"),
        n_unmatched_positive=("unmatched_positive", "sum"),
    )
    coverage_summary["n_unmatched"] = (
        coverage_summary["n_original"] - coverage_summary["n_retained"]
    )
    coverage_summary["n_retained_negative"] = (
        coverage_summary["n_retained"]
        - coverage_summary["n_retained_positive"]
    )
    coverage_summary.to_csv(
        OUT / "development_patient_coverage_by_cancer.tsv", sep="\t"
    )
    external_summary = []
    for database_cancer, model_cancer in CANCER_MAP.items():
        source_file = (
            EXTERNAL_SOURCE
            / database_cancer
            / "X_45_log2TPM_as_downloaded.tsv"
        )
        external_X = pd.read_csv(source_file, sep="\t", index_col=0)
        if external_X.index.duplicated().any():
            raise ValueError(
                "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
            )
        absent = sorted(set(features44) - set(external_X.columns))
        if absent:
            raise ValueError(
                "TPM-compatible 44-member feature construction: integrity check failed; condition: input or artifact integrity check."
            )
        external_X = external_X.loc[:, features44].apply(
            pd.to_numeric, errors="raise"
        )
        cohort_out = EXTERNAL_OUT / database_cancer
        cohort_out.mkdir(parents=True, exist_ok=True)
        external_X.to_csv(
            cohort_out / "X_44_log2TPM_plus_1.tsv",
            sep="\t",
            index_label="patient_id",
        )
        external_summary.append(
            {
                "database_cancer": database_cancer,
                "model_cancer": model_cancer,
                "n_patients": len(external_X),
                "n_features": external_X.shape[1],
                "n_missing_cells": int(external_X.isna().sum().sum()),
            }
        )
    pd.DataFrame(external_summary).to_csv(
        EXTERNAL_OUT / "CPTAC_44_member_input_summary.tsv",
        sep="\t",
        index=False,
    )
    new_spec = {
        "version": "TPM_44_members",
        "expression_feature_order": features44,
        "excluded_feature": "LINC00032",
        "exclusion_reason": "Not matched by the predefined Ensembl mapping in the selected TCGA Toil dataset",
        "TCGA_source_unit": "log2(TPM+0.001)",
        "model_input_unit": "log2(TPM+1)",
        "CPTAC_values": "Kept as downloaded; no distribution alignment",
        "patient_selection": "Original development patients with matched Toil primary-tumor RNA",
        "n_original_development": len(meta),
        "n_available_development": len(patient_X),
        "model_retraining_required": True,
        "old_model_weights_or_thresholds_reused": False,
    }
    with (OUT / "TPM_44_feature_specification.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(new_spec, handle, ensure_ascii=False, indent=2)
    _public_summary(globals(), "TPM-compatible 44-member feature construction")


#### TCGA/CPTAC TPM input compatibility


In [ ]:
@_nb9_registry.register("s35_0")
def _nb9_stage_s35_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL"
        )
    )
    TRAIN_DIR = BASE / "08_TPM_retraining/01_expression_inputs_44"
    CPTAC_DIR = BASE / "08_TPM_retraining/CPTAC_inputs_44"
    OLD_REVIEW = BASE / "07_CPTAC_expression_scale_review"
    OUT = BASE / "08_TPM_retraining/01b_TPM_scale_review"
    OUT.mkdir(parents=True, exist_ok=True)
    CANCER_MAP = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    with (TRAIN_DIR / "TPM_44_feature_specification.json").open(
        "r", encoding="utf-8"
    ) as handle:
        specification = json.load(handle)
    features = specification["expression_feature_order"]
    train = pd.read_csv(
        TRAIN_DIR / "X_development_log2TPM_plus_1.tsv", sep="\t", index_col=0
    )
    train.index = train.index.astype(str)
    train = train.loc[:, features].apply(pd.to_numeric, errors="raise")
    meta = pd.read_csv(
        TRAIN_DIR / "development_metadata.tsv",
        sep="\t",
        usecols=["patient_id", "acronym"],
        dtype=str,
    ).set_index("patient_id")
    if train.index.duplicated().any() or meta.index.duplicated().any():
        raise ValueError(
            "TCGA/CPTAC TPM input compatibility: integrity check failed; condition: input or artifact integrity check."
        )
    if set(train.index) != set(meta.index):
        raise ValueError(
            "TCGA/CPTAC TPM input compatibility: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.loc[train.index]
    if not np.isfinite(train.to_numpy()).all():
        raise ValueError(
            "TCGA/CPTAC TPM input compatibility: integrity check failed; condition: input or artifact integrity check."
        )
    gene_rows = []
    patient_rows = []
    cohort_rows = []
    for database_cancer, model_cancer in CANCER_MAP.items():
        external = pd.read_csv(
            CPTAC_DIR / database_cancer / "X_44_log2TPM_plus_1.tsv",
            sep="\t",
            index_col=0,
        )
        external.index = external.index.astype(str)
        external = external.loc[:, features].apply(
            pd.to_numeric, errors="raise"
        )
        if external.index.duplicated().any():
            raise ValueError(
                "TCGA/CPTAC TPM input compatibility: integrity check failed; condition: input or artifact integrity check."
            )
        if not np.isfinite(external.to_numpy()).all():
            raise ValueError(
                "TCGA/CPTAC TPM input compatibility: integrity check failed; condition: input or artifact integrity check."
            )
        same_cancer = train.loc[meta.index[meta["acronym"].eq(model_cancer)]]
        if same_cancer.empty:
            raise ValueError(
                "TCGA/CPTAC TPM input compatibility: integrity check failed; condition: input or artifact integrity check."
            )
        reference_sd = same_cancer.std(ddof=0).replace(0, np.nan)
        median_shift = (
            external.median() - same_cancer.median()
        ) / reference_sd
        p01 = same_cancer.quantile(0.01)
        p99 = same_cancer.quantile(0.99)
        outside = external.lt(p01) | external.gt(p99)
        diagnostic_z = (external - same_cancer.mean()) / reference_sd
        for gene in features:
            gene_rows.append(
                {
                    "database_cancer": database_cancer,
                    "model_cancer": model_cancer,
                    "feature": gene,
                    "n_TCGA": len(same_cancer),
                    "n_CPTAC": len(external),
                    "TCGA_median": same_cancer[gene].median(),
                    "CPTAC_median": external[gene].median(),
                    "TCGA_SD": reference_sd[gene],
                    "median_shift_in_same_cancer_TCGA_SD": median_shift[gene],
                    "CPTAC_fraction_outside_TCGA_P01_P99": outside[
                        gene
                    ].mean(),
                }
            )
        patient_qc = pd.DataFrame(
            {
                "patient_id": external.index,
                "database_cancer": database_cancer,
                "n_features_outside_same_cancer_TCGA_P01_P99": outside.sum(
                    axis=1
                ).to_numpy(),
                "n_features_abs_same_cancer_z_gt_3": diagnostic_z.abs()
                .gt(3)
                .sum(axis=1)
                .to_numpy(),
            }
        )
        patient_rows.append(patient_qc)
        cohort_rows.append(
            {
                "database_cancer": database_cancer,
                "n_TCGA": len(same_cancer),
                "n_CPTAC": len(external),
                "n_features": len(features),
                "median_abs_gene_median_shift_SD": median_shift.abs().median(),
                "n_genes_abs_median_shift_gt_1SD": int(
                    median_shift.abs().gt(1).sum()
                ),
                "n_genes_abs_median_shift_gt_3SD": int(
                    median_shift.abs().gt(3).sum()
                ),
                "median_n_features_outside_same_cancer_P01_P99": outside.sum(
                    axis=1
                ).median(),
            }
        )
    gene_table = pd.DataFrame(gene_rows)
    cohort_table = pd.DataFrame(cohort_rows)
    core_table = gene_table.loc[
        gene_table["feature"].isin(["MTAP", "CDKN2A", "CDKN2B"])
    ].copy()
    old_file = OLD_REVIEW / "core_three_genes_scale_comparison.tsv"
    if old_file.is_file():
        old = pd.read_csv(old_file, sep="\t")
        old = old[
            [
                "database_cancer",
                "feature",
                "median_shift_in_same_cancer_TCGA_SD",
            ]
        ].rename(
            columns={
                "median_shift_in_same_cancer_TCGA_SD": "old_EBPP_median_shift_SD"
            }
        )
        core_table = core_table.merge(
            old,
            on=["database_cancer", "feature"],
            how="left",
            validate="one_to_one",
        )
        core_table["absolute_shift_reduced"] = (
            core_table["median_shift_in_same_cancer_TCGA_SD"].abs()
            < core_table["old_EBPP_median_shift_SD"].abs()
        )
    gene_table.to_csv(
        OUT / "TPM_44_gene_scale_comparison.tsv", sep="\t", index=False
    )
    core_table.to_csv(
        OUT / "TPM_core_genes_vs_previous_scale.tsv", sep="\t", index=False
    )
    cohort_table.to_csv(
        OUT / "TPM_cohort_scale_summary.tsv", sep="\t", index=False
    )
    pd.concat(patient_rows, ignore_index=True).to_csv(
        OUT / "TPM_patient_input_range_diagnostics.tsv", sep="\t", index=False
    )
    display_columns = [
        "database_cancer",
        "feature",
        "TCGA_median",
        "CPTAC_median",
        "median_shift_in_same_cancer_TCGA_SD",
    ]
    if "old_EBPP_median_shift_SD" in core_table.columns:
        display_columns += [
            "old_EBPP_median_shift_SD",
            "absolute_shift_reduced",
        ]
    _public_summary(globals(), "TCGA/CPTAC TPM input compatibility")


## 4. Model Training in the TCGA Development Set

Train only on development patients. The final TPM model branches are identified below. Earlier regional analyses are retained to reconstruct the development history; all stages execute in source dependency order.


### Model 1: Three-gene L2 logistic regression


#### Final TPM three-gene L2 fitting, locking, and OOF comparison


In [ ]:
@_nb9_registry.register("s39_0")
def _nb9_stage_s39_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        roc_curve,
        precision_recall_curve,
    )

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL/08_TPM_retraining"
        )
    )
    INPUT = BASE / "01_expression_inputs_44"
    PREVIOUS_LOCK = BASE / "04_locked_models_and_thresholds"
    PREVIOUS_EXTERNAL = BASE / "05_CPTAC_external_validation"
    CPTAC_INPUT = BASE / "CPTAC_inputs_44"
    OUT = BASE / "06_three_gene_L2_comparison"
    LOCK = OUT / "locked_baseline"
    FIGURES = OUT / "figures"
    for directory in [OUT, LOCK, FIGURES]:
        directory.mkdir(parents=True, exist_ok=True)
    X_FILE = INPUT / "X_development_log2TPM_plus_1.tsv"
    META_FILE = INPUT / "development_metadata.tsv"
    PREVIOUS_OOF_FILE = (
        PREVIOUS_LOCK / "records/development_OOF_predictions.tsv"
    )
    PREVIOUS_MANIFEST_FILE = PREVIOUS_LOCK / "locked_manifest.json"
    PREVIOUS_THRESHOLDS_FILE = (
        PREVIOUS_LOCK / "thresholds/locked_thresholds.json"
    )
    DL_MANIFEST_FILE = PREVIOUS_LOCK / "records/DL_training_manifest.json"
    EXTERNAL_PREDICTIONS_FILE = (
        PREVIOUS_EXTERNAL / "CPTAC_patient_predictions.tsv"
    )
    CORE_GENES = ["MTAP", "CDKN2A", "CDKN2B"]
    L2_NAME = "TPM3_L2"
    DL_NAME = "TPM44_DL_3seed"
    EN_NAME = "TPM44_ElasticNet"
    MODEL_NAMES = [DL_NAME, EN_NAME, L2_NAME]
    L2_SETTINGS = {
        "C": 1.0,
        "solver": "lbfgs",
        "penalty": "l2",
        "class_weight": None,
        "max_iter": 5000,
        "tol": 1e-06,
        "random_state": 41,
    }
    N_BOOTSTRAPS = 1000
    RANDOM_SEED = 20261017
    CANCER_MAP = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    PROBABILITY_COLUMNS = {
        DL_NAME: "prob_DL",
        EN_NAME: "prob_ElasticNet",
        L2_NAME: "prob_L2",
    }
    COLORS = {DL_NAME: "#33539E", EN_NAME: "#A5678E", L2_NAME: "#7FACD6"}
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    for path in [
        X_FILE,
        META_FILE,
        PREVIOUS_OOF_FILE,
        PREVIOUS_MANIFEST_FILE,
        PREVIOUS_THRESHOLDS_FILE,
        DL_MANIFEST_FILE,
        EXTERNAL_PREDICTIONS_FILE,
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def save_json(value, path):
        with Path(path).open("w", encoding="utf-8") as handle:
            json.dump(
                value, handle, ensure_ascii=False, indent=2, allow_nan=False
            )

    def wilson_interval(successes, n):
        if n == 0:
            return (np.nan, np.nan)
        z = 1.959963984540054
        p = successes / n
        denominator = 1 + z * z / n
        center = (p + z * z / (2 * n)) / denominator
        half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
        return (center - half, center + half)

    def performance(y, probability):
        y = np.asarray(y, dtype=int)
        p = np.asarray(probability, dtype=float)
        if not np.isfinite(p).all():
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        if ((p < 0) | (p > 1)).any():
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        has_both = len(np.unique(y)) == 2
        predicted = p >= 0.5
        tp = int(((y == 1) & predicted).sum())
        fp = int(((y == 0) & predicted).sum())
        fn = int(((y == 1) & ~predicted).sum())
        tn = int(((y == 0) & ~predicted).sum())
        return {
            "n_patients": len(y),
            "n_positive": int((y == 1).sum()),
            "n_negative": int((y == 0).sum()),
            "positive_fraction": float((y == 1).mean()),
            "AUROC": roc_auc_score(y, p) if has_both else np.nan,
            "AP": average_precision_score(y, p) if has_both else np.nan,
            "Brier_score": brier_score_loss(y, p),
            "log_loss": log_loss(
                y, np.clip(p, 1e-07, 1 - 1e-07), labels=[0, 1]
            ),
            "precision_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "sensitivity_at_05": tp / (tp + fn) if tp + fn else np.nan,
            "specificity_at_05": tn / (tn + fp) if tn + fp else np.nan,
            "TP_at_05": tp,
            "FP_at_05": fp,
            "FN_at_05": fn,
            "TN_at_05": tn,
        }

    def classify_groups(probability, thresholds):
        p = np.asarray(probability, dtype=float)
        groups = np.full(len(p), "Uncertain", dtype=object)
        low = thresholds["low_threshold"]
        high = thresholds["high_threshold"]
        if low is not None:
            groups[p <= low] = "Predicted_non_deep"
        if high is not None:
            groups[p >= high] = "Predicted_deep_deletion"
        return groups

    def summarize_groups(y, probability, thresholds, model, cohort):
        y = np.asarray(y, dtype=int)
        groups = classify_groups(probability, thresholds)
        rows = []
        total_positive = int(y.sum())
        for group in [
            "Predicted_deep_deletion",
            "Uncertain",
            "Predicted_non_deep",
        ]:
            mask = groups == group
            n = int(mask.sum())
            n_positive = int(y[mask].sum())
            if group == "Predicted_deep_deletion":
                value_name = "PPV"
                successes = n_positive
            elif group == "Predicted_non_deep":
                value_name = "NPV"
                successes = n - n_positive
            else:
                value_name = "DNA_positive_fraction"
                successes = n_positive
            low_ci, high_ci = wilson_interval(successes, n)
            rows.append(
                {
                    "cohort": cohort,
                    "model": model,
                    "group": group,
                    "n_patients": n,
                    "coverage_fraction": n / len(y),
                    "n_DNA_positive": n_positive,
                    "fraction_all_DNA_positives": (
                        n_positive / total_positive
                        if total_positive
                        else np.nan
                    ),
                    "value_name": value_name,
                    "value": successes / n if n else np.nan,
                    "Wilson_CI95_low": low_ci,
                    "Wilson_CI95_high": high_ci,
                }
            )
        return rows

    with PREVIOUS_MANIFEST_FILE.open("r", encoding="utf-8") as handle:
        previous_manifest = json.load(handle)
    for relative_path, expected_hash in previous_manifest[
        "locked_file_hashes"
    ].items():
        path = PREVIOUS_LOCK / relative_path
        if not path.is_file() or sha256_file(path) != expected_hash:
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
    threshold_rules = previous_manifest["threshold_rules"]
    with PREVIOUS_THRESHOLDS_FILE.open("r", encoding="utf-8") as handle:
        previous_thresholds = json.load(handle)
    with DL_MANIFEST_FILE.open("r", encoding="utf-8") as handle:
        dl_manifest = json.load(handle)
    cancer_categories = sorted(dl_manifest["cancer_to_index"])
    X = pd.read_csv(X_FILE, sep="\t", index_col=0)
    X.index = X.index.astype(str)
    if X.index.duplicated().any():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    X = X.loc[:, CORE_GENES].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    meta = pd.read_csv(
        META_FILE, sep="\t", dtype={"patient_id": str, "acronym": str}
    )
    if meta["patient_id"].duplicated().any():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.set_index("patient_id")
    if set(meta.index) != set(X.index):
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.loc[X.index]
    y = pd.to_numeric(meta["label_MTAP_deep"], errors="raise").to_numpy(
        dtype=int
    )
    if not np.isin(y, [0, 1]).all():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    previous_oof = pd.read_csv(
        PREVIOUS_OOF_FILE, sep="\t", dtype={"patient_id": str}
    )
    if previous_oof["patient_id"].duplicated().any():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    previous_oof = previous_oof.set_index("patient_id")
    if set(previous_oof.index) != set(X.index):
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    previous_oof = previous_oof.loc[X.index]
    if not np.array_equal(
        previous_oof["label_MTAP_deep"].to_numpy(dtype=int), y
    ):
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        previous_oof["acronym"].astype(str).to_numpy(),
        meta["acronym"].astype(str).to_numpy(),
    ):
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    folds = previous_oof["outer_fold"].to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    if not set(meta["acronym"]).issubset(cancer_categories):
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    X_model = X.copy()
    X_model["acronym"] = meta["acronym"].astype(str)

    def make_pipeline():
        preprocessing = ColumnTransformer(
            transformers=[
                ("expression", StandardScaler(), CORE_GENES),
                (
                    "cancer",
                    OneHotEncoder(
                        categories=[cancer_categories],
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                    ["acronym"],
                ),
            ],
            remainder="drop",
            sparse_threshold=0,
        )
        return Pipeline(
            [
                ("preprocessing", preprocessing),
                ("model", LogisticRegression(**L2_SETTINGS)),
            ]
        )

    def select_thresholds(y, probability):
        table = (
            pd.DataFrame({"probability": probability, "positive": y})
            .groupby("probability")
            .agg(n=("positive", "size"), positives=("positive", "sum"))
        )
        high_table = table.sort_index(ascending=False).copy()
        high_table["selected_n"] = high_table["n"].cumsum()
        high_table["selected_positive"] = high_table["positives"].cumsum()
        high_table["PPV"] = (
            high_table["selected_positive"] / high_table["selected_n"]
        )
        high_table["Wilson_low"] = [
            wilson_interval(int(k), int(n))[0]
            for k, n in zip(
                high_table["selected_positive"], high_table["selected_n"]
            )
        ]
        high_table["eligible"] = (
            high_table["selected_n"].ge(threshold_rules["positive_min_n"])
            & high_table["PPV"].ge(threshold_rules["positive_PPV_target"])
            & high_table["Wilson_low"].ge(
                threshold_rules["positive_Wilson_low_min"]
            )
        )
        eligible_high = high_table.loc[high_table["eligible"]]
        high = (
            float(eligible_high["selected_n"].idxmax())
            if not eligible_high.empty
            else None
        )
        low_table = table.sort_index().copy()
        low_table["selected_n"] = low_table["n"].cumsum()
        low_table["selected_positive"] = low_table["positives"].cumsum()
        low_table["selected_negative"] = (
            low_table["selected_n"] - low_table["selected_positive"]
        )
        low_table["NPV"] = (
            low_table["selected_negative"] / low_table["selected_n"]
        )
        low_table["Wilson_low"] = [
            wilson_interval(int(k), int(n))[0]
            for k, n in zip(
                low_table["selected_negative"], low_table["selected_n"]
            )
        ]
        low_table["eligible"] = (
            low_table["selected_n"].ge(threshold_rules["negative_min_n"])
            & low_table["NPV"].ge(threshold_rules["negative_NPV_target"])
            & low_table["Wilson_low"].ge(
                threshold_rules["negative_Wilson_low_min"]
            )
        )
        if high is not None:
            low_table["eligible"] &= low_table.index.to_numpy() < high
        eligible_low = low_table.loc[low_table["eligible"]]
        low = (
            float(eligible_low["selected_n"].idxmax())
            if not eligible_low.empty
            else None
        )
        return (
            {
                "low_threshold": low,
                "high_threshold": high,
                "positive_rule_met": high is not None,
                "negative_rule_met": low is not None,
            },
            high_table,
            low_table,
        )

    baseline_manifest_file = LOCK / "locked_manifest.json"
    source_hashes = {
        str(path): sha256_file(path)
        for path in [
            X_FILE,
            META_FILE,
            PREVIOUS_OOF_FILE,
            PREVIOUS_MANIFEST_FILE,
            PREVIOUS_THRESHOLDS_FILE,
        ]
    }
    if baseline_manifest_file.exists():
        with baseline_manifest_file.open("r", encoding="utf-8") as handle:
            baseline_manifest = json.load(handle)
        if baseline_manifest["source_hashes"] != source_hashes:
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        if baseline_manifest["L2_settings"] != L2_SETTINGS:
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        if baseline_manifest["threshold_rules"] != threshold_rules:
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        for name, expected_hash in baseline_manifest[
            "locked_file_hashes"
        ].items():
            path = LOCK / name
            if not path.is_file() or sha256_file(path) != expected_hash:
                raise ValueError(
                    "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
                )
        bundle = joblib.load(LOCK / "three_gene_L2.joblib")
        cached_oof = (
            pd.read_csv(
                LOCK / "development_OOF_predictions.tsv",
                sep="\t",
                dtype={"patient_id": str},
            )
            .set_index("patient_id")
            .loc[X.index]
        )
        p_l2_oof = cached_oof["prob_L2"].to_numpy()
        with (LOCK / "thresholds.json").open("r", encoding="utf-8") as handle:
            l2_thresholds = json.load(handle)
    else:
        p_l2_oof = np.full(len(X_model), np.nan)
        fold_records = []
        for fold in [1, 2, 3, 4, 5]:
            train_indices = np.flatnonzero(folds != fold)
            valid_indices = np.flatnonzero(folds == fold)
            pipeline = make_pipeline()
            pipeline.fit(X_model.iloc[train_indices], y[train_indices])
            n_iterations = int(np.max(pipeline.named_steps["model"].n_iter_))
            if n_iterations >= L2_SETTINGS["max_iter"]:
                raise RuntimeError(
                    "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
                )
            probability = pipeline.predict_proba(X_model.iloc[valid_indices])[
                :, 1
            ]
            p_l2_oof[valid_indices] = probability
            result = performance(y[valid_indices], probability)
            fold_records.append(
                {
                    "outer_fold": fold,
                    "n_train": len(train_indices),
                    **result,
                    "n_iterations": n_iterations,
                }
            )
        if not np.isfinite(p_l2_oof).all():
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        final_pipeline = make_pipeline()
        final_pipeline.fit(X_model, y)
        final_iterations = int(
            np.max(final_pipeline.named_steps["model"].n_iter_)
        )
        if final_iterations >= L2_SETTINGS["max_iter"]:
            raise RuntimeError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        bundle = {
            "pipeline": final_pipeline,
            "feature_names": CORE_GENES,
            "input_columns": CORE_GENES + ["acronym"],
            "expression_unit": "log2(TPM+1)",
            "L2_settings": L2_SETTINGS,
        }
        l2_thresholds, high_table, low_table = select_thresholds(y, p_l2_oof)
        joblib.dump(bundle, LOCK / "three_gene_L2.joblib")
        pd.DataFrame(
            {
                "patient_id": X.index,
                "acronym": meta["acronym"].to_numpy(),
                "label_MTAP_deep": y,
                "outer_fold": folds,
                "prob_L2": p_l2_oof,
            }
        ).to_csv(
            LOCK / "development_OOF_predictions.tsv", sep="\t", index=False
        )
        pd.DataFrame(fold_records).to_csv(
            LOCK / "outer_fold_performance.tsv", sep="\t", index=False
        )
        high_table.to_csv(
            LOCK / "high_threshold_candidates.tsv",
            sep="\t",
            index_label="threshold",
        )
        low_table.to_csv(
            LOCK / "low_threshold_candidates.tsv",
            sep="\t",
            index_label="threshold",
        )
        save_json(l2_thresholds, LOCK / "thresholds.json")
        feature_names = final_pipeline.named_steps[
            "preprocessing"
        ].get_feature_names_out()
        pd.DataFrame(
            {
                "feature": feature_names,
                "coefficient": final_pipeline.named_steps[
                    "model"
                ].coef_.ravel(),
            }
        ).to_csv(LOCK / "final_model_coefficients.tsv", sep="\t", index=False)
        lock_files = sorted(
            (
                path
                for path in LOCK.iterdir()
                if path.is_file() and path != baseline_manifest_file
            )
        )
        baseline_manifest = {
            "model": L2_NAME,
            "source_hashes": source_hashes,
            "L2_settings": L2_SETTINGS,
            "threshold_rules": threshold_rules,
            "feature_order": CORE_GENES,
            "expression_unit": "log2(TPM+1)",
            "outer_folds": "Inherited original DL patient folds",
            "CPTAC_used_to_fit_model_or_threshold": False,
            "analysis_status": "Additional baseline comparison after previous CPTAC model results were already examined",
            "locked_file_hashes": {
                path.name: sha256_file(path) for path in lock_files
            },
        }
        save_json(baseline_manifest, baseline_manifest_file)
    development = previous_oof.copy()
    development["prob_L2"] = p_l2_oof
    development_rows = [
        {
            "model": name,
            **performance(
                y, development[PROBABILITY_COLUMNS[name]].to_numpy()
            ),
        }
        for name in MODEL_NAMES
    ]
    development_results = pd.DataFrame(development_rows)
    development_results.to_csv(
        OUT / "three_model_development_performance.tsv", sep="\t", index=False
    )
    development.to_csv(
        OUT / "three_model_development_OOF.tsv",
        sep="\t",
        index_label="patient_id",
    )
    development_group_rows = []
    all_thresholds = {
        DL_NAME: previous_thresholds[DL_NAME],
        EN_NAME: previous_thresholds[EN_NAME],
        L2_NAME: l2_thresholds,
    }
    for name in MODEL_NAMES:
        development_group_rows.extend(
            summarize_groups(
                y,
                development[PROBABILITY_COLUMNS[name]].to_numpy(),
                all_thresholds[name],
                name,
                "Development_OOF",
            )
        )
    pd.DataFrame(development_group_rows).to_csv(
        OUT / "three_model_development_groups.tsv", sep="\t", index=False
    )


#### Fixed L2 baselines and development out-of-fold predictions


In [ ]:
@_nb9_registry.register("s05_0")
def _nb9_stage_s05_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import time
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import joblib
    from sklearn.base import clone
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        roc_curve,
        precision_recall_curve,
        confusion_matrix,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    PREVIOUS_DIR = (
        PROJECT / "data_processed/MTAP_RNA_prediction/02_labels_and_split"
    )
    COHORT_FILE = PREVIOUS_DIR / "TCGA_prediction_cohort_with_split.tsv"
    RNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA"
        / "EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv"
    )
    OUTPUT_DIR = (
        PROJECT / "data_processed/MTAP_RNA_prediction/03_baseline_models"
    )
    FIGURE_DIR = OUTPUT_DIR / "figures"
    MODEL_DIR = OUTPUT_DIR / "development_models"
    for directory in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    EXPRESSION_CACHE = OUTPUT_DIR / "three_gene_patient_expression.tsv"
    RANDOM_SEED = 20261016
    N_FOLDS = 5
    REGULARIZATION_C = 1.0
    DESCRIPTIVE_THRESHOLD = 0.5
    GENE_ENTREZ = {"MTAP": "4507", "CDKN2A": "1029", "CDKN2B": "1030"}
    GENES = list(GENE_ENTREZ)
    MTAP_PALETTE = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]
    for path in [COHORT_FILE, RNA_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
    cohort = pd.read_csv(
        COHORT_FILE,
        sep="\t",
        dtype={
            "patient_id": "string",
            "acronym": "string",
            "dataset_split": "string",
        },
    )
    required_columns = {
        "patient_id",
        "acronym",
        "dataset_split",
        "label_MTAP_deep",
        "MTAP_GISTIC_call",
        "CDKN2A_GISTIC_call",
        "CDKN2B_GISTIC_call",
        "MTAP_RNA",
    }
    missing = required_columns.difference(cohort.columns)
    if missing:
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    if cohort["patient_id"].isna().any():
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    if cohort["patient_id"].duplicated().any():
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    if not cohort["dataset_split"].isin(["development", "test"]).all():
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    if not cohort["label_MTAP_deep"].isin([0, 1]).all():
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    if (
        not cohort["label_MTAP_deep"]
        .eq(cohort["MTAP_GISTIC_call"].eq(-2).astype(int))
        .all()
    ):
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    cohort = cohort.sort_values("patient_id").reset_index(drop=True)
    patient_ids = cohort["patient_id"].tolist()

    def parse_gene_id(value):
        value = str(value).strip()
        parts = value.split("|")
        symbol = parts[0].strip()
        entrez = ""
        if len(parts) >= 2:
            entrez = parts[-1].strip()
            if entrez.endswith(".0"):
                entrez = entrez[:-2]
        return (symbol, entrez)

    if EXPRESSION_CACHE.is_file():
        expression = pd.read_csv(
            EXPRESSION_CACHE, sep="\t", dtype={"patient_id": "string"}
        )
        if expression["patient_id"].duplicated().any():
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        if set(expression["patient_id"]) != set(patient_ids):
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        needed_expression = {"patient_id", *[f"{gene}_RNA" for gene in GENES]}
        if not needed_expression.issubset(expression.columns):
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
    else:
        header = pd.read_csv(RNA_FILE, sep="\t", nrows=0).columns.tolist()
        gene_column = header[0]
        patient_set = set(patient_ids)
        mapping_records = []
        for column in header[1:]:
            parts = str(column).strip().split("-")
            if (
                len(parts) >= 4
                and parts[0] == "TCGA"
                and (parts[3][:2] == "01")
            ):
                patient = "-".join(parts[:3])
                if patient in patient_set:
                    mapping_records.append(
                        {"RNA_column": column, "patient_id": patient}
                    )
        column_map = pd.DataFrame(mapping_records)
        if column_map.empty:
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        if set(column_map["patient_id"]) != patient_set:
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        selected_columns = column_map["RNA_column"].tolist()
        matching_rows = {gene: [] for gene in GENES}
        matching_audit = []
        start_time = time.time()
        n_scanned = 0
        reader = pd.read_csv(
            RNA_FILE,
            sep="\t",
            usecols=[gene_column] + selected_columns,
            chunksize=128,
        )
        for chunk in reader:
            n_scanned += len(chunk)
            for row_index, raw_id in chunk[gene_column].items():
                symbol, entrez = parse_gene_id(raw_id)
                for gene, expected_entrez in GENE_ENTREZ.items():
                    id_match = entrez == expected_entrez
                    symbol_match = symbol == gene
                    if not (id_match or symbol_match):
                        continue
                    if symbol_match and entrez and (entrez != expected_entrez):
                        raise ValueError(
                            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
                        )
                    if symbol in GENE_ENTREZ and symbol != gene:
                        raise ValueError(
                            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
                        )
                    values = pd.to_numeric(
                        chunk.loc[row_index, selected_columns], errors="coerce"
                    )
                    matching_rows[gene].append(values)
                    matching_audit.append(
                        {
                            "target_gene": gene,
                            "expected_Entrez": expected_entrez,
                            "RNA_gene_id": str(raw_id),
                            "match_method": (
                                "Entrez_verified"
                                if id_match
                                else "Exact_symbol"
                            ),
                        }
                    )
            if n_scanned % 4096 == 0:
                found = {
                    gene: len(rows) for gene, rows in matching_rows.items()
                }
        audit = pd.DataFrame(matching_audit)
        audit.to_csv(
            OUTPUT_DIR / "three_gene_RNA_mapping_audit.tsv",
            sep="\t",
            index=False,
        )
        for gene, rows in matching_rows.items():
            if len(rows) != 1:
                raise ValueError(
                    "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
                )
        expression = pd.DataFrame({"patient_id": patient_ids})
        for gene in GENES:
            values = matching_rows[gene][0]
            sample_values = column_map.copy()
            sample_values["expression"] = sample_values["RNA_column"].map(
                values
            )
            numeric = sample_values["expression"].to_numpy(dtype=float)
            if not np.isfinite(numeric).all():
                raise ValueError(
                    "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
                )
            patient_values = sample_values.groupby("patient_id")[
                "expression"
            ].mean()
            expression[f"{gene}_RNA"] = expression["patient_id"].map(
                patient_values
            )
        expression.to_csv(EXPRESSION_CACHE, sep="\t", index=False)
        column_map.to_csv(
            OUTPUT_DIR / "three_gene_RNA_sample_map.tsv", sep="\t", index=False
        )
    cohort = cohort.rename(columns={"MTAP_RNA": "MTAP_RNA_previous"})
    cohort = cohort.merge(
        expression[["patient_id"] + [f"{gene}_RNA" for gene in GENES]],
        on="patient_id",
        how="left",
        validate="one_to_one",
    )
    rna_features = [f"{gene}_RNA" for gene in GENES]
    for column in rna_features + ["MTAP_RNA_previous"]:
        cohort[column] = pd.to_numeric(cohort[column], errors="coerce")
    values = cohort[rna_features].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    if (values < 0).any():
        negative_summary = cohort[rna_features].lt(0).sum()
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    previous_mtap = cohort["MTAP_RNA_previous"].to_numpy(dtype=float)
    if not np.isfinite(previous_mtap).all():
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    mtap_consistent = np.allclose(
        cohort["MTAP_RNA"].to_numpy(dtype=float),
        previous_mtap,
        rtol=1e-05,
        atol=1e-05,
    )
    if not mtap_consistent:
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    for gene in GENES:
        cohort[f"{gene}_log2_RNA"] = np.log2(cohort[f"{gene}_RNA"] + 1)
    cohort.to_csv(
        OUTPUT_DIR / "cohort_with_three_gene_expression.tsv",
        sep="\t",
        index=False,
    )
    development = (
        cohort.loc[cohort["dataset_split"].eq("development")]
        .sort_values("patient_id")
        .reset_index(drop=True)
    )
    y = development["label_MTAP_deep"].astype(int).to_numpy()
    if min(np.bincount(y, minlength=2)) < N_FOLDS:
        raise ValueError(
            "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
        )
    cv = StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED
    )
    splits = list(cv.split(development, y))
    fold_assignment = np.full(len(development), -1, dtype=int)
    for fold, (_, validation_index) in enumerate(splits, start=1):
        fold_assignment[validation_index] = fold
    fold_file = OUTPUT_DIR / "development_fixed_CV_folds.tsv"
    fold_table = development[
        ["patient_id", "acronym", "label_MTAP_deep"]
    ].copy()
    fold_table["CV_fold"] = fold_assignment
    if fold_file.is_file():
        existing = pd.read_csv(
            fold_file, sep="\t", dtype={"patient_id": "string"}
        )
        if existing["patient_id"].duplicated().any():
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        expected = fold_table.set_index("patient_id").sort_index()
        observed = existing.set_index("patient_id").sort_index()
        if not expected.index.equals(observed.index):
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        for column in ["label_MTAP_deep", "CV_fold"]:
            if not np.array_equal(
                expected[column].to_numpy(), observed[column].to_numpy()
            ):
                raise ValueError(
                    "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
                )
    else:
        fold_table.to_csv(fold_file, sep="\t", index=False)

    def make_logistic():
        return LogisticRegression(
            C=REGULARIZATION_C,
            penalty="l2",
            solver="lbfgs",
            class_weight=None,
            max_iter=3000,
            random_state=RANDOM_SEED,
        )

    def make_expression_model(columns):
        preprocessing = ColumnTransformer(
            transformers=[("RNA", StandardScaler(), columns)], remainder="drop"
        )
        return Pipeline(
            [("preprocessing", preprocessing), ("classifier", make_logistic())]
        )

    cancer_preprocessing = ColumnTransformer(
        transformers=[
            (
                "cancer",
                OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                ["acronym"],
            )
        ],
        remainder="drop",
    )
    models = {
        "Cancer_only": Pipeline(
            [
                ("preprocessing", cancer_preprocessing),
                ("classifier", make_logistic()),
            ]
        ),
        "MTAP_only": make_expression_model(["MTAP_log2_RNA"]),
        "MTAP_CDKN2A": make_expression_model(
            ["MTAP_log2_RNA", "CDKN2A_log2_RNA"]
        ),
        "MTAP_CDKN2B": make_expression_model(
            ["MTAP_log2_RNA", "CDKN2B_log2_RNA"]
        ),
        "Three_gene": make_expression_model(
            ["MTAP_log2_RNA", "CDKN2A_log2_RNA", "CDKN2B_log2_RNA"]
        ),
    }
    X = development[
        ["acronym", "MTAP_log2_RNA", "CDKN2A_log2_RNA", "CDKN2B_log2_RNA"]
    ].copy()

    def evaluate_predictions(labels, probabilities):
        labels = np.asarray(labels, dtype=int)
        probabilities = np.asarray(probabilities, dtype=float)
        predictions = (probabilities >= DESCRIPTIVE_THRESHOLD).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            labels, predictions, labels=[0, 1]
        ).ravel()
        has_both = np.unique(labels).size == 2
        return {
            "n_patients": int(len(labels)),
            "n_positive": int(labels.sum()),
            "n_negative": int((labels == 0).sum()),
            "prevalence": float(labels.mean()),
            "AUROC": (
                float(roc_auc_score(labels, probabilities))
                if has_both
                else np.nan
            ),
            "AUPRC_AP": (
                float(average_precision_score(labels, probabilities))
                if has_both
                else np.nan
            ),
            "Brier_score": float(brier_score_loss(labels, probabilities)),
            "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
            "threshold": DESCRIPTIVE_THRESHOLD,
            "sensitivity_at_05": tp / (tp + fn) if tp + fn > 0 else np.nan,
            "specificity_at_05": tn / (tn + fp) if tn + fp > 0 else np.nan,
            "PPV_at_05": tp / (tp + fp) if tp + fp > 0 else np.nan,
            "NPV_at_05": tn / (tn + fn) if tn + fn > 0 else np.nan,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
        }

    prediction_table = development[
        [
            "patient_id",
            "acronym",
            "label_MTAP_deep",
            "MTAP_GISTIC_call",
            "CDKN2A_GISTIC_call",
            "CDKN2B_GISTIC_call",
        ]
    ].copy()
    prediction_table["CV_fold"] = fold_assignment
    overall_records = []
    fold_records = []
    for model_name, template in models.items():
        oof_probability = np.full(len(development), np.nan, dtype=float)
        for fold, (train_index, validation_index) in enumerate(
            splits, start=1
        ):
            estimator = clone(template)
            estimator.fit(X.iloc[train_index], y[train_index])
            probability = estimator.predict_proba(X.iloc[validation_index])[
                :, 1
            ]
            oof_probability[validation_index] = probability
            fold_result = evaluate_predictions(
                y[validation_index], probability
            )
            fold_records.append(
                {"model": model_name, "fold": fold, **fold_result}
            )
        if not np.isfinite(oof_probability).all():
            raise ValueError(
                "Fixed L2 baselines and development out-of-fold predictions: integrity check failed; condition: input or artifact integrity check."
            )
        prediction_table[f"probability_{model_name}"] = oof_probability
        overall_records.append(
            {
                "model": model_name,
                "evaluation": "Development_5fold_OOF",
                **evaluate_predictions(y, oof_probability),
            }
        )
        development_fit = clone(template).fit(X, y)
        joblib.dump(
            development_fit,
            MODEL_DIR / f"{model_name}_development_fit_NOT_FINAL.joblib",
        )
    overall = pd.DataFrame(overall_records)
    fold_statistics = pd.DataFrame(fold_records)
    prediction_table.to_csv(
        OUTPUT_DIR / "development_OOF_predictions.tsv", sep="\t", index=False
    )
    overall.to_csv(
        OUTPUT_DIR / "baseline_OOF_overall_metrics.tsv", sep="\t", index=False
    )
    fold_statistics.to_csv(
        OUTPUT_DIR / "baseline_metrics_by_fold.tsv", sep="\t", index=False
    )
    cancer_records = []
    for cancer, subset in prediction_table.groupby("acronym", observed=True):
        labels = subset["label_MTAP_deep"].to_numpy()
        n_positive = int(labels.sum())
        n_negative = int((labels == 0).sum())
        if n_positive == 0 or n_negative == 0:
            support = "Single_class_no_AUROC"
        elif n_positive < 10 or n_negative < 20:
            support = "Low_support_interpret_cautiously"
        else:
            support = "Descriptive_adequate_counts"
        for model_name in models:
            result = evaluate_predictions(
                labels, subset[f"probability_{model_name}"].to_numpy()
            )
            cancer_records.append(
                {
                    "acronym": cancer,
                    "model": model_name,
                    "support_status": support,
                    **result,
                }
            )
    cancer_statistics = pd.DataFrame(cancer_records)
    cancer_statistics.to_csv(
        OUTPUT_DIR / "baseline_OOF_metrics_by_cancer.tsv",
        sep="\t",
        index=False,
    )
    subgroup_masks = {
        "All_MTAP_deep": prediction_table["label_MTAP_deep"].eq(1),
        "All_MTAP_non_deep": prediction_table["label_MTAP_deep"].eq(0),
        "CDKN2A_deep_but_MTAP_non_deep": prediction_table[
            "CDKN2A_GISTIC_call"
        ].eq(-2)
        & prediction_table["label_MTAP_deep"].eq(0),
        "CDKN2B_deep_but_MTAP_non_deep": prediction_table[
            "CDKN2B_GISTIC_call"
        ].eq(-2)
        & prediction_table["label_MTAP_deep"].eq(0),
        "MTAP_deep_without_CDKN2B_deep": prediction_table[
            "label_MTAP_deep"
        ].eq(1)
        & prediction_table["CDKN2B_GISTIC_call"].ne(-2),
    }
    subgroup_records = []
    for subgroup_name, mask in subgroup_masks.items():
        subset = prediction_table.loc[mask]
        if subset.empty:
            continue
        for model_name in models:
            probability = subset[f"probability_{model_name}"].to_numpy()
            subgroup_records.append(
                {
                    "subgroup": subgroup_name,
                    "model": model_name,
                    "n_patients": len(subset),
                    "n_DNA_positive": int(subset["label_MTAP_deep"].sum()),
                    "mean_predicted_probability": float(probability.mean()),
                    "median_predicted_probability": float(
                        np.median(probability)
                    ),
                    "fraction_predicted_positive_at_05": float(
                        (probability >= DESCRIPTIVE_THRESHOLD).mean()
                    ),
                }
            )
    subgroup_statistics = pd.DataFrame(subgroup_records)
    subgroup_statistics.to_csv(
        OUTPUT_DIR / "baseline_OOF_DNA_subgroup_review.tsv",
        sep="\t",
        index=False,
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


#### Development-only probability-calibration sensitivity analysis


In [ ]:
@_nb9_registry.register("s07_0")
def _nb9_stage_s07_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import joblib
    from sklearn.base import clone
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    BASELINE_DIR = (
        PROJECT / "data_processed/MTAP_RNA_prediction/03_baseline_models"
    )
    COHORT_FILE = BASELINE_DIR / "cohort_with_three_gene_expression.tsv"
    OOF_FILE = BASELINE_DIR / "development_OOF_predictions.tsv"
    BASE_MODEL_FILE = (
        BASELINE_DIR
        / "development_models"
        / "Three_gene_development_fit_NOT_FINAL.joblib"
    )
    OUTPUT_DIR = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction/05_probability_calibration"
    )
    FIGURE_DIR = OUTPUT_DIR / "figures"
    MODEL_DIR = OUTPUT_DIR / "development_models"
    for directory in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    RANDOM_SEED = 20261016
    INNER_FOLDS = 5
    METHODS = ["Uncalibrated", "Sigmoid", "Isotonic"]
    FEATURE_COLUMNS = [
        "acronym",
        "MTAP_log2_RNA",
        "CDKN2A_log2_RNA",
        "CDKN2B_log2_RNA",
    ]
    PROBABILITY_BINS = [0.0, 0.01, 0.025, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
    COLORS = {
        "Uncalibrated": "#33539E",
        "Sigmoid": "#7FACD6",
        "Isotonic": "#A5678E",
    }
    for path in [COHORT_FILE, OOF_FILE, BASE_MODEL_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
            )
    cohort = pd.read_csv(
        COHORT_FILE,
        sep="\t",
        dtype={"patient_id": "string", "acronym": "string"},
    )
    previous_oof = pd.read_csv(
        OOF_FILE, sep="\t", dtype={"patient_id": "string"}
    )
    required_cohort = {
        "patient_id",
        "dataset_split",
        "label_MTAP_deep",
        *FEATURE_COLUMNS,
    }
    required_oof = {
        "patient_id",
        "label_MTAP_deep",
        "CV_fold",
        "probability_Three_gene",
    }
    for name, frame, required in [
        ("Expression cohort", cohort, required_cohort),
        ("Existing OOF predictions", previous_oof, required_oof),
    ]:
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(
                "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
            )
        if (
            frame["patient_id"].isna().any()
            or frame["patient_id"].duplicated().any()
        ):
            raise ValueError(
                "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
            )
    development = cohort.loc[cohort["dataset_split"].eq("development")].copy()
    if set(development["patient_id"]) != set(previous_oof["patient_id"]):
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    reference = previous_oof[
        ["patient_id", "label_MTAP_deep", "CV_fold", "probability_Three_gene"]
    ].rename(
        columns={
            "label_MTAP_deep": "reference_label",
            "probability_Three_gene": "reference_probability",
        }
    )
    development = (
        development.merge(
            reference, on="patient_id", how="left", validate="one_to_one"
        )
        .sort_values("patient_id")
        .reset_index(drop=True)
    )
    if (
        not development["label_MTAP_deep"]
        .eq(development["reference_label"])
        .all()
    ):
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    if not development["label_MTAP_deep"].isin([0, 1]).all():
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    if development["acronym"].isna().any():
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    numeric_columns = ["MTAP_log2_RNA", "CDKN2A_log2_RNA", "CDKN2B_log2_RNA"]
    for column in numeric_columns:
        development[column] = pd.to_numeric(
            development[column], errors="coerce"
        )
    if not np.isfinite(
        development[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    if development["CV_fold"].isna().any():
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    fold_values = pd.to_numeric(development["CV_fold"], errors="coerce")
    if fold_values.isna().any() or not fold_values.isin([1, 2, 3, 4, 5]).all():
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    development["CV_fold"] = fold_values.astype(int)
    reference_prob = development["reference_probability"].to_numpy(dtype=float)
    if (
        not np.isfinite(reference_prob).all()
        or (reference_prob < 0).any()
        or (reference_prob > 1).any()
    ):
        raise ValueError(
            "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
        )
    X = development[FEATURE_COLUMNS].copy()
    y = development["label_MTAP_deep"].astype(int).to_numpy()
    base_template = clone(joblib.load(BASE_MODEL_FILE))

    def evaluate(labels, probabilities):
        return {
            "AUROC": roc_auc_score(labels, probabilities),
            "AUPRC_AP": average_precision_score(labels, probabilities),
            "Brier_score": brier_score_loss(labels, probabilities),
            "log_loss": log_loss(labels, probabilities, labels=[0, 1]),
        }

    oof = {method: np.full(len(development), np.nan) for method in METHODS}
    fold_records = []
    for fold in sorted(development["CV_fold"].unique()):
        validation_index = np.flatnonzero(
            development["CV_fold"].eq(fold).to_numpy()
        )
        training_index = np.flatnonzero(
            development["CV_fold"].ne(fold).to_numpy()
        )
        X_train = X.iloc[training_index]
        y_train = y[training_index]
        X_valid = X.iloc[validation_index]
        y_valid = y[validation_index]
        if min(np.bincount(y_train, minlength=2)) < INNER_FOLDS:
            raise ValueError(
                "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
            )
        raw_model = clone(base_template).fit(X_train, y_train)
        raw_probability = raw_model.predict_proba(X_valid)[:, 1]
        if not np.allclose(
            raw_probability,
            reference_prob[validation_index],
            rtol=1e-06,
            atol=1e-07,
        ):
            raise ValueError(
                "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
            )
        oof["Uncalibrated"][validation_index] = raw_probability
        fold_records.append(
            {
                "outer_fold": int(fold),
                "method": "Uncalibrated",
                **evaluate(y_valid, raw_probability),
            }
        )
        for display_name, calibration_method in [
            ("Sigmoid", "sigmoid"),
            ("Isotonic", "isotonic"),
        ]:
            inner_cv = StratifiedKFold(
                n_splits=INNER_FOLDS,
                shuffle=True,
                random_state=RANDOM_SEED + int(fold),
            )
            calibrated_model = CalibratedClassifierCV(
                estimator=clone(base_template),
                method=calibration_method,
                cv=inner_cv,
                ensemble=False,
                n_jobs=1,
            )
            calibrated_model.fit(X_train, y_train)
            probability = calibrated_model.predict_proba(X_valid)[:, 1]
            oof[display_name][validation_index] = probability
            fold_records.append(
                {
                    "outer_fold": int(fold),
                    "method": display_name,
                    **evaluate(y_valid, probability),
                }
            )
    prediction_table = development[
        ["patient_id", "acronym", "label_MTAP_deep", "CV_fold"]
    ].copy()
    overall_records = []
    for method in METHODS:
        probabilities = oof[method]
        if (
            not np.isfinite(probabilities).all()
            or (probabilities < 0).any()
            or (probabilities > 1).any()
        ):
            raise ValueError(
                "Development-only probability-calibration sensitivity analysis: integrity check failed; condition: input or artifact integrity check."
            )
        prediction_table[f"probability_{method}"] = probabilities
        overall_records.append(
            {
                "method": method,
                "evaluation": "Development_outer_5fold_OOF",
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                **evaluate(y, probabilities),
            }
        )
    overall = pd.DataFrame(overall_records)
    fold_table = pd.DataFrame(fold_records)
    baseline_row = overall.loc[overall["method"].eq("Uncalibrated")].iloc[0]
    for metric in ["Brier_score", "log_loss"]:
        overall[f"{metric}_difference_vs_uncalibrated"] = (
            overall[metric] - baseline_row[metric]
        )
    prediction_table.to_csv(
        OUTPUT_DIR / "calibration_comparison_OOF_predictions.tsv",
        sep="\t",
        index=False,
    )
    overall.to_csv(
        OUTPUT_DIR / "calibration_comparison_overall_metrics.tsv",
        sep="\t",
        index=False,
    )
    fold_table.to_csv(
        OUTPUT_DIR / "calibration_comparison_metrics_by_fold.tsv",
        sep="\t",
        index=False,
    )
    calibration_records = []
    for method in METHODS:
        temporary = pd.DataFrame({"label": y, "probability": oof[method]})
        temporary["bin"] = pd.cut(
            temporary["probability"],
            bins=PROBABILITY_BINS,
            include_lowest=True,
        )
        for interval, subset in temporary.groupby("bin", observed=True):
            calibration_records.append(
                {
                    "method": method,
                    "probability_bin": str(interval),
                    "n_patients": len(subset),
                    "n_positive": int(subset["label"].sum()),
                    "mean_predicted_probability": subset["probability"].mean(),
                    "observed_positive_fraction": subset["label"].mean(),
                }
            )
    calibration_table = pd.DataFrame(calibration_records)
    calibration_table.to_csv(
        OUTPUT_DIR / "calibration_bins_all_methods.tsv", sep="\t", index=False
    )
    cancer_records = []
    for cancer, indices in development.groupby(
        "acronym", observed=True
    ).groups.items():
        indices = np.asarray(list(indices), dtype=int)
        labels = y[indices]
        n_positive = int(labels.sum())
        n_negative = int((labels == 0).sum())
        both_classes = np.unique(labels).size == 2
        for method in METHODS:
            probability = oof[method][indices]
            cancer_records.append(
                {
                    "acronym": cancer,
                    "method": method,
                    "n_patients": len(indices),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "AUROC": (
                        roc_auc_score(labels, probability)
                        if both_classes
                        else np.nan
                    ),
                    "AUPRC_AP": (
                        average_precision_score(labels, probability)
                        if both_classes
                        else np.nan
                    ),
                    "Brier_score": brier_score_loss(labels, probability),
                    "mean_predicted_probability": probability.mean(),
                    "observed_positive_fraction": labels.mean(),
                }
            )
    pd.DataFrame(cancer_records).to_csv(
        OUTPUT_DIR / "calibration_metrics_by_cancer.tsv", sep="\t", index=False
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


### Model 2: 44-gene Elastic Net


#### Final 44-member TPM Elastic Net training and locking


In [ ]:
@_nb9_registry.register("s37_0")
def _nb9_stage_s37_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import importlib.util
    import json
    import shutil
    import warnings
    import joblib
    import numpy as np
    import pandas as pd
    import torch
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        roc_curve,
        precision_recall_curve,
    )

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL"
        )
    )
    BRANCH = BASE / "08_TPM_retraining"
    INPUT = BRANCH / "01_expression_inputs_44"
    DL_DIR = BRANCH / "02_DL_training"
    EN_DIR = BRANCH / "03_same_input_ElasticNet"
    LOCK = BRANCH / "04_locked_models_and_thresholds"
    EXTERNAL_OUT = BRANCH / "05_CPTAC_external_validation"
    CPTAC_X_DIR = BRANCH / "CPTAC_inputs_44"
    CPTAC_META_DIR = BASE / "06_CPTAC_RNA_pairing_v2"
    for directory in [EN_DIR, LOCK, EXTERNAL_OUT]:
        directory.mkdir(parents=True, exist_ok=True)
    FIGURES = EXTERNAL_OUT / "figures"
    FIGURES.mkdir(exist_ok=True)
    X_FILE = INPUT / "X_development_log2TPM_plus_1.tsv"
    META_FILE = INPUT / "development_metadata.tsv"
    SPEC_FILE = INPUT / "TPM_44_feature_specification.json"
    DL_OOF_FILE = DL_DIR / "development_MTAP_OOF_predictions.tsv"
    DL_CHECKPOINTS = {
        seed: DL_DIR / "models" / f"development_candidate_seed{seed}.pt"
        for seed in [41, 42, 43]
    }
    CANCER_MAP = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    DL_NAME = "TPM44_DL_3seed"
    EN_NAME = "TPM44_ElasticNet"
    MODEL_NAMES = [DL_NAME, EN_NAME]
    RANDOM_SEED = 20261017
    N_JOBS = 4
    PARAM_GRID = {
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.1, 0.5, 0.9],
    }
    THRESHOLD_RULES = {
        "positive_PPV_target": 0.9,
        "positive_Wilson_low_min": 0.85,
        "positive_min_n": 30,
        "negative_NPV_target": 0.98,
        "negative_Wilson_low_min": 0.97,
        "negative_min_n": 100,
    }
    N_BOOTSTRAPS = 1000
    COLORS = {DL_NAME: "#33539E", EN_NAME: "#A5678E"}
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    required_files = [
        X_FILE,
        META_FILE,
        SPEC_FILE,
        DL_OOF_FILE,
        *DL_CHECKPOINTS.values(),
    ]
    for path in required_files:
        if not path.is_file():
            raise FileNotFoundError(path)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def save_json(value, path):
        with Path(path).open("w", encoding="utf-8") as handle:
            json.dump(
                value, handle, ensure_ascii=False, indent=2, allow_nan=False
            )

    def wilson_interval(successes, n):
        if n == 0:
            return (np.nan, np.nan)
        z = 1.959963984540054
        p = successes / n
        denominator = 1 + z * z / n
        center = (p + z * z / (2 * n)) / denominator
        half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
        return (center - half, center + half)

    def performance(y, probability):
        y = np.asarray(y, dtype=int)
        p = np.asarray(probability, dtype=float)
        if not np.isfinite(p).all():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        if ((p < 0) | (p > 1)).any():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        positive = y == 1
        negative = y == 0
        predicted = p >= 0.5
        tp = int((positive & predicted).sum())
        fp = int((negative & predicted).sum())
        fn = int((positive & ~predicted).sum())
        tn = int((negative & ~predicted).sum())
        has_both = len(np.unique(y)) == 2
        clipped = np.clip(p, 1e-07, 1 - 1e-07)
        return {
            "n_patients": len(y),
            "n_positive": int(positive.sum()),
            "n_negative": int(negative.sum()),
            "positive_fraction": float(positive.mean()),
            "AUROC": roc_auc_score(y, p) if has_both else np.nan,
            "AP": average_precision_score(y, p) if has_both else np.nan,
            "Brier_score": brier_score_loss(y, p),
            "log_loss": log_loss(y, clipped, labels=[0, 1]),
            "precision_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "sensitivity_at_05": tp / (tp + fn) if tp + fn else np.nan,
            "specificity_at_05": tn / (tn + fp) if tn + fp else np.nan,
            "TP_at_05": tp,
            "FP_at_05": fp,
            "FN_at_05": fn,
            "TN_at_05": tn,
        }

    def classify_groups(probability, threshold):
        p = np.asarray(probability, dtype=float)
        groups = np.full(len(p), "Uncertain", dtype=object)
        low = threshold["low_threshold"]
        high = threshold["high_threshold"]
        if low is not None:
            groups[p <= low] = "Predicted_non_deep"
        if high is not None:
            groups[p >= high] = "Predicted_deep_deletion"
        return groups

    def group_summary(y, probability, threshold, model, cohort):
        y = np.asarray(y, dtype=int)
        groups = classify_groups(probability, threshold)
        rows = []
        n_positive = int(y.sum())
        for group in [
            "Predicted_deep_deletion",
            "Uncertain",
            "Predicted_non_deep",
        ]:
            selected = groups == group
            n = int(selected.sum())
            n_pos = int(y[selected].sum())
            if group == "Predicted_deep_deletion":
                value_name = "PPV"
                successes = n_pos
            elif group == "Predicted_non_deep":
                value_name = "NPV"
                successes = n - n_pos
            else:
                value_name = "DNA_positive_fraction"
                successes = n_pos
            low_ci, high_ci = wilson_interval(successes, n)
            rows.append(
                {
                    "cohort": cohort,
                    "model": model,
                    "group": group,
                    "n_patients": n,
                    "coverage_fraction": n / len(y),
                    "n_DNA_positive": n_pos,
                    "fraction_all_DNA_positives": (
                        n_pos / n_positive if n_positive else np.nan
                    ),
                    "value_name": value_name,
                    "value": successes / n if n else np.nan,
                    "Wilson_CI95_low": low_ci,
                    "Wilson_CI95_high": high_ci,
                }
            )
        return rows

    with SPEC_FILE.open("r", encoding="utf-8") as handle:
        input_spec = json.load(handle)
    features = input_spec["expression_feature_order"]
    if len(features) != 44 or len(set(features)) != 44:
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    X = pd.read_csv(X_FILE, sep="\t", index_col=0)
    X.index = X.index.astype(str)
    if X.index.duplicated().any():
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    X = X.loc[:, features].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    meta = pd.read_csv(
        META_FILE, sep="\t", dtype={"patient_id": str, "acronym": str}
    )
    if meta["patient_id"].duplicated().any():
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.set_index("patient_id")
    if set(meta.index) != set(X.index):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.loc[X.index]
    y = pd.to_numeric(meta["label_MTAP_deep"], errors="raise").to_numpy(
        dtype=int
    )
    if not np.isin(y, [0, 1]).all():
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    dl_oof = pd.read_csv(DL_OOF_FILE, sep="\t", dtype={"patient_id": str})
    if dl_oof["patient_id"].duplicated().any():
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    dl_oof = dl_oof.set_index("patient_id")
    if set(dl_oof.index) != set(X.index):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    dl_oof = dl_oof.loc[X.index]
    if not np.array_equal(dl_oof["label_MTAP_deep"].to_numpy(dtype=int), y):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        dl_oof["acronym"].astype(str).to_numpy(),
        meta["acronym"].astype(str).to_numpy(),
    ):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    folds = dl_oof["outer_fold"].to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    p_dl_oof = dl_oof["prob_MTAP_deep"].to_numpy(dtype=float)
    seed_columns = [f"prob_seed_{seed}" for seed in [41, 42, 43]]
    if not np.allclose(
        dl_oof[seed_columns].mean(axis=1), p_dl_oof, atol=1e-06
    ):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    checkpoints = {
        seed: torch.load(path, map_location="cpu", weights_only=False)
        for seed, path in DL_CHECKPOINTS.items()
    }
    reference_checkpoint = checkpoints[41]
    for seed, checkpoint in checkpoints.items():
        if checkpoint["feature_names"] != features:
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        if checkpoint["expression_unit"] != "log2(TPM+1)":
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        for key in [
            "region_indices",
            "core_indices",
            "cancer_to_index",
            "architecture",
            "target_Entrez_order",
            "MTAP_INDEX",
        ]:
            if checkpoint[key] != reference_checkpoint[key]:
                raise ValueError(
                    "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
                )
        for key in ["mean", "std"]:
            if not np.allclose(
                checkpoint["preprocessing"][key],
                reference_checkpoint["preprocessing"][key],
            ):
                raise ValueError(
                    "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
                )
    raw32 = X.to_numpy(dtype=np.float32)
    expected_mean = raw32.mean(axis=0)
    expected_std = raw32.std(axis=0, ddof=0)
    expected_std = np.where(expected_std < 1e-08, 1.0, expected_std)
    if not np.allclose(
        reference_checkpoint["preprocessing"]["mean"],
        expected_mean,
        rtol=1e-05,
        atol=1e-05,
    ):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.allclose(
        reference_checkpoint["preprocessing"]["std"],
        expected_std,
        rtol=1e-05,
        atol=1e-05,
    ):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    cancer_categories = sorted(reference_checkpoint["cancer_to_index"])
    if not set(meta["acronym"]).issubset(cancer_categories):
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    X_model = X.copy()
    X_model["acronym"] = meta["acronym"].astype(str)

    def make_en_pipeline():
        expression_pipeline = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median", keep_empty_features=True),
                ),
                ("scaler", StandardScaler()),
            ]
        )
        preprocessing = ColumnTransformer(
            transformers=[
                ("expression", expression_pipeline, features),
                (
                    "cancer",
                    OneHotEncoder(
                        categories=[cancer_categories],
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                    ["acronym"],
                ),
            ],
            remainder="drop",
            sparse_threshold=0,
        )
        return Pipeline(
            [
                ("preprocessing", preprocessing),
                (
                    "model",
                    LogisticRegression(
                        penalty="elasticnet",
                        solver="saga",
                        max_iter=10000,
                        tol=0.0001,
                        class_weight=None,
                        random_state=41,
                    ),
                ),
            ]
        )

    en_model_file = EN_DIR / "development_candidate.joblib"
    en_oof_file = EN_DIR / "development_OOF_predictions.tsv"
    en_manifest_file = EN_DIR / "training_manifest.json"
    input_hashes = {
        str(path): sha256_file(path)
        for path in [
            X_FILE,
            META_FILE,
            SPEC_FILE,
            DL_OOF_FILE,
            *DL_CHECKPOINTS.values(),
        ]
    }
    en_cache_complete = all(
        (
            path.exists()
            for path in [en_model_file, en_oof_file, en_manifest_file]
        )
    )
    if en_cache_complete:
        with en_manifest_file.open("r", encoding="utf-8") as handle:
            en_training_manifest = json.load(handle)
        if en_training_manifest["input_hashes"] != input_hashes:
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        en_predictions = (
            pd.read_csv(en_oof_file, sep="\t", dtype={"patient_id": str})
            .set_index("patient_id")
            .loc[X.index]
        )
        if not np.array_equal(
            en_predictions["label_MTAP_deep"].to_numpy(dtype=int), y
        ):
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        if not np.array_equal(
            en_predictions["outer_fold"].to_numpy(dtype=int), folds
        ):
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        p_en_oof = en_predictions["prob_ElasticNet"].to_numpy()
        en_bundle = joblib.load(en_model_file)
    else:
        p_en_oof = np.full(len(X), np.nan)
        fold_records = []
        for fold in [1, 2, 3, 4, 5]:
            train_indices = np.flatnonzero(folds != fold)
            valid_indices = np.flatnonzero(folds == fold)
            inner_cv = StratifiedKFold(
                n_splits=3, shuffle=True, random_state=41 + fold
            )
            search = GridSearchCV(
                estimator=make_en_pipeline(),
                param_grid=PARAM_GRID,
                scoring="average_precision",
                cv=inner_cv,
                n_jobs=N_JOBS,
                refit=True,
                error_score="raise",
            )
            search.fit(X_model.iloc[train_indices], y[train_indices])
            best_model = search.best_estimator_
            iterations = int(np.max(best_model.named_steps["model"].n_iter_))
            if iterations >= 10000:
                raise RuntimeError(
                    "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
                )
            p_en_oof[valid_indices] = best_model.predict_proba(
                X_model.iloc[valid_indices]
            )[:, 1]
            en_metrics = performance(y[valid_indices], p_en_oof[valid_indices])
            dl_metrics = performance(y[valid_indices], p_dl_oof[valid_indices])
            fold_records.append(
                {
                    "outer_fold": fold,
                    "n_train": len(train_indices),
                    "n_valid": len(valid_indices),
                    "best_C": float(search.best_params_["model__C"]),
                    "best_l1_ratio": float(
                        search.best_params_["model__l1_ratio"]
                    ),
                    "inner_best_AP": float(search.best_score_),
                    "ElasticNet_AUROC": en_metrics["AUROC"],
                    "ElasticNet_AP": en_metrics["AP"],
                    "DL_AUROC": dl_metrics["AUROC"],
                    "DL_AP": dl_metrics["AP"],
                    "n_iterations": iterations,
                }
            )
        if not np.isfinite(p_en_oof).all():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        final_search = GridSearchCV(
            make_en_pipeline(),
            PARAM_GRID,
            scoring="average_precision",
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=41),
            n_jobs=N_JOBS,
            refit=True,
            error_score="raise",
        )
        final_search.fit(X_model, y)
        final_iterations = int(
            np.max(final_search.best_estimator_.named_steps["model"].n_iter_)
        )
        if final_iterations >= 10000:
            raise RuntimeError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        en_bundle = {
            "pipeline": final_search.best_estimator_,
            "feature_names": features,
            "input_columns": features + ["acronym"],
            "expression_unit": "log2(TPM+1)",
            "best_params": final_search.best_params_,
        }
        joblib.dump(en_bundle, en_model_file)
        pd.DataFrame(fold_records).to_csv(
            EN_DIR / "outer_fold_results.tsv", sep="\t", index=False
        )
        pd.DataFrame(
            {
                "patient_id": X.index,
                "label_MTAP_deep": y,
                "outer_fold": folds,
                "prob_ElasticNet": p_en_oof,
            }
        ).to_csv(en_oof_file, sep="\t", index=False)
        en_training_manifest = {
            "input_hashes": input_hashes,
            "param_grid": PARAM_GRID,
            "best_params": final_search.best_params_,
            "outer_folds": "Inherited DL patient folds",
            "expression_unit": "log2(TPM+1)",
            "n_features": len(features),
            "n_patients": len(X),
            "CPTAC_used_for_training": False,
        }
        save_json(en_training_manifest, en_manifest_file)
    development_oof = pd.DataFrame(
        {
            "patient_id": X.index,
            "acronym": meta["acronym"].to_numpy(),
            "label_MTAP_deep": y,
            "outer_fold": folds,
            "prob_DL": p_dl_oof,
            "prob_ElasticNet": p_en_oof,
        }
    )
    development_comparison = pd.DataFrame(
        [
            {"model": DL_NAME, **performance(y, p_dl_oof)},
            {"model": EN_NAME, **performance(y, p_en_oof)},
        ]
    )
    development_comparison.to_csv(
        EN_DIR / "development_model_comparison.tsv", sep="\t", index=False
    )

    def select_thresholds(y, probability):
        base = pd.DataFrame(
            {
                "probability": np.asarray(probability, dtype=float),
                "positive": np.asarray(y, dtype=int),
            }
        )
        distinct = base.groupby("probability").agg(
            n=("positive", "size"), positives=("positive", "sum")
        )
        high_candidates = distinct.sort_index(ascending=False).copy()
        high_candidates["selected_n"] = high_candidates["n"].cumsum()
        high_candidates["selected_positive"] = high_candidates[
            "positives"
        ].cumsum()
        high_candidates["PPV"] = (
            high_candidates["selected_positive"]
            / high_candidates["selected_n"]
        )
        high_candidates["Wilson_low"] = [
            wilson_interval(int(k), int(n))[0]
            for k, n in zip(
                high_candidates["selected_positive"],
                high_candidates["selected_n"],
            )
        ]
        high_candidates["eligible"] = (
            high_candidates["selected_n"].ge(THRESHOLD_RULES["positive_min_n"])
            & high_candidates["PPV"].ge(THRESHOLD_RULES["positive_PPV_target"])
            & high_candidates["Wilson_low"].ge(
                THRESHOLD_RULES["positive_Wilson_low_min"]
            )
        )
        eligible_high = high_candidates.loc[high_candidates["eligible"]]
        high = None
        if not eligible_high.empty:
            high = float(eligible_high["selected_n"].idxmax())
        low_candidates = distinct.sort_index().copy()
        low_candidates["selected_n"] = low_candidates["n"].cumsum()
        low_candidates["selected_positive"] = low_candidates[
            "positives"
        ].cumsum()
        low_candidates["selected_negative"] = (
            low_candidates["selected_n"] - low_candidates["selected_positive"]
        )
        low_candidates["NPV"] = (
            low_candidates["selected_negative"] / low_candidates["selected_n"]
        )
        low_candidates["Wilson_low"] = [
            wilson_interval(int(k), int(n))[0]
            for k, n in zip(
                low_candidates["selected_negative"],
                low_candidates["selected_n"],
            )
        ]
        low_candidates["eligible"] = (
            low_candidates["selected_n"].ge(THRESHOLD_RULES["negative_min_n"])
            & low_candidates["NPV"].ge(THRESHOLD_RULES["negative_NPV_target"])
            & low_candidates["Wilson_low"].ge(
                THRESHOLD_RULES["negative_Wilson_low_min"]
            )
        )
        if high is not None:
            low_candidates["eligible"] &= (
                low_candidates.index.to_numpy() < high
            )
        eligible_low = low_candidates.loc[low_candidates["eligible"]]
        low = None
        if not eligible_low.empty:
            low = float(eligible_low["selected_n"].idxmax())
        result = {
            "low_threshold": low,
            "high_threshold": high,
            "positive_rule_met": high is not None,
            "negative_rule_met": low is not None,
        }
        return (result, high_candidates, low_candidates)

    NETWORK_SOURCE = '\nimport torch\nfrom torch import nn\n\nclass RegionalMultiTaskNet(nn.Module):\n    def __init__(self, checkpoint):\n        super().__init__()\n\n        self.region_indices = checkpoint["region_indices"]\n        self.core_indices = checkpoint["core_indices"]\n        architecture = checkpoint["architecture"]\n\n        region_hidden = int(architecture["region_hidden"])\n        shared_hidden = int(architecture["shared_hidden"])\n        embedding_dim = int(architecture["cancer_embedding"])\n        dropout = float(architecture["dropout"])\n\n        self.n_targets = len(checkpoint["target_Entrez_order"])\n\n        self.region_encoders = nn.ModuleList([\n            nn.Sequential(\n                nn.Linear(len(indices), region_hidden),\n                nn.ReLU(),\n                nn.Dropout(dropout),\n            )\n            for indices in self.region_indices\n        ])\n\n        self.cancer_embedding = nn.Embedding(\n            max(checkpoint["cancer_to_index"].values()) + 1,\n            embedding_dim,\n        )\n\n        shared_input = (\n            len(self.region_indices) * region_hidden\n            + len(self.core_indices)\n            + embedding_dim\n        )\n\n        self.shared = nn.Sequential(\n            nn.Linear(shared_input, shared_hidden),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n            nn.Linear(shared_hidden, self.n_targets * 3),\n        )\n\n    def forward(self, expression, cancer):\n        regional = [\n            encoder(expression[:, indices])\n            for encoder, indices in zip(\n                self.region_encoders, self.region_indices\n            )\n        ]\n\n        combined = torch.cat(\n            regional + [\n                expression[:, self.core_indices],\n                self.cancer_embedding(cancer),\n            ],\n            dim=1,\n        )\n\n        return self.shared(combined).reshape(\n            -1, self.n_targets, 3\n        )\n'
    LOCK_MANIFEST = LOCK / "locked_manifest.json"
    if LOCK_MANIFEST.exists():
        with LOCK_MANIFEST.open("r", encoding="utf-8") as handle:
            locked_manifest = json.load(handle)
        current_sources = {
            **input_hashes,
            str(en_model_file): sha256_file(en_model_file),
            str(en_oof_file): sha256_file(en_oof_file),
        }
        if locked_manifest["source_hashes"] != current_sources:
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        if locked_manifest["threshold_rules"] != THRESHOLD_RULES:
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        for relative_path, expected_hash in locked_manifest[
            "locked_file_hashes"
        ].items():
            path = LOCK / relative_path
            if not path.is_file() or sha256_file(path) != expected_hash:
                raise ValueError(
                    "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
                )
        with (LOCK / "thresholds/locked_thresholds.json").open(
            "r", encoding="utf-8"
        ) as handle:
            locked_thresholds = json.load(handle)
    else:
        for name in ["models", "records", "thresholds"]:
            (LOCK / name).mkdir(exist_ok=True)
        locked_thresholds = {}
        dev_group_rows = []
        for name, probability in [(DL_NAME, p_dl_oof), (EN_NAME, p_en_oof)]:
            threshold, high_table, low_table = select_thresholds(
                y, probability
            )
            locked_thresholds[name] = threshold
            high_table.to_csv(
                LOCK / "thresholds" / f"{name}_high_threshold_candidates.tsv",
                sep="\t",
                index_label="threshold",
            )
            low_table.to_csv(
                LOCK / "thresholds" / f"{name}_low_threshold_candidates.tsv",
                sep="\t",
                index_label="threshold",
            )
            dev_group_rows.extend(
                group_summary(
                    y, probability, threshold, name, "Development_OOF"
                )
            )
        save_json(
            locked_thresholds, LOCK / "thresholds/locked_thresholds.json"
        )
        pd.DataFrame(dev_group_rows).to_csv(
            LOCK / "thresholds/development_group_summary.tsv",
            sep="\t",
            index=False,
        )
        for seed, source in DL_CHECKPOINTS.items():
            shutil.copy2(source, LOCK / "models" / f"DL_seed{seed}.pt")
        shutil.copy2(en_model_file, LOCK / "models/ElasticNet_locked.joblib")
        (LOCK / "region_multitask_TPM44_network.py").write_text(
            NETWORK_SOURCE, encoding="utf-8"
        )
        development_oof.to_csv(
            LOCK / "records/development_OOF_predictions.tsv",
            sep="\t",
            index=False,
        )
        development_comparison.to_csv(
            LOCK / "records/development_model_comparison.tsv",
            sep="\t",
            index=False,
        )
        shutil.copy2(
            SPEC_FILE, LOCK / "records/TPM_44_feature_specification.json"
        )
        shutil.copy2(
            en_manifest_file,
            LOCK / "records/ElasticNet_training_manifest.json",
        )
        shutil.copy2(
            DL_DIR / "training_manifest.json",
            LOCK / "records/DL_training_manifest.json",
        )
        locked_files = sorted(
            (
                path
                for path in LOCK.rglob("*")
                if path.is_file() and path != LOCK_MANIFEST
            )
        )
        locked_manifest = {
            "version": "TPM44_DL_and_ElasticNet",
            "expression_unit": "log2(TPM+1)",
            "feature_order": features,
            "DL_ensemble": "Arithmetic mean of seeds 41/42/43",
            "threshold_rules": THRESHOLD_RULES,
            "source_hashes": {
                **input_hashes,
                str(en_model_file): sha256_file(en_model_file),
                str(en_oof_file): sha256_file(en_oof_file),
            },
            "locked_file_hashes": {
                str(path.relative_to(LOCK)): sha256_file(path)
                for path in locked_files
            },
            "CPTAC_labels_used_for_parameter_selection": False,
            "threshold_interval_scope": "Development OOF post-selection descriptive intervals",
        }
        save_json(locked_manifest, LOCK_MANIFEST)
    threshold_table = pd.DataFrame(
        [{"model": name, **locked_thresholds[name]} for name in MODEL_NAMES]
    )


#### Three-gene L2 and Elastic Net comparison


In [ ]:
@_nb9_registry.register("s12_0")
def _nb9_stage_s12_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import importlib.util
    import json
    import warnings
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import sklearn
    from sklearn.base import clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import SGDClassifier
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    PROJECT = Path(str(PROJECT_ROOT))
    AUTHOR_DIR = PROJECT / "code_external/greenelab_pancancer"
    AUTHOR_UTIL = AUTHOR_DIR / "scripts/util/tcga_util.py"
    PREDICTION_DIR = PROJECT / "data_processed/MTAP_RNA_prediction"
    EXPRESSION_FILE = (
        PREDICTION_DIR
        / "03_baseline_models/cohort_with_three_gene_expression.tsv"
    )
    PREVIOUS_OOF_FILE = (
        PREDICTION_DIR / "03_baseline_models/development_OOF_predictions.tsv"
    )
    OUTPUT_DIR = PREDICTION_DIR / "07_greenelab_SGD_adaptation"
    FIGURE_DIR = OUTPUT_DIR / "figures"
    MODEL_DIR = OUTPUT_DIR / "models"
    CV_DIR = OUTPUT_DIR / "inner_CV"
    for directory in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR, CV_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    SEED = 20261016
    FEATURES = ["MTAP_log2_RNA", "CDKN2A_log2_RNA", "CDKN2B_log2_RNA"]
    LABEL = "label_MTAP_deep"
    PRIMARY_SELECTION_METRIC = "AP"
    PARAM_GRID = {
        "classify__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0],
        "classify__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9],
    }
    PALETTE = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]
    for file in [AUTHOR_UTIL, EXPRESSION_FILE, PREVIOUS_OOF_FILE]:
        if not file.is_file():
            raise FileNotFoundError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
    spec = importlib.util.spec_from_file_location(
        "greenelab_tcga_util", AUTHOR_UTIL
    )
    author_util = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(author_util)
    get_threshold_metrics = author_util.get_threshold_metrics
    author_hash = hashlib.sha256(AUTHOR_UTIL.read_bytes()).hexdigest()
    expression = pd.read_csv(
        EXPRESSION_FILE, sep="\t", dtype={"patient_id": str}
    )
    previous = pd.read_csv(
        PREVIOUS_OOF_FILE, sep="\t", dtype={"patient_id": str}
    )
    required_expression = {
        "patient_id",
        "acronym",
        "dataset_split",
        LABEL,
        *FEATURES,
    }
    required_previous = {
        "patient_id",
        LABEL,
        "CV_fold",
        "probability_MTAP_only",
        "probability_Three_gene",
    }
    for name, frame, required in [
        ("Expression cache", expression, required_expression),
        ("Existing OOF file", previous, required_previous),
    ]:
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
        if frame["patient_id"].duplicated().any():
            raise ValueError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
    development = expression.loc[
        expression["dataset_split"].eq("development")
    ].copy()
    del expression
    if set(development["patient_id"]) != set(previous["patient_id"]):
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    development = development.set_index("patient_id")
    previous = previous.set_index("patient_id")
    development = development.sort_index()
    previous = previous.reindex(development.index)
    for column in [LABEL, *FEATURES]:
        development[column] = pd.to_numeric(
            development[column], errors="raise"
        )
    if development[LABEL].isna().any():
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    if not development[LABEL].isin([0, 1]).all():
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        development[LABEL].to_numpy(),
        pd.to_numeric(previous[LABEL], errors="raise").to_numpy(),
    ):
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    X = development[FEATURES].astype(float)
    y = development[LABEL].astype(int)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    folds_numeric = pd.to_numeric(previous["CV_fold"], errors="raise")
    if folds_numeric.isna().any() or (folds_numeric % 1 != 0).any():
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    folds = folds_numeric.astype(int)
    if sorted(folds.unique().tolist()) != [1, 2, 3, 4, 5]:
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    if len(y) != 7431 or int(y.sum()) != 558:
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    for column in ["probability_MTAP_only", "probability_Three_gene"]:
        previous[column] = pd.to_numeric(previous[column], errors="raise")
        if (
            not np.isfinite(previous[column]).all()
            or not previous[column].between(0, 1).all()
        ):
            raise ValueError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
    pipeline = Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "classify",
                SGDClassifier(
                    loss="log_loss",
                    penalty="elasticnet",
                    class_weight="balanced",
                    random_state=SEED,
                    max_iter=20000,
                    tol=1e-05,
                    n_iter_no_change=10,
                    shuffle=True,
                ),
            ),
        ]
    )
    SCORING = {"AP": "average_precision", "AUROC": "roc_auc"}
    convergence_records = []

    def fit_search(X_train, y_train, cv, stage):
        search = GridSearchCV(
            estimator=clone(pipeline),
            param_grid=PARAM_GRID,
            scoring=SCORING,
            refit=PRIMARY_SELECTION_METRIC,
            cv=cv,
            n_jobs=1,
            return_train_score=False,
            error_score="raise",
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            search.fit(X_train, y_train)
        convergence_messages = [
            str(item.message)
            for item in caught
            if issubclass(item.category, ConvergenceWarning)
        ]
        for message in convergence_messages:
            convergence_records.append({"stage": stage, "warning": message})
        if convergence_messages:
            pd.DataFrame(convergence_records).to_csv(
                OUTPUT_DIR / "convergence_warnings.tsv", sep="\t", index=False
            )
            raise RuntimeError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
        return search

    def calculate_metrics(labels, scores):
        labels = np.asarray(labels, dtype=int)
        scores = np.asarray(scores, dtype=float)
        if not np.isfinite(scores).all():
            raise ValueError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
        if np.unique(labels).size < 2:
            return None
        return get_threshold_metrics(labels, scores)

    def coefficient_table(fitted_pipeline, stage):
        classifier = fitted_pipeline.named_steps["classify"]
        return pd.DataFrame(
            {
                "stage": stage,
                "feature": FEATURES,
                "coefficient_per_standardized_feature": classifier.coef_[0],
                "intercept": float(classifier.intercept_[0]),
                "alpha": classifier.alpha,
                "l1_ratio": classifier.l1_ratio,
                "n_iter": int(classifier.n_iter_),
            }
        )

    oof_scores = np.full(len(y), np.nan)
    fold_results = []
    coefficient_results = []
    outer_splits = []
    for fold in [1, 2, 3, 4, 5]:
        train_index = np.flatnonzero(folds.to_numpy() != fold)
        valid_index = np.flatnonzero(folds.to_numpy() == fold)
        outer_splits.append((train_index, valid_index))
        if (
            y.iloc[train_index].nunique() != 2
            or y.iloc[valid_index].nunique() != 2
        ):
            raise ValueError(
                "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
            )
        inner_cv = StratifiedKFold(
            n_splits=5, shuffle=True, random_state=SEED + fold
        )
        search = fit_search(
            X.iloc[train_index],
            y.iloc[train_index],
            cv=inner_cv,
            stage=f"outer_fold_{fold}",
        )
        pd.DataFrame(search.cv_results_).to_csv(
            CV_DIR / f"outer_fold_{fold}_grid_results.tsv",
            sep="\t",
            index=False,
        )
        best_pipeline = search.best_estimator_
        valid_scores = best_pipeline.decision_function(X.iloc[valid_index])
        oof_scores[valid_index] = valid_scores
        metrics = calculate_metrics(y.iloc[valid_index], valid_scores)
        fold_results.append(
            {
                "outer_fold": fold,
                "n_train": len(train_index),
                "n_valid": len(valid_index),
                "n_valid_positive": int(y.iloc[valid_index].sum()),
                "best_alpha": search.best_params_["classify__alpha"],
                "best_l1_ratio": search.best_params_["classify__l1_ratio"],
                "inner_best_AP": search.best_score_,
                "outer_AUROC": metrics["auroc"],
                "outer_AP": metrics["aupr"],
                "n_iter": int(best_pipeline.named_steps["classify"].n_iter_),
            }
        )
        coefficient_results.append(
            coefficient_table(best_pipeline, stage=f"outer_fold_{fold}")
        )
        joblib.dump(
            {
                "pipeline": best_pipeline,
                "features": FEATURES,
                "label": LABEL,
                "training_scope": f"development_outer_fold_{fold}",
                "output_type": "uncalibrated_decision_score",
            },
            MODEL_DIR / f"outer_fold_{fold}.joblib",
        )
    if not np.isfinite(oof_scores).all():
        raise ValueError(
            "Three-gene L2 and Elastic Net comparison: integrity check failed; condition: input or artifact integrity check."
        )
    fold_results = pd.DataFrame(fold_results)
    fold_results.to_csv(
        OUTPUT_DIR / "nested_CV_outer_fold_results.tsv", sep="\t", index=False
    )
    pd.concat(coefficient_results, ignore_index=True).to_csv(
        OUTPUT_DIR / "outer_fold_coefficients.tsv", sep="\t", index=False
    )
    oof_output = development[["acronym", LABEL]].copy()
    oof_output["CV_fold"] = folds
    oof_output["score_SGD_ElasticNet"] = oof_scores
    oof_output["probability_MTAP_only"] = previous["probability_MTAP_only"]
    oof_output["probability_Three_gene_L2"] = previous[
        "probability_Three_gene"
    ]
    oof_output.to_csv(
        OUTPUT_DIR / "development_nested_OOF_predictions.tsv",
        sep="\t",
        index_label="patient_id",
    )
    model_scores = {
        "MTAP_only_L2": previous["probability_MTAP_only"].to_numpy(),
        "Three_gene_L2": previous["probability_Three_gene"].to_numpy(),
        "Three_gene_SGD_ElasticNet": oof_scores,
    }
    metric_rows = []
    curve_results = {}
    for name, scores in model_scores.items():
        metrics = calculate_metrics(y, scores)
        curve_results[name] = metrics
        metric_rows.append(
            {
                "model": name,
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                "AUROC": metrics["auroc"],
                "AUPRC_AP": metrics["aupr"],
                "evaluation": (
                    "Nested_outer_5fold_OOF"
                    if name == "Three_gene_SGD_ElasticNet"
                    else "Existing_fixed_5fold_OOF"
                ),
            }
        )
        metrics["roc_df"].to_csv(
            OUTPUT_DIR / f"{name}_ROC_coordinates.tsv", sep="\t", index=False
        )
        metrics["pr_df"].to_csv(
            OUTPUT_DIR / f"{name}_PR_coordinates.tsv", sep="\t", index=False
        )
    performance = pd.DataFrame(metric_rows)
    performance.to_csv(
        OUTPUT_DIR / "development_model_comparison.tsv", sep="\t", index=False
    )
    cancer_rows = []
    for acronym, subset in development.groupby("acronym"):
        positions = development.index.get_indexer(subset.index)
        labels = y.iloc[positions].to_numpy()
        n_positive = int(labels.sum())
        n_negative = int((labels == 0).sum())
        for name, scores in model_scores.items():
            metrics = calculate_metrics(labels, np.asarray(scores)[positions])
            cancer_rows.append(
                {
                    "acronym": acronym,
                    "model": name,
                    "n_patients": len(labels),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "AUROC": (
                        metrics["auroc"] if metrics is not None else np.nan
                    ),
                    "AUPRC_AP": (
                        metrics["aupr"] if metrics is not None else np.nan
                    ),
                    "support_status": (
                        "Single_class"
                        if metrics is None
                        else (
                            "Low_support"
                            if min(n_positive, n_negative) < 10
                            else "Descriptive"
                        )
                    ),
                }
            )
    cancer_results = pd.DataFrame(cancer_rows)
    cancer_results.to_csv(
        OUTPUT_DIR / "development_per_cancer_performance.tsv",
        sep="\t",
        index=False,
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.unicode_minus": False,
        }
    )


#### Regional Elastic Net search with nested evaluation


In [ ]:
@_nb9_registry.register("s14_0")
def _nb9_stage_s14_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import warnings
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import sklearn
    from sklearn.base import clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        roc_curve,
        precision_recall_curve,
    )
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    PROJECT = Path(str(PROJECT_ROOT))
    PRED_DIR = PROJECT / "data_processed/MTAP_RNA_prediction"
    PREVIOUS_DIR = PRED_DIR / "09_9p21_region_model"
    RAW_CACHE = PREVIOUS_DIR / "development_region_raw_expression.tsv"
    CANDIDATE_FILE = (
        PRED_DIR
        / "08_9p21_region_candidate_audit/9p21_3_candidate_gene_RNA_mapping.tsv"
    )
    COHORT_FILE = (
        PRED_DIR / "03_baseline_models/cohort_with_three_gene_expression.tsv"
    )
    PREVIOUS_OOF_FILE = (
        PRED_DIR / "03_baseline_models/development_OOF_predictions.tsv"
    )
    OUT = PREVIOUS_DIR / "signedlog_v2"
    FIG = OUT / "figures"
    MODELS = OUT / "models"
    CV_DIR = OUT / "inner_CV"
    for directory in [OUT, FIG, MODELS, CV_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    SEED = 20261016
    LABEL = "label_MTAP_deep"
    CORE_IDS = ["4507", "1029", "1030"]
    CORE_COLUMNS = {
        "4507": "MTAP_log2_RNA",
        "1029": "CDKN2A_log2_RNA",
        "1030": "CDKN2B_log2_RNA",
    }
    REGION_GRID = {
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.1, 0.5, 0.9],
    }
    PALETTE = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]
    for path in [RAW_CACHE, CANDIDATE_FILE, COHORT_FILE, PREVIOUS_OOF_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
            )
    cohort = pd.read_csv(COHORT_FILE, sep="\t", dtype={"patient_id": str})
    previous = pd.read_csv(
        PREVIOUS_OOF_FILE, sep="\t", dtype={"patient_id": str}
    )
    required = {
        "patient_id",
        "acronym",
        "dataset_split",
        LABEL,
        *CORE_COLUMNS.values(),
    }
    if not required.issubset(cohort.columns):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    if not {"patient_id", LABEL, "CV_fold"}.issubset(previous.columns):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    if (
        cohort["patient_id"].duplicated().any()
        or previous["patient_id"].duplicated().any()
    ):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    dev = (
        cohort.loc[cohort["dataset_split"].eq("development")]
        .copy()
        .set_index("patient_id")
        .sort_index()
    )
    del cohort
    if set(dev.index) != set(previous["patient_id"]):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    previous = previous.set_index("patient_id").reindex(dev.index)
    label_values = pd.to_numeric(dev[LABEL], errors="raise")
    if not label_values.isin([0, 1]).all():
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    y = label_values.astype(int)
    if not np.array_equal(
        y.to_numpy(), pd.to_numeric(previous[LABEL], errors="raise").to_numpy()
    ):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    fold_values = pd.to_numeric(previous["CV_fold"], errors="raise")
    if fold_values.isna().any() or (fold_values % 1 != 0).any():
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    folds = fold_values.astype(int)
    if sorted(folds.unique()) != [1, 2, 3, 4, 5]:
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    if len(y) != 7431 or int(y.sum()) != 558:
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    audit = pd.read_csv(CANDIDATE_FILE, sep="\t", dtype={"Entrez_ID": str})
    candidate = audit.loc[
        audit["mapping_status"].eq("Unique_Entrez_match")
    ].copy()
    candidate["Entrez_ID"] = candidate["Entrez_ID"].str.strip()
    candidate = candidate.sort_values("Entrez_ID").reset_index(drop=True)
    if candidate["Entrez_ID"].duplicated().any():
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    feature_ids = candidate["Entrez_ID"].tolist()
    if not set(CORE_IDS).issubset(feature_ids):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    raw = pd.read_csv(
        RAW_CACHE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    raw.columns = raw.columns.astype(str)
    if raw.index.duplicated().any():
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    if set(raw.index) != set(dev.index) or set(raw.columns) != set(
        feature_ids
    ):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    raw = raw.reindex(index=dev.index, columns=feature_ids)
    raw = raw.apply(pd.to_numeric, errors="raise")
    if np.isinf(raw.to_numpy(dtype=float)).any():
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    coverage = candidate[["DNA_symbol", "Entrez_ID", "RNA_gene_ids"]].copy()
    for name, function in [
        ("n_missing", lambda x: x.isna().sum()),
        ("n_negative", lambda x: x.lt(0).sum()),
        ("n_le_minus_1", lambda x: x.le(-1).sum()),
        ("n_zero", lambda x: x.eq(0).sum()),
        ("minimum", lambda x: x.min()),
        ("median", lambda x: x.median()),
        ("maximum", lambda x: x.max()),
    ]:
        coverage[name] = [
            function(raw[entrez]) for entrez in coverage["Entrez_ID"]
        ]
    coverage.to_csv(
        OUT / "development_expression_scale_audit.tsv", sep="\t", index=False
    )
    negative_by_cancer = []
    for acronym, patient_rows in dev.groupby("acronym"):
        subset = raw.loc[patient_rows.index]
        for entrez in feature_ids:
            negative_by_cancer.append(
                {
                    "acronym": acronym,
                    "Entrez_ID": entrez,
                    "n_patients": len(subset),
                    "n_observed": int(subset[entrez].notna().sum()),
                    "n_negative": int(subset[entrez].lt(0).sum()),
                    "minimum": subset[entrez].min(),
                }
            )
    pd.DataFrame(negative_by_cancer).to_csv(
        OUT / "negative_values_by_cancer.tsv", sep="\t", index=False
    )
    negative_genes = coverage.loc[coverage["n_negative"].gt(0)]
    if negative_genes.empty:
        pass
    X_region = np.sign(raw) * np.log1p(np.abs(raw)) / np.log(2.0)
    if np.isinf(X_region.to_numpy(dtype=float)).any():
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(raw.isna().to_numpy(), X_region.isna().to_numpy()):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    X_region.to_csv(
        OUT / "development_region_signedlog_expression.tsv",
        sep="\t",
        index_label="patient_id",
    )
    candidate.to_csv(
        OUT / "fixed_region_candidate_genes.tsv", sep="\t", index=False
    )
    for entrez, old_column in CORE_COLUMNS.items():
        old = pd.to_numeric(dev[old_column], errors="raise").to_numpy(
            dtype=float
        )
        new = X_region[entrez].to_numpy(dtype=float)
        if not np.allclose(old, new, rtol=1e-07, atol=1e-07, equal_nan=True):
            raise ValueError(
                "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
            )
    X_core = X_region[CORE_IDS].copy()

    def make_pipeline(model):
        return Pipeline(
            [
                (
                    "impute",
                    SimpleImputer(strategy="median", keep_empty_features=True),
                ),
                ("variance", VarianceThreshold(threshold=0.0)),
                ("scale", StandardScaler()),
                ("model", model),
            ]
        )

    core_pipeline = make_pipeline(
        LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            C=1.0,
            class_weight=None,
            max_iter=3000,
            random_state=SEED,
        )
    )
    region_pipeline = make_pipeline(
        LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            class_weight=None,
            max_iter=50000,
            tol=0.0001,
            random_state=SEED,
        )
    )

    def checked_fit(estimator, X, labels, stage):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            estimator.fit(X, labels)
        messages = [
            str(item.message)
            for item in caught
            if issubclass(item.category, ConvergenceWarning)
        ]
        if messages:
            pd.DataFrame({"stage": stage, "warning": messages}).to_csv(
                OUT / "convergence_warnings.tsv", sep="\t", index=False
            )
            raise RuntimeError(
                "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
            )
        return estimator

    def make_search(cv):
        return GridSearchCV(
            estimator=clone(region_pipeline),
            param_grid=REGION_GRID,
            scoring={"AP": "average_precision", "AUROC": "roc_auc"},
            refit="AP",
            cv=cv,
            n_jobs=1,
            return_train_score=False,
            error_score="raise",
        )

    def get_coefficients(pipeline, features, stage):
        retained = pipeline.named_steps["variance"].get_support()
        values = np.zeros(len(features), dtype=float)
        values[retained] = pipeline.named_steps["model"].coef_[0]
        return pd.DataFrame(
            {
                "stage": stage,
                "Entrez_ID": features,
                "retained_after_variance_filter": retained,
                "standardized_coefficient": values,
                "nonzero_coefficient": np.abs(values) > 1e-08,
            }
        )

    oof_core = np.full(len(y), np.nan)
    oof_region = np.full(len(y), np.nan)
    fold_rows = []
    coefficient_rows = []
    outer_splits = []
    for fold in [1, 2, 3, 4, 5]:
        train_index = np.flatnonzero(folds.to_numpy() != fold)
        valid_index = np.flatnonzero(folds.to_numpy() == fold)
        outer_splits.append((train_index, valid_index))
        if (
            y.iloc[train_index].nunique() != 2
            or y.iloc[valid_index].nunique() != 2
        ):
            raise ValueError(
                "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
            )
        baseline = checked_fit(
            clone(core_pipeline),
            X_core.iloc[train_index],
            y.iloc[train_index],
            stage=f"core_fold_{fold}",
        )
        inner_cv = StratifiedKFold(
            n_splits=5, shuffle=True, random_state=SEED + fold
        )
        search = checked_fit(
            make_search(inner_cv),
            X_region.iloc[train_index],
            y.iloc[train_index],
            stage=f"region_fold_{fold}",
        )
        pd.DataFrame(search.cv_results_).to_csv(
            CV_DIR / f"outer_fold_{fold}_search.tsv", sep="\t", index=False
        )
        best = search.best_estimator_
        core_prob = baseline.predict_proba(X_core.iloc[valid_index])[:, 1]
        region_prob = best.predict_proba(X_region.iloc[valid_index])[:, 1]
        oof_core[valid_index] = core_prob
        oof_region[valid_index] = region_prob
        labels = y.iloc[valid_index]
        row = {
            "outer_fold": fold,
            "n_valid": len(valid_index),
            "n_positive": int(labels.sum()),
            "best_C": search.best_params_["model__C"],
            "best_l1_ratio": search.best_params_["model__l1_ratio"],
            "core_AUROC": roc_auc_score(labels, core_prob),
            "region_AUROC": roc_auc_score(labels, region_prob),
            "core_AP": average_precision_score(labels, core_prob),
            "region_AP": average_precision_score(labels, region_prob),
        }
        row["AP_difference_region_minus_core"] = (
            row["region_AP"] - row["core_AP"]
        )
        fold_rows.append(row)
        coefficient_rows.append(
            get_coefficients(best, feature_ids, stage=f"outer_fold_{fold}")
        )
        pd.DataFrame(fold_rows).to_csv(
            OUT / "outer_fold_comparison.tsv", sep="\t", index=False
        )
    if not (np.isfinite(oof_core).all() and np.isfinite(oof_region).all()):
        raise ValueError(
            "Regional Elastic Net search with nested evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    predictions = dev[["acronym", LABEL]].copy()
    predictions["CV_fold"] = folds
    predictions["probability_Three_gene_L2"] = oof_core
    predictions["probability_Region_ElasticNet"] = oof_region
    predictions.to_csv(
        OUT / "development_OOF_predictions.tsv",
        sep="\t",
        index_label="patient_id",
    )
    model_outputs = {
        "Three_gene_L2": oof_core,
        "Region_ElasticNet": oof_region,
    }
    performance = pd.DataFrame(
        [
            {
                "model": name,
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                "AUROC": roc_auc_score(y, probabilities),
                "AP": average_precision_score(y, probabilities),
            }
            for name, probabilities in model_outputs.items()
        ]
    )
    performance.to_csv(
        OUT / "development_model_comparison.tsv", sep="\t", index=False
    )
    fold_results = pd.DataFrame(fold_rows)
    coefficients = pd.concat(coefficient_rows, ignore_index=True).merge(
        candidate[["Entrez_ID", "DNA_symbol"]],
        on="Entrez_ID",
        how="left",
        validate="many_to_one",
    )
    coefficients.to_csv(
        OUT / "outer_fold_region_coefficients.tsv", sep="\t", index=False
    )
    stability = coefficients.groupby(
        ["Entrez_ID", "DNA_symbol"], as_index=False
    ).agg(
        n_outer_folds_nonzero=("nonzero_coefficient", "sum"),
        mean_standardized_coefficient=("standardized_coefficient", "mean"),
        min_standardized_coefficient=("standardized_coefficient", "min"),
        max_standardized_coefficient=("standardized_coefficient", "max"),
    )
    stability.to_csv(
        OUT / "region_coefficient_stability.tsv", sep="\t", index=False
    )
    cancer_rows = []
    for acronym, subset in predictions.groupby("acronym"):
        labels = subset[LABEL].astype(int)
        both_classes = labels.nunique() == 2
        for model, column in [
            ("Three_gene_L2", "probability_Three_gene_L2"),
            ("Region_ElasticNet", "probability_Region_ElasticNet"),
        ]:
            cancer_rows.append(
                {
                    "acronym": acronym,
                    "model": model,
                    "n_patients": len(labels),
                    "n_positive": int(labels.sum()),
                    "AUROC": (
                        roc_auc_score(labels, subset[column])
                        if both_classes
                        else np.nan
                    ),
                    "AP": (
                        average_precision_score(labels, subset[column])
                        if both_classes
                        else np.nan
                    ),
                    "status": (
                        "Single_class"
                        if not both_classes
                        else (
                            "Low_support"
                            if min(int(labels.sum()), int((labels == 0).sum()))
                            < 10
                            else "Descriptive"
                        )
                    ),
                }
            )
    pd.DataFrame(cancer_rows).to_csv(
        OUT / "per_cancer_performance.tsv", sep="\t", index=False
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


#### Regional feature sensitivity: 28 versus eight genes


In [ ]:
@_nb9_registry.register("s17_0")
def _nb9_stage_s17_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import warnings
    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    from sklearn.base import BaseEstimator, TransformerMixin
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.feature_selection import VarianceThreshold
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, average_precision_score
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.utils.validation import check_is_fitted

    PROJECT = Path(str(PROJECT_ROOT))
    PRED_DIR = PROJECT / "data_processed/MTAP_RNA_prediction"
    PREVIOUS_DIR = PRED_DIR / "09_9p21_region_model/signedlog_v2"
    EXPRESSION_FILE = (
        PREVIOUS_DIR / "development_region_signedlog_expression.tsv"
    )
    OOF_FILE = PREVIOUS_DIR / "development_OOF_predictions.tsv"
    CANDIDATE_FILE = PREVIOUS_DIR / "fixed_region_candidate_genes.tsv"
    OUT = PRED_DIR / "10_region_missingness_sensitivity"
    CV_DIR = OUT / "inner_CV"
    MODEL_DIR = OUT / "models"
    for directory in [OUT, CV_DIR, MODEL_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    LABEL = "label_MTAP_deep"
    SEED = 20261016
    CORE_IDS = {"4507", "1029", "1030"}
    PARAM_GRID = {
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.1, 0.5, 0.9],
    }
    for path in [EXPRESSION_FILE, OOF_FILE, CANDIDATE_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
            )
    data = pd.read_csv(OOF_FILE, sep="\t", dtype={"patient_id": str})
    required = {
        "patient_id",
        "acronym",
        LABEL,
        "CV_fold",
        "probability_Three_gene_L2",
        "probability_Region_ElasticNet",
    }
    if not required.issubset(data.columns):
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    if data["patient_id"].duplicated().any():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    data = data.set_index("patient_id").sort_index()
    label_values = pd.to_numeric(data[LABEL], errors="raise")
    if not label_values.isin([0, 1]).all():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    y = label_values.astype(int)
    fold_values = pd.to_numeric(data["CV_fold"], errors="raise")
    if fold_values.isna().any() or (fold_values % 1 != 0).any():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    folds = fold_values.astype(int)
    if sorted(folds.unique().tolist()) != [1, 2, 3, 4, 5]:
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    if len(data) != 7431 or int(y.sum()) != 558:
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    for column in [
        "probability_Three_gene_L2",
        "probability_Region_ElasticNet",
    ]:
        data[column] = pd.to_numeric(data[column], errors="raise")
        if data[column].isna().any() or not data[column].between(0, 1).all():
            raise ValueError(
                "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
            )
    X = pd.read_csv(
        EXPRESSION_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    X.columns = X.columns.astype(str)
    if X.index.duplicated().any() or X.columns.duplicated().any():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    if set(X.index) != set(data.index):
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    candidate = pd.read_csv(CANDIDATE_FILE, sep="\t", dtype={"Entrez_ID": str})
    if candidate["Entrez_ID"].duplicated().any():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    features = candidate["Entrez_ID"].tolist()
    if set(X.columns) != set(features):
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    X = X.reindex(index=data.index, columns=features)
    X = X.apply(pd.to_numeric, errors="raise")
    if np.isinf(X.to_numpy(dtype=float)).any():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    if not CORE_IDS.issubset(X.columns):
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    if X[list(CORE_IDS)].isna().any().any():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )

    class CompleteTrainingFeatures(TransformerMixin, BaseEstimator):

        def fit(self, X, y=None):
            if not isinstance(X, pd.DataFrame):
                raise TypeError(
                    "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
                )
            self.input_features_ = X.columns.tolist()
            self.kept_features_ = X.columns[~X.isna().any(axis=0)].tolist()
            self.dropped_features_ = [
                feature
                for feature in self.input_features_
                if feature not in self.kept_features_
            ]
            if not self.kept_features_:
                raise ValueError(
                    "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
                )
            return self

        def transform(self, X):
            check_is_fitted(self, "kept_features_")
            missing_columns = set(self.kept_features_).difference(X.columns)
            if missing_columns:
                raise ValueError(
                    "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
                )
            return X.loc[:, self.kept_features_].copy()

    pipeline = Pipeline(
        [
            ("complete", CompleteTrainingFeatures()),
            (
                "impute",
                SimpleImputer(strategy="median", keep_empty_features=True),
            ),
            ("variance", VarianceThreshold(threshold=0.0)),
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    penalty="elasticnet",
                    solver="saga",
                    class_weight=None,
                    max_iter=50000,
                    tol=0.0001,
                    random_state=SEED,
                ),
            ),
        ]
    )

    def make_search(cv):
        return GridSearchCV(
            estimator=pipeline,
            param_grid=PARAM_GRID,
            scoring={"AP": "average_precision", "AUROC": "roc_auc"},
            refit="AP",
            cv=cv,
            n_jobs=1,
            return_train_score=False,
            error_score="raise",
        )

    def checked_fit(estimator, train_X, train_y, stage):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            estimator.fit(train_X, train_y)
        messages = [
            str(item.message)
            for item in caught
            if issubclass(item.category, ConvergenceWarning)
        ]
        if messages:
            pd.DataFrame({"stage": stage, "warning": messages}).to_csv(
                OUT / "convergence_warnings.tsv", sep="\t", index=False
            )
            raise RuntimeError(
                "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
            )
        return estimator

    def feature_audit(train_X, valid_X, stage):
        kept = ~train_X.isna().any(axis=0)
        return pd.DataFrame(
            {
                "stage": stage,
                "Entrez_ID": train_X.columns,
                "n_train_missing": train_X.isna().sum().to_numpy(),
                "retained_by_completeness": kept.to_numpy(),
                "n_validation_missing": valid_X.isna().sum().to_numpy(),
            }
        )

    oof_sensitivity = np.full(len(y), np.nan)
    fold_rows = []
    audit_rows = []
    coefficient_rows = []
    outer_splits = []
    for fold in [1, 2, 3, 4, 5]:
        train_index = np.flatnonzero(folds.to_numpy() != fold)
        valid_index = np.flatnonzero(folds.to_numpy() == fold)
        outer_splits.append((train_index, valid_index))
        train_X = X.iloc[train_index]
        valid_X = X.iloc[valid_index]
        train_y = y.iloc[train_index]
        valid_y = y.iloc[valid_index]
        if train_y.nunique() != 2 or valid_y.nunique() != 2:
            raise ValueError(
                "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
            )
        inner_cv = StratifiedKFold(
            n_splits=5, shuffle=True, random_state=SEED + fold
        )
        inner_splits = list(inner_cv.split(train_X, train_y))
        audit_rows.append(
            feature_audit(train_X, valid_X, stage=f"outer_{fold}_refit")
        )
        for inner_fold, (inner_train, inner_valid) in enumerate(
            inner_splits, start=1
        ):
            audit_rows.append(
                feature_audit(
                    train_X.iloc[inner_train],
                    train_X.iloc[inner_valid],
                    stage=f"outer_{fold}_inner_{inner_fold}",
                )
            )
        search = checked_fit(
            make_search(inner_splits),
            train_X,
            train_y,
            stage=f"outer_fold_{fold}",
        )
        pd.DataFrame(search.cv_results_).to_csv(
            CV_DIR / f"outer_fold_{fold}_search.tsv", sep="\t", index=False
        )
        best = search.best_estimator_
        selected = best.named_steps["complete"].kept_features_
        if train_X[selected].isna().any().any():
            raise ValueError(
                "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
            )
        validation_missing = valid_X[selected].isna()
        probabilities = best.predict_proba(valid_X)[:, 1]
        oof_sensitivity[valid_index] = probabilities
        row = {
            "outer_fold": fold,
            "n_train": len(train_X),
            "n_valid": len(valid_X),
            "n_positive_valid": int(valid_y.sum()),
            "n_features_complete_in_training": len(selected),
            "n_valid_patients_requiring_imputation": int(
                validation_missing.any(axis=1).sum()
            ),
            "n_valid_cells_requiring_imputation": int(
                validation_missing.sum().sum()
            ),
            "best_C": search.best_params_["model__C"],
            "best_l1_ratio": search.best_params_["model__l1_ratio"],
            "AUROC": roc_auc_score(valid_y, probabilities),
            "AP": average_precision_score(valid_y, probabilities),
        }
        fold_rows.append(row)
        retained_after_variance = best.named_steps["variance"].get_support()
        coefficient_values = np.zeros(len(selected))
        coefficient_values[retained_after_variance] = best.named_steps[
            "model"
        ].coef_[0]
        coefficient_rows.append(
            pd.DataFrame(
                {
                    "outer_fold": fold,
                    "Entrez_ID": selected,
                    "retained_after_variance": retained_after_variance,
                    "standardized_coefficient": coefficient_values,
                }
            )
        )
        pd.DataFrame(fold_rows).to_csv(
            OUT / "outer_fold_results.tsv", sep="\t", index=False
        )
    feature_audits = pd.concat(audit_rows, ignore_index=True).merge(
        candidate[["Entrez_ID", "DNA_symbol"]],
        on="Entrez_ID",
        how="left",
        validate="many_to_one",
    )
    feature_audits.to_csv(
        OUT / "outer_and_inner_feature_selection_audit.tsv",
        sep="\t",
        index=False,
    )
    outer_audits = feature_audits.loc[
        feature_audits["stage"].str.endswith("_refit")
    ]
    outer_audits.to_csv(
        OUT / "outer_training_feature_selection.tsv", sep="\t", index=False
    )
    pd.concat(coefficient_rows, ignore_index=True).merge(
        candidate[["Entrez_ID", "DNA_symbol"]],
        on="Entrez_ID",
        how="left",
        validate="many_to_one",
    ).to_csv(OUT / "outer_fold_coefficients.tsv", sep="\t", index=False)
    if not np.isfinite(oof_sensitivity).all():
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    data["probability_Complete_training_region"] = oof_sensitivity
    data.to_csv(
        OUT / "development_three_model_OOF.tsv",
        sep="\t",
        index_label="patient_id",
    )
    model_columns = {
        "Three_gene_L2": "probability_Three_gene_L2",
        "Region_28_candidate": "probability_Region_ElasticNet",
        "Region_complete_training_features": "probability_Complete_training_region",
    }
    comparison = pd.DataFrame(
        [
            {
                "model": name,
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                "AUROC": roc_auc_score(y, data[column]),
                "AP": average_precision_score(y, data[column]),
            }
            for name, column in model_columns.items()
        ]
    )
    comparison.to_csv(
        OUT / "development_model_comparison.tsv", sep="\t", index=False
    )
    cancer_rows = []
    for acronym, subset in data.groupby("acronym"):
        labels = subset[LABEL].astype(int)
        both_classes = labels.nunique() == 2
        for name, column in model_columns.items():
            cancer_rows.append(
                {
                    "acronym": acronym,
                    "model": name,
                    "n_patients": len(subset),
                    "n_positive": int(labels.sum()),
                    "AUROC": (
                        roc_auc_score(labels, subset[column])
                        if both_classes
                        else np.nan
                    ),
                    "AP": (
                        average_precision_score(labels, subset[column])
                        if both_classes
                        else np.nan
                    ),
                    "support_status": (
                        "Single_class"
                        if not both_classes
                        else (
                            "Low_support"
                            if min(int(labels.sum()), int((labels == 0).sum()))
                            < 10
                            else "Descriptive"
                        )
                    ),
                }
            )
    pd.DataFrame(cancer_rows).to_csv(
        OUT / "per_cancer_three_model_comparison.tsv", sep="\t", index=False
    )
    final_search = checked_fit(
        make_search(outer_splits), X, y, stage="full_development"
    )
    pd.DataFrame(final_search.cv_results_).to_csv(
        CV_DIR / "full_development_search.tsv", sep="\t", index=False
    )
    final_pipeline = final_search.best_estimator_
    final_features = final_pipeline.named_steps["complete"].kept_features_
    portable_pipeline = Pipeline(final_pipeline.steps[1:])
    if not np.allclose(
        final_pipeline.predict_proba(X),
        portable_pipeline.predict_proba(X[final_features]),
    ):
        raise ValueError(
            "Regional feature sensitivity: 28 versus eight genes: integrity check failed; condition: input or artifact integrity check."
        )
    joblib.dump(
        {
            "pipeline": portable_pipeline,
            "feature_Entrez_order": final_features,
            "pipeline_input": "already_signedlog_transformed",
            "preprocessing_before_pipeline": "sign(x)*log2(1+abs(x)); x=patient mean of source RNA values",
            "feature_selection_rule": "No missing values in development training data",
            "training_scope": "development_only",
            "status": "sensitivity_candidate_not_final",
            "sklearn_version": sklearn.__version__,
        },
        MODEL_DIR / "complete_region_development_candidate.joblib",
    )
    candidate.loc[candidate["Entrez_ID"].isin(final_features)].to_csv(
        OUT / "full_development_retained_genes.tsv", sep="\t", index=False
    )
    with open(OUT / "run_record.json", "w", encoding="utf-8") as handle:
        json.dump(
            {
                "seed": SEED,
                "selection_rule": "Drop any feature missing in current training fold",
                "selection_inside_inner_CV": True,
                "parameter_grid": PARAM_GRID,
                "best_development_parameters": final_search.best_params_,
                "final_retained_Entrez_order": final_features,
                "patients_removed": 0,
                "test_evaluated": False,
                "threshold_selected": False,
                "validation_missing_fallback": "Training median; counts reported",
            },
            handle,
            ensure_ascii=False,
            indent=2,
        )
    _public_summary(
        globals(), "Regional feature sensitivity: 28 versus eight genes"
    )


#### Eight-gene Elastic Net and three-gene L2 comparison


In [ ]:
@_nb9_registry.register("s19_0")
def _nb9_stage_s19_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import json
    import shutil
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        roc_curve,
        precision_recall_curve,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    PRED = PROJECT / "data_processed/MTAP_RNA_prediction"
    EIGHT_MODEL_FILE = (
        PRED
        / "10_region_missingness_sensitivity/models/complete_region_development_candidate.joblib"
    )
    BASELINE_MODEL_FILE = (
        PRED
        / "09_9p21_region_model/signedlog_v2/models/Three_gene_L2_development_candidate.joblib"
    )
    DEV_OOF_FILE = (
        PRED
        / "10_region_missingness_sensitivity/development_three_model_OOF.tsv"
    )
    METADATA_FILE = (
        PRED / "03_baseline_models/cohort_with_three_gene_expression.tsv"
    )
    RNA_FILE = (
        PROJECT
        / "data_raw/PANCAN/TCGA/EBPlusPlusAdjustPANCAN_IlluminaHiSeq_RNASeqV2.geneExp.tsv"
    )
    OUT = PRED / "11_locked_model_TCGA_test"
    LOCK_DIR = OUT / "locked_models"
    FIG_DIR = OUT / "figures"
    for directory in [OUT, LOCK_DIR, FIG_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    MANIFEST_FILE = OUT / "locked_analysis_manifest.json"
    FINAL_RESULTS_FILE = OUT / "TCGA_test_performance.tsv"
    SEED = 20261016
    N_BOOTSTRAPS = 2000
    LABEL = "label_MTAP_deep"
    DEV_MAIN_PROBABILITY = "probability_Complete_training_region"
    RULES = {
        "positive_min_PPV": 0.9,
        "positive_min_Wilson_lower": 0.85,
        "positive_min_n": 30,
        "negative_min_NPV": 0.98,
        "negative_min_Wilson_lower": 0.97,
        "negative_min_n": 100,
    }
    EXPECTED_EIGHT_IDS = {
        "100048912",
        "1029",
        "1030",
        "401494",
        "4300",
        "4507",
        "54914",
        "55958",
    }
    PALETTE = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]

    def sha256_file(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()

    def signed_log2(frame):
        return np.sign(frame) * np.log1p(np.abs(frame)) / np.log(2.0)

    def wilson_interval(k, n):
        k = np.asarray(k, dtype=float)
        n = np.asarray(n, dtype=float)
        with np.errstate(divide="ignore", invalid="ignore"):
            p = k / n
            z = 1.96
            denominator = 1 + z**2 / n
            center = (p + z**2 / (2 * n)) / denominator
            half = (
                z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominator
            )
        return (
            np.where(n > 0, center - half, np.nan),
            np.where(n > 0, center + half, np.nan),
        )

    def calibration_table(labels, probabilities, n_bins=10):
        table = pd.DataFrame(
            {
                "label": np.asarray(labels, dtype=int),
                "probability": np.asarray(probabilities, dtype=float),
            }
        )
        table["bin"] = pd.qcut(
            table["probability"], q=n_bins, duplicates="drop"
        )
        return (
            table.groupby("bin", observed=True)
            .agg(
                n=("label", "size"),
                n_positive=("label", "sum"),
                mean_probability=("probability", "mean"),
                observed_fraction=("label", "mean"),
            )
            .reset_index()
        )

    def metric_row(labels, probabilities, model):
        labels = np.asarray(labels, dtype=int)
        probabilities = np.asarray(probabilities, dtype=float)
        return {
            "model": model,
            "n_patients": len(labels),
            "n_positive": int(labels.sum()),
            "AUROC": roc_auc_score(labels, probabilities),
            "AP": average_precision_score(labels, probabilities),
            "Brier_score": brier_score_loss(labels, probabilities),
            "log_loss": log_loss(labels, probabilities, labels=[0, 1]),
        }

    def choose_thresholds(labels, probabilities):
        frame = pd.DataFrame(
            {
                "probability": np.asarray(probabilities, dtype=float),
                "positive": np.asarray(labels, dtype=int),
            }
        )
        grouped = (
            frame.groupby("probability", sort=True)["positive"]
            .agg(["size", "sum"])
            .rename(columns={"size": "n", "sum": "n_positive"})
        )
        n = grouped["n"].to_numpy()
        positive = grouped["n_positive"].to_numpy()
        thresholds = grouped.index.to_numpy(dtype=float)
        n_high = np.cumsum(n[::-1])[::-1]
        positive_high = np.cumsum(positive[::-1])[::-1]
        ppv = positive_high / n_high
        ppv_low, ppv_high = wilson_interval(positive_high, n_high)
        n_low = np.cumsum(n)
        negative_low = np.cumsum(n - positive)
        npv = negative_low / n_low
        npv_low, npv_high = wilson_interval(negative_low, n_low)
        audit = pd.DataFrame(
            {
                "threshold": thresholds,
                "n_high": n_high,
                "n_positive_high": positive_high,
                "PPV_high": ppv,
                "PPV_high_CI95_low": ppv_low,
                "PPV_high_CI95_high": ppv_high,
                "n_low": n_low,
                "n_negative_low": negative_low,
                "NPV_low": npv,
                "NPV_low_CI95_low": npv_low,
                "NPV_low_CI95_high": npv_high,
            }
        )
        positive_candidates = audit.loc[
            audit["PPV_high"].ge(RULES["positive_min_PPV"])
            & audit["PPV_high_CI95_low"].ge(RULES["positive_min_Wilson_lower"])
            & audit["n_high"].ge(RULES["positive_min_n"])
        ]
        if positive_candidates.empty:
            audit.to_csv(
                OUT / "development_threshold_search.tsv", sep="\t", index=False
            )
            raise RuntimeError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        chosen_positive = positive_candidates.sort_values(
            ["n_positive_high", "threshold"], ascending=[False, True]
        ).iloc[0]
        high_threshold = float(chosen_positive["threshold"])
        negative_candidates = audit.loc[
            audit["threshold"].lt(high_threshold)
            & audit["NPV_low"].ge(RULES["negative_min_NPV"])
            & audit["NPV_low_CI95_low"].ge(RULES["negative_min_Wilson_lower"])
            & audit["n_low"].ge(RULES["negative_min_n"])
        ]
        if negative_candidates.empty:
            audit.to_csv(
                OUT / "development_threshold_search.tsv", sep="\t", index=False
            )
            raise RuntimeError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        chosen_negative = negative_candidates.sort_values(
            "n_low", ascending=False
        ).iloc[0]
        low_threshold = float(chosen_negative["threshold"])
        audit.to_csv(
            OUT / "development_threshold_search.tsv", sep="\t", index=False
        )
        chosen = pd.DataFrame(
            [
                {
                    "group": "Predicted_deep_deletion",
                    "threshold": high_threshold,
                    "rule": "probability >= threshold",
                    "n_development": int(chosen_positive["n_high"]),
                    "predictive_value": chosen_positive["PPV_high"],
                    "Wilson_CI95_low": chosen_positive["PPV_high_CI95_low"],
                },
                {
                    "group": "Predicted_non_deep",
                    "threshold": low_threshold,
                    "rule": "probability <= threshold",
                    "n_development": int(chosen_negative["n_low"]),
                    "predictive_value": chosen_negative["NPV_low"],
                    "Wilson_CI95_low": chosen_negative["NPV_low_CI95_low"],
                },
            ]
        )
        chosen.to_csv(
            OUT / "development_selected_thresholds.tsv", sep="\t", index=False
        )
        return (low_threshold, high_threshold)

    def extract_test_expression(test, feature_ids):
        cache = OUT / "TCGA_test_raw_expression.tsv"
        if cache.is_file():
            raw = pd.read_csv(
                cache, sep="\t", dtype={"patient_id": str}
            ).set_index("patient_id")
            raw.columns = raw.columns.astype(str)
            if (
                raw.index.duplicated().any()
                or set(raw.index) != set(test.index)
                or set(raw.columns) != set(feature_ids)
            ):
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            return raw.reindex(index=test.index, columns=feature_ids).apply(
                pd.to_numeric, errors="raise"
            )
        header = pd.read_csv(RNA_FILE, sep="\t", nrows=0)
        gene_column = header.columns[0]
        patient_set = set(test.index)
        sample_map = {}
        for column in header.columns[1:]:
            barcode = str(column)
            if (
                barcode.startswith("TCGA-")
                and len(barcode) >= 16
                and (barcode[13:15] == "01")
                and (barcode[:12] in patient_set)
            ):
                sample_map[column] = barcode[:12]
        if set(sample_map.values()) != patient_set:
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        pd.DataFrame(
            {
                "RNA_column": list(sample_map),
                "patient_id": list(sample_map.values()),
            }
        ).to_csv(
            OUT / "TCGA_test_RNA_column_mapping.tsv", sep="\t", index=False
        )
        wanted = set(feature_ids)
        extracted = {}
        for number, chunk in enumerate(
            pd.read_csv(
                RNA_FILE,
                sep="\t",
                usecols=[gene_column, *sample_map],
                chunksize=512,
            ),
            start=1,
        ):
            ids = chunk[gene_column].astype(str).str.rsplit("|", n=1).str[-1]
            selected = chunk.loc[ids.isin(wanted)].copy()
            for _, row in selected.iterrows():
                gene_id = str(row[gene_column]).rsplit("|", 1)[-1]
                if gene_id in extracted:
                    raise ValueError(
                        "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                    )
                values = pd.to_numeric(
                    row[list(sample_map)], errors="raise"
                ).astype(float)
                values.index = [sample_map[column] for column in sample_map]
                extracted[gene_id] = values.groupby(level=0).mean()
            if number % 10 == 0:
                pass
        if set(extracted) != wanted:
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        raw = pd.DataFrame(extracted).reindex(
            index=test.index, columns=feature_ids
        )
        raw.to_csv(cache, sep="\t", index_label="patient_id")
        return raw

    def main():
        if FINAL_RESULTS_FILE.is_file():
            return
        for path in [
            EIGHT_MODEL_FILE,
            BASELINE_MODEL_FILE,
            DEV_OOF_FILE,
            METADATA_FILE,
            RNA_FILE,
        ]:
            if not path.is_file():
                raise FileNotFoundError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
        if not MANIFEST_FILE.is_file():
            dev = pd.read_csv(
                DEV_OOF_FILE, sep="\t", dtype={"patient_id": str}
            )
            required = {"patient_id", LABEL, DEV_MAIN_PROBABILITY}
            if not required.issubset(dev.columns):
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            if dev["patient_id"].duplicated().any():
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            labels = pd.to_numeric(dev[LABEL], errors="raise")
            probabilities = pd.to_numeric(
                dev[DEV_MAIN_PROBABILITY], errors="raise"
            )
            if (
                not labels.isin([0, 1]).all()
                or probabilities.isna().any()
                or (not probabilities.between(0, 1).all())
            ):
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            if len(dev) != 7431 or int(labels.sum()) != 558:
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            bundle = joblib.load(EIGHT_MODEL_FILE)
            features = [str(value) for value in bundle["feature_Entrez_order"]]
            if set(features) != EXPECTED_EIGHT_IDS:
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            if bundle.get("pipeline_input") != "already_signedlog_transformed":
                raise ValueError(
                    "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
                )
            low, high = choose_thresholds(labels, probabilities)
            calibration_table(labels, probabilities).to_csv(
                OUT / "development_calibration_bins.tsv", sep="\t", index=False
            )
            pd.DataFrame(
                [metric_row(labels, probabilities, "Eight_member")]
            ).to_csv(
                OUT / "development_probability_diagnostics.tsv",
                sep="\t",
                index=False,
            )
            locked_main = LOCK_DIR / "Eight_member_locked.joblib"
            locked_baseline = LOCK_DIR / "Three_gene_L2_locked.joblib"
            shutil.copyfile(EIGHT_MODEL_FILE, locked_main)
            shutil.copyfile(BASELINE_MODEL_FILE, locked_baseline)
            manifest = {
                "primary_model": "Eight_member",
                "comparator": "Three_gene_L2",
                "feature_Entrez_order": features,
                "expression_transform": "sign(x)*log2(1+abs(x)); x=patient mean source RNA",
                "additional_probability_calibration": False,
                "rules": RULES,
                "low_threshold": low,
                "high_threshold": high,
                "development_OOF_sha256": sha256_file(DEV_OOF_FILE),
                "main_model_sha256": sha256_file(locked_main),
                "baseline_model_sha256": sha256_file(locked_baseline),
                "seed": SEED,
                "thresholds_selected_without_test": True,
            }
            with open(MANIFEST_FILE, "x", encoding="utf-8") as handle:
                json.dump(manifest, handle, ensure_ascii=False, indent=2)
        with open(MANIFEST_FILE, encoding="utf-8") as handle:
            manifest = json.load(handle)
        locked_main = LOCK_DIR / "Eight_member_locked.joblib"
        locked_baseline = LOCK_DIR / "Three_gene_L2_locked.joblib"
        if (
            sha256_file(locked_main) != manifest["main_model_sha256"]
            or sha256_file(locked_baseline)
            != manifest["baseline_model_sha256"]
            or sha256_file(DEV_OOF_FILE) != manifest["development_OOF_sha256"]
        ):
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        main_bundle = joblib.load(locked_main)
        baseline_bundle = joblib.load(locked_baseline)
        features = manifest["feature_Entrez_order"]
        baseline_features = [
            str(value) for value in baseline_bundle["feature_Entrez_order"]
        ]
        if not set(baseline_features).issubset(features):
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        metadata = pd.read_csv(
            METADATA_FILE, sep="\t", dtype={"patient_id": str}
        )
        if metadata["patient_id"].duplicated().any():
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        test = (
            metadata.loc[metadata["dataset_split"].eq("test")]
            .copy()
            .set_index("patient_id")
            .sort_index()
        )
        dev_ids = set(
            pd.read_csv(DEV_OOF_FILE, sep="\t", usecols=["patient_id"])[
                "patient_id"
            ]
        )
        if dev_ids.intersection(test.index):
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        mtap_calls = pd.to_numeric(test["MTAP_GISTIC_call"], errors="raise")
        if not mtap_calls.isin([-2, -1, 0, 1, 2]).all():
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        test_y = mtap_calls.eq(-2).astype(int)
        if len(test) != 1858 or int(test_y.sum()) != 140:
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        raw = extract_test_expression(test, features)
        if np.isinf(raw.to_numpy(dtype=float)).any():
            raise ValueError(
                "Eight-gene Elastic Net and three-gene L2 comparison: integrity check failed; condition: input or artifact integrity check."
            )
        expression_audit = pd.DataFrame(
            {
                "Entrez_ID": features,
                "n_missing": raw.isna().sum().to_numpy(),
                "n_negative": raw.lt(0).sum().to_numpy(),
                "minimum": raw.min().to_numpy(),
                "maximum": raw.max().to_numpy(),
            }
        )
        expression_audit.to_csv(
            OUT / "TCGA_test_expression_audit.tsv", sep="\t", index=False
        )
        X_test = signed_log2(raw)
        predictions = test[["acronym", "MTAP_GISTIC_call"]].copy()
        for optional in ["CDKN2A_GISTIC_call", "CDKN2B_GISTIC_call"]:
            if optional in test.columns:
                predictions[optional] = test[optional]
        predictions[LABEL] = test_y
        predictions["probability_Eight_member"] = main_bundle[
            "pipeline"
        ].predict_proba(X_test[features])[:, 1]
        predictions["probability_Three_gene_L2"] = baseline_bundle[
            "pipeline"
        ].predict_proba(X_test[baseline_features])[:, 1]
        predictions["n_missing_input_features"] = raw.isna().sum(axis=1)
        low = manifest["low_threshold"]
        high = manifest["high_threshold"]
        probabilities = predictions["probability_Eight_member"]
        predictions["RNA_prediction_group"] = "Uncertain"
        predictions.loc[probabilities.ge(high), "RNA_prediction_group"] = (
            "Predicted_deep_deletion"
        )
        predictions.loc[probabilities.le(low), "RNA_prediction_group"] = (
            "Predicted_non_deep"
        )
        predictions.to_csv(
            OUT / "TCGA_test_patient_predictions.tsv",
            sep="\t",
            index_label="patient_id",
        )
        outputs = {
            "Eight_member": predictions["probability_Eight_member"],
            "Three_gene_L2": predictions["probability_Three_gene_L2"],
        }
        performance = pd.DataFrame(
            [
                metric_row(test_y, probability, name)
                for name, probability in outputs.items()
            ]
        )
        rng = np.random.default_rng(SEED)
        y_array = test_y.to_numpy()
        positive_index = np.flatnonzero(y_array == 1)
        negative_index = np.flatnonzero(y_array == 0)
        boot = {name: np.zeros((N_BOOTSTRAPS, 2)) for name in outputs}
        for iteration in range(N_BOOTSTRAPS):
            sampled = np.concatenate(
                [
                    rng.choice(
                        positive_index, size=len(positive_index), replace=True
                    ),
                    rng.choice(
                        negative_index, size=len(negative_index), replace=True
                    ),
                ]
            )
            for name, probability in outputs.items():
                sampled_p = probability.to_numpy()[sampled]
                boot[name][iteration] = [
                    roc_auc_score(y_array[sampled], sampled_p),
                    average_precision_score(y_array[sampled], sampled_p),
                ]
        for metric_index, metric in enumerate(["AUROC", "AP"]):
            for name in outputs:
                ci_low, ci_high = np.quantile(
                    boot[name][:, metric_index], [0.025, 0.975]
                )
                performance.loc[
                    performance["model"].eq(name), f"{metric}_CI95_low"
                ] = ci_low
                performance.loc[
                    performance["model"].eq(name), f"{metric}_CI95_high"
                ] = ci_high
        group_rows = []
        for group in [
            "Predicted_deep_deletion",
            "Uncertain",
            "Predicted_non_deep",
        ]:
            subset = predictions.loc[
                predictions["RNA_prediction_group"].eq(group)
            ]
            n = len(subset)
            n_positive = int(subset[LABEL].sum())
            if group == "Predicted_deep_deletion":
                k = n_positive
                value_name = "PPV"
            elif group == "Predicted_non_deep":
                k = n - n_positive
                value_name = "NPV"
            else:
                k = n_positive
                value_name = "DNA_positive_fraction"
            ci_low, ci_high = wilson_interval(k, n)
            group_rows.append(
                {
                    "group": group,
                    "n_patients": n,
                    "coverage_fraction": n / len(predictions),
                    "n_DNA_positive": n_positive,
                    "value_name": value_name,
                    "value": k / n if n else np.nan,
                    "Wilson_CI95_low": float(ci_low),
                    "Wilson_CI95_high": float(ci_high),
                }
            )
        group_results = pd.DataFrame(group_rows)
        group_results.to_csv(
            OUT / "TCGA_test_three_group_results.tsv", sep="\t", index=False
        )
        cancer_rows = []
        for acronym, subset in predictions.groupby("acronym"):
            labels = subset[LABEL].astype(int)
            for name in outputs:
                probability = subset[f"probability_{name}"]
                cancer_rows.append(
                    {
                        "acronym": acronym,
                        "model": name,
                        "n_patients": len(subset),
                        "n_positive": int(labels.sum()),
                        "AUROC": (
                            roc_auc_score(labels, probability)
                            if labels.nunique() == 2
                            else np.nan
                        ),
                        "AP": (
                            average_precision_score(labels, probability)
                            if labels.nunique() == 2
                            else np.nan
                        ),
                        "support_status": (
                            "Single_class"
                            if labels.nunique() < 2
                            else (
                                "Low_support"
                                if min(
                                    int(labels.sum()), int((labels == 0).sum())
                                )
                                < 10
                                else "Descriptive"
                            )
                        ),
                    }
                )
        pd.DataFrame(cancer_rows).to_csv(
            OUT / "TCGA_test_per_cancer_performance.tsv", sep="\t", index=False
        )
        plt.rcParams.update(
            {
                "font.family": "DejaVu Sans",
                "font.size": 9,
                "pdf.fonttype": 42,
                "ps.fonttype": 42,
            }
        )
        fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), facecolor="white")
        for (name, probability), color in zip(
            outputs.items(), [PALETTE[0], PALETTE[4]]
        ):
            fpr, tpr, _ = roc_curve(test_y, probability)
            precision, recall, _ = precision_recall_curve(test_y, probability)
            axes[0].plot(
                fpr,
                tpr,
                color=color,
                label=f"{name}: AUC={roc_auc_score(test_y, probability):.3f}",
            )
            axes[1].plot(
                recall,
                precision,
                color=color,
                label=f"{name}: AP={average_precision_score(test_y, probability):.3f}",
            )
            calibration = calibration_table(test_y, probability)
            calibration.to_csv(
                OUT / f"TCGA_test_{name}_calibration.tsv",
                sep="\t",
                index=False,
            )
            axes[2].plot(
                calibration["mean_probability"],
                calibration["observed_fraction"],
                marker="o",
                markersize=4,
                color=color,
                label=name,
            )
        axes[0].plot([0, 1], [0, 1], "--", color="#AAAAAA", linewidth=0.8)
        axes[1].axhline(
            test_y.mean(), linestyle="--", color="#AAAAAA", linewidth=0.8
        )
        axes[2].plot([0, 1], [0, 1], "--", color="#AAAAAA", linewidth=0.8)
        axes[0].set(
            xlabel="False positive rate",
            ylabel="True positive rate",
            title="TCGA held-out ROC",
        )
        axes[1].set(
            xlabel="Recall",
            ylabel="Precision",
            title="TCGA held-out precision-recall",
        )
        axes[2].set(
            xlabel="Mean predicted probability",
            ylabel="Observed deep-deletion fraction",
            title="Calibration: quantile bins",
        )
        for ax in axes:
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.02)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            ax.legend(
                frameon=False,
                fontsize=8,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.2),
            )
        fig.subplots_adjust(
            left=0.06, right=0.99, bottom=0.3, top=0.9, wspace=0.32
        )
        for extension in ["pdf", "png"]:
            fig.savefig(
                FIG_DIR / f"TCGA_locked_test_evaluation.{extension}",
                dpi=350,
                bbox_inches="tight",
                facecolor="white",
            )
        plt.show()
        performance.to_csv(FINAL_RESULTS_FILE, sep="\t", index=False)

    main()
    _public_summary(
        globals(), "Eight-gene Elastic Net and three-gene L2 comparison"
    )


#### Forty-five-feature Elastic Net development


In [ ]:
@_nb9_registry.register("s26_0")
def _nb9_stage_s26_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import time
    import numpy as np
    import pandas as pd
    import joblib
    import matplotlib.pyplot as plt
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    ROOT = PROJECT / "data_processed/MTAP_RNA_prediction"
    INPUT = ROOT / "15_region_multitask_DL/01_training_inputs"
    DL_DIR = ROOT / "15_region_multitask_DL/02_training_v1"
    OUT = ROOT / "15_region_multitask_DL/03_same_input_ElasticNet"
    X_FILE = INPUT / "X_development_signedlog.tsv"
    METADATA_FILE = INPUT / "development_metadata.tsv"
    DL_FILE = DL_DIR / "development_MTAP_OOF_predictions.tsv"
    for path in [X_FILE, METADATA_FILE, DL_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
            )
    MODEL_DIR = OUT / "models"
    FIGURE_DIR = OUT / "figures"
    for directory in [OUT, MODEL_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    SEED = 2026
    INNER_FOLDS = 3
    N_JOBS = 4
    MAX_ITER = 10000
    TOL = 0.0001
    N_BOOTSTRAPS = 2000
    PARAM_GRID = {
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.1, 0.5, 0.9],
    }
    started = time.monotonic()
    metadata = pd.read_csv(
        METADATA_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    expression = pd.read_csv(X_FILE, sep="\t", index_col=0)
    dl = pd.read_csv(DL_FILE, sep="\t", dtype={"patient_id": str}).set_index(
        "patient_id"
    )
    expression.index = expression.index.astype(str)
    for name, table in [
        ("metadata", metadata),
        ("expression", expression),
        ("DL OOF", dl),
    ]:
        if table.index.duplicated().any():
            raise ValueError(
                "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
            )
    required_metadata = {"acronym", "dataset_split", "label_MTAP_deep"}
    required_dl = {
        "acronym",
        "label_MTAP_deep",
        "outer_fold",
        "prob_MTAP_deep",
    }
    if not required_metadata.issubset(metadata.columns):
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if not required_dl.issubset(dl.columns):
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if set(expression.index) != set(metadata.index) or set(dl.index) != set(
        metadata.index
    ):
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if not metadata["dataset_split"].eq("development").all():
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    expression = expression.reindex(metadata.index)
    dl = dl.reindex(metadata.index)
    expression = expression.apply(pd.to_numeric, errors="raise")
    if expression.columns.duplicated().any():
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if np.isinf(expression.to_numpy(dtype=float)).any():
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    y_series = pd.to_numeric(metadata["label_MTAP_deep"], errors="raise")
    if y_series.isna().any() or not y_series.isin([0, 1]).all():
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    y = y_series.to_numpy(dtype=int)
    dl_label = pd.to_numeric(dl["label_MTAP_deep"], errors="raise").to_numpy()
    if not np.array_equal(y, dl_label):
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        metadata["acronym"].to_numpy(), dl["acronym"].to_numpy()
    ):
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    fold_values = pd.to_numeric(dl["outer_fold"], errors="raise")
    if (
        fold_values.isna().any()
        or not fold_values.eq(fold_values.astype(int)).all()
    ):
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    folds = fold_values.to_numpy(dtype=int)
    unique_folds = sorted(np.unique(folds))
    if len(unique_folds) != 5:
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    p_dl = pd.to_numeric(dl["prob_MTAP_deep"], errors="raise").to_numpy(
        dtype=float
    )
    if not np.isfinite(p_dl).all() or ((p_dl < 0) | (p_dl > 1)).any():
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if len(metadata) != 7431 or y.sum() != 558 or expression.shape[1] != 45:
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    if metadata["acronym"].isna().any():
        raise ValueError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    feature_names = expression.columns.tolist()
    cancer_names = sorted(metadata["acronym"].unique())
    X = expression.copy()
    X["acronym"] = metadata["acronym"].astype(str)

    def make_pipeline():
        numeric_pipeline = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="median", keep_empty_features=True),
                ),
                ("scaler", StandardScaler()),
            ]
        )
        preprocessing = ColumnTransformer(
            transformers=[
                ("expression", numeric_pipeline, feature_names),
                (
                    "cancer",
                    OneHotEncoder(
                        categories=[cancer_names],
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                    ["acronym"],
                ),
            ],
            remainder="drop",
            sparse_threshold=0,
        )
        return Pipeline(
            [
                ("preprocessing", preprocessing),
                (
                    "model",
                    LogisticRegression(
                        penalty="elasticnet",
                        solver="saga",
                        class_weight=None,
                        max_iter=MAX_ITER,
                        tol=TOL,
                        random_state=SEED,
                    ),
                ),
            ]
        )

    def make_search(seed):
        inner_cv = StratifiedKFold(
            n_splits=INNER_FOLDS, shuffle=True, random_state=seed
        )
        return GridSearchCV(
            estimator=make_pipeline(),
            param_grid=PARAM_GRID,
            scoring="average_precision",
            cv=inner_cv,
            refit=True,
            n_jobs=N_JOBS,
            pre_dispatch=N_JOBS,
            error_score="raise",
            return_train_score=False,
        )

    def positive_probability(estimator, values):
        classes = estimator.named_steps["model"].classes_
        positions = np.flatnonzero(classes == 1)
        if len(positions) != 1:
            raise ValueError(
                "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
            )
        return estimator.predict_proba(values)[:, positions[0]]

    p_en = np.full(len(X), np.nan)
    fold_records = []
    for fold in unique_folds:
        train_index = np.flatnonzero(folds != fold)
        valid_index = np.flatnonzero(folds == fold)
        search = make_search(SEED + int(fold))
        search.fit(X.iloc[train_index], y[train_index])
        best_model = search.best_estimator_
        n_iterations = int(best_model.named_steps["model"].n_iter_.max())
        if n_iterations >= MAX_ITER:
            raise RuntimeError(
                "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
            )
        probability = positive_probability(best_model, X.iloc[valid_index])
        p_en[valid_index] = probability
        en_auc = roc_auc_score(y[valid_index], probability)
        en_ap = average_precision_score(y[valid_index], probability)
        dl_auc = roc_auc_score(y[valid_index], p_dl[valid_index])
        dl_ap = average_precision_score(y[valid_index], p_dl[valid_index])
        fold_records.append(
            {
                "outer_fold": int(fold),
                "n_train": len(train_index),
                "n_valid": len(valid_index),
                "best_C": search.best_params_["model__C"],
                "best_l1_ratio": search.best_params_["model__l1_ratio"],
                "inner_best_AP": search.best_score_,
                "n_iterations": n_iterations,
                "ElasticNet_AUROC": en_auc,
                "DL_AUROC": dl_auc,
                "ElasticNet_AP": en_ap,
                "DL_AP": dl_ap,
                "delta_AP_DL_minus_EN": dl_ap - en_ap,
            }
        )
        pd.DataFrame(search.cv_results_).to_csv(
            OUT / f"inner_search_fold{fold}.tsv", sep="\t", index=False
        )
        joblib.dump(best_model, MODEL_DIR / f"ElasticNet_fold{fold}.joblib")
        pd.DataFrame(fold_records).to_csv(
            OUT / "outer_fold_comparison.tsv", sep="\t", index=False
        )
    if not np.isfinite(p_en).all():
        raise RuntimeError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )

    def calculate_metrics(truth, probability):
        predicted = probability >= 0.5
        tn, fp, fn, tp = confusion_matrix(
            truth, predicted, labels=[0, 1]
        ).ravel()
        return {
            "AUROC": roc_auc_score(truth, probability),
            "AP": average_precision_score(truth, probability),
            "Brier_score": brier_score_loss(truth, probability),
            "log_loss": log_loss(truth, probability, labels=[0, 1]),
            "precision_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "sensitivity_at_05": tp / (tp + fn),
            "specificity_at_05": tn / (tn + fp),
            "TP_at_05": int(tp),
            "FP_at_05": int(fp),
            "FN_at_05": int(fn),
            "TN_at_05": int(tn),
        }

    model_probabilities = {
        "Same_input_ElasticNet": p_en,
        "Regional_multitask_DL_3seed": p_dl,
    }
    performance = pd.DataFrame(
        [
            {
                "model": name,
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                **calculate_metrics(y, probability),
            }
            for name, probability in model_probabilities.items()
        ]
    )
    performance.to_csv(
        OUT / "development_model_comparison.tsv", sep="\t", index=False
    )
    patient_results = pd.DataFrame(
        {
            "patient_id": metadata.index,
            "acronym": metadata["acronym"].to_numpy(),
            "label_MTAP_deep": y,
            "outer_fold": folds,
            "prob_ElasticNet": p_en,
            "prob_DL": p_dl,
        }
    )
    patient_results.to_csv(
        OUT / "paired_development_OOF_predictions.tsv", sep="\t", index=False
    )
    cancer_records = []
    for cancer in cancer_names:
        indices = np.flatnonzero(metadata["acronym"].to_numpy() == cancer)
        truth = y[indices]
        for name, probability in model_probabilities.items():
            record = {
                "acronym": cancer,
                "model": name,
                "n_patients": len(indices),
                "n_positive": int(truth.sum()),
                "n_negative": int((truth == 0).sum()),
            }
            if len(np.unique(truth)) == 2:
                record.update(calculate_metrics(truth, probability[indices]))
            else:
                record["AUROC"] = np.nan
                record["AP"] = np.nan
            cancer_records.append(record)
    pd.DataFrame(cancer_records).to_csv(
        OUT / "per_cancer_comparison.tsv", sep="\t", index=False
    )
    rng = np.random.default_rng(SEED)
    positive_index = np.flatnonzero(y == 1)
    negative_index = np.flatnonzero(y == 0)
    metric_functions = {
        "AUROC": roc_auc_score,
        "AP": average_precision_score,
        "Brier_score": brier_score_loss,
    }
    bootstrap_values = {
        name: {metric: np.empty(N_BOOTSTRAPS) for metric in metric_functions}
        for name in model_probabilities
    }
    for iteration in range(N_BOOTSTRAPS):
        sampled_index = np.concatenate(
            [
                rng.choice(
                    positive_index, size=len(positive_index), replace=True
                ),
                rng.choice(
                    negative_index, size=len(negative_index), replace=True
                ),
            ]
        )
        truth = y[sampled_index]
        for name, probability in model_probabilities.items():
            sampled_probability = probability[sampled_index]
            for metric, function in metric_functions.items():
                bootstrap_values[name][metric][iteration] = function(
                    truth, sampled_probability
                )
        if (iteration + 1) % 500 == 0:
            pass
    interval_records = []
    difference_records = []
    dl_name = "Regional_multitask_DL_3seed"
    en_name = "Same_input_ElasticNet"
    for metric, function in metric_functions.items():
        for name, probability in model_probabilities.items():
            low, high = np.quantile(
                bootstrap_values[name][metric], [0.025, 0.975]
            )
            interval_records.append(
                {
                    "model": name,
                    "metric": metric,
                    "estimate": function(y, probability),
                    "CI95_low": low,
                    "CI95_high": high,
                    "interval_scope": "Conditional_on_fixed_OOF_predictions",
                }
            )
        differences = (
            bootstrap_values[dl_name][metric]
            - bootstrap_values[en_name][metric]
        )
        low, high = np.quantile(differences, [0.025, 0.975])
        difference_records.append(
            {
                "metric": metric,
                "difference_DL_minus_ElasticNet": function(y, p_dl)
                - function(y, p_en),
                "CI95_low": low,
                "CI95_high": high,
                "favorable_direction": (
                    "negative" if metric == "Brier_score" else "positive"
                ),
                "CI_excludes_zero": bool(low > 0 or high < 0),
                "interval_scope": "Conditional_on_fixed_OOF_predictions",
            }
        )
    intervals = pd.DataFrame(interval_records)
    differences = pd.DataFrame(difference_records)
    intervals.to_csv(
        OUT / "paired_bootstrap_metric_intervals.tsv", sep="\t", index=False
    )
    differences.to_csv(
        OUT / "paired_bootstrap_DL_minus_ElasticNet.tsv", sep="\t", index=False
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )
    colors = {en_name: "#33539E", dl_name: "#A5678E"}
    labels = {en_name: "Elastic Net", dl_name: "Regional multitask DL"}


#### Model and threshold locking for the regional branch


In [ ]:
@_nb9_registry.register("s27_0")
def _nb9_stage_s27_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from datetime import datetime, timezone
    from importlib import metadata as package_metadata
    import hashlib
    import json
    import numpy as np
    import pandas as pd
    import torch
    import joblib

    PROJECT = Path(str(PROJECT_ROOT))
    ROOT = PROJECT / "data_processed/MTAP_RNA_prediction"
    BASE = ROOT / "15_region_multitask_DL"
    INPUT = BASE / "01_training_inputs"
    DL_DIR = BASE / "02_training_v1"
    EN_DIR = BASE / "03_same_input_ElasticNet"
    OUT = BASE / "04_locked_models_and_thresholds"
    MODEL_DIR = OUT / "models"
    RECORD_DIR = OUT / "records"
    THRESHOLD_DIR = OUT / "thresholds"
    for directory in [OUT, MODEL_DIR, RECORD_DIR, THRESHOLD_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    SEEDS = [41, 42, 43]
    RULES = {
        "positive_PPV_target": 0.9,
        "positive_Wilson_lower_min": 0.85,
        "positive_min_n": 30,
        "negative_NPV_target": 0.98,
        "negative_Wilson_lower_min": 0.97,
        "negative_min_n": 100,
        "high_group_rule": "probability >= high_threshold",
        "low_group_rule": "probability <= low_threshold",
        "threshold_selection": "Maximize group coverage among eligible development OOF cutoffs; require low < high",
        "interval_interpretation": "Descriptive after threshold search; not an external performance guarantee",
    }
    SOURCES = {
        "development_metadata": INPUT / "development_metadata.tsv",
        "development_expression": INPUT / "X_development_signedlog.tsv",
        "prediction_targets": INPUT / "prediction_targets.tsv",
        "expression_mapping": INPUT / "expression_member_mapping.tsv",
        "DL_training_manifest": DL_DIR / "training_manifest.json",
        "DL_OOF": DL_DIR / "development_MTAP_OOF_predictions.tsv",
        "paired_OOF": EN_DIR / "paired_development_OOF_predictions.tsv",
        "EN_model": EN_DIR
        / "models"
        / "same_input_ElasticNet_development_candidate.joblib",
    }
    for seed in SEEDS:
        SOURCES[f"DL_seed_{seed}"] = (
            DL_DIR / "models" / f"development_candidate_seed{seed}.pt"
        )
    for name, path in SOURCES.items():
        if not path.is_file():
            raise FileNotFoundError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )

    def sha256_file(path):
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def write_once(path, content):
        if isinstance(content, str):
            content = content.encode("utf-8")
        if path.exists():
            if path.read_bytes() != content:
                raise RuntimeError(
                    "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
                )
        else:
            path.write_bytes(content)

    def copy_once(source, destination):
        write_once(destination, source.read_bytes())

    def save_table_once(table, path, index=False):
        write_once(path, table.to_csv(sep="\t", index=index))

    source_hashes = {name: sha256_file(path) for name, path in SOURCES.items()}
    manifest_file = OUT / "locked_manifest.json"
    old_manifest = None
    if manifest_file.exists():
        old_manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        if old_manifest["source_hashes"] != source_hashes:
            raise RuntimeError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
        if old_manifest["threshold_rules"] != RULES:
            raise RuntimeError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
    metadata = pd.read_csv(
        SOURCES["development_metadata"], sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    paired = pd.read_csv(
        SOURCES["paired_OOF"], sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    dl_oof = pd.read_csv(
        SOURCES["DL_OOF"], sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    for name, table in [
        ("metadata", metadata),
        ("paired OOF", paired),
        ("DL OOF", dl_oof),
    ]:
        if table.index.duplicated().any():
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
    if set(metadata.index) != set(paired.index) or set(metadata.index) != set(
        dl_oof.index
    ):
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    if not metadata["dataset_split"].eq("development").all():
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    paired = paired.reindex(metadata.index)
    dl_oof = dl_oof.reindex(metadata.index)
    y_series = pd.to_numeric(metadata["label_MTAP_deep"], errors="raise")
    if y_series.isna().any() or not y_series.isin([0, 1]).all():
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    y = y_series.to_numpy(dtype=int)
    if len(y) != 7431 or y.sum() != 558:
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    for name, table in [("paired OOF", paired), ("DL OOF", dl_oof)]:
        labels = pd.to_numeric(
            table["label_MTAP_deep"], errors="raise"
        ).to_numpy()
        if not np.array_equal(y, labels):
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
        if not np.array_equal(
            metadata["acronym"].to_numpy(), table["acronym"].to_numpy()
        ):
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
    paired_folds = pd.to_numeric(
        paired["outer_fold"], errors="raise"
    ).to_numpy()
    dl_folds = pd.to_numeric(dl_oof["outer_fold"], errors="raise").to_numpy()
    if (
        not np.isfinite(paired_folds).all()
        or not np.array_equal(paired_folds, dl_folds)
        or len(np.unique(paired_folds)) != 5
    ):
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    probabilities = {
        "Regional_multitask_DL_3seed": pd.to_numeric(
            paired["prob_DL"], errors="raise"
        ).to_numpy(dtype=float),
        "Same_input_ElasticNet": pd.to_numeric(
            paired["prob_ElasticNet"], errors="raise"
        ).to_numpy(dtype=float),
    }
    for name, probability in probabilities.items():
        if (
            not np.isfinite(probability).all()
            or ((probability < 0) | (probability > 1)).any()
        ):
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
    if not np.allclose(
        probabilities["Regional_multitask_DL_3seed"],
        pd.to_numeric(dl_oof["prob_MTAP_deep"], errors="raise").to_numpy(),
        rtol=0,
        atol=1e-10,
    ):
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    expression_header = pd.read_csv(
        SOURCES["development_expression"], sep="\t", nrows=0
    ).columns.tolist()
    feature_names = expression_header[1:]
    if len(feature_names) != 45:
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    targets = pd.read_csv(
        SOURCES["prediction_targets"], sep="\t", dtype={"Entrez_ID": str}
    ).sort_values("target_index")
    target_ids = targets["Entrez_ID"].tolist()
    checkpoints = []
    for seed in SEEDS:
        checkpoint = torch.load(
            SOURCES[f"DL_seed_{seed}"], map_location="cpu", weights_only=True
        )
        if checkpoint["feature_names"] != feature_names:
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
        if checkpoint["target_Entrez_order"] != target_ids:
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
        mtap_index = checkpoint["MTAP_INDEX"]
        if target_ids[mtap_index] != "4507":
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
        for field in ["median", "mean", "std"]:
            values = np.asarray(
                checkpoint["preprocessing"][field], dtype=float
            )
            if values.shape != (45,) or not np.isfinite(values).all():
                raise ValueError(
                    "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
                )
        if (np.asarray(checkpoint["preprocessing"]["std"]) <= 0).any():
            raise ValueError(
                "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
            )
        if checkpoints:
            reference = checkpoints[0]
            for field in [
                "feature_names",
                "target_Entrez_order",
                "MTAP_INDEX",
                "region_indices",
                "core_indices",
                "cancer_to_index",
                "architecture",
                "preprocessing",
            ]:
                if checkpoint[field] != reference[field]:
                    raise ValueError(
                        "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
                    )
        checkpoints.append(checkpoint)
        copy_once(SOURCES[f"DL_seed_{seed}"], MODEL_DIR / f"DL_seed{seed}.pt")
    network_source = 'import torch\nfrom torch import nn\n\n\nclass RegionMultitaskNet(nn.Module):\n    def __init__(\n        self,\n        n_targets,\n        region_indices,\n        core_indices,\n        cancer_to_index,\n        architecture,\n    ):\n        super().__init__()\n\n        self.region_indices = region_indices\n        self.core_indices = core_indices\n        self.n_targets = n_targets\n\n        region_hidden = architecture["region_hidden"]\n        shared_hidden = architecture["shared_hidden"]\n        embedding_size = architecture["cancer_embedding"]\n        dropout = architecture["dropout"]\n\n        self.region_encoders = nn.ModuleList([\n            nn.Sequential(\n                nn.Linear(len(indices), region_hidden),\n                nn.ReLU(),\n                nn.Dropout(dropout),\n            )\n            for indices in region_indices\n        ])\n\n        n_cancers = max(cancer_to_index.values()) + 1\n\n        self.cancer_embedding = nn.Embedding(\n            n_cancers, embedding_size\n        )\n\n        combined_dimension = (\n            len(region_indices) * region_hidden\n            + len(core_indices)\n            + embedding_size\n        )\n\n        self.shared = nn.Sequential(\n            nn.Linear(combined_dimension, shared_hidden),\n            nn.ReLU(),\n            nn.Dropout(dropout),\n        )\n\n        self.output = nn.Linear(\n            shared_hidden, n_targets * 3\n        )\n\n    def forward(self, expression, cancer):\n        regional = [\n            encoder(expression[:, indices])\n            for encoder, indices in zip(\n                self.region_encoders,\n                self.region_indices,\n            )\n        ]\n\n        combined = torch.cat(\n            regional + [\n                expression[:, self.core_indices],\n                self.cancer_embedding(cancer),\n            ],\n            dim=1,\n        )\n\n        return self.output(\n            self.shared(combined)\n        ).reshape(-1, self.n_targets, 3)\n\n\ndef model_from_checkpoint(checkpoint):\n    model = RegionMultitaskNet(\n        n_targets=len(checkpoint["target_Entrez_order"]),\n        region_indices=checkpoint["region_indices"],\n        core_indices=checkpoint["core_indices"],\n        cancer_to_index=checkpoint["cancer_to_index"],\n        architecture=checkpoint["architecture"],\n    )\n\n    model.load_state_dict(\n        checkpoint["state_dict"], strict=True\n    )\n\n    model.eval()\n    return model\n'
    write_once(OUT / "region_multitask_network.py", network_source)
    namespace = {}
    exec(
        compile(network_source, "region_multitask_network.py", "exec"),
        namespace,
    )
    for seed, checkpoint in zip(SEEDS, checkpoints):
        model = namespace["model_from_checkpoint"](checkpoint)
        with torch.no_grad():
            logits = model(torch.zeros(2, 45), torch.ones(2, dtype=torch.long))
            expected_shape = (2, len(target_ids), 3)
            if tuple(logits.shape) != expected_shape:
                raise ValueError(
                    "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
                )
            if not torch.isfinite(logits).all():
                raise ValueError(
                    "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
                )
        del model
    en_bundle = joblib.load(SOURCES["EN_model"])
    if en_bundle["expression_feature_names"] != feature_names:
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    if en_bundle["input_columns"] != feature_names + ["acronym"]:
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        en_bundle["pipeline"].named_steps["model"].classes_, np.array([0, 1])
    ):
        raise ValueError(
            "Model and threshold locking for the regional branch: integrity check failed; condition: input or artifact integrity check."
        )
    copy_once(SOURCES["EN_model"], MODEL_DIR / "ElasticNet_locked.joblib")

    def wilson_interval(successes, totals):
        successes = np.asarray(successes, dtype=float)
        totals = np.asarray(totals, dtype=float)
        z = 1.959963984540054
        proportion = successes / totals
        denominator = 1 + z**2 / totals
        center = (proportion + z**2 / (2 * totals)) / denominator
        half_width = (
            z
            * np.sqrt(
                proportion * (1 - proportion) / totals + z**2 / (4 * totals**2)
            )
            / denominator
        )
        return (center - half_width, center + half_width)

    def make_threshold_table(truth, probability):
        grouped = (
            pd.DataFrame({"probability": probability, "positive": truth})
            .groupby("probability", sort=True)
            .agg(n=("positive", "size"), n_positive=("positive", "sum"))
            .reset_index()
        )
        n = grouped["n"].to_numpy()
        positive = grouped["n_positive"].to_numpy()
        n_high = np.cumsum(n[::-1])[::-1]
        tp_high = np.cumsum(positive[::-1])[::-1]
        n_low = np.cumsum(n)
        fn_low = np.cumsum(positive)
        tn_low = n_low - fn_low
        ppv = tp_high / n_high
        npv = tn_low / n_low
        ppv_lower, ppv_upper = wilson_interval(tp_high, n_high)
        npv_lower, npv_upper = wilson_interval(tn_low, n_low)
        return pd.DataFrame(
            {
                "threshold": grouped["probability"],
                "n_high": n_high,
                "TP_high": tp_high,
                "FP_high": n_high - tp_high,
                "PPV_high": ppv,
                "PPV_Wilson_low": ppv_lower,
                "PPV_Wilson_high": ppv_upper,
                "sensitivity_high": tp_high / truth.sum(),
                "n_low": n_low,
                "TN_low": tn_low,
                "FN_low": fn_low,
                "NPV_low": npv,
                "NPV_Wilson_low": npv_lower,
                "NPV_Wilson_high": npv_upper,
                "fraction_all_positives_in_low": fn_low / truth.sum(),
            }
        )

    def select_thresholds(table):
        eligible_high = table.loc[
            table["n_high"].ge(RULES["positive_min_n"])
            & table["PPV_high"].ge(RULES["positive_PPV_target"])
            & table["PPV_Wilson_low"].ge(RULES["positive_Wilson_lower_min"])
        ]
        if eligible_high.empty:
            high = None
        else:
            high = float(eligible_high.iloc[0]["threshold"])
        eligible_low = table.loc[
            table["n_low"].ge(RULES["negative_min_n"])
            & table["NPV_low"].ge(RULES["negative_NPV_target"])
            & table["NPV_Wilson_low"].ge(RULES["negative_Wilson_lower_min"])
        ]
        if high is not None:
            eligible_low = eligible_low.loc[eligible_low["threshold"].lt(high)]
        if eligible_low.empty:
            low = None
        else:
            low = float(eligible_low.iloc[-1]["threshold"])
        return (low, high)

    threshold_records = []
    group_records = []
    locked_thresholds = {}
    patient_groups = pd.DataFrame(
        {
            "patient_id": metadata.index,
            "acronym": metadata["acronym"].to_numpy(),
            "label_MTAP_deep": y,
            "outer_fold": paired_folds.astype(int),
        }
    )
    short_names = {
        "Regional_multitask_DL_3seed": "DL",
        "Same_input_ElasticNet": "ElasticNet",
    }
    for name, probability in probabilities.items():
        table = make_threshold_table(y, probability)
        low, high = select_thresholds(table)
        short_name = short_names[name]
        save_table_once(
            table, THRESHOLD_DIR / f"{short_name}_candidate_thresholds.tsv"
        )
        locked_thresholds[name] = {
            "low_threshold": low,
            "high_threshold": high,
            "positive_rule_met": high is not None,
            "negative_rule_met": low is not None,
        }
        groups = np.full(len(y), "Uncertain", dtype=object)
        if low is not None:
            groups[probability <= low] = "Predicted_non_deep"
        if high is not None:
            groups[probability >= high] = "Predicted_deep_deletion"
        patient_groups[f"{short_name}_probability"] = probability
        patient_groups[f"{short_name}_group"] = groups
        threshold_records.append({"model": name, **locked_thresholds[name]})
        for group_name in [
            "Predicted_deep_deletion",
            "Uncertain",
            "Predicted_non_deep",
        ]:
            mask = groups == group_name
            n_group = int(mask.sum())
            n_positive = int(y[mask].sum())
            n_negative = n_group - n_positive
            if group_name == "Predicted_non_deep":
                value_name = "NPV"
                successes = n_negative
            elif group_name == "Predicted_deep_deletion":
                value_name = "PPV"
                successes = n_positive
            else:
                value_name = "DNA_positive_fraction"
                successes = n_positive
            if n_group:
                value = successes / n_group
                lower, upper = wilson_interval(successes, n_group)
                lower = float(lower)
                upper = float(upper)
            else:
                value = np.nan
                lower = np.nan
                upper = np.nan
            group_records.append(
                {
                    "model": name,
                    "group": group_name,
                    "n_patients": n_group,
                    "coverage_fraction": n_group / len(y),
                    "n_DNA_positive": n_positive,
                    "n_DNA_negative": n_negative,
                    "fraction_all_DNA_positives": n_positive / y.sum(),
                    "value_name": value_name,
                    "value": value,
                    "Wilson_CI95_low": lower,
                    "Wilson_CI95_high": upper,
                    "evaluation_scope": "Development_OOF_after_threshold_search",
                }
            )
    threshold_summary = pd.DataFrame(threshold_records)
    group_summary = pd.DataFrame(group_records)
    save_table_once(
        threshold_summary, THRESHOLD_DIR / "locked_threshold_summary.tsv"
    )
    save_table_once(
        group_summary, THRESHOLD_DIR / "development_group_summary.tsv"
    )
    save_table_once(
        patient_groups, THRESHOLD_DIR / "development_patient_groups.tsv"
    )
    write_once(
        THRESHOLD_DIR / "locked_thresholds.json",
        json.dumps(locked_thresholds, ensure_ascii=False, indent=2),
    )
    copy_once(
        SOURCES["prediction_targets"], RECORD_DIR / "prediction_targets.tsv"
    )
    copy_once(
        SOURCES["expression_mapping"],
        RECORD_DIR / "expression_member_mapping.tsv",
    )
    copy_once(
        SOURCES["development_metadata"],
        RECORD_DIR / "development_metadata.tsv",
    )
    copy_once(
        SOURCES["DL_training_manifest"],
        RECORD_DIR / "DL_training_manifest.json",
    )
    copy_once(
        SOURCES["paired_OOF"],
        RECORD_DIR / "paired_development_OOF_predictions.tsv",
    )
    feature_specification = {
        "expression_feature_order": feature_names,
        "expression_transform": "sign(x)*log2(1+abs(x))",
        "expression_input_scale": "Training EB++ expression scale; external expression compatibility must be checked",
        "DL_ensemble_rule": "Arithmetic mean of the three seed probabilities",
        "DL_target_class_order": {
            "0": "GISTIC -2: deep deletion",
            "1": "GISTIC -1: shallow deletion",
            "2": "GISTIC 0/+1/+2: no deletion",
        },
        "group_interpretation": {
            "Predicted_deep_deletion": "RNA-predicted MTAP deep deletion",
            "Predicted_non_deep": "RNA-predicted absence of MTAP deep deletion; not equivalent to copy-number neutral",
            "Uncertain": "No high-confidence binary assignment",
        },
    }
    write_once(
        RECORD_DIR / "feature_and_prediction_specification.json",
        json.dumps(feature_specification, ensure_ascii=False, indent=2),
    )
    versions = {}
    for package in ["numpy", "pandas", "torch", "scikit-learn", "joblib"]:
        try:
            versions[package] = package_metadata.version(package)
        except package_metadata.PackageNotFoundError:
            versions[package] = "not installed"
    locked_hashes = {
        str(path.relative_to(OUT)): sha256_file(path)
        for path in sorted(OUT.rglob("*"))
        if path.is_file() and path != manifest_file
    }
    manifest = {
        "created_UTC": (
            old_manifest["created_UTC"]
            if old_manifest is not None
            else datetime.now(timezone.utc).isoformat()
        ),
        "status": "Frozen_candidates_for_external_validation",
        "primary_candidate": "Regional_multitask_DL_3seed",
        "comparator": "Same_input_ElasticNet",
        "DL_seeds": SEEDS,
        "n_development_patients": int(len(y)),
        "n_MTAP_positive": int(y.sum()),
        "source_paths": {name: str(path) for name, path in SOURCES.items()},
        "source_hashes": source_hashes,
        "threshold_rules": RULES,
        "locked_thresholds": locked_thresholds,
        "locked_file_hashes": locked_hashes,
        "package_versions": versions,
        "test_used": False,
        "CPTAC_used": False,
        "models_retrained": False,
        "limitations": [
            "Thresholds selected using development OOF predictions.",
            "Post-search Wilson intervals are descriptive.",
            "External PPV/NPV targets are not guaranteed.",
            "Predicted non-deep is not predicted copy-number neutral.",
            "Bulk thresholds are not validated single-cell thresholds.",
        ],
    }
    write_once(
        manifest_file, json.dumps(manifest, ensure_ascii=False, indent=2)
    )
    for name, thresholds in locked_thresholds.items():
        if not thresholds["positive_rule_met"]:
            pass
        if not thresholds["negative_rule_met"]:
            pass
    _public_summary(
        globals(), "Model and threshold locking for the regional branch"
    )


### Model 3: Deep learning


#### Independent retraining of the 44-member TPM DL branch


In [ ]:
@_nb9_registry.register("s34_0")
def _nb9_stage_s34_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import copy
    import json
    import random
    import time
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
    )

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL"
        )
    )
    INPUT = BASE / "08_TPM_retraining/01_expression_inputs_44"
    OUT = BASE / "08_TPM_retraining/02_DL_training"
    OUT.mkdir(parents=True, exist_ok=True)
    MODEL_DIR = OUT / "models"
    MODEL_DIR.mkdir(exist_ok=True)
    X_FILE = INPUT / "X_development_log2TPM_plus_1.tsv"
    META_FILE = INPUT / "development_metadata.tsv"
    SPEC_FILE = INPUT / "TPM_44_feature_specification.json"
    Y_FILE = BASE / "01_training_inputs/Y_development_three_class.tsv"
    TARGET_FILE = BASE / "01_training_inputs/prediction_targets.tsv"
    OLD_OOF_FILE = BASE / "02_training_v1/development_MTAP_OOF_predictions.tsv"
    OLD_CHECKPOINT = (
        BASE / "04_locked_models_and_thresholds/models/DL_seed41.pt"
    )
    SEEDS = [41, 42, 43]
    BATCH_SIZE = 256
    MAX_EPOCHS = 150
    PATIENCE = 15
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.01
    AUXILIARY_WEIGHT = 0.3
    INNER_VALID_FRACTION = 0.15
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    for path in [
        X_FILE,
        META_FILE,
        SPEC_FILE,
        Y_FILE,
        TARGET_FILE,
        OLD_OOF_FILE,
        OLD_CHECKPOINT,
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)
    FINAL_PERFORMANCE = OUT / "development_MTAP_OOF_performance.tsv"
    if FINAL_PERFORMANCE.exists():
        raise FileExistsError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    with SPEC_FILE.open("r", encoding="utf-8") as handle:
        input_spec = json.load(handle)
    features = input_spec["expression_feature_order"]
    if len(features) != 44 or len(set(features)) != 44:
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    X_df = pd.read_csv(X_FILE, sep="\t", index_col=0)
    X_df.index = X_df.index.astype(str)
    if X_df.index.duplicated().any():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    X_df = X_df.loc[:, features].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(X_df.to_numpy()).all():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    meta = pd.read_csv(
        META_FILE, sep="\t", dtype={"patient_id": str, "acronym": str}
    )
    if meta["patient_id"].duplicated().any():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.set_index("patient_id").loc[X_df.index].copy()
    patient_ids = X_df.index.to_numpy()
    X_raw = X_df.to_numpy(dtype=np.float32)
    binary_labels = pd.to_numeric(
        meta["label_MTAP_deep"], errors="raise"
    ).to_numpy(dtype=int)
    old_oof = pd.read_csv(OLD_OOF_FILE, sep="\t", dtype={"patient_id": str})
    if old_oof["patient_id"].duplicated().any():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    fold_map = old_oof.set_index("patient_id")["outer_fold"]
    folds_series = pd.Series(patient_ids, index=patient_ids).map(fold_map)
    if folds_series.isna().any():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    folds = folds_series.to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    targets = pd.read_csv(TARGET_FILE, sep="\t", dtype=str)
    targets["target_index"] = pd.to_numeric(
        targets["target_index"], errors="raise"
    ).astype(int)
    targets = targets.sort_values("target_index")
    target_ids = targets["Entrez_ID"].tolist()
    target_symbols = targets["DNA_symbol"].tolist()
    if len(target_ids) != 35:
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    MTAP_INDEX = target_ids.index("4507")
    Y_df = pd.read_csv(Y_FILE, sep="\t", index_col=0)
    Y_df.index = Y_df.index.astype(str)
    Y_df.columns = Y_df.columns.astype(str)
    if Y_df.index.duplicated().any():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    Y_df = Y_df.loc[patient_ids, target_ids]
    if Y_df.isna().any().any():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    Y = Y_df.to_numpy(dtype=np.int64)
    if not np.isin(Y, [0, 1, 2]).all():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal((Y[:, MTAP_INDEX] == 0).astype(int), binary_labels):
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    old_checkpoint = torch.load(
        OLD_CHECKPOINT, map_location="cpu", weights_only=False
    )
    old_features = old_checkpoint["feature_names"]
    old_region_indices = old_checkpoint["region_indices"]
    architecture = old_checkpoint["architecture"]
    cancer_to_index = old_checkpoint["cancer_to_index"]
    new_position = {gene: position for position, gene in enumerate(features)}
    region_indices = []
    for old_indices in old_region_indices:
        old_region_genes = [old_features[position] for position in old_indices]
        new_indices = [
            new_position[gene]
            for gene in old_region_genes
            if gene in new_position
        ]
        if not new_indices:
            raise ValueError(
                "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
            )
        region_indices.append(new_indices)
    covered_positions = [
        position for region in region_indices for position in region
    ]
    if len(covered_positions) != len(features) or set(
        covered_positions
    ) != set(range(len(features))):
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    core_indices = [
        features.index(gene) for gene in ["MTAP", "CDKN2A", "CDKN2B"]
    ]
    unknown_cancers = set(meta["acronym"]) - set(cancer_to_index)
    if unknown_cancers:
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    cancer_codes = (
        meta["acronym"].map(cancer_to_index).to_numpy(dtype=np.int64)
    )

    class RegionalMultiTaskNet(nn.Module):

        def __init__(self):
            super().__init__()
            region_hidden = int(architecture["region_hidden"])
            shared_hidden = int(architecture["shared_hidden"])
            embedding_dim = int(architecture["cancer_embedding"])
            dropout = float(architecture["dropout"])
            self.region_encoders = nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Linear(len(indices), region_hidden),
                        nn.ReLU(),
                        nn.Dropout(dropout),
                    )
                    for indices in region_indices
                ]
            )
            n_cancer_embeddings = max(cancer_to_index.values()) + 1
            self.cancer_embedding = nn.Embedding(
                n_cancer_embeddings, embedding_dim
            )
            shared_input = (
                len(region_indices) * region_hidden
                + len(core_indices)
                + embedding_dim
            )
            self.shared = nn.Sequential(
                nn.Linear(shared_input, shared_hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(shared_hidden, len(target_ids) * 3),
            )

        def forward(self, expression, cancer):
            regional = [
                encoder(expression[:, indices])
                for encoder, indices in zip(
                    self.region_encoders, region_indices
                )
            ]
            combined = torch.cat(
                regional
                + [expression[:, core_indices], self.cancer_embedding(cancer)],
                dim=1,
            )
            return self.shared(combined).reshape(-1, len(target_ids), 3)

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False

    def fit_preprocessing(values):
        mean = values.mean(axis=0)
        std = values.std(axis=0, ddof=0)
        std = np.where(std < 1e-08, 1.0, std)
        return {"mean": mean.astype(np.float32), "std": std.astype(np.float32)}

    def transform(values, preprocessing):
        return (
            (values - preprocessing["mean"]) / preprocessing["std"]
        ).astype(np.float32)

    def make_loader(indices, preprocessing, seed):
        generator = torch.Generator()
        generator.manual_seed(seed)
        dataset = TensorDataset(
            torch.from_numpy(transform(X_raw[indices], preprocessing)),
            torch.from_numpy(cancer_codes[indices]),
            torch.from_numpy(Y[indices]),
        )
        return DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            generator=generator,
        )

    def train_one_epoch(model, loader, optimizer):
        model.train()
        total_loss = 0.0
        total_patients = 0
        for expression, cancer, target in loader:
            expression = expression.to(DEVICE)
            cancer = cancer.to(DEVICE)
            target = target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(expression, cancer)
            mtap_logits = logits[:, MTAP_INDEX, :]
            binary_logit = mtap_logits[:, 0] - torch.logsumexp(
                mtap_logits[:, 1:], dim=1
            )
            binary_target = (target[:, MTAP_INDEX] == 0).float()
            main_loss = F.binary_cross_entropy_with_logits(
                binary_logit, binary_target
            )
            auxiliary_loss = F.cross_entropy(
                logits.reshape(-1, 3), target.reshape(-1)
            )
            loss = main_loss + AUXILIARY_WEIGHT * auxiliary_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            batch_n = len(expression)
            total_loss += float(loss.item()) * batch_n
            total_patients += batch_n
        return total_loss / total_patients

    @torch.no_grad()
    def predict(model, indices, preprocessing):
        model.eval()
        results = []
        for start in range(0, len(indices), 1024):
            current = indices[start : start + 1024]
            expression = torch.from_numpy(
                transform(X_raw[current], preprocessing)
            ).to(DEVICE)
            cancer = torch.from_numpy(cancer_codes[current]).to(DEVICE)
            probability = torch.softmax(model(expression, cancer), dim=-1)
            results.append(probability.cpu().numpy())
        return np.concatenate(results, axis=0)

    def new_optimizer(model):
        return torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )

    def save_checkpoint(model, preprocessing, path, extra):
        torch.save(
            {
                "state_dict": {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                },
                "feature_names": features,
                "target_Entrez_order": target_ids,
                "target_symbols": target_symbols,
                "MTAP_INDEX": MTAP_INDEX,
                "region_indices": region_indices,
                "core_indices": core_indices,
                "cancer_to_index": cancer_to_index,
                "preprocessing": {
                    key: value.tolist() for key, value in preprocessing.items()
                },
                "architecture": architecture,
                "expression_unit": "log2(TPM+1)",
                "network_class": "RegionalMultiTaskNet",
                "extra": extra,
            },
            path,
        )

    start_time = time.time()
    n_patients = len(patient_ids)
    n_targets = len(target_ids)
    seed_oof = np.full(
        (len(SEEDS), n_patients, n_targets, 3), np.nan, dtype=np.float32
    )
    fold_seed_records = []
    for fold in [1, 2, 3, 4, 5]:
        outer_train = np.flatnonzero(folds != fold)
        outer_valid = np.flatnonzero(folds == fold)
        inner_train, inner_valid = train_test_split(
            outer_train,
            test_size=INNER_VALID_FRACTION,
            random_state=2026 + fold,
            stratify=binary_labels[outer_train],
        )
        inner_preprocessing = fit_preprocessing(X_raw[inner_train])
        outer_preprocessing = fit_preprocessing(X_raw[outer_train])
        for seed_position, seed in enumerate(SEEDS):
            set_seed(seed)
            model = RegionalMultiTaskNet().to(DEVICE)
            optimizer = new_optimizer(model)
            loader = make_loader(inner_train, inner_preprocessing, seed)
            best_ap = -np.inf
            best_epoch = 1
            epochs_without_improvement = 0
            for epoch in range(1, MAX_EPOCHS + 1):
                loss = train_one_epoch(model, loader, optimizer)
                inner_probability = predict(
                    model, inner_valid, inner_preprocessing
                )[:, MTAP_INDEX, 0]
                inner_ap = average_precision_score(
                    binary_labels[inner_valid], inner_probability
                )
                if inner_ap > best_ap + 1e-06:
                    best_ap = inner_ap
                    best_epoch = epoch
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1
                if epoch == 1 or epoch % 20 == 0:
                    pass
                if epochs_without_improvement >= PATIENCE:
                    break
            del model, optimizer
            set_seed(seed)
            model = RegionalMultiTaskNet().to(DEVICE)
            optimizer = new_optimizer(model)
            loader = make_loader(outer_train, outer_preprocessing, seed)
            for epoch in range(1, best_epoch + 1):
                train_one_epoch(model, loader, optimizer)
            outer_probability = predict(
                model, outer_valid, outer_preprocessing
            )
            seed_oof[seed_position, outer_valid] = outer_probability
            mtap_probability = outer_probability[:, MTAP_INDEX, 0]
            fold_seed_records.append(
                {
                    "outer_fold": fold,
                    "seed": seed,
                    "n_train": len(outer_train),
                    "n_valid": len(outer_valid),
                    "n_valid_positive": int(binary_labels[outer_valid].sum()),
                    "selected_epochs": best_epoch,
                    "inner_best_AP": best_ap,
                    "outer_AUROC": roc_auc_score(
                        binary_labels[outer_valid], mtap_probability
                    ),
                    "outer_AP": average_precision_score(
                        binary_labels[outer_valid], mtap_probability
                    ),
                }
            )
            save_checkpoint(
                model,
                outer_preprocessing,
                MODEL_DIR / f"fold{fold}_seed{seed}.pt",
                {
                    "outer_fold": fold,
                    "seed": seed,
                    "epochs": best_epoch,
                    "status": "outer_fold_model",
                },
            )
            pd.DataFrame(fold_seed_records).to_csv(
                OUT / "outer_fold_seed_results.tsv", sep="\t", index=False
            )
            del model, optimizer
        fold_probability = seed_oof[:, outer_valid, MTAP_INDEX, 0].mean(axis=0)
    if not np.isfinite(seed_oof).all():
        raise ValueError(
            "Independent retraining of the 44-member TPM DL branch: integrity check failed; condition: input or artifact integrity check."
        )
    mean_oof = seed_oof.mean(axis=0)
    mtap_oof = mean_oof[:, MTAP_INDEX, 0]
    oof_table = pd.DataFrame(
        {
            "patient_id": patient_ids,
            "acronym": meta["acronym"].to_numpy(),
            "label_MTAP_deep": binary_labels,
            "outer_fold": folds,
            "prob_MTAP_deep": mtap_oof,
        }
    )
    for seed_position, seed in enumerate(SEEDS):
        oof_table[f"prob_seed_{seed}"] = seed_oof[
            seed_position, :, MTAP_INDEX, 0
        ]
    oof_table.to_csv(
        OUT / "development_MTAP_OOF_predictions.tsv", sep="\t", index=False
    )
    np.savez_compressed(
        OUT / "development_all_target_OOF.npz",
        probabilities=mean_oof,
        patient_id=patient_ids.astype(str),
        target_Entrez=np.asarray(target_ids, dtype=str),
        target_symbols=np.asarray(target_symbols, dtype=str),
    )

    def performance(y, probability):
        probability = np.clip(probability.astype(float), 1e-07, 1 - 1e-07)
        predicted = probability >= 0.5
        tp = int(((y == 1) & predicted).sum())
        fp = int(((y == 0) & predicted).sum())
        fn = int(((y == 1) & ~predicted).sum())
        tn = int(((y == 0) & ~predicted).sum())
        return {
            "n_patients": len(y),
            "n_positive": int(y.sum()),
            "AUROC": roc_auc_score(y, probability),
            "AP": average_precision_score(y, probability),
            "Brier_score": brier_score_loss(y, probability),
            "log_loss": log_loss(y, probability),
            "precision_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "sensitivity_at_05": tp / (tp + fn),
            "specificity_at_05": tn / (tn + fp),
            "TP_at_05": tp,
            "FP_at_05": fp,
            "FN_at_05": fn,
            "TN_at_05": tn,
        }

    result = performance(binary_labels, mtap_oof)
    result["model"] = "TPM44_regional_multitask_DL_3seed"
    performance_table = pd.DataFrame([result])
    seed_results = []
    for position, seed in enumerate(SEEDS):
        row = performance(binary_labels, seed_oof[position, :, MTAP_INDEX, 0])
        row["seed"] = seed
        seed_results.append(row)
    seed_table = pd.DataFrame(seed_results)
    seed_table.to_csv(OUT / "seed_stability.tsv", sep="\t", index=False)
    fold_seed_table = pd.DataFrame(fold_seed_records)
    final_epochs = max(1, int(np.median(fold_seed_table["selected_epochs"])))
    all_indices = np.arange(n_patients)
    full_preprocessing = fit_preprocessing(X_raw)
    for seed in SEEDS:
        set_seed(seed)
        model = RegionalMultiTaskNet().to(DEVICE)
        optimizer = new_optimizer(model)
        loader = make_loader(all_indices, full_preprocessing, seed)
        for epoch in range(1, final_epochs + 1):
            loss = train_one_epoch(model, loader, optimizer)
            if epoch == 1 or epoch % 20 == 0:
                pass
        save_checkpoint(
            model,
            full_preprocessing,
            MODEL_DIR / f"development_candidate_seed{seed}.pt",
            {
                "seed": seed,
                "epochs": final_epochs,
                "status": "development_candidate",
            },
        )
        del model, optimizer
    manifest = {
        "version": "TPM44_regional_multitask_DL",
        "expression_unit": "log2(TPM+1)",
        "n_development_patients": n_patients,
        "n_MTAP_positive": int(binary_labels.sum()),
        "feature_names": features,
        "target_Entrez_order": target_ids,
        "target_symbols": target_symbols,
        "region_indices": region_indices,
        "core_indices": core_indices,
        "cancer_to_index": cancer_to_index,
        "architecture": architecture,
        "seeds": SEEDS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "auxiliary_weight": AUXILIARY_WEIGHT,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "final_epochs": final_epochs,
        "outer_folds": "Inherited original patient assignments",
        "old_weights_reused": False,
        "old_preprocessing_reused": False,
        "old_thresholds_reused": False,
        "CPTAC_evaluated": False,
        "test_evaluated": False,
    }
    with (OUT / "training_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    performance_table.to_csv(FINAL_PERFORMANCE, sep="\t", index=False)
    _public_summary(
        globals(), "Independent retraining of the 44-member TPM DL branch"
    )


#### Regional multitask DL training and seed analysis


In [ ]:
@_nb9_registry.register("s24_0")
def _nb9_stage_s24_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import copy
    import json
    import random
    import time
    import numpy as np
    import pandas as pd
    import torch
    from torch import nn
    from torch.nn import functional as F
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        precision_score,
        recall_score,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
    )
    import matplotlib.pyplot as plt

    PROJECT = Path(str(PROJECT_ROOT))
    ROOT = PROJECT / "data_processed/MTAP_RNA_prediction"
    INPUT = ROOT / "15_region_multitask_DL/01_training_inputs"
    OUT = ROOT / "15_region_multitask_DL/02_training_v1"
    FOLD_FILE = ROOT / "03_baseline_models/development_OOF_predictions.tsv"
    FILES = {
        "X": INPUT / "X_development_signedlog.tsv",
        "Y": INPUT / "Y_development_three_class.tsv",
        "metadata": INPUT / "development_metadata.tsv",
        "targets": INPUT / "prediction_targets.tsv",
        "members": INPUT / "expression_member_mapping.tsv",
    }
    for path in [*FILES.values(), FOLD_FILE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
    MODEL_DIR = OUT / "models"
    FIGURE_DIR = OUT / "figures"
    for directory in [OUT, MODEL_DIR, FIGURE_DIR]:
        directory.mkdir(parents=True, exist_ok=True)
    SEEDS = [41, 42, 43]
    BATCH_SIZE = 256
    MAX_EPOCHS = 150
    PATIENCE = 15
    MIN_AP_IMPROVEMENT = 0.0001
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 0.01
    DROPOUT = 0.15
    REGION_HIDDEN = 16
    SHARED_HIDDEN = 64
    CANCER_EMBEDDING = 8
    AUXILIARY_WEIGHT = 0.3
    INNER_VALID_FRACTION = 0.15
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    torch.set_num_threads(4)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    if DEVICE.type == "cuda":
        pass

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    metadata = pd.read_csv(
        FILES["metadata"], sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    X_df = pd.read_csv(FILES["X"], sep="\t", index_col=0)
    Y_df = pd.read_csv(FILES["Y"], sep="\t", index_col=0)
    targets = pd.read_csv(FILES["targets"], sep="\t", dtype={"Entrez_ID": str})
    members = pd.read_csv(
        FILES["members"], sep="\t", dtype=str, keep_default_na=False
    )
    X_df.index = X_df.index.astype(str)
    Y_df.index = Y_df.index.astype(str)
    Y_df.columns = Y_df.columns.astype(str)
    for name, table in [("metadata", metadata), ("X", X_df), ("Y", Y_df)]:
        if table.index.duplicated().any():
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
    if set(X_df.index) != set(metadata.index) or set(Y_df.index) != set(
        metadata.index
    ):
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    if not metadata["dataset_split"].eq("development").all():
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    targets = targets.sort_values("target_index").reset_index(drop=True)
    expected_indices = np.arange(len(targets))
    if not np.array_equal(
        targets["target_index"].to_numpy(), expected_indices
    ):
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    target_ids = targets["Entrez_ID"].tolist()
    if set(Y_df.columns) != set(target_ids):
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    X_df = X_df.reindex(metadata.index)
    Y_df = Y_df.reindex(index=metadata.index, columns=target_ids)
    X_df = X_df.apply(pd.to_numeric, errors="raise")
    Y_df = Y_df.apply(pd.to_numeric, errors="raise")
    if Y_df.isna().any().any():
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    if not Y_df.isin([-100, 0, 1, 2]).all().all():
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    X = X_df.to_numpy(dtype=np.float32)
    Y = Y_df.to_numpy(dtype=np.int64)
    if np.isinf(X).any():
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    mtap_rows = targets.index[targets["Entrez_ID"].eq("4507")].tolist()
    if len(mtap_rows) != 1:
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    MTAP_INDEX = mtap_rows[0]
    if (Y[:, MTAP_INDEX] == -100).any():
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    y_binary = (Y[:, MTAP_INDEX] == 0).astype(np.int64)
    fixed_binary = pd.to_numeric(
        metadata["label_MTAP_deep"], errors="raise"
    ).to_numpy()
    if not np.array_equal(y_binary, fixed_binary):
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    if len(metadata) != 7431 or y_binary.sum() != 558:
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    feature_names = X_df.columns.tolist()
    fold_table = pd.read_csv(FOLD_FILE, sep="\t", dtype={"patient_id": str})
    if "patient_id" not in fold_table.columns:
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )

    def normalize_column_name(name):
        return "".join(
            (character for character in name.lower() if character.isalnum())
        )

    fold_candidates = [
        column
        for column in fold_table.columns
        if normalize_column_name(column)
        in {"fold", "cvfold", "outerfold", "foldid"}
    ]
    if not fold_candidates:
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    fold_table = fold_table.loc[
        fold_table["patient_id"].isin(metadata.index)
    ].copy()
    fold_series_list = []
    for column in fold_candidates:
        per_patient = fold_table.groupby("patient_id")[column].nunique(
            dropna=False
        )
        if not per_patient.eq(1).all():
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
        values = (
            fold_table.drop_duplicates("patient_id")
            .set_index("patient_id")[column]
            .reindex(metadata.index)
        )
        if values.isna().any():
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
        numeric_values = pd.to_numeric(values, errors="raise")
        if not numeric_values.eq(numeric_values.astype(int)).all():
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
        fold_series_list.append(numeric_values.astype(int))
    folds = fold_series_list[0].to_numpy()
    for additional in fold_series_list[1:]:
        if not np.array_equal(folds, additional.to_numpy()):
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
    outer_folds = sorted(np.unique(folds))
    if len(outer_folds) != 5:
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    fold_summary = (
        pd.DataFrame({"fold": folds, "label": y_binary})
        .groupby("fold")
        .agg(n_patients=("label", "size"), n_positive=("label", "sum"))
    )
    if members["author_symbol"].duplicated().any():
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    member_index = members.set_index("author_symbol")
    if not set(feature_names).issubset(member_index.index):
        raise ValueError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    feature_annotation = member_index.reindex(feature_names).reset_index()
    feature_annotation["segment_index_0based"] = pd.to_numeric(
        feature_annotation["segment_index_0based"], errors="raise"
    ).astype(int)
    segment_order = sorted(feature_annotation["segment_index_0based"].unique())
    region_indices = []
    for segment in segment_order:
        indices = np.flatnonzero(
            feature_annotation["segment_index_0based"].to_numpy() == segment
        ).tolist()
        if not indices:
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
        region_indices.append(indices)
    core_indices = []
    for gene in ["MTAP", "CDKN2A", "CDKN2B"]:
        if gene not in feature_names:
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
        core_indices.append(feature_names.index(gene))
    cancer_names = sorted(metadata["acronym"].unique())
    cancer_to_index = {name: i + 1 for i, name in enumerate(cancer_names)}
    cancer_codes = (
        metadata["acronym"].map(cancer_to_index).to_numpy(dtype=np.int64)
    )
    for segment, indices in zip(segment_order, region_indices):
        pass

    def fit_preprocessor(values):
        all_missing = np.isnan(values).all(axis=0)
        median = np.zeros(values.shape[1], dtype=np.float32)
        observed_columns = ~all_missing
        median[observed_columns] = np.nanmedian(
            values[:, observed_columns], axis=0
        )
        filled = np.where(np.isnan(values), median[None, :], values)
        mean = filled.mean(axis=0)
        std = filled.std(axis=0)
        std = np.where(std < 1e-06, 1.0, std)
        return {
            "median": median.astype(np.float32),
            "mean": mean.astype(np.float32),
            "std": std.astype(np.float32),
            "all_missing_columns": np.flatnonzero(all_missing).tolist(),
        }

    def transform_values(values, preprocessing):
        filled = np.where(
            np.isnan(values), preprocessing["median"][None, :], values
        )
        transformed = (
            filled - preprocessing["mean"][None, :]
        ) / preprocessing["std"][None, :]
        if not np.isfinite(transformed).all():
            raise ValueError(
                "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
            )
        return transformed.astype(np.float32)

    class RegionMultitaskNet(nn.Module):

        def __init__(self):
            super().__init__()
            self.region_indices = region_indices
            self.core_indices = core_indices
            self.region_encoders = nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Linear(len(indices), REGION_HIDDEN),
                        nn.ReLU(),
                        nn.Dropout(DROPOUT),
                    )
                    for indices in region_indices
                ]
            )
            self.cancer_embedding = nn.Embedding(
                len(cancer_names) + 1, CANCER_EMBEDDING
            )
            combined_dimension = (
                len(region_indices) * REGION_HIDDEN
                + len(core_indices)
                + CANCER_EMBEDDING
            )
            self.shared = nn.Sequential(
                nn.Linear(combined_dimension, SHARED_HIDDEN),
                nn.ReLU(),
                nn.Dropout(DROPOUT),
            )
            self.output = nn.Linear(SHARED_HIDDEN, Y.shape[1] * 3)

        def forward(self, expression, cancer):
            regional = [
                encoder(expression[:, indices])
                for encoder, indices in zip(
                    self.region_encoders, self.region_indices
                )
            ]
            combined = torch.cat(
                regional
                + [
                    expression[:, self.core_indices],
                    self.cancer_embedding(cancer),
                ],
                dim=1,
            )
            hidden = self.shared(combined)
            return self.output(hidden).reshape(-1, Y.shape[1], 3)

    def initialize_output_priors(model, training_labels):
        probabilities = []
        for target_index in range(training_labels.shape[1]):
            labels = training_labels[:, target_index]
            labels = labels[labels != -100]
            counts = np.bincount(labels, minlength=3).astype(float)
            probabilities.append((counts + 1.0) / (counts.sum() + 3.0))
        bias = np.log(np.asarray(probabilities)).reshape(-1)
        with torch.no_grad():
            model.output.bias.copy_(
                torch.tensor(bias, dtype=torch.float32, device=DEVICE)
            )

    def compute_loss(logits, labels):
        mtap_logits = logits[:, MTAP_INDEX, :]
        mtap_binary_logit = mtap_logits[:, 0] - torch.logsumexp(
            mtap_logits[:, 1:], dim=1
        )
        binary_truth = (labels[:, MTAP_INDEX] == 0).float()
        primary_loss = F.binary_cross_entropy_with_logits(
            mtap_binary_logit, binary_truth
        )
        auxiliary_loss = F.cross_entropy(
            logits.reshape(-1, 3), labels.reshape(-1), ignore_index=-100
        )
        return primary_loss + AUXILIARY_WEIGHT * auxiliary_loss

    def predict_probabilities(model, values, cancers):
        model.eval()
        predictions = []
        with torch.no_grad():
            for start in range(0, len(values), 1024):
                end = start + 1024
                batch_x = torch.as_tensor(
                    values[start:end], dtype=torch.float32, device=DEVICE
                )
                batch_c = torch.as_tensor(
                    cancers[start:end], dtype=torch.long, device=DEVICE
                )
                probability = torch.softmax(model(batch_x, batch_c), dim=-1)
                predictions.append(probability.cpu().numpy())
        return np.concatenate(predictions, axis=0)

    def fit_network(
        train_x,
        train_y,
        train_cancer,
        seed,
        epochs,
        valid_x=None,
        valid_y=None,
        valid_cancer=None,
        description="",
    ):
        set_seed(seed)
        model = RegionMultitaskNet().to(DEVICE)
        initialize_output_priors(model, train_y)
        dataset = TensorDataset(
            torch.as_tensor(train_x, dtype=torch.float32),
            torch.as_tensor(train_y, dtype=torch.long),
            torch.as_tensor(train_cancer, dtype=torch.long),
        )
        generator = torch.Generator()
        generator.manual_seed(seed)
        loader = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            generator=generator,
        )
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
        history = []
        best_ap = -np.inf
        best_epoch = 1
        waiting = 0
        for epoch in range(1, epochs + 1):
            model.train()
            total_loss = 0.0
            n_processed = 0
            for batch_x, batch_y, batch_c in loader:
                batch_x = batch_x.to(DEVICE)
                batch_y = batch_y.to(DEVICE)
                batch_c = batch_c.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits = model(batch_x, batch_c)
                loss = compute_loss(logits, batch_y)
                if not torch.isfinite(loss):
                    raise RuntimeError(
                        "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
                    )
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
                total_loss += loss.item() * len(batch_x)
                n_processed += len(batch_x)
            mean_loss = total_loss / n_processed
            record = {"epoch": epoch, "train_loss": mean_loss}
            if valid_x is not None:
                valid_probability = predict_probabilities(
                    model, valid_x, valid_cancer
                )[:, MTAP_INDEX, 0]
                valid_binary = (valid_y[:, MTAP_INDEX] == 0).astype(int)
                valid_ap = average_precision_score(
                    valid_binary, valid_probability
                )
                record["inner_valid_AP"] = valid_ap
                if valid_ap > best_ap + MIN_AP_IMPROVEMENT:
                    best_ap = valid_ap
                    best_epoch = epoch
                    waiting = 0
                else:
                    waiting += 1
                if epoch == 1 or epoch % 10 == 0:
                    pass
                history.append(record)
                if waiting >= PATIENCE:
                    break
            else:
                history.append(record)
                if epoch == 1 or epoch % 20 == 0:
                    pass
        if valid_x is None:
            best_epoch = epochs
            best_ap = np.nan
        return (model, best_epoch, best_ap, history)

    def save_checkpoint(model, preprocessing, path, extra=None):
        payload = {
            "state_dict": {
                key: value.detach().cpu()
                for key, value in model.state_dict().items()
            },
            "feature_names": feature_names,
            "target_Entrez_order": target_ids,
            "target_symbols": targets["DNA_symbol"].tolist(),
            "MTAP_INDEX": MTAP_INDEX,
            "region_indices": region_indices,
            "core_indices": core_indices,
            "cancer_to_index": cancer_to_index,
            "preprocessing": {
                "median": preprocessing["median"].tolist(),
                "mean": preprocessing["mean"].tolist(),
                "std": preprocessing["std"].tolist(),
            },
            "architecture": {
                "region_hidden": REGION_HIDDEN,
                "shared_hidden": SHARED_HIDDEN,
                "cancer_embedding": CANCER_EMBEDDING,
                "dropout": DROPOUT,
            },
            "extra": extra or {},
        }
        torch.save(payload, path)

    n_patients = len(metadata)
    n_targets = Y.shape[1]
    seed_oof = np.full((len(SEEDS), n_patients), np.nan, dtype=np.float32)
    oof_all_targets = np.full(
        (n_patients, n_targets, 3), np.nan, dtype=np.float32
    )
    training_records = []
    selected_epochs = []
    started = time.monotonic()
    for fold in outer_folds:
        outer_train = np.flatnonzero(folds != fold)
        outer_valid = np.flatnonzero(folds == fold)
        inner_train, inner_valid = train_test_split(
            outer_train,
            test_size=INNER_VALID_FRACTION,
            random_state=2026 + int(fold),
            stratify=y_binary[outer_train],
        )
        inner_preprocessing = fit_preprocessor(X[inner_train])
        inner_train_x = transform_values(X[inner_train], inner_preprocessing)
        inner_valid_x = transform_values(X[inner_valid], inner_preprocessing)
        outer_preprocessing = fit_preprocessor(X[outer_train])
        outer_train_x = transform_values(X[outer_train], outer_preprocessing)
        outer_valid_x = transform_values(X[outer_valid], outer_preprocessing)
        fold_probabilities = []
        for seed_index, seed in enumerate(SEEDS):
            description = f"Fold {fold}/seed{seed}/early stopping"
            inner_model, best_epoch, best_ap, history = fit_network(
                inner_train_x,
                Y[inner_train],
                cancer_codes[inner_train],
                seed=seed,
                epochs=MAX_EPOCHS,
                valid_x=inner_valid_x,
                valid_y=Y[inner_valid],
                valid_cancer=cancer_codes[inner_valid],
                description=description,
            )
            pd.DataFrame(history).to_csv(
                OUT / f"inner_history_fold{fold}_seed{seed}.tsv",
                sep="\t",
                index=False,
            )
            del inner_model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            selected_epochs.append(best_epoch)
            outer_model, _, _, refit_history = fit_network(
                outer_train_x,
                Y[outer_train],
                cancer_codes[outer_train],
                seed=seed,
                epochs=best_epoch,
                description=f"Fold {fold}/seed{seed}/refit",
            )
            probability = predict_probabilities(
                outer_model, outer_valid_x, cancer_codes[outer_valid]
            )
            fold_probabilities.append(probability)
            mtap_probability = probability[:, MTAP_INDEX, 0]
            seed_oof[seed_index, outer_valid] = mtap_probability
            training_records.append(
                {
                    "outer_fold": fold,
                    "seed": seed,
                    "n_train": len(outer_train),
                    "n_valid": len(outer_valid),
                    "selected_epochs": best_epoch,
                    "inner_best_AP": best_ap,
                    "outer_AUROC": roc_auc_score(
                        y_binary[outer_valid], mtap_probability
                    ),
                    "outer_AP": average_precision_score(
                        y_binary[outer_valid], mtap_probability
                    ),
                }
            )
            save_checkpoint(
                outer_model,
                outer_preprocessing,
                MODEL_DIR / f"fold{fold}_seed{seed}.pt",
                extra={
                    "outer_fold": int(fold),
                    "seed": seed,
                    "epochs": best_epoch,
                },
            )
            pd.DataFrame(refit_history).to_csv(
                OUT / f"refit_history_fold{fold}_seed{seed}.tsv",
                sep="\t",
                index=False,
            )
            del outer_model
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
        fold_mean = np.mean(fold_probabilities, axis=0)
        oof_all_targets[outer_valid] = fold_mean
        pd.DataFrame(training_records).to_csv(
            OUT / "outer_fold_seed_results.tsv", sep="\t", index=False
        )
        np.savez_compressed(
            OUT / "OOF_progress.npz",
            probabilities=oof_all_targets,
            seed_MTAP_probabilities=seed_oof,
            patient_id=metadata.index.to_numpy(dtype=str),
        )
    if np.isnan(oof_all_targets).any() or np.isnan(seed_oof).any():
        raise RuntimeError(
            "Regional multitask DL training and seed analysis: integrity check failed; condition: input or artifact integrity check."
        )
    mtap_oof = oof_all_targets[:, MTAP_INDEX, 0]

    def binary_metrics(truth, probability):
        predicted = probability >= 0.5
        tn, fp, fn, tp = confusion_matrix(
            truth, predicted, labels=[0, 1]
        ).ravel()
        return {
            "AUROC": roc_auc_score(truth, probability),
            "AP": average_precision_score(truth, probability),
            "Brier_score": brier_score_loss(truth, probability),
            "log_loss": log_loss(truth, probability, labels=[0, 1]),
            "precision_at_05": precision_score(
                truth, predicted, zero_division=0
            ),
            "sensitivity_at_05": recall_score(
                truth, predicted, zero_division=0
            ),
            "specificity_at_05": tn / (tn + fp),
            "TP_at_05": int(tp),
            "FP_at_05": int(fp),
            "FN_at_05": int(fn),
            "TN_at_05": int(tn),
        }

    performance = pd.DataFrame(
        [
            {
                "model": "Regional_multitask_DL_3seed",
                "n_patients": n_patients,
                "n_positive": int(y_binary.sum()),
                **binary_metrics(y_binary, mtap_oof),
            }
        ]
    )
    performance.to_csv(
        OUT / "development_MTAP_OOF_performance.tsv", sep="\t", index=False
    )
    seed_results = pd.DataFrame(
        [
            {"seed": seed, **binary_metrics(y_binary, seed_oof[i])}
            for i, seed in enumerate(SEEDS)
        ]
    )
    seed_results.to_csv(
        OUT / "development_seed_stability.tsv", sep="\t", index=False
    )
    oof_table = metadata[["acronym", "label_MTAP_deep"]].copy()
    oof_table["outer_fold"] = folds
    oof_table["prob_MTAP_deep"] = mtap_oof
    for i, seed in enumerate(SEEDS):
        oof_table[f"prob_seed_{seed}"] = seed_oof[i]
    oof_table.to_csv(
        OUT / "development_MTAP_OOF_predictions.tsv",
        sep="\t",
        index=True,
        index_label="patient_id",
    )
    np.savez_compressed(
        OUT / "development_all_target_OOF.npz",
        probabilities=oof_all_targets,
        patient_id=metadata.index.to_numpy(dtype=str),
        target_Entrez=np.asarray(target_ids, dtype=str),
        target_symbols=targets["DNA_symbol"].to_numpy(dtype=str),
    )
    cancer_results = []
    for cancer in cancer_names:
        indices = np.flatnonzero(metadata["acronym"].to_numpy() == cancer)
        truth = y_binary[indices]
        record = {
            "acronym": cancer,
            "n_patients": len(indices),
            "n_positive": int(truth.sum()),
            "n_negative": int((truth == 0).sum()),
        }
        if len(np.unique(truth)) == 2:
            record.update(binary_metrics(truth, mtap_oof[indices]))
        else:
            record["AUROC"] = np.nan
            record["AP"] = np.nan
        cancer_results.append(record)
    pd.DataFrame(cancer_results).to_csv(
        OUT / "development_per_cancer_performance.tsv", sep="\t", index=False
    )
    target_results = []
    for target_index, row in targets.iterrows():
        observed = Y[:, target_index] != -100
        truth_class = Y[observed, target_index]
        truth_deep = (truth_class == 0).astype(int)
        probability = oof_all_targets[observed, target_index, 0]
        predicted_class = np.argmax(
            oof_all_targets[observed, target_index, :], axis=1
        )
        record = {
            "DNA_symbol": row["DNA_symbol"],
            "Entrez_ID": row["Entrez_ID"],
            "n_observed": int(observed.sum()),
            "n_deep": int(truth_deep.sum()),
            "three_class_accuracy": float(
                np.mean(predicted_class == truth_class)
            ),
        }
        if len(np.unique(truth_deep)) == 2:
            record["deep_AUROC"] = roc_auc_score(truth_deep, probability)
            record["deep_AP"] = average_precision_score(
                truth_deep, probability
            )
        target_results.append(record)
    pd.DataFrame(target_results).to_csv(
        OUT / "development_all_target_performance.tsv", sep="\t", index=False
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


## 5. Model Validation in the TCGA Holdout Set

Evaluate all three frozen models on the original held-out TCGA patients. No model fitting or threshold selection uses these holdout predictions.


### Shared holdout input audit


#### Original 20 percent TCGA holdout and frozen model inputs


In [ ]:
@_nb9_registry.register("s72_0")
def _nb9_stage_s72_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from datetime import datetime
    import gzip
    import hashlib
    import importlib.util
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import torch
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    PRED = PROJECT / "data_processed/MTAP_RNA_prediction"
    BASE = PRED / "15_region_multitask_DL/08_TPM_retraining"
    INPUT = BASE / "01_expression_inputs_44"
    LOCK = BASE / "04_locked_models_and_thresholds"
    META_FILE = (
        PRED / "03_baseline_models/cohort_with_three_gene_expression.tsv"
    )
    DEV_META_FILE = INPUT / "development_metadata.tsv"
    DEV_X_FILE = INPUT / "X_development_log2TPM_plus_1.tsv"
    SPEC_FILE = INPUT / "TPM_44_feature_specification.json"
    SOURCE_CACHE = INPUT / "development_sample_log2TPM_plus_0001.tsv"
    TOIL_FILE = PROJECT / "data_raw/PANCAN/TCGA/Toil_TPM/tcga_RSEM_gene_tpm.gz"
    L2_FILE = (
        BASE
        / "06_three_gene_L2_comparison/locked_baseline/three_gene_L2.joblib"
    )
    EN_FILE = LOCK / "models/ElasticNet_locked.joblib"
    NETWORK_FILE = LOCK / "region_multitask_TPM44_network.py"
    DL_FILES = [LOCK / f"models/DL_seed{s}.pt" for s in [41, 42, 43]]
    SEED = 20261101
    N_BOOTSTRAP = 1000
    THRESHOLD = 0.5
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    OUT = BASE / "07_TCGA_original_holdout_evaluation" / RUN_ID
    required = [
        META_FILE,
        DEV_META_FILE,
        DEV_X_FILE,
        SPEC_FILE,
        SOURCE_CACHE,
        TOIL_FILE,
        L2_FILE,
        EN_FILE,
        NETWORK_FILE,
        *DL_FILES,
    ]
    missing = [str(p) for p in required if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    OUT.mkdir(parents=True, exist_ok=False)

    def sha256_file(path):
        h = hashlib.sha256()
        with Path(path).open("rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    model_files = [L2_FILE, EN_FILE, NETWORK_FILE, *DL_FILES]
    model_hashes_before = {str(p): sha256_file(p) for p in model_files}
    meta = pd.read_csv(META_FILE, sep="\t", dtype=str)
    dev_meta = pd.read_csv(DEV_META_FILE, sep="\t", dtype=str)
    for name, frame in [
        ("Original patient metadata", meta),
        ("Development cohort", dev_meta),
    ]:
        if frame["patient_id"].duplicated().any():
            raise ValueError(
                "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
            )
    test = (
        meta.loc[meta["dataset_split"].eq("test")]
        .copy()
        .set_index("patient_id")
        .sort_index()
    )
    calls = pd.to_numeric(test["MTAP_GISTIC_call"], errors="raise")
    if not calls.isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    test["label_MTAP_deep"] = calls.eq(-2).astype(int)
    if len(test) != 1858 or test["label_MTAP_deep"].sum() != 140:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    if set(test.index) & set(dev_meta["patient_id"]):
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    with SPEC_FILE.open(encoding="utf-8") as f:
        spec = json.load(f)
    features = spec["expression_feature_order"]
    if (
        len(features) != 44
        or len(set(features)) != 44
        or spec["TCGA_source_unit"] != "log2(TPM+0.001)"
        or (spec["model_input_unit"] != "log2(TPM+1)")
    ):
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    cache = pd.read_csv(SOURCE_CACHE, sep="\t", index_col=0)
    cache.index = cache.index.astype(str)
    cache.columns = cache.columns.astype(str)
    if cache.index.duplicated().any():
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    if not set(features).issubset(cache.columns):
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    cache = cache.loc[:, features].apply(pd.to_numeric, errors="raise")
    if not np.isfinite(cache.to_numpy()).all():
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    cache_patients = set(cache.index.str[:12])
    if not cache_patients.issubset(set(dev_meta["patient_id"])):
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    with gzip.open(TOIL_FILE, "rt") as f:
        header = f.readline().rstrip("\r\n").split("\t")
    sample_columns = header[1:]
    if len(sample_columns) != len(set(sample_columns)):
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    column_position = {s: i for i, s in enumerate(sample_columns)}
    missing_cache_samples = [
        s for s in cache.index if s not in column_position
    ]
    if missing_cache_samples:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    rng_reference = np.random.default_rng(SEED)
    reference_ids = []
    for gene in features:
        reference_ids.extend([cache[gene].idxmin(), cache[gene].idxmax()])
    reference_ids.extend(
        rng_reference.choice(
            cache.index.to_numpy(), size=min(96, len(cache)), replace=False
        ).tolist()
    )
    reference_ids = list(dict.fromkeys(reference_ids))
    reference_positions = np.array(
        [column_position[s] for s in reference_ids], dtype=int
    )
    reference_values = cache.loc[reference_ids, features].to_numpy().T
    reference_variation = np.ptp(reference_values, axis=0)
    probe_indices = np.argsort(reference_variation)[
        -min(8, len(reference_ids)) :
    ]
    test_sample_ids = []
    test_sample_positions = []
    for i, sample_id in enumerate(sample_columns):
        parts = sample_id.split("-")
        if (
            len(parts) >= 4
            and sample_id[:12] in test.index
            and (parts[3][:2] == "01")
        ):
            test_sample_ids.append(sample_id)
            test_sample_positions.append(i)
    if not test_sample_ids:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    test_sample_positions = np.asarray(test_sample_positions, dtype=int)
    gene_matches = {g: [] for g in features}
    test_source_vectors = {}
    with gzip.open(TOIL_FILE, "rt") as f:
        next(f)
        for row_number, line in enumerate(f, start=1):
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) != len(header):
                raise ValueError(
                    "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
                )
            source_gene_id = fields[0]
            values = fields[1:]
            try:
                probe = np.array(
                    [
                        float(values[reference_positions[j]])
                        for j in probe_indices
                    ],
                    dtype=float,
                )
            except ValueError:
                continue
            candidate_indices = np.flatnonzero(
                np.isclose(
                    reference_values[:, probe_indices],
                    probe[None, :],
                    rtol=0,
                    atol=1e-07,
                ).all(axis=1)
            )
            if len(candidate_indices):
                full_reference = np.array(
                    [float(values[j]) for j in reference_positions],
                    dtype=float,
                )
                for gene_index in candidate_indices:
                    if np.allclose(
                        full_reference,
                        reference_values[gene_index],
                        rtol=0,
                        atol=1e-07,
                    ):
                        gene = features[gene_index]
                        gene_matches[gene].append(source_gene_id)
                        if len(gene_matches[gene]) == 1:
                            test_source_vectors[gene] = np.array(
                                [
                                    float(values[j])
                                    for j in test_sample_positions
                                ],
                                dtype=float,
                            )
            if row_number % 10000 == 0:
                pass
    mapping_records = [
        {
            "gene": gene,
            "n_matching_source_rows": len(gene_matches[gene]),
            "source_gene_ids": ";".join(gene_matches[gene]),
            "identity_check": "Development_source_expression_exact_match",
            "n_reference_samples": len(reference_ids),
        }
        for gene in features
    ]
    pd.DataFrame(mapping_records).to_csv(
        OUT / "01_source_gene_identity_audit.tsv", sep="\t", index=False
    )
    ambiguous = {
        gene: ids for gene, ids in gene_matches.items() if len(ids) != 1
    }
    if ambiguous:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    matched_source_ids = [gene_matches[g][0] for g in features]
    if len(set(matched_source_ids)) != len(features):
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    source_log = pd.DataFrame(test_source_vectors, index=test_sample_ids).loc[
        :, features
    ]
    source_log.index.name = "sample_id"
    sample_mapping = pd.DataFrame(
        {
            "sample_id": test_sample_ids,
            "patient_id": [s[:12] for s in test_sample_ids],
        }
    )
    sample_mapping.to_csv(
        OUT / "02_test_Toil_sample_mapping.tsv", sep="\t", index=False
    )
    patient_rows = {}
    for patient, group in sample_mapping.groupby("patient_id", sort=True):
        sample_ids = group["sample_id"].tolist()
        values = source_log.loc[sample_ids].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            continue
        mean_source_log = np.logaddexp.reduce(
            values * np.log(2), axis=0
        ) / np.log(2) - np.log2(len(sample_ids))
        patient_rows[patient] = np.logaddexp2(mean_source_log, np.log2(0.999))
    X = pd.DataFrame.from_dict(patient_rows, orient="index", columns=features)
    X.index.name = "patient_id"
    X = X.loc[[p for p in test.index if p in X.index]]
    if len(X) == 0 or not np.isfinite(X.to_numpy()).all():
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    availability = test[["acronym", "label_MTAP_deep"]].copy()
    sample_counts = sample_mapping.groupby("patient_id").size()
    availability["n_Toil_primary_samples"] = (
        availability.index.to_series().map(sample_counts).fillna(0).astype(int)
    )
    availability["included"] = availability.index.isin(X.index)
    availability["status"] = np.where(
        availability["included"],
        "Included",
        np.where(
            availability["n_Toil_primary_samples"].eq(0),
            "No_matching_Toil_primary_RNA",
            "Nonfinite_expression",
        ),
    )
    availability.to_csv(
        OUT / "03_all_original_test_patients_availability.tsv", sep="\t"
    )
    X.to_csv(OUT / "04_X_test_log2TPM_plus_1.tsv", sep="\t")
    evaluation_meta = test.loc[X.index].copy()
    y = evaluation_meta["label_MTAP_deep"].to_numpy(dtype=int)
    if len(np.unique(y)) != 2:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    l2_bundle = joblib.load(L2_FILE)
    en_bundle = joblib.load(EN_FILE)
    if list(l2_bundle["feature_names"]) != ["MTAP", "CDKN2A", "CDKN2B"]:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    if list(en_bundle["feature_names"]) != features:
        raise ValueError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    model_input = X.copy()
    model_input["acronym"] = evaluation_meta["acronym"]

    def frozen_pipeline_probability(bundle, frame):
        pipeline = bundle["pipeline"]
        classes = list(pipeline.classes_)
        if 1 not in classes:
            raise ValueError(
                "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
            )
        return pipeline.predict_proba(frame)[:, classes.index(1)]


### Model 1: Three-gene L2 logistic regression


#### Frozen L2 prediction on the TCGA holdout


In [ ]:
@_nb9_registry.register("s72_117_l2")
def _nb9_stage_s72_117_l2():
    probabilities = {}
    probabilities["Three_gene_L2"] = frozen_pipeline_probability(
        l2_bundle, model_input
    )


### Model 2: 44-gene Elastic Net


#### Frozen 44-gene Elastic Net prediction on the TCGA holdout


In [ ]:
@_nb9_registry.register("s72_117_en")
def _nb9_stage_s72_117_en():
    probabilities["Elastic_Net"] = frozen_pipeline_probability(
        en_bundle, model_input
    )


### Model 3: Deep learning


#### Frozen three-seed TPM DL ensemble on the TCGA holdout


In [ ]:
@_nb9_registry.register("s72_118")
def _nb9_stage_s72_118():
    module_spec = importlib.util.spec_from_file_location(
        "heldout_locked_TPM44_network", NETWORK_FILE
    )
    network_module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(network_module)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    seed_predictions = []
    for checkpoint_file in DL_FILES:
        checkpoint = torch.load(
            checkpoint_file, map_location="cpu", weights_only=False
        )
        if list(checkpoint["feature_names"]) != features:
            raise ValueError(
                "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
            )
        network = network_module.RegionalMultiTaskNet(checkpoint)
        network.load_state_dict(checkpoint["state_dict"], strict=True)
        network.to(device).eval()
        preprocessing = checkpoint["preprocessing"]
        mean = np.asarray(preprocessing["mean"], dtype=np.float32)
        std = np.asarray(preprocessing["std"], dtype=np.float32)
        if not np.isfinite(mean).all() or not (std > 0).all():
            raise ValueError(
                "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
            )
        values = (X.to_numpy(dtype=np.float32) - mean) / std
        cancer_mapping = checkpoint["cancer_to_index"]
        unknown = set(evaluation_meta["acronym"]) - set(cancer_mapping)
        if unknown:
            raise ValueError(
                "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
            )
        cancer_codes = (
            evaluation_meta["acronym"]
            .map(cancer_mapping)
            .to_numpy(dtype=np.int64)
        )
        batches = []
        with torch.no_grad():
            for start in range(0, len(X), 512):
                expression_tensor = torch.from_numpy(
                    values[start : start + 512]
                ).to(device)
                cancer_tensor = torch.from_numpy(
                    cancer_codes[start : start + 512]
                ).to(device)
                output = torch.softmax(
                    network(expression_tensor, cancer_tensor), dim=-1
                )
                batches.append(
                    output[:, checkpoint["MTAP_INDEX"], 0].cpu().numpy()
                )
        seed_predictions.append(np.concatenate(batches))
        del network
    probabilities["Bulk_DL"] = np.mean(seed_predictions, axis=0)


### Matched holdout predictions


#### Probability validation and aligned holdout predictions


In [ ]:
@_nb9_registry.register("s72_125")
def _nb9_stage_s72_125():
    for name, probability in probabilities.items():
        if (
            len(probability) != len(X)
            or not np.isfinite(probability).all()
            or ((probability < 0) | (probability > 1)).any()
        ):
            raise ValueError(
                "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
            )
    predictions = evaluation_meta[
        ["acronym", "MTAP_GISTIC_call", "label_MTAP_deep"]
    ].copy()
    for name, probability in probabilities.items():
        predictions[f"prob_{name}"] = probability
        predictions[f"pred_{name}_at_0.50"] = (
            probability >= THRESHOLD
        ).astype(int)
    predictions.to_csv(
        OUT / "05_TCGA_original_holdout_three_model_predictions.tsv", sep="\t"
    )


## 6. CPTAC External Validation

Use frozen model parameters with the source CPTAC eligibility, pairing, and expression-scale checks.


### Frozen external inference


#### CPTAC frozen Elastic Net and DL inference and metrics


In [ ]:
@_nb9_registry.register("s37_113")
def _nb9_stage_s37_113():
    network_file = LOCK / "region_multitask_TPM44_network.py"
    module_spec = importlib.util.spec_from_file_location(
        "locked_TPM44_network", network_file
    )
    network_module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(network_module)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    locked_dl = []
    for seed in [41, 42, 43]:
        checkpoint = torch.load(
            LOCK / "models" / f"DL_seed{seed}.pt",
            map_location="cpu",
            weights_only=False,
        )
        network = network_module.RegionalMultiTaskNet(checkpoint)
        network.load_state_dict(checkpoint["state_dict"], strict=True)
        network.to(device)
        network.eval()
        locked_dl.append((seed, checkpoint, network))
    locked_en_bundle = joblib.load(LOCK / "models/ElasticNet_locked.joblib")
    if locked_en_bundle["feature_names"] != features:
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )

    @torch.no_grad()
    def predict_locked_dl(frame, acronym):
        seed_probabilities = []
        for seed, checkpoint, network in locked_dl:
            expected_features = checkpoint["feature_names"]
            values = frame.loc[:, expected_features].to_numpy(dtype=np.float32)
            preprocessing = checkpoint["preprocessing"]
            mean = np.asarray(preprocessing["mean"], dtype=np.float32)
            std = np.asarray(preprocessing["std"], dtype=np.float32)
            if not np.isfinite(values).all():
                raise ValueError(
                    "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
                )
            values = (values - mean) / std
            cancer_mapping = checkpoint["cancer_to_index"]
            if acronym not in cancer_mapping:
                raise ValueError(
                    "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
                )
            codes = np.full(
                len(frame), cancer_mapping[acronym], dtype=np.int64
            )
            batches = []
            for start in range(0, len(frame), 1024):
                expression = torch.from_numpy(values[start : start + 1024]).to(
                    device
                )
                cancer = torch.from_numpy(codes[start : start + 1024]).to(
                    device
                )
                probabilities = torch.softmax(
                    network(expression, cancer), dim=-1
                )
                batches.append(
                    probabilities[:, checkpoint["MTAP_INDEX"], 0].cpu().numpy()
                )
            seed_probabilities.append(np.concatenate(batches))
        return np.mean(seed_probabilities, axis=0)

    external_tables = []
    external_file_hashes = {}
    for database_cancer, model_cancer in CANCER_MAP.items():
        expression_file = (
            CPTAC_X_DIR / database_cancer / "X_44_log2TPM_plus_1.tsv"
        )
        metadata_file = (
            CPTAC_META_DIR / database_cancer / "paired_patient_metadata.tsv"
        )
        expression = pd.read_csv(expression_file, sep="\t", index_col=0)
        expression.index = expression.index.astype(str)
        if expression.index.duplicated().any():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        expression = expression.loc[:, features].apply(
            pd.to_numeric, errors="raise"
        )
        if not np.isfinite(expression.to_numpy()).all():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        probability_dl = predict_locked_dl(expression, model_cancer)
        en_input = expression.copy()
        en_input["acronym"] = model_cancer
        probability_en = locked_en_bundle["pipeline"].predict_proba(en_input)[
            :, 1
        ]
        external_meta = pd.read_csv(
            metadata_file,
            sep="\t",
            usecols=["patient_id", "acronym", "MTAP_GISTIC_call"],
            dtype={"patient_id": str, "acronym": str},
        )
        if external_meta["patient_id"].duplicated().any():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        external_meta = external_meta.set_index("patient_id")
        if set(external_meta.index) != set(expression.index):
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        external_meta = external_meta.loc[expression.index]
        if set(external_meta["acronym"]) != {model_cancer}:
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        calls = pd.to_numeric(
            external_meta["MTAP_GISTIC_call"], errors="raise"
        )
        if (calls.notna() & ~calls.isin([-2, -1, 0, 1, 2])).any():
            raise ValueError(
                "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
            )
        labels = pd.Series(pd.NA, index=expression.index, dtype="Int64")
        observed = calls.notna()
        labels.loc[observed] = calls.loc[observed].eq(-2).astype(int)
        result = pd.DataFrame(
            {
                "patient_id": expression.index,
                "database_cancer": database_cancer,
                "acronym": model_cancer,
                "MTAP_GISTIC_call": calls.to_numpy(),
                "label_MTAP_deep": labels.to_numpy(),
                "prob_DL": probability_dl,
                "prob_ElasticNet": probability_en,
                "group_DL": classify_groups(
                    probability_dl, locked_thresholds[DL_NAME]
                ),
                "group_ElasticNet": classify_groups(
                    probability_en, locked_thresholds[EN_NAME]
                ),
            }
        )
        external_tables.append(result)
        external_file_hashes[str(expression_file)] = sha256_file(
            expression_file
        )
        external_file_hashes[str(metadata_file)] = sha256_file(metadata_file)
    external_predictions = pd.concat(external_tables, ignore_index=True)
    if external_predictions["patient_id"].duplicated().any():
        duplicated = external_predictions.loc[
            external_predictions["patient_id"].duplicated(keep=False),
            ["patient_id", "database_cancer"],
        ]
        duplicated.to_csv(
            EXTERNAL_OUT / "duplicate_patient_review.tsv",
            sep="\t",
            index=False,
        )
        raise ValueError(
            "Forty-four-member Elastic Net training and frozen CPTAC validation: integrity check failed; condition: input or artifact integrity check."
        )
    external_predictions.to_csv(
        EXTERNAL_OUT / "CPTAC_patient_predictions.tsv", sep="\t", index=False
    )
    evaluation = external_predictions.loc[
        external_predictions["label_MTAP_deep"].notna()
    ].copy()
    evaluation["label_MTAP_deep"] = evaluation["label_MTAP_deep"].astype(int)
    probability_columns = {DL_NAME: "prob_DL", EN_NAME: "prob_ElasticNet"}
    metric_rows = []
    group_rows = []
    difference_rows = []
    subsets = [("ALL_CPTAC", evaluation)]
    subsets += [
        (
            database_cancer,
            evaluation.loc[
                evaluation["database_cancer"].eq(database_cancer)
            ].copy(),
        )
        for database_cancer in CANCER_MAP
    ]
    for subset_position, (cohort, data) in enumerate(subsets):
        data = data.reset_index(drop=True)
        if data.empty:
            continue
        yy = data["label_MTAP_deep"].to_numpy(dtype=int)
        point_metrics = {}
        bootstrap_metrics = {
            name: {metric: [] for metric in ["AUROC", "AP", "Brier_score"]}
            for name in MODEL_NAMES
        }
        for name in MODEL_NAMES:
            point_metrics[name] = performance(
                yy, data[probability_columns[name]].to_numpy()
            )
            group_rows.extend(
                group_summary(
                    yy,
                    data[probability_columns[name]].to_numpy(),
                    locked_thresholds[name],
                    name,
                    cohort,
                )
            )
        strata = list(
            data.groupby(
                ["database_cancer", "label_MTAP_deep"], sort=True
            ).indices.values()
        )
        rng = np.random.default_rng(RANDOM_SEED + subset_position)
        for bootstrap in range(N_BOOTSTRAPS):
            sampled = np.concatenate(
                [
                    rng.choice(indices, size=len(indices), replace=True)
                    for indices in strata
                ]
            )
            y_boot = yy[sampled]
            has_both = len(np.unique(y_boot)) == 2
            for name in MODEL_NAMES:
                p_boot = data[probability_columns[name]].to_numpy()[sampled]
                bootstrap_metrics[name]["AUROC"].append(
                    roc_auc_score(y_boot, p_boot) if has_both else np.nan
                )
                bootstrap_metrics[name]["AP"].append(
                    average_precision_score(y_boot, p_boot)
                    if has_both
                    else np.nan
                )
                bootstrap_metrics[name]["Brier_score"].append(
                    brier_score_loss(y_boot, p_boot)
                )
        n_positive = int(yy.sum())
        n_negative = int((yy == 0).sum())
        if min(n_positive, n_negative) == 0:
            support = "Single_class_no_AUROC_or_AP"
        elif min(n_positive, n_negative) < 10:
            support = "Very_limited_class_support"
        elif min(n_positive, n_negative) < 20:
            support = "Limited_class_support"
        else:
            support = "Descriptive_evaluation"
        for name in MODEL_NAMES:
            row = {
                "cohort": cohort,
                "model": name,
                **point_metrics[name],
                "support_status": support,
            }
            for metric in ["AUROC", "AP", "Brier_score"]:
                values = np.asarray(
                    bootstrap_metrics[name][metric], dtype=float
                )
                values = values[np.isfinite(values)]
                if len(values):
                    low, high = np.quantile(values, [0.025, 0.975])
                else:
                    low, high = (np.nan, np.nan)
                row[f"{metric}_CI95_low"] = low
                row[f"{metric}_CI95_high"] = high
            metric_rows.append(row)
        for metric in ["AUROC", "AP", "Brier_score"]:
            difference = np.asarray(
                bootstrap_metrics[DL_NAME][metric], dtype=float
            ) - np.asarray(bootstrap_metrics[EN_NAME][metric], dtype=float)
            difference = difference[np.isfinite(difference)]
            if len(difference):
                low, high = np.quantile(difference, [0.025, 0.975])
            else:
                low, high = (np.nan, np.nan)
            difference_rows.append(
                {
                    "cohort": cohort,
                    "metric": metric,
                    "difference_DL_minus_ElasticNet": point_metrics[DL_NAME][
                        metric
                    ]
                    - point_metrics[EN_NAME][metric],
                    "CI95_low": low,
                    "CI95_high": high,
                    "favorable_direction": (
                        "negative" if metric == "Brier_score" else "positive"
                    ),
                    "interval_scope": "Conditional_on_frozen_models; cancer_by_label_stratified_patient_bootstrap",
                }
            )
    metrics_table = pd.DataFrame(metric_rows)
    groups_table = pd.DataFrame(group_rows)
    differences_table = pd.DataFrame(difference_rows)
    metrics_table.to_csv(
        EXTERNAL_OUT / "CPTAC_external_performance.tsv", sep="\t", index=False
    )
    groups_table.to_csv(
        EXTERNAL_OUT / "CPTAC_locked_threshold_group_results.tsv",
        sep="\t",
        index=False,
    )
    differences_table.to_csv(
        EXTERNAL_OUT / "CPTAC_paired_model_differences.tsv",
        sep="\t",
        index=False,
    )


#### CPTAC frozen L2 inference and matched three-model metrics


In [ ]:
@_nb9_registry.register("s39_95")
def _nb9_stage_s39_95():
    external = pd.read_csv(
        EXTERNAL_PREDICTIONS_FILE,
        sep="\t",
        dtype={"patient_id": str, "database_cancer": str, "acronym": str},
    )
    if external["patient_id"].duplicated().any():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    external["prob_L2"] = np.nan
    external_input_hashes = {}
    for database_cancer, model_cancer in CANCER_MAP.items():
        expression_file = (
            CPTAC_INPUT / database_cancer / "X_44_log2TPM_plus_1.tsv"
        )
        expression = pd.read_csv(expression_file, sep="\t", index_col=0)
        expression.index = expression.index.astype(str)
        if expression.index.duplicated().any():
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        selected = external["database_cancer"].eq(database_cancer)
        ids = external.loc[selected, "patient_id"].tolist()
        if set(expression.index) != set(ids):
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        if set(external.loc[selected, "acronym"]) != {model_cancer}:
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        expression = expression.loc[ids, CORE_GENES].apply(
            pd.to_numeric, errors="raise"
        )
        if not np.isfinite(expression.to_numpy()).all():
            raise ValueError(
                "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
            )
        model_input = expression.copy()
        model_input["acronym"] = model_cancer
        external.loc[selected, "prob_L2"] = bundle["pipeline"].predict_proba(
            model_input
        )[:, 1]
        external_input_hashes[str(expression_file)] = sha256_file(
            expression_file
        )
    if external["prob_L2"].isna().any():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    external["group_L2"] = classify_groups(external["prob_L2"], l2_thresholds)
    external.to_csv(
        OUT / "CPTAC_three_model_patient_predictions.tsv",
        sep="\t",
        index=False,
    )
    observed_labels = pd.to_numeric(
        external["label_MTAP_deep"], errors="raise"
    )
    evaluation = external.loc[observed_labels.notna()].copy()
    evaluation["label_MTAP_deep"] = observed_labels.loc[
        evaluation.index
    ].astype(int)
    if not evaluation["label_MTAP_deep"].isin([0, 1]).all():
        raise ValueError(
            "Final TPM three-model comparison and L2 locking: integrity check failed; condition: input or artifact integrity check."
        )
    metric_rows = []
    group_rows = []
    difference_rows = []
    subsets = [("ALL_CPTAC", evaluation)]
    subsets += [
        (cancer, evaluation.loc[evaluation["database_cancer"].eq(cancer)])
        for cancer in CANCER_MAP
    ]
    for subset_position, (cohort, frame) in enumerate(subsets):
        frame = frame.reset_index(drop=True)
        if frame.empty:
            continue
        yy = frame["label_MTAP_deep"].to_numpy(dtype=int)
        probabilities = {
            name: frame[PROBABILITY_COLUMNS[name]].to_numpy(dtype=float)
            for name in MODEL_NAMES
        }
        points = {
            name: performance(yy, probabilities[name]) for name in MODEL_NAMES
        }
        samples = {
            name: {metric: [] for metric in ["AUROC", "AP", "Brier_score"]}
            for name in MODEL_NAMES
        }
        strata = list(
            frame.groupby(
                ["database_cancer", "label_MTAP_deep"], sort=True
            ).indices.values()
        )
        rng = np.random.default_rng(RANDOM_SEED + subset_position)
        for bootstrap in range(N_BOOTSTRAPS):
            indices = np.concatenate(
                [
                    rng.choice(positions, size=len(positions), replace=True)
                    for positions in strata
                ]
            )
            y_boot = yy[indices]
            has_both = len(np.unique(y_boot)) == 2
            for name in MODEL_NAMES:
                p_boot = probabilities[name][indices]
                samples[name]["AUROC"].append(
                    roc_auc_score(y_boot, p_boot) if has_both else np.nan
                )
                samples[name]["AP"].append(
                    average_precision_score(y_boot, p_boot)
                    if has_both
                    else np.nan
                )
                samples[name]["Brier_score"].append(
                    brier_score_loss(y_boot, p_boot)
                )
        smallest_class = min(int((yy == 1).sum()), int((yy == 0).sum()))
        if smallest_class == 0:
            support = "Single_class_no_AUROC_or_AP"
        elif smallest_class < 10:
            support = "Very_limited_class_support"
        elif smallest_class < 20:
            support = "Limited_class_support"
        else:
            support = "Descriptive_evaluation"
        for name in MODEL_NAMES:
            row = {
                "cohort": cohort,
                "model": name,
                **points[name],
                "support_status": support,
            }
            for metric in ["AUROC", "AP", "Brier_score"]:
                values = np.asarray(samples[name][metric], dtype=float)
                values = values[np.isfinite(values)]
                if len(values):
                    low, high = np.quantile(values, [0.025, 0.975])
                else:
                    low, high = (np.nan, np.nan)
                row[f"{metric}_CI95_low"] = low
                row[f"{metric}_CI95_high"] = high
            metric_rows.append(row)
            group_rows.extend(
                summarize_groups(
                    yy, probabilities[name], all_thresholds[name], name, cohort
                )
            )
        for main_model in [DL_NAME, EN_NAME]:
            for metric in ["AUROC", "AP", "Brier_score"]:
                differences = np.asarray(
                    samples[main_model][metric], dtype=float
                ) - np.asarray(samples[L2_NAME][metric], dtype=float)
                differences = differences[np.isfinite(differences)]
                if len(differences):
                    low, high = np.quantile(differences, [0.025, 0.975])
                else:
                    low, high = (np.nan, np.nan)
                difference_rows.append(
                    {
                        "cohort": cohort,
                        "main_model": main_model,
                        "comparator": L2_NAME,
                        "metric": metric,
                        "difference_main_minus_L2": points[main_model][metric]
                        - points[L2_NAME][metric],
                        "CI95_low": low,
                        "CI95_high": high,
                        "favorable_direction": (
                            "negative"
                            if metric == "Brier_score"
                            else "positive"
                        ),
                        "interval_scope": "Conditional_on_frozen_models;cancer_by_label_stratified_patient_bootstrap",
                    }
                )
    external_metrics = pd.DataFrame(metric_rows)
    external_groups = pd.DataFrame(group_rows)
    external_differences = pd.DataFrame(difference_rows)
    external_metrics.to_csv(
        OUT / "CPTAC_three_model_performance.tsv", sep="\t", index=False
    )
    external_groups.to_csv(
        OUT / "CPTAC_three_model_locked_threshold_groups.tsv",
        sep="\t",
        index=False,
    )
    external_differences.to_csv(
        OUT / "CPTAC_paired_comparison_vs_L2.tsv", sep="\t", index=False
    )
    yy = evaluation["label_MTAP_deep"].to_numpy(dtype=int)


### External validation diagnostics


#### Frozen-model transfer-error diagnostics


In [ ]:
@_nb9_registry.register("s40_0")
def _nb9_stage_s40_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import importlib.util
    import hashlib
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import torch
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL/08_TPM_retraining"
        )
    )
    INPUT = BASE / "01_expression_inputs_44"
    DL_LOCK = BASE / "04_locked_models_and_thresholds"
    L2_DIR = BASE / "06_three_gene_L2_comparison"
    L2_LOCK = L2_DIR / "locked_baseline"
    CPTAC_INPUT = BASE / "CPTAC_inputs_44"
    OUT = BASE / "07_transfer_error_diagnosis"
    OUT.mkdir(parents=True, exist_ok=True)
    FIGURES = OUT / "figures"
    FIGURES.mkdir(exist_ok=True)
    EXTERNAL_FILE = L2_DIR / "CPTAC_three_model_patient_predictions.tsv"
    DL_NAME = "TPM44_DL_3seed"
    L2_NAME = "TPM3_L2"
    MODEL_NAMES = [DL_NAME, L2_NAME]
    PROBABILITY_COLUMNS = {DL_NAME: "prob_DL", L2_NAME: "prob_L2"}
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    CANCER_MAP = {
        "BRCA": "BRCA",
        "CCRCC": "KIRC",
        "COAD": "COAD",
        "GBM": "GBM",
        "HNSCC": "HNSC",
        "LSCC": "LUSC",
        "LUAD": "LUAD",
        "OV": "OV",
        "PDAC": "PAAD",
        "UCEC": "UCEC",
    }
    DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )

    def read_json(path):
        with Path(path).open("r", encoding="utf-8") as handle:
            return json.load(handle)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def verify_lock(folder):
        manifest = read_json(folder / "locked_manifest.json")
        for relative_path, expected_hash in manifest[
            "locked_file_hashes"
        ].items():
            path = folder / relative_path
            if not path.is_file() or sha256_file(path) != expected_hash:
                raise ValueError(
                    "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
                )
        return manifest

    def wilson_interval(successes, n):
        if n == 0:
            return (np.nan, np.nan)
        z = 1.959963984540054
        p = successes / n
        denominator = 1 + z * z / n
        center = (p + z * z / (2 * n)) / denominator
        half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
        return (center - half, center + half)

    def assign_groups(probability, thresholds):
        p = np.asarray(probability, dtype=float)
        groups = np.full(len(p), "Uncertain", dtype=object)
        low = thresholds["low_threshold"]
        high = thresholds["high_threshold"]
        if low is not None:
            groups[p <= low] = "Predicted_non_deep"
        if high is not None:
            groups[p >= high] = "Predicted_deep_deletion"
        return groups

    def error_summary(y, probability, thresholds):
        y = np.asarray(y, dtype=int)
        p = np.asarray(probability, dtype=float)
        positive = p >= 0.5
        tp = int(((y == 1) & positive).sum())
        fp = int(((y == 0) & positive).sum())
        fn = int(((y == 1) & ~positive).sum())
        tn = int(((y == 0) & ~positive).sum())
        n_pos = tp + fn
        n_neg = fp + tn
        groups = assign_groups(p, thresholds)
        high = groups == "Predicted_deep_deletion"
        low = groups == "Predicted_non_deep"
        return {
            "n_patients": len(y),
            "n_positive": n_pos,
            "n_negative": n_neg,
            "TP_at_05": tp,
            "FP_at_05": fp,
            "FN_at_05": fn,
            "TN_at_05": tn,
            "sensitivity_at_05": tp / n_pos if n_pos else np.nan,
            "false_positive_rate_at_05": fp / n_neg if n_neg else np.nan,
            "PPV_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "n_locked_high": int(high.sum()),
            "locked_high_TP": int(y[high].sum()),
            "locked_high_FP": int((y[high] == 0).sum()),
            "locked_high_PPV": float(y[high].mean()) if high.any() else np.nan,
            "n_locked_low": int(low.sum()),
            "locked_low_FN": int(y[low].sum()),
            "locked_low_NPV": (
                float((y[low] == 0).mean()) if low.any() else np.nan
            ),
        }

    verify_lock(DL_LOCK)
    verify_lock(L2_LOCK)
    dl_thresholds = read_json(DL_LOCK / "thresholds/locked_thresholds.json")[
        DL_NAME
    ]
    l2_thresholds = read_json(L2_LOCK / "thresholds.json")
    thresholds = {DL_NAME: dl_thresholds, L2_NAME: l2_thresholds}
    network_file = DL_LOCK / "region_multitask_TPM44_network.py"
    module_spec = importlib.util.spec_from_file_location(
        "diagnostic_locked_TPM44_network", network_file
    )
    network_module = importlib.util.module_from_spec(module_spec)
    module_spec.loader.exec_module(network_module)
    dl_models = []
    for seed in [41, 42, 43]:
        checkpoint = torch.load(
            DL_LOCK / "models" / f"DL_seed{seed}.pt",
            map_location="cpu",
            weights_only=False,
        )
        network = network_module.RegionalMultiTaskNet(checkpoint)
        network.load_state_dict(checkpoint["state_dict"], strict=True)
        network.to(DEVICE)
        network.eval()
        dl_models.append((checkpoint, network))
    features = dl_models[0][0]["feature_names"]
    l2_bundle = joblib.load(L2_LOCK / "three_gene_L2.joblib")
    if l2_bundle["feature_names"] != CORE:
        raise ValueError(
            "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
        )

    @torch.no_grad()
    def predict_dl(expression, cancers):
        probabilities = []
        for checkpoint, network in dl_models:
            values = expression.loc[:, checkpoint["feature_names"]].to_numpy(
                dtype=np.float32
            )
            if not np.isfinite(values).all():
                raise ValueError(
                    "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
                )
            mean = np.asarray(
                checkpoint["preprocessing"]["mean"], dtype=np.float32
            )
            std = np.asarray(
                checkpoint["preprocessing"]["std"], dtype=np.float32
            )
            if not np.isfinite(std).all() or (std <= 0).any():
                raise ValueError(
                    "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
                )
            standardized = (values - mean) / std
            codes = pd.Series(np.asarray(cancers, dtype=str)).map(
                checkpoint["cancer_to_index"]
            )
            if codes.isna().any():
                raise ValueError(
                    "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
                )
            codes = codes.to_numpy(dtype=np.int64)
            batches = []
            for start in range(0, len(expression), 512):
                xx = torch.from_numpy(standardized[start : start + 512]).to(
                    DEVICE
                )
                cc = torch.from_numpy(codes[start : start + 512]).to(DEVICE)
                probability = torch.softmax(network(xx, cc), dim=-1)
                batches.append(
                    probability[:, checkpoint["MTAP_INDEX"], 0].cpu().numpy()
                )
            probabilities.append(np.concatenate(batches))
        return np.mean(probabilities, axis=0)

    def predict_l2(expression, cancers):
        model_input = expression.loc[:, CORE].copy()
        model_input["acronym"] = np.asarray(cancers, dtype=str)
        return l2_bundle["pipeline"].predict_proba(model_input)[:, 1]

    train_X = pd.read_csv(
        INPUT / "X_development_log2TPM_plus_1.tsv", sep="\t", index_col=0
    )
    train_X.index = train_X.index.astype(str)
    train_X = train_X.loc[:, features].apply(pd.to_numeric, errors="raise")
    meta = pd.read_csv(
        INPUT / "development_metadata.tsv",
        sep="\t",
        dtype={"patient_id": str, "acronym": str},
    ).set_index("patient_id")
    dl_oof = pd.read_csv(
        DL_LOCK / "records/development_OOF_predictions.tsv",
        sep="\t",
        dtype={"patient_id": str},
    ).set_index("patient_id")
    l2_oof = pd.read_csv(
        L2_LOCK / "development_OOF_predictions.tsv",
        sep="\t",
        dtype={"patient_id": str},
    ).set_index("patient_id")
    for name, frame in [
        ("train_X", train_X),
        ("meta", meta),
        ("dl_oof", dl_oof),
        ("l2_oof", l2_oof),
    ]:
        if frame.index.duplicated().any():
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
        if set(frame.index) != set(train_X.index):
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
    meta = meta.loc[train_X.index]
    dl_oof = dl_oof.loc[train_X.index]
    l2_oof = l2_oof.loc[train_X.index]
    y_train = pd.to_numeric(meta["label_MTAP_deep"], errors="raise").to_numpy(
        dtype=int
    )
    for name, frame in [("DL", dl_oof), ("L2", l2_oof)]:
        if not np.array_equal(
            frame["label_MTAP_deep"].to_numpy(dtype=int), y_train
        ):
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
    if not np.array_equal(
        dl_oof["outer_fold"].to_numpy(dtype=int),
        l2_oof["outer_fold"].to_numpy(dtype=int),
    ):
        raise ValueError(
            "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
        )
    p_dl_fit = predict_dl(train_X, meta["acronym"].to_numpy())
    p_l2_fit = predict_l2(train_X, meta["acronym"].to_numpy())
    development = pd.DataFrame(
        {
            "patient_id": train_X.index,
            "acronym": meta["acronym"].to_numpy(),
            "label_MTAP_deep": y_train,
            "DL_OOF": dl_oof["prob_DL"].to_numpy(),
            "DL_final_fit": p_dl_fit,
            "L2_OOF": l2_oof["prob_L2"].to_numpy(),
            "L2_final_fit": p_l2_fit,
        }
    )
    development.to_csv(
        OUT / "development_OOF_vs_final_fit_predictions.tsv",
        sep="\t",
        index=False,
    )
    distribution_rows = []
    for model, oof_column, fit_column in [
        (DL_NAME, "DL_OOF", "DL_final_fit"),
        (L2_NAME, "L2_OOF", "L2_final_fit"),
    ]:
        for truth_group, mask in [
            ("All", np.ones(len(development), dtype=bool)),
            ("DNA_deep", y_train == 1),
            ("DNA_non_deep", y_train == 0),
        ]:
            difference = (
                development.loc[mask, fit_column].to_numpy()
                - development.loc[mask, oof_column].to_numpy()
            )
            for source, column in [
                ("OOF", oof_column),
                ("Final_fit_NOT_validation", fit_column),
            ]:
                p = development.loc[mask, column].to_numpy()
                groups = assign_groups(p, thresholds[model])
                distribution_rows.append(
                    {
                        "model": model,
                        "DNA_group": truth_group,
                        "prediction_source": source,
                        "n": len(p),
                        "probability_P10": np.quantile(p, 0.1),
                        "probability_median": np.median(p),
                        "probability_P90": np.quantile(p, 0.9),
                        "fraction_ge_05": np.mean(p >= 0.5),
                        "fraction_locked_high": np.mean(
                            groups == "Predicted_deep_deletion"
                        ),
                        "median_final_minus_OOF": np.median(difference),
                    }
                )
    distribution_table = pd.DataFrame(distribution_rows)
    distribution_table.to_csv(
        OUT / "development_probability_distribution_review.tsv",
        sep="\t",
        index=False,
    )
    external = pd.read_csv(
        EXTERNAL_FILE,
        sep="\t",
        dtype={"patient_id": str, "database_cancer": str, "acronym": str},
    )
    if external["patient_id"].duplicated().any():
        raise ValueError(
            "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
        )
    external["label_MTAP_deep"] = pd.to_numeric(
        external["label_MTAP_deep"], errors="raise"
    )
    external["MTAP_GISTIC_call"] = pd.to_numeric(
        external["MTAP_GISTIC_call"], errors="raise"
    )
    if external["label_MTAP_deep"].isna().any():
        raise ValueError(
            "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
        )
    external["label_MTAP_deep"] = external["label_MTAP_deep"].astype(int)
    if not np.array_equal(
        external["MTAP_GISTIC_call"].eq(-2).astype(int),
        external["label_MTAP_deep"],
    ):
        raise ValueError(
            "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
        )
    inference_rows = []
    gene_shift_rows = []
    patient_diagnostic_tables = []
    for database_cancer, model_cancer in CANCER_MAP.items():
        mask = external["database_cancer"].eq(database_cancer)
        cohort = external.loc[mask].copy()
        if cohort.empty:
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
        path = CPTAC_INPUT / database_cancer / "X_44_log2TPM_plus_1.tsv"
        expression = pd.read_csv(path, sep="\t", index_col=0)
        expression.index = expression.index.astype(str)
        if expression.index.duplicated().any():
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
        ids = cohort["patient_id"].tolist()
        if set(ids) != set(expression.index):
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
        expression = expression.loc[ids, features].apply(
            pd.to_numeric, errors="raise"
        )
        if not np.isfinite(expression.to_numpy()).all():
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
        cancers = np.repeat(model_cancer, len(expression))
        recomputed = {
            DL_NAME: predict_dl(expression, cancers),
            L2_NAME: predict_l2(expression, cancers),
        }
        for model in MODEL_NAMES:
            saved = cohort[PROBABILITY_COLUMNS[model]].to_numpy(dtype=float)
            difference = np.abs(saved - recomputed[model])
            maximum_difference = float(difference.max())
            inference_rows.append(
                {
                    "database_cancer": database_cancer,
                    "model": model,
                    "n_patients": len(expression),
                    "max_absolute_prediction_difference": maximum_difference,
                    "consistent_at_1e_minus_5": maximum_difference <= 1e-05,
                }
            )
            if maximum_difference > 1e-05:
                pd.DataFrame(inference_rows).to_csv(
                    OUT / "inference_consistency.tsv", sep="\t", index=False
                )
                raise ValueError(
                    "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
                )
        reference_ids = meta.index[meta["acronym"].eq(model_cancer)]
        reference = train_X.loc[reference_ids]
        if reference.empty:
            raise ValueError(
                "Frozen-model transfer-error diagnostics: integrity check failed; condition: input or artifact integrity check."
            )
        reference_sd = reference.std(ddof=0).replace(0, np.nan)
        z = (expression - reference.mean()) / reference_sd
        median_shift = (
            expression.median() - reference.median()
        ) / reference_sd
        for gene in features:
            gene_shift_rows.append(
                {
                    "database_cancer": database_cancer,
                    "feature": gene,
                    "TCGA_median": reference[gene].median(),
                    "CPTAC_median": expression[gene].median(),
                    "median_shift_in_TCGA_SD": median_shift[gene],
                    "is_core_gene": gene in CORE,
                }
            )
        patient_qc = cohort[
            [
                "patient_id",
                "database_cancer",
                "acronym",
                "MTAP_GISTIC_call",
                "label_MTAP_deep",
                "prob_DL",
                "prob_L2",
            ]
        ].copy()
        patient_qc["mean_abs_expression_z"] = z.abs().mean(axis=1).to_numpy()
        patient_qc["n_features_abs_z_gt_3"] = (
            z.abs().gt(3).sum(axis=1).to_numpy()
        )
        for gene in CORE:
            patient_qc[f"{gene}_log2TPM"] = expression[gene].to_numpy()
            patient_qc[f"{gene}_diagnostic_z"] = z[gene].to_numpy()
        patient_diagnostic_tables.append(patient_qc)
    inference_table = pd.DataFrame(inference_rows)
    gene_shift_table = pd.DataFrame(gene_shift_rows)
    patient_diagnostics = pd.concat(
        patient_diagnostic_tables, ignore_index=True
    )
    inference_table.to_csv(
        OUT / "inference_consistency.tsv", sep="\t", index=False
    )
    gene_shift_table.to_csv(
        OUT / "CPTAC_gene_expression_shift.tsv", sep="\t", index=False
    )
    for model, short_name in [(DL_NAME, "DL"), (L2_NAME, "L2")]:
        p = patient_diagnostics[PROBABILITY_COLUMNS[model]].to_numpy()
        yy = patient_diagnostics["label_MTAP_deep"].to_numpy(dtype=int)
        patient_diagnostics[f"{short_name}_error_at_05"] = np.select(
            [
                (yy == 1) & (p >= 0.5),
                (yy == 0) & (p >= 0.5),
                (yy == 1) & (p < 0.5),
            ],
            ["TP", "FP", "FN"],
            default="TN",
        )
        patient_diagnostics[f"{short_name}_locked_group"] = assign_groups(
            p, thresholds[model]
        )
    patient_diagnostics.to_csv(
        OUT / "CPTAC_patient_error_diagnostics.tsv", sep="\t", index=False
    )
    cohort_rows = []
    gistic_rows = []
    expression_error_rows = []
    subsets = [("ALL_CPTAC", patient_diagnostics)]
    subsets += [
        (
            cancer,
            patient_diagnostics.loc[
                patient_diagnostics["database_cancer"].eq(cancer)
            ],
        )
        for cancer in CANCER_MAP
    ]
    for cohort_name, frame in subsets:
        yy = frame["label_MTAP_deep"].to_numpy(dtype=int)
        for model, short_name in [(DL_NAME, "DL"), (L2_NAME, "L2")]:
            probability = frame[PROBABILITY_COLUMNS[model]].to_numpy()
            cohort_rows.append(
                {
                    "cohort": cohort_name,
                    "model": model,
                    **error_summary(yy, probability, thresholds[model]),
                }
            )
            for call in [-2, -1, 0, 1, 2]:
                selected = frame["MTAP_GISTIC_call"].eq(call)
                group = frame.loc[selected]
                if group.empty:
                    continue
                p = group[PROBABILITY_COLUMNS[model]].to_numpy()
                n = len(group)
                n_predicted_positive = int((p >= 0.5).sum())
                low_ci, high_ci = wilson_interval(n_predicted_positive, n)
                gistic_rows.append(
                    {
                        "cohort": cohort_name,
                        "model": model,
                        "MTAP_GISTIC_call": call,
                        "n_patients": n,
                        "n_predicted_positive_at_05": n_predicted_positive,
                        "predicted_positive_fraction_at_05": n_predicted_positive
                        / n,
                        "rate_interpretation": (
                            "Sensitivity"
                            if call == -2
                            else "False_positive_rate"
                        ),
                        "Wilson_CI95_low": low_ci,
                        "Wilson_CI95_high": high_ci,
                        "median_probability": np.median(p),
                    }
                )
            for error in ["TP", "FP", "FN", "TN"]:
                group = frame.loc[frame[f"{short_name}_error_at_05"].eq(error)]
                if group.empty:
                    continue
                expression_error_rows.append(
                    {
                        "cohort": cohort_name,
                        "model": model,
                        "error_at_05": error,
                        "n_patients": len(group),
                        "median_probability": group[
                            PROBABILITY_COLUMNS[model]
                        ].median(),
                        "median_mean_abs_expression_z": group[
                            "mean_abs_expression_z"
                        ].median(),
                        "median_n_features_abs_z_gt_3": group[
                            "n_features_abs_z_gt_3"
                        ].median(),
                        "median_MTAP_expression_z": group[
                            "MTAP_diagnostic_z"
                        ].median(),
                        "median_CDKN2A_expression_z": group[
                            "CDKN2A_diagnostic_z"
                        ].median(),
                        "median_CDKN2B_expression_z": group[
                            "CDKN2B_diagnostic_z"
                        ].median(),
                    }
                )
    cohort_errors = pd.DataFrame(cohort_rows)
    gistic_errors = pd.DataFrame(gistic_rows)
    expression_errors = pd.DataFrame(expression_error_rows)
    cohort_errors.to_csv(
        OUT / "CPTAC_error_summary_by_cancer.tsv", sep="\t", index=False
    )
    gistic_errors.to_csv(
        OUT / "CPTAC_error_summary_by_GISTIC.tsv", sep="\t", index=False
    )
    expression_errors.to_csv(
        OUT / "CPTAC_expression_shift_by_error_group.tsv",
        sep="\t",
        index=False,
    )


#### CPTAC DNA and prediction review


In [ ]:
@_nb9_registry.register("s41_0")
def _nb9_stage_s41_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from itertools import chain
    import csv
    import gzip
    import hashlib
    import json
    import re
    import numpy as np
    import pandas as pd
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    ROOT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL"
    )
    BASE = ROOT / "08_TPM_retraining"
    RAW = PROJECT / "data_raw/CPTAC/LinkedOmicsKB"
    PREDICTION_FILE = (
        BASE
        / "06_three_gene_L2_comparison"
        / "CPTAC_three_model_patient_predictions.tsv"
    )
    FEATURE_MAPPING_FILE = (
        ROOT / "06_CPTAC_RNA_pairing" / "locked_45_feature_Ensembl_mapping.tsv"
    )
    DL_THRESHOLDS_FILE = (
        BASE
        / "04_locked_models_and_thresholds"
        / "thresholds/locked_thresholds.json"
    )
    L2_THRESHOLDS_FILE = (
        BASE
        / "06_three_gene_L2_comparison"
        / "locked_baseline/thresholds.json"
    )
    OUT = BASE / "08_neutral_sensitivity_and_DNA_review"
    OUT.mkdir(parents=True, exist_ok=True)
    FIGURES = OUT / "figures"
    FIGURES.mkdir(exist_ok=True)
    DL_NAME = "TPM44_DL_3seed"
    L2_NAME = "TPM3_L2"
    MODEL_COLUMNS = {DL_NAME: "prob_DL", L2_NAME: "prob_L2"}
    CANCERS = [
        "BRCA",
        "CCRCC",
        "COAD",
        "GBM",
        "HNSCC",
        "LSCC",
        "LUAD",
        "OV",
        "PDAC",
        "UCEC",
    ]
    GENES = ["MTAP", "CDKN2A", "CDKN2B"]
    EXPECTED_ENTREZ = {"MTAP": "4507", "CDKN2A": "1029", "CDKN2B": "1030"}
    CONTINUOUS_FILES = {}
    PURITY_FILE = None
    N_BOOTSTRAPS = 1000
    RANDOM_SEED = 20261017
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    for path in [
        RAW,
        PREDICTION_FILE,
        FEATURE_MAPPING_FILE,
        DL_THRESHOLDS_FILE,
        L2_THRESHOLDS_FILE,
    ]:
        if not path.exists():
            raise FileNotFoundError(path)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def read_json(path):
        with Path(path).open("r", encoding="utf-8") as handle:
            return json.load(handle)

    def wilson_interval(successes, n):
        if n == 0:
            return (np.nan, np.nan)
        z = 1.959963984540054
        p = successes / n
        denominator = 1 + z * z / n
        center = (p + z * z / (2 * n)) / denominator
        half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
        return (center - half, center + half)

    def performance(y, probability):
        y = np.asarray(y, dtype=int)
        p = np.asarray(probability, dtype=float)
        both_classes = len(np.unique(y)) == 2
        predicted = p >= 0.5
        tp = int(((y == 1) & predicted).sum())
        fp = int(((y == 0) & predicted).sum())
        fn = int(((y == 1) & ~predicted).sum())
        tn = int(((y == 0) & ~predicted).sum())
        return {
            "n_patients": len(y),
            "n_positive": int((y == 1).sum()),
            "n_control": int((y == 0).sum()),
            "positive_fraction": float(y.mean()),
            "AUROC": roc_auc_score(y, p) if both_classes else np.nan,
            "AP": average_precision_score(y, p) if both_classes else np.nan,
            "Brier_score": brier_score_loss(y, p),
            "TP_at_05": tp,
            "FP_at_05": fp,
            "FN_at_05": fn,
            "TN_at_05": tn,
            "PPV_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "sensitivity_at_05": tp / (tp + fn) if tp + fn else np.nan,
            "specificity_at_05": tn / (tn + fp) if tn + fp else np.nan,
        }

    def assign_groups(probability, thresholds):
        p = np.asarray(probability, dtype=float)
        result = np.full(len(p), "Uncertain", dtype=object)
        if thresholds["low_threshold"] is not None:
            result[p <= thresholds["low_threshold"]] = "Predicted_non_deep"
        if thresholds["high_threshold"] is not None:
            result[p >= thresholds["high_threshold"]] = (
                "Predicted_deep_deletion"
            )
        return result

    def group_results(y, probability, thresholds):
        y = np.asarray(y, dtype=int)
        groups = assign_groups(probability, thresholds)
        rows = []
        n_all_positive = int(y.sum())
        for group in [
            "Predicted_deep_deletion",
            "Uncertain",
            "Predicted_non_deep",
        ]:
            selected = groups == group
            n = int(selected.sum())
            n_positive = int(y[selected].sum())
            if group == "Predicted_deep_deletion":
                value_name = "PPV"
                successes = n_positive
            elif group == "Predicted_non_deep":
                value_name = "NPV"
                successes = n - n_positive
            else:
                value_name = "DNA_positive_fraction"
                successes = n_positive
            low, high = wilson_interval(successes, n)
            rows.append(
                {
                    "group": group,
                    "n_patients": n,
                    "n_DNA_positive": n_positive,
                    "coverage_fraction": n / len(y),
                    "fraction_all_DNA_positives": (
                        n_positive / n_all_positive
                        if n_all_positive
                        else np.nan
                    ),
                    "value_name": value_name,
                    "value": successes / n if n else np.nan,
                    "Wilson_CI95_low": low,
                    "Wilson_CI95_high": high,
                }
            )
        return rows

    data = pd.read_csv(
        PREDICTION_FILE,
        sep="\t",
        dtype={"patient_id": str, "database_cancer": str, "acronym": str},
    )
    if data["patient_id"].duplicated().any():
        raise ValueError(
            "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
        )
    data["MTAP_GISTIC_call"] = pd.to_numeric(
        data["MTAP_GISTIC_call"], errors="raise"
    )
    data["label_MTAP_deep"] = pd.to_numeric(
        data["label_MTAP_deep"], errors="raise"
    )
    if data[["MTAP_GISTIC_call", "label_MTAP_deep"]].isna().any().any():
        raise ValueError(
            "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
        )
    if not data["MTAP_GISTIC_call"].isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
        )
    data["label_MTAP_deep"] = data["label_MTAP_deep"].astype(int)
    if not np.array_equal(
        data["MTAP_GISTIC_call"].eq(-2).astype(int), data["label_MTAP_deep"]
    ):
        raise ValueError(
            "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
        )
    for column in MODEL_COLUMNS.values():
        data[column] = pd.to_numeric(data[column], errors="raise")
        if (
            not np.isfinite(data[column]).all()
            or not data[column].between(0, 1).all()
        ):
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
    thresholds = {
        DL_NAME: read_json(DL_THRESHOLDS_FILE)[DL_NAME],
        L2_NAME: read_json(L2_THRESHOLDS_FILE),
    }
    metric_rows = []
    threshold_rows = []
    comparisons = {
        "Deep_vs_non_deep": data,
        "Deep_vs_neutral": data.loc[
            data["MTAP_GISTIC_call"].isin([-2, 0])
        ].copy(),
    }
    for comparison_position, (comparison, frame) in enumerate(
        comparisons.items()
    ):
        subsets = [("ALL_CPTAC", frame)]
        subsets += [
            (cancer, frame.loc[frame["database_cancer"].eq(cancer)])
            for cancer in CANCERS
        ]
        for cohort, subset in subsets:
            subset = subset.reset_index(drop=True)
            if subset.empty:
                continue
            y = subset["label_MTAP_deep"].to_numpy(dtype=int)
            bootstrap_values = {
                model: {"AUROC": [], "AP": []} for model in MODEL_COLUMNS
            }
            if cohort == "ALL_CPTAC":
                strata = list(
                    subset.groupby(
                        ["database_cancer", "label_MTAP_deep"], sort=True
                    ).indices.values()
                )
                rng = np.random.default_rng(RANDOM_SEED + comparison_position)
                for bootstrap in range(N_BOOTSTRAPS):
                    indices = np.concatenate(
                        [
                            rng.choice(
                                positions, size=len(positions), replace=True
                            )
                            for positions in strata
                        ]
                    )
                    y_boot = y[indices]
                    has_both = len(np.unique(y_boot)) == 2
                    for model, column in MODEL_COLUMNS.items():
                        p_boot = subset[column].to_numpy()[indices]
                        bootstrap_values[model]["AUROC"].append(
                            roc_auc_score(y_boot, p_boot)
                            if has_both
                            else np.nan
                        )
                        bootstrap_values[model]["AP"].append(
                            average_precision_score(y_boot, p_boot)
                            if has_both
                            else np.nan
                        )
            for model, column in MODEL_COLUMNS.items():
                p = subset[column].to_numpy()
                result = performance(y, p)
                row = {
                    "comparison": comparison,
                    "cohort": cohort,
                    "model": model,
                    **result,
                }
                for metric in ["AUROC", "AP"]:
                    values = np.asarray(
                        bootstrap_values[model][metric], dtype=float
                    )
                    values = values[np.isfinite(values)]
                    if len(values):
                        low, high = np.quantile(values, [0.025, 0.975])
                    else:
                        low, high = (np.nan, np.nan)
                    row[f"{metric}_CI95_low"] = low
                    row[f"{metric}_CI95_high"] = high
                metric_rows.append(row)
                for group_row in group_results(y, p, thresholds[model]):
                    threshold_rows.append(
                        {
                            "comparison": comparison,
                            "cohort": cohort,
                            "model": model,
                            **group_row,
                        }
                    )
    metrics = pd.DataFrame(metric_rows)
    groups = pd.DataFrame(threshold_rows)
    metrics.to_csv(
        OUT / "fixed_prediction_two_comparison_performance.tsv",
        sep="\t",
        index=False,
    )
    groups.to_csv(
        OUT / "fixed_threshold_two_comparison_groups.tsv",
        sep="\t",
        index=False,
    )
    mapping = pd.read_csv(FEATURE_MAPPING_FILE, sep="\t", dtype=str)
    gene_aliases = {}
    for gene in GENES:
        row = mapping.loc[mapping["Entrez_ID"].eq(EXPECTED_ENTREZ[gene])]
        if len(row) != 1:
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        ensembl = re.sub("\\.\\d+$", "", row.iloc[0]["Ensembl_ID"])
        gene_aliases[gene] = {gene, EXPECTED_ENTREZ[gene], ensembl}

    def identify_gene(value):
        value = value.strip()
        tokens = [value] + value.split("|")
        normalized = set()
        for token in tokens:
            token = token.strip()
            normalized.add(re.sub("\\.\\d+$", "", token))
        matches = [
            gene
            for gene, aliases in gene_aliases.items()
            if normalized.intersection(aliases)
        ]
        if len(matches) > 1:
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        return matches[0] if matches else None

    def read_three_gene_matrix(path, patients):
        path = Path(path)
        opener = gzip.open if path.name.lower().endswith(".gz") else open
        patient_set = set(patients)
        with opener(path, "rt", encoding="utf-8-sig", newline="") as handle:
            reader = csv.reader(handle, delimiter="\t")
            header = next(reader)
            first_row = next(reader)
            header = [value.strip() for value in header]
            if len(first_row) == len(header) + 1:
                sample_names = header
            elif len(first_row) == len(header):
                sample_names = header[1:]
            else:
                raise ValueError(
                    "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
                )
            selected_positions = {}
            for position, sample in enumerate(sample_names):
                if sample.endswith("_A"):
                    continue
                candidate = sample[:-2] if sample.endswith("_T") else sample
                if candidate in patient_set:
                    if candidate in selected_positions:
                        raise ValueError(
                            "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
                        )
                    selected_positions[candidate] = position
            matched_patients = [
                patient
                for patient in patients
                if patient in selected_positions
            ]
            if not matched_patients:
                raise ValueError(
                    "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
                )
            positions = [
                selected_positions[patient] for patient in matched_patients
            ]
            found = {}
            for row in chain([first_row], reader):
                if not row:
                    continue
                gene = identify_gene(row[0])
                if gene is None:
                    continue
                if gene in found:
                    raise ValueError(
                        "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
                    )
                if len(row) != len(sample_names) + 1:
                    raise ValueError(
                        "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
                    )
                values = pd.Series(
                    [row[position + 1] for position in positions]
                ).replace(["", "NA", "NaN", "nan", "N/A", "."], np.nan)
                found[gene] = pd.to_numeric(values, errors="raise").to_numpy(
                    dtype=float
                )
        result = pd.DataFrame(found, index=matched_patients).reindex(
            index=patients, columns=GENES
        )
        result.index.name = "patient_id"
        if np.isinf(result.to_numpy()).any():
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        return result

    def find_continuous_file(cancer):
        if cancer in CONTINUOUS_FILES:
            path = Path(CONTINUOUS_FILES[cancer])
            if not path.is_file():
                raise FileNotFoundError(path)
            return (path, "Explicitly_specified_log2_ratio")
        candidates = []
        for path in RAW.rglob("*"):
            if not path.is_file():
                continue
            name = path.name.lower()
            if not re.match(f"^{re.escape(cancer.lower())}(?:[_\\-.])", name):
                continue
            is_text = name.endswith((".txt", ".tsv", ".txt.gz", ".tsv.gz"))
            is_dna = any((token in name for token in ["cnv", "cna", "copy"]))
            if (
                is_text
                and is_dna
                and ("log2" in name)
                and ("gistic" not in name)
            ):
                candidates.append(path)
        if len(candidates) == 0:
            return (None, "Not_found")
        if len(candidates) > 1:
            return (
                None,
                "Multiple_candidates: " + " | ".join(map(str, candidates)),
            )
        return (candidates[0], "Unique_log2_DNA_candidate")

    patient_tables = []
    file_review_rows = []
    source_hashes = {}
    for cancer in CANCERS:
        cohort = data.loc[data["database_cancer"].eq(cancer)].copy()
        patients = cohort["patient_id"].tolist()
        gistic_file = RAW / f"{cancer}_WES_CNV_gene_gistic_level.txt"
        if not gistic_file.is_file():
            raise FileNotFoundError(gistic_file)
        gistic = read_three_gene_matrix(gistic_file, patients)
        if gistic["MTAP"].isna().any():
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        if not np.array_equal(
            gistic["MTAP"].to_numpy(), cohort["MTAP_GISTIC_call"].to_numpy()
        ):
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        observed_calls = gistic.to_numpy()
        finite_calls = observed_calls[np.isfinite(observed_calls)]
        if not np.isin(finite_calls, [-2, -1, 0, 1, 2]).all():
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        source_hashes[str(gistic_file)] = sha256_file(gistic_file)
        for gene in GENES:
            cohort[f"{gene}_GISTIC_review"] = gistic[gene].to_numpy()
        continuous_file, continuous_status = find_continuous_file(cancer)
        n_mtap_continuous = 0
        continuous_error = ""
        if continuous_file is not None:
            try:
                continuous = read_three_gene_matrix(continuous_file, patients)
                if continuous["MTAP"].notna().sum() == 0:
                    raise ValueError(
                        "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
                    )
                for gene in GENES:
                    cohort[f"{gene}_DNA_log2ratio"] = continuous[
                        gene
                    ].to_numpy()
                n_mtap_continuous = int(continuous["MTAP"].notna().sum())
                source_hashes[str(continuous_file)] = sha256_file(
                    continuous_file
                )
                continuous_status = "Read_successfully"
            except Exception as error:
                continuous_error = f"{type(error).__name__}: {error}"
                continuous_status = "Read_failed"
        for gene in GENES:
            column = f"{gene}_DNA_log2ratio"
            if column not in cohort.columns:
                cohort[column] = np.nan
        file_review_rows.append(
            {
                "database_cancer": cancer,
                "n_fixed_patients": len(cohort),
                "GISTIC_file": str(gistic_file),
                "continuous_file": (
                    str(continuous_file) if continuous_file is not None else ""
                ),
                "continuous_status": continuous_status,
                "n_MTAP_continuous_available": n_mtap_continuous,
                "error": continuous_error,
            }
        )
        patient_tables.append(cohort)
    review = pd.concat(patient_tables, ignore_index=True)
    file_review = pd.DataFrame(file_review_rows)
    file_review.to_csv(
        OUT / "continuous_DNA_file_review.tsv", sep="\t", index=False
    )
    review["ABSOLUTE_purity"] = np.nan
    review["purity_source"] = ""
    purity_status = "Not_provided"
    if PURITY_FILE is not None:
        purity = pd.read_csv(
            PURITY_FILE,
            sep="\t",
            dtype={
                "database_cancer": str,
                "patient_id": str,
                "purity_source": str,
            },
        )
        required = {
            "database_cancer",
            "patient_id",
            "ABSOLUTE_purity",
            "purity_source",
        }
        if not required.issubset(purity.columns):
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        if purity.duplicated(["database_cancer", "patient_id"]).any():
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        purity["ABSOLUTE_purity"] = pd.to_numeric(
            purity["ABSOLUTE_purity"], errors="raise"
        )
        observed = purity["ABSOLUTE_purity"].dropna()
        if not observed.between(0, 1).all():
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        has_value = purity["ABSOLUTE_purity"].notna()
        if (
            purity.loc[has_value, "purity_source"]
            .fillna("")
            .str.strip()
            .eq("")
            .any()
        ):
            raise ValueError(
                "CPTAC DNA and prediction review: integrity check failed; condition: input or artifact integrity check."
            )
        review = review.drop(
            columns=["ABSOLUTE_purity", "purity_source"]
        ).merge(
            purity[list(required)],
            on=["database_cancer", "patient_id"],
            how="left",
            validate="one_to_one",
        )
        source_hashes[str(PURITY_FILE)] = sha256_file(PURITY_FILE)
        purity_status = "Provided_without_automatic_source_validation"
    a = review["CDKN2A_GISTIC_review"]
    b = review["CDKN2B_GISTIC_review"]
    review["CDKN2A_B_pattern"] = np.select(
        [
            a.isna() | b.isna(),
            a.eq(-2) & b.eq(-2),
            a.eq(-2) & ~b.eq(-2),
            ~a.eq(-2) & b.eq(-2),
        ],
        ["Unknown", "Both_A_B_deep", "A_only_deep", "B_only_deep"],
        default="Neither_A_B_deep",
    )
    y = review["label_MTAP_deep"].to_numpy(dtype=int)
    for model, short_name in [(DL_NAME, "DL"), (L2_NAME, "L2")]:
        p = review[MODEL_COLUMNS[model]].to_numpy()
        review[f"{short_name}_error_at_05"] = np.select(
            [
                (y == 1) & (p >= 0.5),
                (y == 0) & (p >= 0.5),
                (y == 1) & (p < 0.5),
            ],
            ["TP", "FP", "FN"],
            default="TN",
        )
        review[f"{short_name}_locked_group"] = assign_groups(
            p, thresholds[model]
        )
    review.to_csv(
        OUT / "patient_DNA_and_prediction_review.tsv", sep="\t", index=False
    )
    dna_summary_rows = []
    pattern_rows = []
    subsets = [("ALL_CPTAC", review)]
    subsets += [
        (cancer, review.loc[review["database_cancer"].eq(cancer)])
        for cancer in CANCERS
    ]
    for cohort_name, frame in subsets:
        for model, short_name in [(DL_NAME, "DL"), (L2_NAME, "L2")]:
            error_column = f"{short_name}_error_at_05"
            for (call, error), group in frame.groupby(
                ["MTAP_GISTIC_call", error_column], dropna=False
            ):
                row = {
                    "cohort": cohort_name,
                    "model": model,
                    "MTAP_GISTIC_call": int(call),
                    "error_at_05": error,
                    "n_patients": len(group),
                    "median_probability": group[MODEL_COLUMNS[model]].median(),
                    "n_purity_available": int(
                        group["ABSOLUTE_purity"].notna().sum()
                    ),
                    "median_purity": group["ABSOLUTE_purity"].median(),
                }
                for gene in GENES:
                    column = f"{gene}_DNA_log2ratio"
                    values = group[column].dropna()
                    row[f"{gene}_n_continuous"] = len(values)
                    row[f"{gene}_log2ratio_Q1"] = values.quantile(0.25)
                    row[f"{gene}_log2ratio_median"] = values.median()
                    row[f"{gene}_log2ratio_Q3"] = values.quantile(0.75)
                dna_summary_rows.append(row)
            counts = frame.groupby(
                ["MTAP_GISTIC_call", error_column, "CDKN2A_B_pattern"],
                dropna=False,
            ).size()
            for (call, error, pattern), n in counts.items():
                pattern_rows.append(
                    {
                        "cohort": cohort_name,
                        "model": model,
                        "MTAP_GISTIC_call": int(call),
                        "error_at_05": error,
                        "CDKN2A_B_pattern": pattern,
                        "n_patients": int(n),
                    }
                )
    dna_summary = pd.DataFrame(dna_summary_rows)
    patterns = pd.DataFrame(pattern_rows)
    dna_summary.to_csv(
        OUT / "DNA_signal_by_call_and_prediction_error.tsv",
        sep="\t",
        index=False,
    )
    patterns.to_csv(
        OUT / "CDKN2A_B_patterns_by_prediction_error.tsv",
        sep="\t",
        index=False,
    )
    colors = {DL_NAME: "#33539E", L2_NAME: "#A5678E"}
    from sklearn.metrics import roc_curve, precision_recall_curve


#### CPTAC fixed comparison summaries


In [ ]:
@_nb9_registry.register("s42_0")
def _nb9_stage_s42_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import json
    import numpy as np
    import pandas as pd
    from scipy.stats import mannwhitneyu
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    BASE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/15_region_multitask_DL/08_TPM_retraining"
        )
    )
    INPUT_FILE = (
        BASE
        / "08_neutral_sensitivity_and_DNA_review"
        / "patient_DNA_and_prediction_review.tsv"
    )
    OUT = BASE / "09_continuous_DNA_error_comparison"
    OUT.mkdir(parents=True, exist_ok=True)
    FIGURES = OUT / "figures"
    FIGURES.mkdir(exist_ok=True)
    MODELS = {"TPM44_DL_3seed": "prob_DL", "TPM3_L2": "prob_L2"}
    CANCERS = [
        "BRCA",
        "CCRCC",
        "COAD",
        "GBM",
        "HNSCC",
        "LSCC",
        "LUAD",
        "OV",
        "PDAC",
        "UCEC",
    ]
    GENES = ["MTAP", "CDKN2A", "CDKN2B"]
    MIN_N_PER_GROUP = 5
    N_BOOTSTRAPS = 2000
    RANDOM_SEED = 20261017
    DIAGNOSTIC_THRESHOLD = 0.5
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    if not INPUT_FILE.is_file():
        raise FileNotFoundError(INPUT_FILE)
    data = pd.read_csv(
        INPUT_FILE,
        sep="\t",
        dtype={"patient_id": str, "database_cancer": str, "acronym": str},
    )
    required = {
        "patient_id",
        "database_cancer",
        "MTAP_GISTIC_call",
        "label_MTAP_deep",
        *MODELS.values(),
        *[f"{gene}_DNA_log2ratio" for gene in GENES],
    }
    missing_columns = required - set(data.columns)
    if missing_columns:
        raise ValueError(
            "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if data.duplicated(["database_cancer", "patient_id"]).any():
        raise ValueError(
            "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    numeric_columns = [
        "MTAP_GISTIC_call",
        "label_MTAP_deep",
        *MODELS.values(),
        *[f"{gene}_DNA_log2ratio" for gene in GENES],
    ]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="raise")
    if data[["MTAP_GISTIC_call", "label_MTAP_deep"]].isna().any().any():
        raise ValueError(
            "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if not data["MTAP_GISTIC_call"].isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        data["MTAP_GISTIC_call"].eq(-2).astype(int),
        data["label_MTAP_deep"].to_numpy(dtype=int),
    ):
        raise ValueError(
            "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    for column in MODELS.values():
        if (
            not np.isfinite(data[column]).all()
            or not data[column].between(0, 1).all()
        ):
            raise ValueError(
                "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
            )
    for gene in GENES:
        values = data[f"{gene}_DNA_log2ratio"].dropna()
        if np.isinf(values.to_numpy()).any():
            raise ValueError(
                "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
            )
        if values.empty:
            raise ValueError(
                "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
            )
    coverage_rows = []
    for cancer in CANCERS:
        cohort = data.loc[data["database_cancer"].eq(cancer)]
        for gene in GENES:
            values = cohort[f"{gene}_DNA_log2ratio"].dropna()
            suspicious_discrete = (
                len(values) > 0 and values.isin([-2, -1, 0, 1, 2]).all()
            )
            coverage_rows.append(
                {
                    "database_cancer": cancer,
                    "gene": gene,
                    "n_patients": len(cohort),
                    "n_continuous_available": len(values),
                    "n_unique_values": values.nunique(),
                    "minimum": values.min(),
                    "median": values.median(),
                    "maximum": values.max(),
                    "all_values_are_GISTIC_categories": suspicious_discrete,
                }
            )
    coverage = pd.DataFrame(coverage_rows)
    coverage.to_csv(
        OUT / "continuous_DNA_numeric_audit.tsv", sep="\t", index=False
    )
    if coverage["all_values_are_GISTIC_categories"].any():
        raise ValueError(
            "CPTAC fixed comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    long_tables = []
    for model, probability_column in MODELS.items():
        frame = data.copy()
        y = frame["label_MTAP_deep"].to_numpy(dtype=int)
        p = frame[probability_column].to_numpy()
        frame["model"] = model
        frame["probability"] = p
        frame["prediction_error"] = np.select(
            [
                (y == 1) & (p >= DIAGNOSTIC_THRESHOLD),
                (y == 0) & (p >= DIAGNOSTIC_THRESHOLD),
                (y == 1) & (p < DIAGNOSTIC_THRESHOLD),
            ],
            ["TP", "FP", "FN"],
            default="TN",
        )
        long_tables.append(frame)
    long_data = pd.concat(long_tables, ignore_index=True)
    long_data.to_csv(
        OUT / "patient_level_DNA_error_long_table.tsv", sep="\t", index=False
    )

    def bootstrap_median_difference(error_values, correct_values, rng):
        estimates = np.empty(N_BOOTSTRAPS)
        for i in range(N_BOOTSTRAPS):
            error_sample = rng.choice(
                error_values, size=len(error_values), replace=True
            )
            correct_sample = rng.choice(
                correct_values, size=len(correct_values), replace=True
            )
            estimates[i] = np.median(error_sample) - np.median(correct_sample)
        return np.quantile(estimates, [0.025, 0.975])

    def bh_adjust(p_values):
        p_values = np.asarray(p_values, dtype=float)
        adjusted = np.full(len(p_values), np.nan)
        valid_positions = np.flatnonzero(np.isfinite(p_values))
        if len(valid_positions) == 0:
            return adjusted
        valid_p = p_values[valid_positions]
        order = np.argsort(valid_p)
        ordered_p = valid_p[order]
        m = len(ordered_p)
        ordered_q = ordered_p * m / np.arange(1, m + 1)
        ordered_q = np.minimum.accumulate(ordered_q[::-1])[::-1]
        ordered_q = np.clip(ordered_q, 0, 1)
        adjusted[valid_positions[order]] = ordered_q
        return adjusted

    rng = np.random.default_rng(RANDOM_SEED)
    comparison_rows = []
    for model in MODELS:
        for cancer in CANCERS:
            for call in [-2, -1, 0, 1, 2]:
                subset = long_data.loc[
                    long_data["model"].eq(model)
                    & long_data["database_cancer"].eq(cancer)
                    & long_data["MTAP_GISTIC_call"].eq(call)
                ]
                if subset.empty:
                    continue
                if call == -2:
                    error_name = "FN"
                    correct_name = "TP"
                else:
                    error_name = "FP"
                    correct_name = "TN"
                error_group = subset.loc[
                    subset["prediction_error"].eq(error_name)
                ]
                correct_group = subset.loc[
                    subset["prediction_error"].eq(correct_name)
                ]
                for gene in GENES:
                    column = f"{gene}_DNA_log2ratio"
                    error_values = (
                        error_group[column].dropna().to_numpy(dtype=float)
                    )
                    correct_values = (
                        correct_group[column].dropna().to_numpy(dtype=float)
                    )
                    n_error = len(error_values)
                    n_correct = len(correct_values)
                    error_median = (
                        np.median(error_values) if n_error else np.nan
                    )
                    correct_median = (
                        np.median(correct_values) if n_correct else np.nan
                    )
                    difference = (
                        error_median - correct_median
                        if n_error and n_correct
                        else np.nan
                    )
                    row = {
                        "model": model,
                        "database_cancer": cancer,
                        "MTAP_GISTIC_call": call,
                        "gene": gene,
                        "comparison": f"{error_name}_vs_{correct_name}",
                        "error_group": error_name,
                        "correct_group": correct_name,
                        "n_error_total": len(error_group),
                        "n_correct_total": len(correct_group),
                        "n_error_continuous": n_error,
                        "n_correct_continuous": n_correct,
                        "error_Q1": (
                            np.quantile(error_values, 0.25)
                            if n_error
                            else np.nan
                        ),
                        "error_median": error_median,
                        "error_Q3": (
                            np.quantile(error_values, 0.75)
                            if n_error
                            else np.nan
                        ),
                        "correct_Q1": (
                            np.quantile(correct_values, 0.25)
                            if n_correct
                            else np.nan
                        ),
                        "correct_median": correct_median,
                        "correct_Q3": (
                            np.quantile(correct_values, 0.75)
                            if n_correct
                            else np.nan
                        ),
                        "median_difference_error_minus_correct": difference,
                        "median_difference_CI95_low": np.nan,
                        "median_difference_CI95_high": np.nan,
                        "rank_biserial_error_minus_correct": np.nan,
                        "p_value": np.nan,
                        "status": "descriptive_small_or_absent_group",
                    }
                    if min(n_error, n_correct) >= MIN_N_PER_GROUP:
                        test = mannwhitneyu(
                            error_values,
                            correct_values,
                            alternative="two-sided",
                            method="auto",
                        )
                        rank_biserial = (
                            2 * test.statistic / (n_error * n_correct) - 1
                        )
                        ci_low, ci_high = bootstrap_median_difference(
                            error_values, correct_values, rng
                        )
                        row.update(
                            {
                                "median_difference_CI95_low": ci_low,
                                "median_difference_CI95_high": ci_high,
                                "rank_biserial_error_minus_correct": rank_biserial,
                                "p_value": test.pvalue,
                                "status": "exploratory_test",
                            }
                        )
                    comparison_rows.append(row)
    results = pd.DataFrame(comparison_rows)
    results["q_BH_all_new_tests"] = bh_adjust(results["p_value"])
    results.to_csv(
        OUT / "within_cancer_GISTIC_continuous_DNA_comparisons.tsv",
        sep="\t",
        index=False,
    )
    mtap_results = results.loc[results["gene"].eq("MTAP")].copy()
    mtap_results.to_csv(
        OUT / "MTAP_continuous_DNA_comparisons.tsv", sep="\t", index=False
    )
    call_summary_rows = []
    for cancer in CANCERS:
        for call in [-2, -1, 0, 1, 2]:
            subset = data.loc[
                data["database_cancer"].eq(cancer)
                & data["MTAP_GISTIC_call"].eq(call)
            ]
            if subset.empty:
                continue
            for gene in GENES:
                values = subset[f"{gene}_DNA_log2ratio"].dropna()
                call_summary_rows.append(
                    {
                        "database_cancer": cancer,
                        "MTAP_GISTIC_call": call,
                        "gene": gene,
                        "n_patients": len(subset),
                        "n_continuous": len(values),
                        "minimum": values.min(),
                        "P10": values.quantile(0.1),
                        "Q1": values.quantile(0.25),
                        "median": values.median(),
                        "Q3": values.quantile(0.75),
                        "P90": values.quantile(0.9),
                        "maximum": values.max(),
                    }
                )
    call_summary = pd.DataFrame(call_summary_rows)
    call_summary.to_csv(
        OUT / "continuous_DNA_distribution_by_GISTIC.tsv",
        sep="\t",
        index=False,
    )
    plot_rng = np.random.default_rng(RANDOM_SEED)


## 7. Single-Cell Transfer: snRNA-seq and scRNA-seq

Keep GBM LOPO fitting/adaptation distinct from GSE327167 patient-level external evaluation. Sum malignant-cell and whole-tissue counts per patient using the source all-gene UMI denominators. The supplied GSE327167 continuation still needs its verified upstream setup.


### GBM snRNA-seq preprocessing and model transfer


#### Single-cell input structure audit


In [ ]:
@_nb9_registry.register("s45_0")
def _nb9_stage_s45_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import importlib.util

    PROJECT = Path(str(PROJECT_ROOT))
    SEARCH_DIRS = [
        PROJECT / "data_raw" / "GBM",
        PROJECT / "data_processed" / "CPTAC_GBM_joint_analysis",
    ]
    H5AD_FILE = None
    if importlib.util.find_spec("h5py") is None:
        raise ImportError(
            "Single-cell input structure audit: integrity check failed; condition: input or artifact integrity check."
        )
    import h5py

    if H5AD_FILE is not None:
        paths = [Path(H5AD_FILE)]
        if not paths[0].is_file():
            raise FileNotFoundError(paths[0])
    else:
        paths = sorted(
            {
                p
                for directory in SEARCH_DIRS
                if directory.exists()
                for p in directory.rglob("*.h5ad")
            }
        )
    for i, p in enumerate(paths, 1):
        pass
    if not paths:
        pass

    def decode_attr(value):
        if isinstance(value, bytes):
            return value.decode("utf-8")
        return value

    def describe_matrix(node):
        """Inspect matrix shape and encoding without loading count values."""
        if isinstance(node, h5py.Dataset):
            return {
                "encoding": "dense",
                "shape": tuple(node.shape),
                "dtype": str(node.dtype),
            }
        shape = node.attrs.get("shape", None)
        return {
            "encoding": decode_attr(
                node.attrs.get("encoding-type", "unknown")
            ),
            "shape": (
                tuple((int(x) for x in shape)) if shape is not None else None
            ),
            "dtype": str(node["data"].dtype) if "data" in node else None,
        }

    inspect_paths = paths if H5AD_FILE is not None else paths[:10]
    for path in inspect_paths:
        try:
            with h5py.File(path, "r") as f:
                if "X" in f:
                    pass
                if "layers" in f:
                    for name, node in f["layers"].items():
                        pass
                if "raw" in f and "X" in f["raw"]:
                    pass
                for axis in ["obs", "var"]:
                    if axis not in f:
                        continue
                    group = f[axis]
                    if axis == "obs":
                        keywords = (
                            "patient",
                            "case",
                            "sample",
                            "donor",
                            "celltype",
                            "cell_type",
                            "annotation",
                            "tumor",
                            "malignan",
                            "cluster",
                            "mtap",
                        )
                        for name, node in group.items():
                            if not any((k in name.lower() for k in keywords)):
                                continue
                            if (
                                isinstance(node, h5py.Group)
                                and "categories" in node
                            ):
                                values = node["categories"][:]
                                values = [decode_attr(x) for x in values]
                            elif isinstance(node, h5py.Dataset):
                                pass
        except Exception as exc:
            pass
    _public_summary(globals(), "Single-cell input structure audit")


#### Single-nucleus HDF5 metadata inspection


In [ ]:
@_nb9_registry.register("s46_0")
def _nb9_stage_s46_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import os
    import h5py
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    GBM = PROJECT / "data_processed/CPTAC_GBM_joint_analysis"
    EXPR_FILE = GBM / "CPTAC_GBM_16patients_SCT_merged.h5ad"
    ANNOT_FILE = (
        GBM
        / "scVI_exploratory/run_20261008_151430_065424"
        / "annotation/scVI_manual_annotated.h5ad"
    )
    OUT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "16_scRNA_transfer/01_source_resolution"
    )
    OUT.mkdir(parents=True, exist_ok=True)
    for p in [EXPR_FILE, ANNOT_FILE]:
        if not p.is_file():
            raise FileNotFoundError(p)

    def decode_scalar(x):
        return x.decode("utf-8") if isinstance(x, bytes) else x

    def read_vector(node):
        """Read ordinary arrays and AnnData categorical encodings."""
        if isinstance(node, h5py.Dataset):
            values = node[:]
            if values.dtype.kind in {"O", "S", "U"}:
                return np.array(
                    [decode_scalar(x) for x in values], dtype=object
                )
            return values
        if "codes" in node and "categories" in node:
            codes = node["codes"][:]
            categories = read_vector(node["categories"])
            values = np.empty(len(codes), dtype=object)
            values[:] = pd.NA
            valid = codes >= 0
            values[valid] = categories[codes[valid]]
            return values
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )

    def read_obs(path, wanted):
        with h5py.File(path, "r") as f:
            obs = f["obs"]
            index_name = decode_scalar(obs.attrs["_index"])
            cell_ids = read_vector(obs[index_name])
            result = pd.DataFrame(index=pd.Index(cell_ids, name="cell_id"))
            for name in wanted:
                if name in obs:
                    result[name] = read_vector(obs[name])
        if not result.index.is_unique:
            raise ValueError(
                "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
            )
        return result

    expr_obs = read_obs(
        EXPR_FILE,
        ["patient_id", "MTAP_GISTIC_call", "nCount_RNA", "nCount_SCT"],
    )
    annot_obs = read_obs(
        ANNOT_FILE,
        [
            "patient_id",
            "MTAP_GISTIC_call",
            "cell_type_manual",
            "annotation_status",
            "scvi_cluster",
            "annotation_review_note",
        ],
    )
    required_expr = {"patient_id", "MTAP_GISTIC_call"}
    required_annot = {
        "patient_id",
        "MTAP_GISTIC_call",
        "cell_type_manual",
        "annotation_status",
    }
    if not required_expr.issubset(expr_obs.columns):
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )
    if not required_annot.issubset(annot_obs.columns):
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )
    if (
        len(expr_obs) != len(annot_obs)
        or not expr_obs.index.isin(annot_obs.index).all()
    ):
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )
    annot_obs = annot_obs.reindex(expr_obs.index)
    for name in ["patient_id", "MTAP_GISTIC_call"]:
        left = expr_obs[name]
        right = annot_obs[name]
        if left.isna().any() or right.isna().any():
            raise ValueError(
                "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
            )
        if name == "MTAP_GISTIC_call":
            left = pd.to_numeric(left, errors="raise")
            right = pd.to_numeric(right, errors="raise")
        else:
            left = left.astype(str)
            right = right.astype(str)
        if not left.eq(right).all():
            bad = pd.DataFrame(
                {"expression_file": left, "annotation_file": right}
            ).loc[~left.eq(right)]
            bad.to_csv(OUT / f"{name}_conflicts.tsv", sep="\t")
            raise ValueError(
                "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
            )
    meta = expr_obs.copy()
    for name in [
        "cell_type_manual",
        "annotation_status",
        "scvi_cluster",
        "annotation_review_note",
    ]:
        if name in annot_obs:
            meta[name] = annot_obs[name]
    meta["MTAP_GISTIC_call"] = pd.to_numeric(
        meta["MTAP_GISTIC_call"], errors="raise"
    ).astype(int)
    if not meta["MTAP_GISTIC_call"].isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )
    n_calls = meta.groupby("patient_id")["MTAP_GISTIC_call"].nunique()
    if (n_calls != 1).any():
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )
    tumor_mask = meta["cell_type_manual"].eq("Tumor").fillna(False)
    tumor = meta.loc[tumor_mask].copy()
    if tumor.empty:
        raise ValueError(
            "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
        )
    tumor["patient_label_MTAP_deep"] = (
        tumor["MTAP_GISTIC_call"] == -2
    ).astype(int)
    all_counts = meta.groupby("patient_id").size().rename("n_all_nuclei")
    tumor_counts = tumor.groupby("patient_id").size().rename("n_tumor_nuclei")
    patients = (
        meta.groupby("patient_id")["MTAP_GISTIC_call"]
        .first()
        .to_frame()
        .join(all_counts)
        .join(tumor_counts)
    )
    patients["n_tumor_nuclei"] = (
        patients["n_tumor_nuclei"].fillna(0).astype(int)
    )
    patients["patient_label_MTAP_deep"] = (
        patients["MTAP_GISTIC_call"] == -2
    ).astype(int)
    patients["DNA_state"] = patients["MTAP_GISTIC_call"].map(
        {
            -2: "Deep_deletion",
            -1: "Shallow_deletion",
            0: "Copy_neutral",
            1: "Gain",
            2: "Amplification",
        }
    )
    patients.to_csv(OUT / "patient_DNA_and_cell_coverage.tsv", sep="\t")
    tumor.to_csv(
        OUT / "Tumor_nucleus_metadata.tsv.gz", sep="\t", compression="gzip"
    )
    sample_rows = np.linspace(
        0, len(expr_obs) - 1, num=min(200, len(expr_obs)), dtype=int
    )
    row_sums = []
    stored_values = []
    with h5py.File(EXPR_FILE, "r") as f:
        x = f["X"]
        encoding = decode_scalar(x.attrs.get("encoding-type", ""))
        if not isinstance(x, h5py.Group) or encoding != "csr_matrix":
            raise ValueError(
                "Single-nucleus HDF5 metadata inspection: integrity check failed; condition: input or artifact integrity check."
            )
        for row in sample_rows:
            start, end = x["indptr"][int(row) : int(row) + 2]
            values = np.asarray(
                x["data"][int(start) : int(end)], dtype=np.float64
            )
            row_sums.append(values.sum())
            stored_values.append(values)
    values = np.concatenate(stored_values)
    finite = values[np.isfinite(values)]
    if len(finite):
        pass
    audit = pd.DataFrame(
        {"X_row_sum": row_sums}, index=expr_obs.index[sample_rows]
    )
    for name in ["nCount_RNA", "nCount_SCT"]:
        if name not in expr_obs:
            continue
        expected = pd.to_numeric(
            expr_obs.loc[audit.index, name], errors="coerce"
        )
        audit[name] = expected
        valid = expected.notna() & (expected > 0)
        if valid.any():
            relative_error = (
                audit.loc[valid, "X_row_sum"] - expected.loc[valid]
            ).abs() / expected.loc[valid]
    audit.to_csv(OUT / "sampled_X_row_sum_audit.tsv", sep="\t")
    search_dirs = [PROJECT / "data_raw/GBM", PROJECT / "data_raw/CPTAC", GBM]
    candidates = []
    seen = set()
    for directory in search_dirs:
        if not directory.exists():
            continue
        for current, dirs, files in os.walk(directory):
            dirs[:] = [
                d
                for d in dirs
                if not d.startswith(".")
                and d not in {"__pycache__", "figures"}
            ]
            for name in files:
                lower = name.lower()
                is_candidate = lower.endswith(
                    (
                        ".rds",
                        ".rdata",
                        ".h5seurat",
                        ".h5",
                        ".mtx",
                        ".mtx.gz",
                        ".loom",
                    )
                ) or (
                    lower.endswith((".tsv", ".tsv.gz", ".csv", ".csv.gz"))
                    and any(
                        (
                            k in lower
                            for k in ("count", "barcode", "feature", "gene")
                        )
                    )
                )
                if not is_candidate:
                    continue
                path = Path(current) / name
                if path in seen:
                    continue
                seen.add(path)
                candidates.append(
                    {
                        "path": str(path),
                        "size_MB": path.stat().st_size / 1024**2,
                    }
                )
    inventory = pd.DataFrame(candidates, columns=["path", "size_MB"])
    inventory.to_csv(
        OUT / "possible_original_count_files.tsv", sep="\t", index=False
    )
    if inventory.empty:
        pass
    else:
        priority = (
            inventory["path"]
            .str.lower()
            .str.endswith(
                (".rds", ".rdata", ".h5seurat", ".h5", ".mtx", ".mtx.gz")
            )
        )
        display = pd.concat(
            [inventory.loc[priority], inventory.loc[~priority]]
        )
        if len(display) > 60:
            pass
    _public_summary(globals(), "Single-nucleus HDF5 metadata inspection")


#### Single-nucleus HDF5 metadata confirmation


In [ ]:
@_nb9_registry.register("s47_0")
def _nb9_stage_s47_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import os
    import h5py
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    GBM = PROJECT / "data_processed/CPTAC_GBM_joint_analysis"
    EXPR_FILE = GBM / "CPTAC_GBM_16patients_SCT_merged.h5ad"
    ANNOT_FILE = (
        GBM
        / "scVI_exploratory/run_20261008_151430_065424"
        / "annotation/scVI_manual_annotated.h5ad"
    )
    OUT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "16_scRNA_transfer/01_source_resolution"
    )
    OUT.mkdir(parents=True, exist_ok=True)
    for p in [EXPR_FILE, ANNOT_FILE]:
        if not p.is_file():
            raise FileNotFoundError(p)

    def decode_scalar(x):
        return x.decode("utf-8") if isinstance(x, bytes) else x

    def read_vector(node):
        """Read ordinary arrays and AnnData categorical encodings."""
        if isinstance(node, h5py.Dataset):
            values = node[:]
            if values.dtype.kind in {"O", "S", "U"}:
                return np.array(
                    [decode_scalar(x) for x in values], dtype=object
                )
            return values
        if "codes" in node and "categories" in node:
            codes = node["codes"][:]
            categories = read_vector(node["categories"])
            values = np.empty(len(codes), dtype=object)
            values[:] = pd.NA
            valid = codes >= 0
            values[valid] = categories[codes[valid]]
            return values
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )

    def read_obs(path, wanted):
        with h5py.File(path, "r") as f:
            obs = f["obs"]
            index_name = decode_scalar(obs.attrs["_index"])
            cell_ids = read_vector(obs[index_name])
            result = pd.DataFrame(index=pd.Index(cell_ids, name="cell_id"))
            for name in wanted:
                if name in obs:
                    result[name] = read_vector(obs[name])
        if not result.index.is_unique:
            raise ValueError(
                "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
            )
        return result

    expr_obs = read_obs(
        EXPR_FILE,
        ["patient_id", "MTAP_GISTIC_call", "nCount_RNA", "nCount_SCT"],
    )
    annot_obs = read_obs(
        ANNOT_FILE,
        [
            "patient_id",
            "MTAP_GISTIC_call",
            "cell_type_manual",
            "annotation_status",
            "scvi_cluster",
            "annotation_review_note",
        ],
    )
    required_expr = {"patient_id", "MTAP_GISTIC_call"}
    required_annot = {
        "patient_id",
        "MTAP_GISTIC_call",
        "cell_type_manual",
        "annotation_status",
    }
    if not required_expr.issubset(expr_obs.columns):
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )
    if not required_annot.issubset(annot_obs.columns):
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )
    if (
        len(expr_obs) != len(annot_obs)
        or not expr_obs.index.isin(annot_obs.index).all()
    ):
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )
    annot_obs = annot_obs.reindex(expr_obs.index)
    for name in ["patient_id", "MTAP_GISTIC_call"]:
        left = expr_obs[name]
        right = annot_obs[name]
        if left.isna().any() or right.isna().any():
            raise ValueError(
                "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
            )
        if name == "MTAP_GISTIC_call":
            left = pd.to_numeric(left, errors="raise")
            right = pd.to_numeric(right, errors="raise")
        else:
            left = left.astype(str)
            right = right.astype(str)
        if not left.eq(right).all():
            bad = pd.DataFrame(
                {"expression_file": left, "annotation_file": right}
            ).loc[~left.eq(right)]
            bad.to_csv(OUT / f"{name}_conflicts.tsv", sep="\t")
            raise ValueError(
                "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
            )
    meta = expr_obs.copy()
    for name in [
        "cell_type_manual",
        "annotation_status",
        "scvi_cluster",
        "annotation_review_note",
    ]:
        if name in annot_obs:
            meta[name] = annot_obs[name]
    meta["MTAP_GISTIC_call"] = pd.to_numeric(
        meta["MTAP_GISTIC_call"], errors="raise"
    ).astype(int)
    if not meta["MTAP_GISTIC_call"].isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )
    n_calls = meta.groupby("patient_id")["MTAP_GISTIC_call"].nunique()
    if (n_calls != 1).any():
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )
    tumor_mask = meta["cell_type_manual"].eq("Tumor").fillna(False)
    tumor = meta.loc[tumor_mask].copy()
    if tumor.empty:
        raise ValueError(
            "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
        )
    tumor["patient_label_MTAP_deep"] = (
        tumor["MTAP_GISTIC_call"] == -2
    ).astype(int)
    all_counts = meta.groupby("patient_id").size().rename("n_all_nuclei")
    tumor_counts = tumor.groupby("patient_id").size().rename("n_tumor_nuclei")
    patients = (
        meta.groupby("patient_id")["MTAP_GISTIC_call"]
        .first()
        .to_frame()
        .join(all_counts)
        .join(tumor_counts)
    )
    patients["n_tumor_nuclei"] = (
        patients["n_tumor_nuclei"].fillna(0).astype(int)
    )
    patients["patient_label_MTAP_deep"] = (
        patients["MTAP_GISTIC_call"] == -2
    ).astype(int)
    patients["DNA_state"] = patients["MTAP_GISTIC_call"].map(
        {
            -2: "Deep_deletion",
            -1: "Shallow_deletion",
            0: "Copy_neutral",
            1: "Gain",
            2: "Amplification",
        }
    )
    patients.to_csv(OUT / "patient_DNA_and_cell_coverage.tsv", sep="\t")
    tumor.to_csv(
        OUT / "Tumor_nucleus_metadata.tsv.gz", sep="\t", compression="gzip"
    )
    sample_rows = np.linspace(
        0, len(expr_obs) - 1, num=min(200, len(expr_obs)), dtype=int
    )
    row_sums = []
    stored_values = []
    with h5py.File(EXPR_FILE, "r") as f:
        x = f["X"]
        encoding = decode_scalar(x.attrs.get("encoding-type", ""))
        if not isinstance(x, h5py.Group) or encoding != "csr_matrix":
            raise ValueError(
                "Single-nucleus HDF5 metadata confirmation: integrity check failed; condition: input or artifact integrity check."
            )
        for row in sample_rows:
            start, end = x["indptr"][int(row) : int(row) + 2]
            values = np.asarray(
                x["data"][int(start) : int(end)], dtype=np.float64
            )
            row_sums.append(values.sum())
            stored_values.append(values)
    values = np.concatenate(stored_values)
    finite = values[np.isfinite(values)]
    if len(finite):
        pass
    audit = pd.DataFrame(
        {"X_row_sum": row_sums}, index=expr_obs.index[sample_rows]
    )
    for name in ["nCount_RNA", "nCount_SCT"]:
        if name not in expr_obs:
            continue
        expected = pd.to_numeric(
            expr_obs.loc[audit.index, name], errors="coerce"
        )
        audit[name] = expected
        valid = expected.notna() & (expected > 0)
        if valid.any():
            relative_error = (
                audit.loc[valid, "X_row_sum"] - expected.loc[valid]
            ).abs() / expected.loc[valid]
    audit.to_csv(OUT / "sampled_X_row_sum_audit.tsv", sep="\t")
    search_dirs = [PROJECT / "data_raw/GBM", PROJECT / "data_raw/CPTAC", GBM]
    candidates = []
    seen = set()
    for directory in search_dirs:
        if not directory.exists():
            continue
        for current, dirs, files in os.walk(directory):
            dirs[:] = [
                d
                for d in dirs
                if not d.startswith(".")
                and d not in {"__pycache__", "figures"}
            ]
            for name in files:
                lower = name.lower()
                is_candidate = lower.endswith(
                    (
                        ".rds",
                        ".rdata",
                        ".h5seurat",
                        ".h5",
                        ".mtx",
                        ".mtx.gz",
                        ".loom",
                    )
                ) or (
                    lower.endswith((".tsv", ".tsv.gz", ".csv", ".csv.gz"))
                    and any(
                        (
                            k in lower
                            for k in ("count", "barcode", "feature", "gene")
                        )
                    )
                )
                if not is_candidate:
                    continue
                path = Path(current) / name
                if path in seen:
                    continue
                seen.add(path)
                candidates.append(
                    {
                        "path": str(path),
                        "size_MB": path.stat().st_size / 1024**2,
                    }
                )
    inventory = pd.DataFrame(candidates, columns=["path", "size_MB"])
    inventory.to_csv(
        OUT / "possible_original_count_files.tsv", sep="\t", index=False
    )
    if inventory.empty:
        pass
    else:
        priority = (
            inventory["path"]
            .str.lower()
            .str.endswith(
                (".rds", ".rdata", ".h5seurat", ".h5", ".mtx", ".mtx.gz")
            )
        )
        display = pd.concat(
            [inventory.loc[priority], inventory.loc[~priority]]
        )
        if len(display) > 60:
            pass
    _public_summary(globals(), "Single-nucleus HDF5 metadata confirmation")


#### Representative HDF5 structure inspection


In [ ]:
@_nb9_registry.register("s48_0")
def _nb9_stage_s48_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import h5py
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    SOURCE = PROJECT / "data_raw/GBM/NIHMS1665743/snRNA"
    OUT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "16_scRNA_transfer/02_HDF5_expression_review"
    )
    OUT.mkdir(parents=True, exist_ok=True)
    files = sorted(SOURCE.glob("*_seurat.hdf5"))
    if not files:
        raise FileNotFoundError(
            "Representative HDF5 structure inspection: integrity check failed; condition: input or artifact integrity check."
        )
    for path in files:
        pass
    inventory = []
    for path in files:
        patient = path.name.removesuffix("_seurat.hdf5")
        with h5py.File(path, "r") as f:
            records = []

            def visitor(name, obj):
                if isinstance(obj, h5py.Dataset):
                    interesting = (
                        obj.ndim == 2
                        or any(
                            (
                                key in name.lower()
                                for key in (
                                    "count",
                                    "raw",
                                    "umi",
                                    "barcode",
                                    "cellid",
                                )
                            )
                        )
                        or name.endswith(("/indices", "/indptr"))
                    )
                    if interesting:
                        records.append(
                            {
                                "patient_id": patient,
                                "file": str(path),
                                "dataset": name,
                                "shape": str(tuple(obj.shape)),
                                "dtype": str(obj.dtype),
                            }
                        )

            f.visititems(visitor)
            inventory.extend(records)
            for group_name in ["layers", "assays"]:
                if group_name in f and isinstance(f[group_name], h5py.Group):
                    pass
            matrix_rows = [
                r for r in records if len(f[r["dataset"]].shape) == 2
            ]
            for row in matrix_rows:
                pass
    inventory = pd.DataFrame(inventory)
    inventory.to_csv(
        OUT / "HDF5_expression_dataset_inventory.tsv", sep="\t", index=False
    )
    example = files[0]

    def as_text(x):
        return x.decode("utf-8") if isinstance(x, bytes) else str(x)

    results = []
    with h5py.File(example, "r") as f:
        required = ["row_attrs/Gene", "col_attrs/CellID"]
        missing = [name for name in required if name not in f]
        if missing:
            raise ValueError(
                "Representative HDF5 structure inspection: integrity check failed; condition: input or artifact integrity check."
            )
        n_genes = f["row_attrs/Gene"].size
        n_cells = f["col_attrs/CellID"].size
        selected_cells = np.unique(
            np.linspace(0, n_cells - 1, num=min(8, n_cells), dtype=int)
        )
        expected_counts = {}
        for field in ["nCount_RNA", "nCount_SCT"]:
            location = f"col_attrs/{field}"
            if location in f:
                values = np.asarray(f[location][:]).ravel()
                if len(values) != n_cells:
                    raise ValueError(
                        "Representative HDF5 structure inspection: integrity check failed; condition: input or artifact integrity check."
                    )
                expected_counts[field] = values[selected_cells].astype(float)
        matrices = []

        def find_matrices(name, obj):
            if not isinstance(obj, h5py.Dataset):
                return
            if obj.ndim != 2 or obj.dtype.kind not in "iuf":
                return
            if obj.shape in [(n_genes, n_cells), (n_cells, n_genes)]:
                matrices.append(name)

        f.visititems(find_matrices)
        check_names = [
            name
            for name in matrices
            if (
                "count" in name.lower()
                or "raw" in name.lower()
                or "umi" in name.lower()
                or (name == "matrix")
                or name.lower().endswith("/rna")
            )
            and "scale" not in name.lower()
        ]
        for name in check_names:
            ds = f[name]
            if n_genes == n_cells:
                raise ValueError(
                    "Representative HDF5 structure inspection: integrity check failed; condition: input or artifact integrity check."
                )
            genes_by_cells = ds.shape == (n_genes, n_cells)
            sums = np.zeros(len(selected_cells), dtype=np.float64)
            n_values = 0
            n_nonfinite = 0
            n_negative = 0
            n_noninteger = 0
            for start in range(0, n_genes, 512):
                end = min(start + 512, n_genes)
                if genes_by_cells:
                    block = np.asarray(
                        ds[start:end, selected_cells], dtype=np.float64
                    )
                else:
                    block = np.asarray(
                        ds[selected_cells, start:end], dtype=np.float64
                    ).T
                sums += block.sum(axis=0)
                finite_mask = np.isfinite(block)
                values = block[finite_mask]
                n_values += block.size
                n_nonfinite += int((~finite_mask).sum())
                n_negative += int((values < 0).sum())
                n_noninteger += int(
                    (
                        ~np.isclose(
                            values, np.rint(values), atol=1e-05, rtol=0
                        )
                    ).sum()
                )
            row = {
                "file": example.name,
                "matrix": name,
                "n_sampled_cells": len(selected_cells),
                "n_checked_values": n_values,
                "n_nonfinite": n_nonfinite,
                "n_negative": n_negative,
                "noninteger_fraction_among_finite": (
                    n_noninteger / (n_values - n_nonfinite)
                    if n_values > n_nonfinite
                    else np.nan
                ),
            }
            for field, expected in expected_counts.items():
                valid = (
                    np.isfinite(expected) & np.isfinite(sums) & (expected > 0)
                )
                row[f"median_relative_error_vs_{field}"] = (
                    float(
                        np.median(
                            np.abs(sums[valid] - expected[valid])
                            / expected[valid]
                        )
                    )
                    if valid.any()
                    else np.nan
                )
            results.append(row)
            for key, value in row.items():
                if key not in {"file", "matrix"}:
                    pass
    if results:
        pd.DataFrame(results).to_csv(
            OUT / "representative_matrix_count_comparison.tsv",
            sep="\t",
            index=False,
        )
    _public_summary(globals(), "Representative HDF5 structure inspection")


#### Public GBM source and annotation resolution


In [ ]:
@_nb9_registry.register("s49_0")
def _nb9_stage_s49_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import re
    import urllib.parse
    import urllib.request
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    PATIENT_FILE = (
        BASE / "01_source_resolution/patient_DNA_and_cell_coverage.tsv"
    )
    OUT = BASE / "03_GDC_original_counts_search"
    OUT.mkdir(parents=True, exist_ok=True)
    if not PATIENT_FILE.is_file():
        raise FileNotFoundError(PATIENT_FILE)
    patients = pd.read_csv(PATIENT_FILE, sep="\t", dtype={"patient_id": str})
    required = {"patient_id", "MTAP_GISTIC_call"}
    if not required.issubset(patients.columns):
        raise ValueError(
            "Public GBM source and annotation resolution: integrity check failed; condition: input or artifact integrity check."
        )
    if patients["patient_id"].isna().any():
        raise ValueError(
            "Public GBM source and annotation resolution: integrity check failed; condition: input or artifact integrity check."
        )
    if patients["patient_id"].duplicated().any():
        raise ValueError(
            "Public GBM source and annotation resolution: integrity check failed; condition: input or artifact integrity check."
        )
    patient_ids = patients["patient_id"].tolist()
    filters = {
        "op": "in",
        "content": {"field": "cases.submitter_id", "value": patient_ids},
    }
    fields = [
        "file_id",
        "file_name",
        "file_size",
        "data_category",
        "data_type",
        "data_format",
        "experimental_strategy",
        "access",
        "md5sum",
        "cases.submitter_id",
        "cases.project.project_id",
        "analysis.workflow_type",
    ]
    hits = []
    offset = 0
    page_size = 500
    while True:
        params = {
            "filters": json.dumps(filters),
            "fields": ",".join(fields),
            "format": "JSON",
            "size": page_size,
            "from": offset,
        }
        url = "https://api.gdc.cancer.gov/files?" + urllib.parse.urlencode(
            params
        )
        request = urllib.request.Request(
            url, headers={"Accept": "application/json"}
        )
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.load(response)
        except Exception as exc:
            raise RuntimeError(
                "Public GBM source and annotation resolution: integrity check failed; condition: input or artifact integrity check."
            ) from exc
        data = payload["data"]
        batch = data.get("hits", [])
        total = int(data["pagination"]["total"])
        hits.extend(batch)
        if len(hits) >= total:
            break
        if not batch:
            raise RuntimeError(
                "Public GBM source and annotation resolution: integrity check failed; condition: input or artifact integrity check."
            )
        offset += len(batch)
    with open(OUT / "GDC_file_metadata.json", "w", encoding="utf-8") as handle:
        json.dump(hits, handle, ensure_ascii=False, indent=2)
    records = []
    for hit in hits:
        cases = hit.get("cases", [])
        case_ids = sorted(
            {
                case.get("submitter_id", "")
                for case in cases
                if case.get("submitter_id")
            }
        )
        projects = sorted(
            {
                case.get("project", {}).get("project_id", "")
                for case in cases
                if case.get("project", {}).get("project_id")
            }
        )
        analysis = hit.get("analysis") or {}
        if isinstance(analysis, dict):
            workflow = analysis.get("workflow_type", "")
        else:
            workflow = ""
        records.append(
            {
                "file_id": hit.get("file_id", ""),
                "file_name": hit.get("file_name", ""),
                "file_size_MB": float(hit.get("file_size") or 0) / 1024**2,
                "data_category": hit.get("data_category", ""),
                "data_type": hit.get("data_type", ""),
                "data_format": hit.get("data_format", ""),
                "experimental_strategy": hit.get("experimental_strategy", ""),
                "access": hit.get("access", ""),
                "workflow_type": workflow,
                "patient_ids": ";".join(case_ids),
                "projects": ";".join(projects),
                "md5sum": hit.get("md5sum", ""),
            }
        )
    columns = [
        "file_id",
        "file_name",
        "file_size_MB",
        "data_category",
        "data_type",
        "data_format",
        "experimental_strategy",
        "access",
        "workflow_type",
        "patient_ids",
        "projects",
        "md5sum",
    ]
    table = pd.DataFrame(records, columns=columns)
    table.to_csv(OUT / "all_patient_GDC_files.tsv", sep="\t", index=False)
    if table.empty:
        pass
    else:
        summary = (
            table.groupby(
                [
                    "experimental_strategy",
                    "data_type",
                    "data_format",
                    "access",
                ],
                dropna=False,
            )
            .size()
            .rename("n_files")
            .reset_index()
        )
        summary.to_csv(
            OUT / "GDC_file_type_summary.tsv", sep="\t", index=False
        )
        text = (
            table[
                [
                    "file_name",
                    "experimental_strategy",
                    "data_type",
                    "workflow_type",
                ]
            ]
            .fillna("")
            .astype(str)
            .agg(" ".join, axis=1)
        )
        explicit_single_cell = text.str.contains(
            "single[\\s_-]*(cell|nucle)|scrna|snrna|10x|chromium|cell[\\s_-]*ranger|seurat",
            case=False,
            regex=True,
        )
        rna_or_expression = text.str.contains(
            "rna|expression|transcript", case=False, regex=True
        ) | table["data_category"].str.contains(
            "Transcriptome", case=False, na=False
        )
        candidates = table.loc[explicit_single_cell | rna_or_expression].copy()
        candidates["explicit_single_cell_keyword"] = explicit_single_cell.loc[
            candidates.index
        ]
        candidates.to_csv(
            OUT / "RNA_and_single_cell_candidates.tsv", sep="\t", index=False
        )
        show_columns = [
            "file_name",
            "experimental_strategy",
            "data_type",
            "data_format",
            "access",
            "file_size_MB",
            "explicit_single_cell_keyword",
        ]
        display = candidates.sort_values(
            ["explicit_single_cell_keyword", "access", "file_size_MB"],
            ascending=[False, True, True],
        )
        if len(display) > 60:
            pass
        matched = {
            case_id
            for value in table["patient_ids"]
            for case_id in value.split(";")
            if case_id
        }
    _public_summary(globals(), "Public GBM source and annotation resolution")


#### GBM UMI matrix acquisition and barcode matching


In [ ]:
@_nb9_registry.register("s50_0")
def _nb9_stage_s50_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import gzip
    import hashlib
    import json
    import shutil
    import tarfile
    import urllib.request
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    SEARCH = BASE / "03_GDC_original_counts_search"
    PATIENT_FILE = (
        BASE / "01_source_resolution/patient_DNA_and_cell_coverage.tsv"
    )
    TUMOR_FILE = BASE / "01_source_resolution/Tumor_nucleus_metadata.tsv.gz"
    GDC_FILE = SEARCH / "all_patient_GDC_files.tsv"
    DOWNLOAD = PROJECT / "data_raw/GBM/NIHMS1665743/snRNA_UMI_MEX"
    OUT = BASE / "04_UMI_download_and_barcode_matching"
    DOWNLOAD.mkdir(parents=True, exist_ok=True)
    OUT.mkdir(parents=True, exist_ok=True)
    for path in [PATIENT_FILE, TUMOR_FILE, GDC_FILE]:
        if not path.is_file():
            raise FileNotFoundError(path)
    patients = pd.read_csv(PATIENT_FILE, sep="\t", dtype={"patient_id": str})
    tumor = pd.read_csv(
        TUMOR_FILE, sep="\t", dtype={"patient_id": str, "cell_barcode": str}
    )
    files = pd.read_csv(GDC_FILE, sep="\t", dtype=str, keep_default_na=False)
    if "cell_barcode" not in tumor.columns:
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    if patients["patient_id"].duplicated().any():
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    expected_patients = set(patients["patient_id"])
    if set(tumor["patient_id"]) != expected_patients:
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    if tumor.duplicated(["patient_id", "cell_barcode"]).any():
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    mask = (
        files["experimental_strategy"].eq("scRNA-Seq")
        & files["data_type"].eq("Gene Expression Quantification")
        & files["data_format"].eq("MEX")
        & files["access"].eq("open")
        & files["file_name"].str.endswith(
            ".qc_filtered_feature_bc_matrix.tar.gz"
        )
    )
    selected = files.loc[mask].copy()

    def resolve_patient(value):
        ids = {x.strip() for x in str(value).split(";") if x.strip()}
        if len(ids) != 1 or not ids.issubset(expected_patients):
            raise ValueError(
                "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
            )
        return next(iter(ids))

    selected["patient_id"] = selected["patient_ids"].map(resolve_patient)
    selected.to_csv(
        OUT / "selected_QC_filtered_MEX_files.tsv", sep="\t", index=False
    )
    coverage = (
        selected.groupby("patient_id")
        .size()
        .reindex(sorted(expected_patients), fill_value=0)
    )
    if not coverage.eq(1).all():
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    if selected["file_id"].duplicated().any():
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    if not selected["md5sum"].str.fullmatch("[0-9a-fA-F]{32}").all():
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )
    total_mb = pd.to_numeric(selected["file_size_MB"], errors="raise").sum()

    def md5_file(path):
        h = hashlib.md5()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    def download_checked(file_id, destination, expected_md5):
        expected_md5 = expected_md5.lower()
        if destination.is_file():
            if md5_file(destination) == expected_md5:
                return
            raise ValueError(
                "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
            )
        partial = destination.with_name(destination.name + ".part")
        url = f"https://api.gdc.cancer.gov/data/{file_id}"
        last_error = None
        for attempt in range(1, 4):
            try:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "MTAP-research"}
                )
                with urllib.request.urlopen(
                    request, timeout=60
                ) as response, open(partial, "wb") as handle:
                    shutil.copyfileobj(
                        response, handle, length=8 * 1024 * 1024
                    )
                actual_md5 = md5_file(partial)
                if actual_md5 != expected_md5:
                    raise ValueError(
                        "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
                    )
                partial.replace(destination)
                return
            except Exception as exc:
                last_error = exc
        raise RuntimeError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        ) from last_error

    def extract_mex(archive, directory):
        directory.mkdir(parents=True, exist_ok=True)
        accepted = {
            "matrix": {"matrix.mtx", "matrix.mtx.gz"},
            "features": {
                "features.tsv",
                "features.tsv.gz",
                "genes.tsv",
                "genes.tsv.gz",
            },
            "barcodes": {"barcodes.tsv", "barcodes.tsv.gz"},
        }
        outputs = {}
        with tarfile.open(archive, "r:gz") as tf:
            members = [m for m in tf.getmembers() if m.isfile()]
            for role, names in accepted.items():
                matches = [m for m in members if Path(m.name).name in names]
                if len(matches) != 1:
                    raise ValueError(
                        "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
                    )
                member = matches[0]
                target = directory / Path(member.name).name
                partial = target.with_name(target.name + ".part")
                source = tf.extractfile(member)
                if source is None:
                    raise ValueError(
                        "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
                    )
                with source, open(partial, "wb") as handle:
                    shutil.copyfileobj(source, handle, length=8 * 1024 * 1024)
                partial.replace(target)
                outputs[role] = target
        return outputs

    def open_text(path):
        if path.name.endswith(".gz"):
            return gzip.open(path, "rt", encoding="utf-8")
        return open(path, "r", encoding="utf-8")

    def read_mtx_dimensions(path):
        with open_text(path) as handle:
            header = handle.readline().strip()
            if not header.lower().startswith(
                "%%matrixmarket matrix coordinate"
            ):
                raise ValueError(
                    "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
                )
            for line in handle:
                line = line.strip()
                if line and (not line.startswith("%")):
                    values = line.split()
                    if len(values) != 3:
                        raise ValueError(
                            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
                        )
                    return (header, tuple(map(int, values)))
        raise ValueError(
            "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
        )

    audit_rows = []
    for row in selected.sort_values("patient_id").itertuples(index=False):
        patient = row.patient_id
        patient_dir = DOWNLOAD / patient
        patient_dir.mkdir(parents=True, exist_ok=True)
        archive = patient_dir / Path(row.file_name).name
        download_checked(row.file_id, archive, row.md5sum)
        extracted = extract_mex(archive, patient_dir / "matrix")
        header, (n_features, n_barcodes, nnz) = read_mtx_dimensions(
            extracted["matrix"]
        )
        with open_text(extracted["barcodes"]) as handle:
            barcodes = [line.rstrip("\r\n").split("\t")[0] for line in handle]
        with open_text(extracted["features"]) as handle:
            feature_count = sum((1 for _ in handle))
        if len(barcodes) != n_barcodes:
            raise ValueError(
                "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
            )
        if feature_count != n_features:
            raise ValueError(
                "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
            )
        if len(set(barcodes)) != len(barcodes):
            raise ValueError(
                "GBM UMI matrix acquisition and barcode matching: integrity check failed; condition: input or artifact integrity check."
            )
        existing = tumor.loc[tumor["patient_id"].eq(patient), "cell_barcode"]
        matched = existing.isin(set(barcodes))
        missing = existing.loc[~matched]
        if len(missing):
            missing.to_frame().to_csv(
                OUT / f"{patient}_unmatched_Tumor_barcodes.tsv",
                sep="\t",
                index=False,
            )
        audit_rows.append(
            {
                "patient_id": patient,
                "GDC_file_id": row.file_id,
                "matrix_file": str(extracted["matrix"]),
                "features_file": str(extracted["features"]),
                "barcodes_file": str(extracted["barcodes"]),
                "matrix_header": header,
                "n_features": n_features,
                "n_barcodes": n_barcodes,
                "n_stored_entries": nnz,
                "n_existing_Tumor_nuclei": len(existing),
                "n_matched_Tumor_nuclei": int(matched.sum()),
                "n_unmatched_Tumor_nuclei": len(missing),
                "archive_MD5_verified": True,
            }
        )
        pd.DataFrame(audit_rows).to_csv(
            OUT / "UMI_MEX_download_and_matching.tsv", sep="\t", index=False
        )
    audit = pd.DataFrame(audit_rows)
    if audit["n_unmatched_Tumor_nuclei"].sum() > 0:
        pass
    _public_summary(
        globals(), "GBM UMI matrix acquisition and barcode matching"
    )


#### GBM raw UMI source verification


In [ ]:
@_nb9_registry.register("s51_0")
def _nb9_stage_s51_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import shutil
    import h5py
    import numpy as np
    import pandas as pd

    PROJECT = Path(str(PROJECT_ROOT))
    H5AD = (
        PROJECT
        / "data_processed/CPTAC_GBM_joint_analysis"
        / "CPTAC_GBM_16patients_SCT_merged.h5ad"
    )
    TUMOR_FILE = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "16_scRNA_transfer/01_source_resolution"
        / "Tumor_nucleus_metadata.tsv.gz"
    )

    def decode(x):
        return x.decode("utf-8") if isinstance(x, bytes) else x

    def read_vector(node):
        if isinstance(node, h5py.Dataset):
            values = node[:]
            if values.dtype.kind in {"O", "S", "U"}:
                return np.array([decode(x) for x in values], dtype=object)
            return values
        if "codes" in node and "categories" in node:
            codes = node["codes"][:]
            categories = read_vector(node["categories"])
            values = np.empty(len(codes), dtype=object)
            values[:] = pd.NA
            valid = codes >= 0
            values[valid] = categories[codes[valid]]
            return values
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )

    tumor = pd.read_csv(
        TUMOR_FILE, sep="\t", dtype={"cell_id": str, "patient_id": str}
    )
    if not {"cell_id", "patient_id"}.issubset(tumor.columns):
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    with h5py.File(H5AD, "r") as f:
        obs = f["obs"]
        index_name = decode(obs.attrs["_index"])
        reference = pd.DataFrame(
            {
                "cell_id": read_vector(obs[index_name]),
                "patient_id": read_vector(obs["patient_id"]),
                "cell_barcode": read_vector(obs["cell_barcode"]),
            }
        )
    if reference.isna().any().any():
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    if reference["cell_id"].duplicated().any():
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    if tumor["cell_id"].isna().any() or tumor["cell_id"].duplicated().any():
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    reference = reference.set_index("cell_id")
    if not tumor["cell_id"].isin(reference.index).all():
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    matched = reference.loc[tumor["cell_id"]]
    if not np.array_equal(
        tumor["patient_id"].astype(str).to_numpy(),
        matched["patient_id"].astype(str).to_numpy(),
    ):
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    barcodes = matched["cell_barcode"].astype(str).to_numpy()
    if "cell_barcode" in tumor.columns:
        if not np.array_equal(
            tumor["cell_barcode"].astype(str).to_numpy(), barcodes
        ):
            raise ValueError(
                "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
            )
    else:
        tumor["cell_barcode"] = barcodes
    if tumor.duplicated(["patient_id", "cell_barcode"]).any():
        raise ValueError(
            "GBM raw UMI source verification: integrity check failed; condition: input or artifact integrity check."
        )
    backup = TUMOR_FILE.with_name(
        "Tumor_nucleus_metadata.before_barcode_fix.tsv.gz"
    )
    if not backup.exists():
        shutil.copy2(TUMOR_FILE, backup)
    temporary = TUMOR_FILE.with_name("Tumor_nucleus_metadata.repaired.tmp.gz")
    tumor.to_csv(temporary, sep="\t", index=False, compression="gzip")
    temporary.replace(TUMOR_FILE)
    _public_summary(globals(), "GBM raw UMI source verification")


#### Patient-level malignant-cell 44-gene pseudobulk construction


In [ ]:
@_nb9_registry.register("s53_0")
def _nb9_stage_s53_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import gc
    import gzip
    import json
    import numpy as np
    import pandas as pd
    from scipy import sparse
    from scipy.io import mmread

    PROJECT = Path(str(PROJECT_ROOT))
    MODEL_ROOT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL"
    )
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    MEX_MANIFEST = (
        BASE
        / "04_UMI_download_and_barcode_matching"
        / "UMI_MEX_download_and_matching.tsv"
    )
    TUMOR_FILE = BASE / "01_source_resolution/Tumor_nucleus_metadata.tsv.gz"
    PATIENT_FILE = (
        BASE / "01_source_resolution/patient_DNA_and_cell_coverage.tsv"
    )
    TRAIN_EXPRESSION = (
        MODEL_ROOT
        / "08_TPM_retraining/01_expression_inputs_44"
        / "X_development_log2TPM_plus_1.tsv"
    )
    GENE_MAPPING = (
        MODEL_ROOT
        / "06_CPTAC_RNA_pairing"
        / "locked_45_feature_Ensembl_mapping.tsv"
    )
    OUT = BASE / "05_Tumor_UMI_and_pseudobulk"
    CELL_OUT = OUT / "patient_cell_caches"
    OUT.mkdir(parents=True, exist_ok=True)
    CELL_OUT.mkdir(parents=True, exist_ok=True)
    for path in [
        MEX_MANIFEST,
        TUMOR_FILE,
        PATIENT_FILE,
        TRAIN_EXPRESSION,
        GENE_MAPPING,
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)
    completion_file = OUT / "preparation_manifest.json"
    if completion_file.exists():
        raise FileExistsError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    header = pd.read_csv(TRAIN_EXPRESSION, sep="\t", nrows=0).columns.tolist()
    features = [x for x in header if x not in {"patient_id", "Unnamed: 0"}]
    if len(features) != 44 or len(set(features)) != 44:
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    mapping = pd.read_csv(
        GENE_MAPPING, sep="\t", dtype=str, keep_default_na=False
    )
    feature_mapping = []
    for feature in features:
        exact_rows = mapping.loc[mapping.eq(feature).any(axis=1)]
        if exact_rows.empty:
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        ensembl_ids = set()
        for value in exact_rows.to_numpy().ravel():
            value = str(value).strip()
            if (
                pd.Series([value])
                .str.fullmatch("ENSG\\d+(?:\\.\\d+)?")
                .iloc[0]
            ):
                ensembl_ids.add(value.split(".")[0])
        if len(ensembl_ids) != 1:
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        feature_mapping.append(
            {
                "feature_order": len(feature_mapping),
                "feature": feature,
                "Ensembl_ID": next(iter(ensembl_ids)),
            }
        )
    feature_mapping = pd.DataFrame(feature_mapping)
    if feature_mapping["Ensembl_ID"].duplicated().any():
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    feature_mapping.to_csv(
        OUT / "fixed_44_feature_mapping.tsv", sep="\t", index=False
    )
    target_ids = feature_mapping["Ensembl_ID"].tolist()
    mex = pd.read_csv(MEX_MANIFEST, sep="\t", dtype={"patient_id": str})
    tumor = pd.read_csv(
        TUMOR_FILE,
        sep="\t",
        dtype={"patient_id": str, "cell_id": str, "cell_barcode": str},
    )
    patients = pd.read_csv(
        PATIENT_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    required = {"patient_id", "cell_id", "cell_barcode", "MTAP_GISTIC_call"}
    if not required.issubset(tumor.columns):
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    if mex["patient_id"].duplicated().any():
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    if not patients.index.is_unique:
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    if tumor.duplicated(["patient_id", "cell_barcode"]).any():
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    if tumor["cell_id"].isna().any() or tumor["cell_id"].duplicated().any():
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )
    expected = set(patients.index)
    if (
        set(mex["patient_id"]) != expected
        or set(tumor["patient_id"]) != expected
    ):
        raise ValueError(
            "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
        )

    def open_text(path):
        path = Path(path)
        if path.name.endswith(".gz"):
            return gzip.open(path, "rt", encoding="utf-8")
        return open(path, "r", encoding="utf-8")

    pb_counts = {}
    pb_logcpm = {}
    patient_audit = []
    coverage_rows = []
    for record in mex.sort_values("patient_id").itertuples(index=False):
        patient = record.patient_id
        matrix_path = Path(record.matrix_file)
        features_path = Path(record.features_file)
        barcodes_path = Path(record.barcodes_file)
        for path in [matrix_path, features_path, barcodes_path]:
            if not path.is_file():
                raise FileNotFoundError(path)
        gene_table = pd.read_csv(
            features_path,
            sep="\t",
            header=None,
            dtype=str,
            keep_default_na=False,
        )
        if gene_table.shape[1] not in {2, 3}:
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        with open_text(barcodes_path) as handle:
            barcodes = pd.Index([line.rstrip("\r\n") for line in handle])
        if not barcodes.is_unique:
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        gene_ids = gene_table.iloc[:, 0].str.replace(
            "\\.\\d+$", "", regex=True
        )
        if gene_table.shape[1] == 3:
            gene_expression_mask = (
                gene_table.iloc[:, 2].eq("Gene Expression").to_numpy()
            )
            if not gene_expression_mask.any():
                raise ValueError(
                    "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
                )
        else:
            if not gene_ids.str.fullmatch("ENSG\\d+").all():
                raise ValueError(
                    "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
                )
            gene_expression_mask = np.ones(len(gene_table), dtype=bool)
        expressed_gene_ids = pd.Index(gene_ids.loc[gene_expression_mask])
        if not expressed_gene_ids.is_unique:
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        positions = expressed_gene_ids.get_indexer(target_ids)
        for feature, ensembl, position in zip(features, target_ids, positions):
            coverage_rows.append(
                {
                    "patient_id": patient,
                    "feature": feature,
                    "Ensembl_ID": ensembl,
                    "present_in_matrix": position >= 0,
                }
            )
        pd.DataFrame(coverage_rows).to_csv(
            OUT / "44_feature_coverage.tsv", sep="\t", index=False
        )
        if (positions < 0).any():
            missing = [features[i] for i in np.flatnonzero(positions < 0)]
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        cell_meta = tumor.loc[tumor["patient_id"].eq(patient)].copy()
        calls = pd.to_numeric(cell_meta["MTAP_GISTIC_call"], errors="raise")
        expected_call = int(patients.loc[patient, "MTAP_GISTIC_call"])
        if calls.isna().any() or not calls.eq(expected_call).all():
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        cell_positions = barcodes.get_indexer(cell_meta["cell_barcode"])
        if (cell_positions < 0).any():
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        with open(matrix_path, "rb") as raw_handle:
            if matrix_path.name.endswith(".gz"):
                with gzip.GzipFile(fileobj=raw_handle) as handle:
                    raw = mmread(handle)
            else:
                raw = mmread(raw_handle)
        if not sparse.issparse(raw):
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        if raw.shape != (len(gene_table), len(barcodes)):
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        for start in range(0, len(raw.data), 1000000):
            values = raw.data[start : start + 1000000]
            if not np.isfinite(values).all() or (values < 0).any():
                raise ValueError(
                    "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
                )
            if not np.issubdtype(values.dtype, np.integer):
                if not np.equal(values, np.rint(values)).all():
                    raise ValueError(
                        "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
                    )
            if values.size and values.max() > np.iinfo(np.int64).max:
                raise ValueError(
                    "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
                )
        raw = raw.astype(np.int64, copy=False).tocsc()
        counts = raw[gene_expression_mask, :][:, cell_positions].tocsc()
        del raw
        library_umi = np.asarray(counts.sum(axis=0)).ravel().astype(np.int64)
        if (library_umi <= 0).any():
            raise ValueError(
                "Patient-level malignant-cell 44-gene pseudobulk construction: integrity check failed; condition: input or artifact integrity check."
            )
        region_counts = counts[positions, :].T.tocsr()
        region_total = (
            np.asarray(region_counts.sum(axis=0)).ravel().astype(np.int64)
        )
        total_umi = int(library_umi.sum())
        logcpm = np.log2(
            1.0 + region_total.astype(np.float64) / total_umi * 1000000.0
        )
        pb_counts[patient] = region_total
        pb_logcpm[patient] = logcpm
        sparse.save_npz(
            CELL_OUT / f"{patient}_Tumor_44_UMI.npz", region_counts
        )
        cell_meta["total_gene_expression_UMI"] = library_umi
        cell_meta["region_44_UMI"] = np.asarray(
            region_counts.sum(axis=1)
        ).ravel()
        cell_meta.to_csv(
            CELL_OUT / f"{patient}_Tumor_metadata.tsv.gz",
            sep="\t",
            index=False,
            compression="gzip",
        )
        patient_audit.append(
            {
                "patient_id": patient,
                "MTAP_GISTIC_call": expected_call,
                "patient_label_MTAP_deep": int(expected_call == -2),
                "n_tumor_nuclei": len(cell_meta),
                "n_gene_expression_features": counts.shape[0],
                "n_model_features_present": len(positions),
                "total_Tumor_UMI": total_umi,
                "median_UMI_per_nucleus": float(np.median(library_umi)),
                "n_model_features_zero_in_pseudobulk": int(
                    (region_total == 0).sum()
                ),
                "annotation_status": (
                    ";".join(
                        sorted(
                            cell_meta["annotation_status"]
                            .dropna()
                            .astype(str)
                            .unique()
                        )
                    )
                    if "annotation_status" in cell_meta
                    else "Not_recorded"
                ),
            }
        )
        pd.DataFrame(patient_audit).to_csv(
            OUT / "patient_input_audit.tsv", sep="\t", index=False
        )
        del counts, region_counts
        gc.collect()
    counts_df = pd.DataFrame.from_dict(
        pb_counts, orient="index", columns=features
    ).sort_index()
    expression_df = pd.DataFrame.from_dict(
        pb_logcpm, orient="index", columns=features
    ).sort_index()
    counts_df.index.name = "patient_id"
    expression_df.index.name = "patient_id"
    counts_df.to_csv(OUT / "Tumor_pseudobulk_44_UMI.tsv", sep="\t")
    expression_df.to_csv(
        OUT / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv", sep="\t"
    )
    audit = pd.DataFrame(patient_audit).sort_values("patient_id")
    audit.to_csv(OUT / "patient_input_audit.tsv", sep="\t", index=False)
    assert len(expression_df) == len(expected)
    assert expression_df.shape[1] == 44
    assert np.isfinite(expression_df.to_numpy()).all()
    assert int(audit["n_tumor_nuclei"].sum()) == len(tumor)
    manifest = {
        "status": "completed",
        "n_patients": len(expression_df),
        "n_tumor_nuclei": len(tumor),
        "n_features": len(features),
        "feature_order": features,
        "input_type": "QC-filtered sc/snRNA gene counts",
        "pseudobulk_transform": "log2(1 + summed_gene_UMI / summed_all_gene_expression_UMI * 1e6)",
        "annotation": "Existing Tumor annotation retained",
        "training_performed": False,
        "source_mex_manifest": str(MEX_MANIFEST),
    }
    with open(completion_file, "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    _public_summary(
        globals(),
        "Patient-level malignant-cell 44-gene pseudobulk construction",
    )


#### GBM three-gene L2 leave-one-patient-out evaluation


In [ ]:
@_nb9_registry.register("s55_0")
def _nb9_stage_s55_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import json
    import re
    import numpy as np
    import pandas as pd
    import torch
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import LeaveOneOut
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        confusion_matrix,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    TRANSFER = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    INPUT = TRANSFER / "05_Tumor_UMI_and_pseudobulk"
    EXPRESSION_FILE = INPUT / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    METADATA_FILE = INPUT / "patient_input_audit.tsv"
    MODEL_BASE = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL/08_TPM_retraining"
    )
    MODEL_DIRS = [
        MODEL_BASE / "04_locked_models_and_thresholds",
        MODEL_BASE / "02_DL_training",
    ]
    OUT = TRANSFER / "06_LOPO_baseline_and_DL_interface"
    OUT.mkdir(parents=True, exist_ok=True)
    for path in [EXPRESSION_FILE, METADATA_FILE]:
        if not path.is_file():
            raise FileNotFoundError(path)
    X = pd.read_csv(EXPRESSION_FILE, sep="\t", index_col="patient_id")
    meta = pd.read_csv(
        METADATA_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    if not X.index.is_unique or not meta.index.is_unique:
        raise ValueError(
            "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    if set(X.index) != set(meta.index):
        raise ValueError(
            "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    X = X.sort_index()
    meta = meta.reindex(X.index)
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(
            "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    if not set(CORE).issubset(X.columns):
        raise ValueError(
            "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    calls = pd.to_numeric(meta["MTAP_GISTIC_call"], errors="raise")
    if calls.isna().any() or not calls.isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
        )
    y = calls.eq(-2).astype(int)
    if "patient_label_MTAP_deep" in meta:
        saved_y = pd.to_numeric(
            meta["patient_label_MTAP_deep"], errors="raise"
        )
        if not saved_y.eq(y).all():
            raise ValueError(
                "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
            )
    predictions = []
    coefficient_rows = []
    loo = LeaveOneOut()
    for fold, (train_idx, valid_idx) in enumerate(loo.split(X), start=1):
        train_ids = X.index[train_idx]
        valid_ids = X.index[valid_idx]
        X_train = X.loc[train_ids, CORE]
        X_valid = X.loc[valid_ids, CORE]
        y_train = y.loc[train_ids]
        if y_train.nunique() != 2:
            raise ValueError(
                "GBM three-gene L2 leave-one-patient-out evaluation: integrity check failed; condition: input or artifact integrity check."
            )
        model = Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        solver="lbfgs",
                        class_weight=None,
                        max_iter=5000,
                        random_state=41,
                    ),
                ),
            ]
        )
        model.fit(X_train, y_train)
        positive_index = list(model.named_steps["model"].classes_).index(1)
        probability = float(model.predict_proba(X_valid)[0, positive_index])
        patient = valid_ids[0]
        predictions.append(
            {
                "patient_id": patient,
                "outer_fold": fold,
                "MTAP_GISTIC_call": int(calls.loc[patient]),
                "label_MTAP_deep": int(y.loc[patient]),
                "probability": probability,
                "prediction_at_05": int(probability >= 0.5),
                "n_training_patients": len(train_ids),
                "n_training_positive": int(y_train.sum()),
            }
        )
        coefficients = model.named_steps["model"].coef_[0]
        for gene, coefficient in zip(CORE, coefficients):
            coefficient_rows.append(
                {
                    "outer_fold": fold,
                    "held_out_patient": patient,
                    "gene": gene,
                    "standardized_coefficient": float(coefficient),
                }
            )
    pred = pd.DataFrame(predictions)
    p = pred["probability"].to_numpy()
    truth = pred["label_MTAP_deep"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(truth, p >= 0.5, labels=[0, 1]).ravel()
    metrics = pd.DataFrame(
        [
            {
                "model": "snRNA_three_gene_L2",
                "evaluation": "Leave_one_patient_out",
                "n_patients": len(pred),
                "n_positive": int(truth.sum()),
                "AUROC": roc_auc_score(truth, p),
                "AP": average_precision_score(truth, p),
                "Brier_score": brier_score_loss(truth, p),
                "log_loss": log_loss(
                    truth, np.clip(p, 1e-07, 1 - 1e-07), labels=[0, 1]
                ),
                "TP_at_05": int(tp),
                "FP_at_05": int(fp),
                "FN_at_05": int(fn),
                "TN_at_05": int(tn),
                "sensitivity_at_05": tp / (tp + fn),
                "specificity_at_05": tn / (tn + fp),
                "PPV_at_05": tp / (tp + fp) if tp + fp else np.nan,
            }
        ]
    )
    pred.to_csv(
        OUT / "three_gene_L2_LOPO_predictions.tsv", sep="\t", index=False
    )
    metrics.to_csv(
        OUT / "three_gene_L2_LOPO_metrics.tsv", sep="\t", index=False
    )
    pd.DataFrame(coefficient_rows).to_csv(
        OUT / "three_gene_L2_LOPO_coefficients.tsv", sep="\t", index=False
    )

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    def describe(value, depth=0):
        """Summarize checkpoint structure without displaying full weight matrices."""
        if isinstance(value, torch.Tensor):
            return {
                "type": "Tensor",
                "shape": list(value.shape),
                "dtype": str(value.dtype),
            }
        if isinstance(value, np.ndarray):
            result = {
                "type": "ndarray",
                "shape": list(value.shape),
                "dtype": str(value.dtype),
            }
            if value.size <= 100:
                result["values"] = value.tolist()
            return result
        if isinstance(value, dict):
            if depth > 6:
                return {"type": "dict", "keys": [str(k) for k in value.keys()]}
            return {str(k): describe(v, depth + 1) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            if len(value) <= 100:
                return [describe(v, depth + 1) for v in value]
            return {
                "type": type(value).__name__,
                "length": len(value),
                "first_items": [describe(v, depth + 1) for v in value[:5]],
            }
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return {
            "type": type(value).__name__,
            "representation": repr(value)[:500],
        }

    report = []

    def add(text):
        report.append(str(text))

    checkpoints = []
    chosen_dir = None
    for directory in MODEL_DIRS:
        if directory.exists():
            found = sorted(
                (
                    p
                    for p in directory.rglob("*")
                    if p.is_file() and p.suffix.lower() in {".pt", ".pth"}
                )
            )
            if found:
                checkpoints = found
                chosen_dir = directory
                break
    add("===== DL checkpoint candidates =====")
    add(f"Selected directory: {chosen_dir}")
    for path in checkpoints:
        add(str(path))
    seed41 = [
        p
        for p in checkpoints
        if re.search("seed[_-]?41(?!\\d)", p.stem, flags=re.I)
    ]
    if len(seed41) == 1:
        checkpoint_path = seed41[0]
        add("\n===== Seed-41 checkpoint structure =====")
        add(f"File: {checkpoint_path}")
        add(f"SHA256：{sha256_file(checkpoint_path)}")
        try:
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=False
            )
            add(
                json.dumps(
                    describe(checkpoint),
                    ensure_ascii=False,
                    indent=2,
                    default=str,
                )
            )
            del checkpoint
        except Exception as exc:
            add(f"Read failed: {type(exc).__name__}: {exc}")
    else:
        add(
            "\nA unique seed-41 checkpoint was not found; candidates were recorded."
        )
    sources = sorted(
        {
            p
            for directory in MODEL_DIRS
            if directory.exists()
            for p in directory.rglob("*.py")
            if any(
                (
                    keyword in p.name.lower()
                    for keyword in ("network", "architecture")
                )
            )
        }
    )
    add("\n===== Saved network definition =====")
    if not sources:
        add(
            "No network or architecture Python file was found in these directories."
        )
    for path in sources:
        add(f"\nFile: {path}")
        add(f"SHA256：{sha256_file(path)}")
        if path.stat().st_size <= 300000:
            add(path.read_text(encoding="utf-8"))
        else:
            add("File exceeds 300 KB; full contents were not printed.")
    for directory in MODEL_DIRS:
        if not directory.exists():
            continue
        for path in sorted(directory.rglob("*manifest*.json")):
            if path.stat().st_size > 200000:
                continue
            add(f"\nConfiguration: {path} =====")
            add(path.read_text(encoding="utf-8"))
    report_path = OUT / "DL_loading_interface_review.txt"
    report_path.write_text("\n".join(report), encoding="utf-8")
    _public_summary(
        globals(), "GBM three-gene L2 leave-one-patient-out evaluation"
    )


#### GBM frozen bulk DL with a bounded input adapter


In [ ]:
@_nb9_registry.register("s57_0")
def _nb9_stage_s57_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import importlib.util
    import json
    import sys
    import time
    import numpy as np
    import pandas as pd
    import torch
    from torch import nn
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        log_loss,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    INPUT = BASE / "05_Tumor_UMI_and_pseudobulk"
    EXPRESSION_FILE = INPUT / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    METADATA_FILE = INPUT / "patient_input_audit.tsv"
    L2_FILE = (
        BASE
        / "06_LOPO_baseline_and_DL_interface"
        / "three_gene_L2_LOPO_predictions.tsv"
    )
    LOCKED = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL/08_TPM_retraining"
        / "04_locked_models_and_thresholds"
    )
    NETWORK_FILE = LOCKED / "region_multitask_TPM44_network.py"
    LOCKED_MANIFEST = LOCKED / "locked_manifest.json"
    SEEDS = [41, 42, 43]
    CHECKPOINTS = [LOCKED / "models" / f"DL_seed{seed}.pt" for seed in SEEDS]
    OUT = BASE / "07_frozen_DL_and_region_adapter_v1"
    MODEL_OUT = OUT / "LOPO_adapters"
    FIG_OUT = OUT / "figures"
    ADAPTER_EPOCHS = 200
    ADAPTER_LR = 0.02
    IDENTITY_PENALTY = 0.05
    MAX_LOG_SCALE = 0.5
    MAX_SHIFT = 1.0
    DEVICE = torch.device("cpu")
    torch.set_num_threads(1)
    torch.manual_seed(41)
    np.random.seed(41)
    for path in [
        EXPRESSION_FILE,
        METADATA_FILE,
        L2_FILE,
        NETWORK_FILE,
        LOCKED_MANIFEST,
        *CHECKPOINTS,
    ]:
        if not path.is_file():
            raise FileNotFoundError(path)
    OUT.mkdir(parents=True, exist_ok=True)
    MODEL_OUT.mkdir(parents=True, exist_ok=True)
    FIG_OUT.mkdir(parents=True, exist_ok=True)
    completion_file = OUT / "analysis_manifest.json"
    if completion_file.exists():
        raise FileExistsError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    start_time = time.time()

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    with open(LOCKED_MANIFEST, encoding="utf-8") as handle:
        locked_manifest = json.load(handle)
    locked_hashes = locked_manifest["locked_file_hashes"]
    source_hashes = {}
    for path in [NETWORK_FILE, *CHECKPOINTS]:
        relative = path.relative_to(LOCKED).as_posix()
        if relative not in locked_hashes:
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
        actual = sha256_file(path)
        if actual != locked_hashes[relative]:
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
        source_hashes[str(path)] = actual
    module_name = "mtap_locked_TPM44_network_for_snRNA"
    spec = importlib.util.spec_from_file_location(module_name, NETWORK_FILE)
    network_module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = network_module
    spec.loader.exec_module(network_module)
    Network = network_module.RegionalMultiTaskNet
    checkpoints = [
        torch.load(path, map_location="cpu", weights_only=False)
        for path in CHECKPOINTS
    ]
    reference = checkpoints[0]
    features = list(reference["feature_names"])
    regions = reference["region_indices"]
    mtap_index = int(reference["MTAP_INDEX"])
    if len(features) != 44 or len(set(features)) != 44:
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if len(regions) != 4:
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if sorted((i for region in regions for i in region)) != list(range(44)):
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if str(reference["target_Entrez_order"][mtap_index]) != "4507":
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if reference["target_symbols"][mtap_index] != "MTAP":
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    for cp in checkpoints:
        for key in [
            "feature_names",
            "region_indices",
            "core_indices",
            "target_Entrez_order",
            "target_symbols",
            "MTAP_INDEX",
            "cancer_to_index",
            "architecture",
        ]:
            if cp[key] != reference[key]:
                raise ValueError(
                    "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
                )
        if cp["expression_unit"] != "log2(TPM+1)":
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
        if "GBM" not in cp["cancer_to_index"]:
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
    X = pd.read_csv(EXPRESSION_FILE, sep="\t", index_col="patient_id")
    X.index = X.index.astype(str)
    meta = pd.read_csv(
        METADATA_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    l2 = pd.read_csv(L2_FILE, sep="\t", dtype={"patient_id": str}).set_index(
        "patient_id"
    )
    for name, frame in [
        ("Expression", X),
        ("Patient metadata", meta),
        ("L2 predictions", l2),
    ]:
        if not frame.index.is_unique:
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
    if not set(X.index) == set(meta.index) == set(l2.index):
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if set(X.columns) != set(features) or len(X.columns) != 44:
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    X = X.sort_index().loc[:, features]
    meta = meta.reindex(X.index)
    l2 = l2.reindex(X.index)
    raw_array = X.to_numpy(dtype=np.float32)
    if not np.isfinite(raw_array).all() or (raw_array < 0).any():
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    calls = pd.to_numeric(meta["MTAP_GISTIC_call"], errors="raise")
    if calls.isna().any() or not calls.isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    y = calls.eq(-2).astype(int).to_numpy()
    if not np.array_equal(
        pd.to_numeric(l2["label_MTAP_deep"], errors="raise").to_numpy(), y
    ):
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        pd.to_numeric(l2["MTAP_GISTIC_call"], errors="raise").to_numpy(),
        calls.to_numpy(),
    ):
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    l2_probability = pd.to_numeric(l2["probability"], errors="raise").to_numpy(
        dtype=float
    )
    if (
        not np.isfinite(l2_probability).all()
        or (l2_probability < 0).any()
        or (l2_probability > 1).any()
    ):
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    if len(X) != 16 or int(y.sum()) != 8:
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    raw_tensor = torch.tensor(raw_array, dtype=torch.float32, device=DEVICE)
    y_tensor = torch.tensor(y, dtype=torch.float32, device=DEVICE)

    class FrozenBulkEnsemble(nn.Module):

        def __init__(self, checkpoint_list):
            super().__init__()
            self.networks = nn.ModuleList()
            self.mtap_indices = []
            self.cancer_indices = []
            for i, cp in enumerate(checkpoint_list):
                model = Network(cp)
                model.load_state_dict(cp["state_dict"], strict=True)
                model.to(DEVICE)
                model.eval()
                for parameter in model.parameters():
                    parameter.requires_grad_(False)
                mean = np.asarray(
                    cp["preprocessing"]["mean"], dtype=np.float32
                )
                std = np.asarray(cp["preprocessing"]["std"], dtype=np.float32)
                if (
                    mean.shape != (44,)
                    or std.shape != (44,)
                    or (not np.isfinite(mean).all())
                    or (not np.isfinite(std).all())
                    or (std <= 0).any()
                ):
                    raise ValueError(
                        "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
                    )
                self.register_buffer(
                    f"mean_{i}", torch.tensor(mean, device=DEVICE)
                )
                self.register_buffer(
                    f"std_{i}", torch.tensor(std, device=DEVICE)
                )
                self.networks.append(model)
                self.mtap_indices.append(int(cp["MTAP_INDEX"]))
                self.cancer_indices.append(int(cp["cancer_to_index"]["GBM"]))

        def forward(self, expression, adapter=None):
            positive_logs = []
            negative_logs = []
            for i, network in enumerate(self.networks):
                z = (expression - getattr(self, f"mean_{i}")) / getattr(
                    self, f"std_{i}"
                )
                if adapter is not None:
                    z = adapter(z)
                cancer = torch.full(
                    (len(expression),),
                    self.cancer_indices[i],
                    dtype=torch.long,
                    device=expression.device,
                )
                logits = network(z, cancer)
                mtap_logits = logits[:, self.mtap_indices[i], :]
                log_class = torch.log_softmax(mtap_logits, dim=1)
                positive_logs.append(log_class[:, 0])
                negative_logs.append(torch.logsumexp(log_class[:, 1:], dim=1))
            normalizer = np.log(len(self.networks))
            log_p = (
                torch.logsumexp(torch.stack(positive_logs), dim=0) - normalizer
            )
            log_q = (
                torch.logsumexp(torch.stack(negative_logs), dim=0) - normalizer
            )
            return (log_p, log_q)

    ensemble = FrozenBulkEnsemble(checkpoints).to(DEVICE)
    ensemble.eval()
    assert not any((p.requires_grad for p in ensemble.parameters()))

    class RegionAffineAdapter(nn.Module):

        def __init__(self, region_indices):
            super().__init__()
            membership = torch.empty(44, dtype=torch.long)
            for region_id, indices in enumerate(region_indices):
                membership[indices] = region_id
            self.register_buffer("membership", membership)
            self.scale_raw = nn.Parameter(torch.zeros(4))
            self.shift_raw = nn.Parameter(torch.zeros(4))

        def coefficients(self):
            scale = torch.exp(MAX_LOG_SCALE * torch.tanh(self.scale_raw))
            shift = MAX_SHIFT * torch.tanh(self.shift_raw)
            return (scale, shift)

        def forward(self, z):
            scale, shift = self.coefficients()
            return z * scale[self.membership] + shift[self.membership]

        def identity_penalty(self):
            scale, shift = self.coefficients()
            return (scale - 1.0).square().mean() + shift.square().mean()

    identity_adapter = RegionAffineAdapter(regions).to(DEVICE)
    with torch.no_grad():
        direct_logp, _ = ensemble(raw_tensor)
        identity_logp, _ = ensemble(raw_tensor, identity_adapter)
        identity_error = float(
            (direct_logp.exp() - identity_logp.exp()).abs().max()
        )
        direct_probability = direct_logp.exp().cpu().numpy()
    if identity_error > 1e-07:
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    n_adapter_parameters = sum(
        (p.numel() for p in identity_adapter.parameters())
    )
    del identity_adapter
    adapted_probability = np.full(len(X), np.nan)
    training_rows = []
    coefficient_rows = []
    for held_out in range(len(X)):
        patient = X.index[held_out]
        train_idx = np.array([i for i in range(len(X)) if i != held_out])
        valid_idx = np.array([held_out])
        train_x = raw_tensor[train_idx]
        train_y = y_tensor[train_idx]
        valid_x = raw_tensor[valid_idx]
        if len(np.unique(y[train_idx])) != 2:
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
        adapter = RegionAffineAdapter(regions).to(DEVICE)
        optimizer = torch.optim.Adam(
            adapter.parameters(), lr=ADAPTER_LR, weight_decay=0.0
        )
        ensemble.eval()
        adapter.train()
        initial_bce = None
        for epoch in range(1, ADAPTER_EPOCHS + 1):
            optimizer.zero_grad(set_to_none=True)
            log_p, log_q = ensemble(train_x, adapter)
            bce = -(train_y * log_p + (1.0 - train_y) * log_q).mean()
            regularization = adapter.identity_penalty()
            loss = bce + IDENTITY_PENALTY * regularization
            if not torch.isfinite(loss):
                raise ValueError(
                    "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
                )
            if initial_bce is None:
                initial_bce = float(bce.detach())
            loss.backward()
            for parameter in adapter.parameters():
                if (
                    parameter.grad is None
                    or not torch.isfinite(parameter.grad).all()
                ):
                    raise ValueError(
                        "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
                    )
            torch.nn.utils.clip_grad_norm_(adapter.parameters(), max_norm=5.0)
            optimizer.step()
        adapter.eval()
        with torch.no_grad():
            valid_logp, _ = ensemble(valid_x, adapter)
            probability = float(valid_logp.exp().item())
            train_logp, train_logq = ensemble(train_x, adapter)
            final_bce = float(
                -(train_y * train_logp + (1.0 - train_y) * train_logq)
                .mean()
                .item()
            )
            final_penalty = float(adapter.identity_penalty().item())
            scales, shifts = adapter.coefficients()
            scales = scales.cpu().numpy()
            shifts = shifts.cpu().numpy()
        adapted_probability[held_out] = probability
        training_rows.append(
            {
                "outer_fold": held_out + 1,
                "held_out_patient": patient,
                "n_training_patients": len(train_idx),
                "n_training_positive": int(y[train_idx].sum()),
                "epochs": ADAPTER_EPOCHS,
                "initial_training_BCE": initial_bce,
                "final_training_BCE": final_bce,
                "final_identity_penalty": final_penalty,
            }
        )
        for region_id, (scale, shift) in enumerate(zip(scales, shifts)):
            coefficient_rows.append(
                {
                    "outer_fold": held_out + 1,
                    "held_out_patient": patient,
                    "region_index": region_id,
                    "scale": float(scale),
                    "shift": float(shift),
                }
            )
        torch.save(
            {
                "adapter_state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v in adapter.state_dict().items()
                },
                "held_out_patient": patient,
                "training_patients": X.index[train_idx].tolist(),
                "feature_names": features,
                "region_indices": regions,
                "adapter_input_space": "Frozen_TCGA_mean_std_standardized_log2CPM_plus_1",
                "source_checkpoints": [str(p) for p in CHECKPOINTS],
                "epochs": ADAPTER_EPOCHS,
                "learning_rate": ADAPTER_LR,
                "identity_penalty_weight": IDENTITY_PENALTY,
                "max_log_scale": MAX_LOG_SCALE,
                "max_shift": MAX_SHIFT,
            },
            MODEL_OUT / f"held_out_{patient}_adapter.pt",
        )
        del adapter, optimizer
    if not np.isfinite(adapted_probability).all():
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    for model, cp in zip(ensemble.networks, checkpoints):
        for key, value in model.state_dict().items():
            if not torch.equal(
                value.detach().cpu(), cp["state_dict"][key].detach().cpu()
            ):
                raise ValueError(
                    "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
                )
    if any(
        (parameter.grad is not None for parameter in ensemble.parameters())
    ):
        raise ValueError(
            "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
        )
    predictions = pd.DataFrame(
        {
            "patient_id": X.index,
            "MTAP_GISTIC_call": calls.to_numpy(dtype=int),
            "label_MTAP_deep": y,
            "Frozen_bulk_DL": direct_probability,
            "Region_adapter_DL_LOPO": adapted_probability,
            "Three_gene_L2_LOPO": l2_probability,
        }
    )
    predictions.to_csv(
        OUT / "three_model_patient_predictions.tsv", sep="\t", index=False
    )
    pd.DataFrame(training_rows).to_csv(
        OUT / "adapter_training_summary.tsv", sep="\t", index=False
    )
    pd.DataFrame(coefficient_rows).to_csv(
        OUT / "adapter_region_coefficients.tsv", sep="\t", index=False
    )
    model_names = [
        "Frozen_bulk_DL",
        "Region_adapter_DL_LOPO",
        "Three_gene_L2_LOPO",
    ]
    metric_rows = []
    for name in model_names:
        probability = predictions[name].to_numpy(dtype=float)
        if (
            not np.isfinite(probability).all()
            or (probability < 0).any()
            or (probability > 1).any()
        ):
            raise ValueError(
                "GBM frozen bulk DL with a bounded input adapter: integrity check failed; condition: input or artifact integrity check."
            )
        tn, fp, fn, tp = confusion_matrix(
            y, probability >= 0.5, labels=[0, 1]
        ).ravel()
        metric_rows.append(
            {
                "model": name,
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                "AUROC": roc_auc_score(y, probability),
                "AP": average_precision_score(y, probability),
                "Brier_score": brier_score_loss(y, probability),
                "log_loss": log_loss(
                    y, np.clip(probability, 1e-07, 1 - 1e-07), labels=[0, 1]
                ),
                "TP_at_05": int(tp),
                "FP_at_05": int(fp),
                "FN_at_05": int(fn),
                "TN_at_05": int(tn),
                "sensitivity_at_05": tp / (tp + fn),
                "specificity_at_05": tn / (tn + fp),
                "PPV_at_05": tp / (tp + fp) if tp + fp else np.nan,
            }
        )
    metrics = pd.DataFrame(metric_rows)
    metrics.to_csv(OUT / "three_model_performance.tsv", sep="\t", index=False)
    colors = {
        "Frozen_bulk_DL": "#33539E",
        "Region_adapter_DL_LOPO": "#7FACD6",
        "Three_gene_L2_LOPO": "#A5678E",
    }
    labels = {
        "Frozen_bulk_DL": "Frozen bulk DL",
        "Region_adapter_DL_LOPO": "Region adapter DL (LOPO)",
        "Three_gene_L2_LOPO": "Three-gene L2 (LOPO)",
    }
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


#### Patient-level L2 prediction interface


In [ ]:
@_nb9_registry.register("s60_0")
def _nb9_stage_s60_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    import joblib
    import numpy as np
    import pandas as pd

    MODEL_FILE = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer/08_L2_cell_number_stability_and_candidate/models/GBM_snRNA_three_gene_L2_candidate.joblib"
        )
    )
    if not MODEL_FILE.is_file():
        raise FileNotFoundError(
            "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
        )
    bundle = joblib.load(MODEL_FILE)
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    pipeline = bundle["pipeline"]
    if list(bundle["feature_names"]) != CORE:
        raise ValueError(
            "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
        )
    if bundle["expression_unit"] != "log2(CPM+1)":
        raise ValueError(
            "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
        )
    classes = list(pipeline.named_steps["model"].classes_)
    if classes != [0, 1]:
        raise ValueError(
            "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
        )

    def predict_MTAP_patient(patient_id, core_counts, total_gene_umi):
        """Predict one patient from uncorrected tumor-cell UMI counts.

        core_counts contains MTAP, CDKN2A, and CDKN2B counts from the same
        selected cells. total_gene_umi contains aligned all-gene Gene Expression
        UMI totals. The output is a patient-level model score, not a DNA label.
        """
        if not isinstance(patient_id, str) or not patient_id.strip():
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        if not isinstance(core_counts, pd.DataFrame):
            raise TypeError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        if not isinstance(total_gene_umi, pd.Series):
            raise TypeError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        if len(core_counts.columns) != 3 or set(core_counts.columns) != set(
            CORE
        ):
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        if (
            not core_counts.index.is_unique
            or not total_gene_umi.index.is_unique
            or core_counts.index.hasnans
            or total_gene_umi.index.hasnans
            or (not core_counts.index.equals(total_gene_umi.index))
        ):
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        n_cells = len(core_counts)
        if n_cells < 50:
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        counts = core_counts.loc[:, CORE].to_numpy(dtype=float)
        libraries = total_gene_umi.to_numpy(dtype=float)
        if (
            not np.isfinite(counts).all()
            or not np.isfinite(libraries).all()
            or (counts < 0).any()
            or (libraries <= 0).any()
        ):
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        if (
            not np.equal(counts, np.rint(counts)).all()
            or not np.equal(libraries, np.rint(libraries)).all()
        ):
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        if (counts.sum(axis=1) > libraries).any():
            raise ValueError(
                "Patient-level L2 prediction interface: integrity check failed; condition: input or artifact integrity check."
            )
        gene_sums = counts.sum(axis=0)
        total_umi = float(libraries.sum())
        logcpm = np.log2(1.0 + gene_sums / total_umi * 1000000.0)
        model_input = pd.DataFrame([logcpm], columns=CORE)
        score = float(pipeline.predict_proba(model_input)[0, 1])
        flags = ["Independent_external_validation_pending"]
        if n_cells < 500:
            flags.append("Limited_cell_number")
        if (gene_sums == 0).all():
            flags.append("All_three_genes_zero_requires_review")
        return {
            "patient_id": patient_id,
            "n_tumor_nuclei": n_cells,
            "MTAP_deep_deletion_score": score,
            "input_log2CPM_plus_1": {
                gene: float(value) for gene, value in zip(CORE, logcpm)
            },
            "quality_flags": flags,
            "prediction_unit": "Patient_Tumor_pseudobulk",
            "DNA_label_assigned": False,
        }

    _public_summary(globals(), "Patient-level L2 prediction interface")


#### GBM LOPO three-model predictions including 44-gene Elastic Net


In [ ]:
@_nb9_registry.register("s62_0")
def _nb9_stage_s62_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import warnings
    import numpy as np
    import pandas as pd
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        roc_curve,
        precision_recall_curve,
    )

    ROOT = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
        )
    )
    PB_DIR = ROOT / "05_Tumor_UMI_and_pseudobulk"
    X_PATH = PB_DIR / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    FEATURE_PATH = PB_DIR / "fixed_44_feature_mapping.tsv"
    PRED_PATH = (
        ROOT
        / "07_frozen_DL_and_region_adapter_v1"
        / "three_model_patient_predictions.tsv"
    )
    OUT = ROOT / "09_snRNA_adapter_L2_ElasticNet_comparison"
    OUT.mkdir(parents=True, exist_ok=True)
    SEED = 20261017
    C_FIXED = 1.0
    L1_RATIO_FIXED = 0.5
    THRESHOLD = 0.5
    N_BOOTSTRAP = 2000
    ADAPTER_EPOCHS = 200
    ADAPTER_LR = 0.02
    IDENTITY_PENALTY = 0.05
    MAX_LOG_SCALE = 0.5
    MAX_SHIFT = 1.0
    N_ADAPTER_REGIONS = 4
    N_ADAPTER_PARAMETERS = 8
    DL_MTAP_TARGET_INDEX = 26
    DL_DEEP_DELETION_CLASS_INDEX = 0
    DL_FROZEN_SEEDS = 3
    MODELS = ["Region_adapter_DL_LOPO", "L2_LOPO", "ElasticNet_LOPO"]
    LABELS = {
        "Region_adapter_DL_LOPO": "Bulk DL + Input Adapter\nLOPO",
        "L2_LOPO": "Three-gene L2\nLOPO",
        "ElasticNet_LOPO": "44-gene Elastic Net\nLOPO",
    }
    COLORS = {
        "Region_adapter_DL_LOPO": "#A5678E",
        "L2_LOPO": "#33539E",
        "ElasticNet_LOPO": "#7FACD6",
    }
    METRICS = [
        "AUROC",
        "AP",
        "Balanced_accuracy",
        "Sensitivity",
        "Specificity",
        "Accuracy",
        "Precision",
        "F1",
        "Brier",
    ]
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9,
            "legend.fontsize": 8,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.dpi": 400,
        }
    )
    for path in [X_PATH, FEATURE_PATH, PRED_PATH]:
        if not path.is_file():
            raise FileNotFoundError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
    expression = pd.read_csv(X_PATH, sep="\t", dtype={"patient_id": str})
    feature_table = pd.read_csv(FEATURE_PATH, sep="\t")
    existing = pd.read_csv(PRED_PATH, sep="\t", dtype={"patient_id": str})
    for name, frame in [
        ("expression", expression),
        ("existing_predictions", existing),
    ]:
        if "patient_id" not in frame.columns:
            raise ValueError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
        if frame["patient_id"].isna().any():
            raise ValueError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
        if frame["patient_id"].duplicated().any():
            duplicated = frame.loc[
                frame["patient_id"].duplicated(keep=False), "patient_id"
            ].tolist()
            raise ValueError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
    required_prediction_columns = {
        "patient_id",
        "label_MTAP_deep",
        "Three_gene_L2_LOPO",
        "Region_adapter_DL_LOPO",
    }
    missing_prediction_columns = required_prediction_columns - set(
        existing.columns
    )
    if missing_prediction_columns:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    expression_patients = set(expression["patient_id"])
    prediction_patients = set(existing["patient_id"])
    if expression_patients != prediction_patients:
        only_expression = sorted(expression_patients - prediction_patients)
        only_prediction = sorted(prediction_patients - expression_patients)
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    required_feature_columns = {"feature", "feature_order"}
    missing_feature_columns = required_feature_columns - set(
        feature_table.columns
    )
    if missing_feature_columns:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    feature_table["feature_order"] = pd.to_numeric(
        feature_table["feature_order"], errors="raise"
    )
    feature_table = feature_table.sort_values("feature_order")
    features = feature_table["feature"].astype(str).tolist()
    if len(features) != 44:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    if len(set(features)) != 44:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    missing_expression_genes = set(features) - set(expression.columns)
    if missing_expression_genes:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    existing = existing.set_index("patient_id").sort_index()
    expression = expression.set_index("patient_id").loc[existing.index]
    if not expression.index.equals(existing.index):
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    X = expression[features].apply(pd.to_numeric, errors="raise")
    y_series = pd.to_numeric(existing["label_MTAP_deep"], errors="raise")
    if y_series.isna().any():
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    if not y_series.isin([0, 1]).all():
        invalid_labels = sorted(y_series.loc[~y_series.isin([0, 1])].unique())
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    y = y_series.astype(int).to_numpy()
    X_values = X.to_numpy(dtype=float)
    if not np.isfinite(X_values).all():
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    if (X_values < 0).any():
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    if len(np.unique(y)) != 2:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    class_counts = np.bincount(y, minlength=2)
    if class_counts.min() < 2:
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    elastic_net_probabilities = np.full(len(y), np.nan)
    fold_records = []
    coefficient_records = []
    for held_out_index, patient_id in enumerate(X.index):
        train_mask = np.arange(len(y)) != held_out_index
        training_labels = y[train_mask]
        if len(np.unique(training_labels)) != 2:
            raise ValueError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
        pipeline = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        penalty="elasticnet",
                        solver="saga",
                        C=C_FIXED,
                        l1_ratio=L1_RATIO_FIXED,
                        class_weight=None,
                        max_iter=20000,
                        tol=1e-05,
                        random_state=SEED,
                    ),
                ),
            ]
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            pipeline.fit(X_values[train_mask], training_labels)
        classifier = pipeline.named_steps["classifier"]
        positive_class_column = list(classifier.classes_).index(1)
        held_out_probability = pipeline.predict_proba(
            X_values[[held_out_index]]
        )[0, positive_class_column]
        elastic_net_probabilities[held_out_index] = held_out_probability
        fold_records.append(
            {
                "held_out_patient": patient_id,
                "n_training_patients": int(train_mask.sum()),
                "n_training_positive": int(training_labels.sum()),
                "n_training_negative": int((training_labels == 0).sum()),
                "probability_ElasticNet_LOPO": held_out_probability,
                "n_iterations": int(classifier.n_iter_[0]),
                "n_nonzero_gene_coefficients": int(
                    np.count_nonzero(classifier.coef_[0])
                ),
            }
        )
        for gene, coefficient in zip(features, classifier.coef_[0]):
            coefficient_records.append(
                {
                    "held_out_patient": patient_id,
                    "gene": gene,
                    "standardized_coefficient": coefficient,
                }
            )
    if not np.isfinite(elastic_net_probabilities).all():
        raise ValueError(
            "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
        )
    elastic_net_fold_audit = pd.DataFrame(fold_records)
    elastic_net_fold_audit.to_csv(
        OUT / "ElasticNet_LOPO_fold_audit.tsv", sep="\t", index=False
    )
    elastic_net_coefficients = pd.DataFrame(coefficient_records)
    elastic_net_coefficients.to_csv(
        OUT / "ElasticNet_LOPO_coefficients.tsv", sep="\t", index=False
    )
    coefficient_summary = elastic_net_coefficients.groupby(
        "gene", as_index=False
    ).agg(
        median_standardized_coefficient=("standardized_coefficient", "median"),
        mean_standardized_coefficient=("standardized_coefficient", "mean"),
        coefficient_SD=("standardized_coefficient", "std"),
        nonzero_fraction=(
            "standardized_coefficient",
            lambda values: np.mean(np.asarray(values) != 0),
        ),
    )
    coefficient_summary["median_absolute_coefficient"] = coefficient_summary[
        "median_standardized_coefficient"
    ].abs()
    coefficient_summary = coefficient_summary.sort_values(
        "median_absolute_coefficient", ascending=False
    )
    coefficient_summary.to_csv(
        OUT / "ElasticNet_LOPO_gene_stability.tsv", sep="\t", index=False
    )
    predictions = existing[["label_MTAP_deep"]].copy()
    if "MTAP_GISTIC_call" in existing.columns:
        predictions["MTAP_GISTIC_call"] = existing["MTAP_GISTIC_call"]
    predictions["Region_adapter_DL_LOPO"] = pd.to_numeric(
        existing["Region_adapter_DL_LOPO"], errors="raise"
    )
    predictions["L2_LOPO"] = pd.to_numeric(
        existing["Three_gene_L2_LOPO"], errors="raise"
    )
    predictions["ElasticNet_LOPO"] = elastic_net_probabilities
    for model in MODELS:
        probabilities = predictions[model].to_numpy(dtype=float)
        if not np.isfinite(probabilities).all():
            raise ValueError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
        if not ((probabilities >= 0) & (probabilities <= 1)).all():
            raise ValueError(
                "GBM three-model comparison including Elastic Net LOPO: integrity check failed; condition: input or artifact integrity check."
            )
    predictions.to_csv(
        OUT / "snRNA_adapter_L2_ElasticNet_patient_predictions.tsv",
        sep="\t",
        index_label="patient_id",
    )

    def safe_ratio(numerator, denominator):
        """Return a missing value when a performance denominator is zero."""
        if denominator == 0:
            return np.nan
        return numerator / denominator

    def evaluate_model(y_true, probabilities, threshold=0.5):
        """Calculate patient-level metrics using the prespecified threshold."""
        predicted_positive = probabilities >= threshold
        true_positive = int(((y_true == 1) & predicted_positive).sum())
        false_positive = int(((y_true == 0) & predicted_positive).sum())
        false_negative = int(((y_true == 1) & ~predicted_positive).sum())
        true_negative = int(((y_true == 0) & ~predicted_positive).sum())
        sensitivity = safe_ratio(true_positive, true_positive + false_negative)
        specificity = safe_ratio(true_negative, true_negative + false_positive)
        return {
            "AUROC": roc_auc_score(y_true, probabilities),
            "AP": average_precision_score(y_true, probabilities),
            "Balanced_accuracy": (sensitivity + specificity) / 2,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Accuracy": safe_ratio(true_positive + true_negative, len(y_true)),
            "Precision": safe_ratio(
                true_positive, true_positive + false_positive
            ),
            "F1": safe_ratio(
                2 * true_positive,
                2 * true_positive + false_positive + false_negative,
            ),
            "Brier": np.mean((probabilities - y_true) ** 2),
            "TP": true_positive,
            "FP": false_positive,
            "FN": false_negative,
            "TN": true_negative,
        }

    def percentile_ci(values):
        """Calculate a percentile bootstrap confidence interval."""
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            return (np.nan, np.nan)
        lower, upper = np.quantile(values, [0.025, 0.975])
        return (float(lower), float(upper))

    point_estimates = {}
    for model in MODELS:
        probability = predictions[model].to_numpy(dtype=float)
        point_estimates[model] = evaluate_model(
            y, probability, threshold=THRESHOLD
        )
    bootstrap_results = {
        model: np.full((N_BOOTSTRAP, len(METRICS)), np.nan) for model in MODELS
    }
    rng = np.random.default_rng(SEED)
    positive_indices = np.flatnonzero(y == 1)
    negative_indices = np.flatnonzero(y == 0)
    for bootstrap_index in range(N_BOOTSTRAP):
        sampled_positive = rng.choice(
            positive_indices, size=len(positive_indices), replace=True
        )
        sampled_negative = rng.choice(
            negative_indices, size=len(negative_indices), replace=True
        )
        sampled_indices = np.concatenate([sampled_positive, sampled_negative])
        sampled_indices = rng.permutation(sampled_indices)
        for model in MODELS:
            probability = predictions[model].to_numpy(dtype=float)
            bootstrap_metrics = evaluate_model(
                y[sampled_indices],
                probability[sampled_indices],
                threshold=THRESHOLD,
            )
            bootstrap_results[model][bootstrap_index] = [
                bootstrap_metrics[metric] for metric in METRICS
            ]


### GSE327167 scRNA-seq patient-level external validation


#### GSE327167 feature mapping, pseudobulk, adaptation, and inference


##### GSE327167 input audit and reference labels

Check input files, supplementary-table patient mapping,
stringent-cell alignment, and fixed 44-feature order before missing-feature
handling. OGM labels are read from a local, untracked reference table.
The evaluated cohort contains 31 patients (4 OGM-detected deep deletions);
these reference labels do not establish copy neutrality for the other patients.
See `docs/GSE327167_INPUTS.md` for local preparation and provenance.


In [ ]:
@_nb9_registry.register("s73_gse327167_setup")
def _nb9_stage_s73_gse327167_setup():
    import os
    from pathlib import Path
    from datetime import datetime
    import gzip
    import hashlib
    import importlib.util
    import json
    import sys
    import warnings
    import joblib
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    from torch import nn
    from scipy import sparse
    from scipy.io import mmread
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        average_precision_score,
        balanced_accuracy_score,
        brier_score_loss,
        confusion_matrix,
        precision_recall_curve,
        roc_auc_score,
        roc_curve,
    )
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    PROJECT = Path(PROJECT_ROOT)
    GSE_DIR = PROJECT / "data_raw/MESO/GSE327167"
    MATRIX_FILE = GSE_DIR / "GSE327167_stringent_scRNA_counts_matrix.mtx.gz"
    GENE_FILE = GSE_DIR / "GSE327167_stringent_scRNA_geneIDs.tsv.gz"
    CELL_FILE = GSE_DIR / "GSE327167_stringent_scRNA_cellIDs.tsv.gz"
    CELL_METADATA_FILE = GSE_DIR / "GSE327167_PM_cell_metadata.tsv.gz"
    SUPPLEMENT_FILE = GSE_DIR / "NIHMS2136244-supplement-4.xlsx"
    TRANSFER_ROOT = PROJECT / "data_processed/MTAP_RNA_prediction" / "16_scRNA_transfer"
    GBM_PB_DIR = TRANSFER_ROOT / "05_Tumor_UMI_and_pseudobulk"
    GBM_X_FILE = GBM_PB_DIR / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    GBM_META_FILE = GBM_PB_DIR / "patient_input_audit.tsv"
    FEATURE_MAPPING_FILE = GBM_PB_DIR / "fixed_44_feature_mapping.tsv"
    L2_MODEL_FILE = (
        TRANSFER_ROOT
        / "08_L2_cell_number_stability_and_candidate"
        / "models"
        / "GBM_snRNA_three_gene_L2_candidate.joblib"
    )
    DL_LOCKED_DIR = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL"
        / "08_TPM_retraining"
        / "04_locked_models_and_thresholds"
    )
    DL_NETWORK_FILE = DL_LOCKED_DIR / "region_multitask_TPM44_network.py"
    DL_MANIFEST_FILE = DL_LOCKED_DIR / "locked_manifest.json"
    DL_SEEDS = [41, 42, 43]
    DL_CHECKPOINT_FILES = [
        DL_LOCKED_DIR / "models" / f"DL_seed{seed}.pt" for seed in DL_SEEDS
    ]
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    OUTPUT_DIR = (
        PROJECT
        / "data_processed"
        / "MTAP_prediction_registry"
        / "external_validation"
        / (RUN_ID + "_GSE327167_31patient_three_models")
    )
    MODEL_OUT = OUTPUT_DIR / "frozen_migrated_models"
    FIGURE_DIR = OUTPUT_DIR / "figures"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    MODEL_OUT.mkdir(parents=True, exist_ok=False)
    FIGURE_DIR.mkdir(parents=True, exist_ok=False)
    CORE_GENES = ["MTAP", "CDKN2A", "CDKN2B"]
    ELASTIC_C = 1.0
    ELASTIC_L1_RATIO = 0.5
    MODEL_THRESHOLD = 0.5
    ADAPTER_EPOCHS = 200
    ADAPTER_LR = 0.02
    IDENTITY_PENALTY = 0.05
    MAX_LOG_SCALE = 0.5
    MAX_SHIFT = 1.0
    MIN_MALIGNANT_CELLS_PRIMARY = 50
    RANDOM_SEED = 20261017
    DEVICE = torch.device("cpu")
    torch.set_num_threads(1)
    torch.manual_seed(41)
    np.random.seed(RANDOM_SEED)
    MTAP_PALETTE = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]
    MODEL_ORDER = ["Three-gene L2", "44-gene Elastic Net", "Bulk DL + input adapter"]
    MODEL_COLORS = {
        "Three-gene L2": MTAP_PALETTE[0],
        "44-gene Elastic Net": MTAP_PALETTE[1],
        "Bulk DL + input adapter": MTAP_PALETTE[4],
    }
    OGM_REFERENCE_FILE = Path(
        os.environ.get(
            "MTAP_GSE327167_OGM_LABELS",
            str(PROJECT / "metadata/GSE327167_OGM_reference_labels.tsv"),
        )
    ).expanduser()
    if not OGM_REFERENCE_FILE.is_absolute():
        OGM_REFERENCE_FILE = PROJECT / OGM_REFERENCE_FILE
    if not OGM_REFERENCE_FILE.is_file():
        raise FileNotFoundError(
            "Local OGM reference labels are required. See docs/GSE327167_INPUTS.md."
        )
    ogm_reference = pd.read_csv(
        OGM_REFERENCE_FILE, sep="\t", dtype="string", keep_default_na=False
    )
    if not {"patient_id", "OGM_deep_deletion_label"}.issubset(ogm_reference.columns):
        raise ValueError(
            "OGM reference requires patient_id and OGM_deep_deletion_label."
        )
    ogm_reference["patient_id"] = ogm_reference["patient_id"].str.strip()
    ogm_reference["OGM_deep_deletion_label"] = pd.to_numeric(
        ogm_reference["OGM_deep_deletion_label"], errors="raise"
    )
    if (
        len(ogm_reference) != 31
        or not ogm_reference["patient_id"].is_unique
        or ogm_reference["patient_id"].eq("").any()
        or (not ogm_reference["OGM_deep_deletion_label"].isin([0, 1]).all())
        or (ogm_reference["OGM_deep_deletion_label"].sum() != 4)
    ):
        raise ValueError(
            "Expected 31 unique evaluable patients: 4 detected and 27 not detected."
        )
    OGM_PATIENTS = ogm_reference["patient_id"].tolist()
    OGM_DEEP_DELETION_PATIENTS = set(
        ogm_reference.loc[ogm_reference["OGM_deep_deletion_label"].eq(1), "patient_id"]
    )

    def require(condition, message):
        if not bool(condition):
            raise RuntimeError(message)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    def clean_text(series):
        result = series.astype("string").str.strip()
        return result.mask(result.isin(["", "NA", "N/A", "nan", "None", "<NA>"]))

    def normalize_identifier(series):
        return clean_text(series).str.replace("\\.0$", "", regex=True)

    def open_text(path):
        path = Path(path)
        if path.name.endswith(".gz"):
            return gzip.open(path, "rt", encoding="utf-8")
        return path.open("r", encoding="utf-8")

    def read_one_column_file(path):
        with open_text(path) as handle:
            values = [line.rstrip("\r\n").split("\t")[0] for line in handle]
        values = pd.Series(values, dtype="string")
        values = normalize_identifier(values).dropna().reset_index(drop=True)
        if len(values) > 0:
            first_value = values.iloc[0].lower()
            if first_value in {"cell_id", "cellid", "barcode", "gene_id", "geneid"}:
                values = values.iloc[1:].reset_index(drop=True)
        return values

    def find_excel_header(path, sheet_name, required_columns, max_rows=40):
        preview = pd.read_excel(
            path, sheet_name=sheet_name, header=None, nrows=max_rows
        )
        required = set(required_columns)
        for row_index in range(len(preview)):
            values = {
                str(value).strip()
                for value in preview.iloc[row_index].dropna().tolist()
            }
            if required.issubset(values):
                return row_index
        raise RuntimeError("Expected Excel header was not found in the inspected rows.")

    def positive_probability(pipeline, frame):
        final_step = pipeline.steps[-1][1]
        classes = list(final_step.classes_)
        require(1 in classes, "GSE327167 upstream integrity check failed: 1 in classes")
        return pipeline.predict_proba(frame)[:, classes.index(1)]

    def calculate_metrics(y_true, probability, threshold=0.5):
        y_true = np.asarray(y_true, dtype=int)
        probability = np.asarray(probability, dtype=float)
        predicted = (probability >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()

        def safe_divide(numerator, denominator):
            if denominator == 0:
                return np.nan
            return float(numerator / denominator)

        return {
            "AUROC": float(roc_auc_score(y_true, probability)),
            "AP": float(average_precision_score(y_true, probability)),
            "Brier": float(brier_score_loss(y_true, probability)),
            "Balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)),
            "Sensitivity": safe_divide(tp, tp + fn),
            "Specificity": safe_divide(tn, tn + fp),
            "PPV": safe_divide(tp, tp + fp),
            "NPV": safe_divide(tn, tn + fn),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
        }

    INPUT_FILES = [
        MATRIX_FILE,
        GENE_FILE,
        CELL_FILE,
        CELL_METADATA_FILE,
        SUPPLEMENT_FILE,
        GBM_X_FILE,
        GBM_META_FILE,
        FEATURE_MAPPING_FILE,
        L2_MODEL_FILE,
        DL_NETWORK_FILE,
        DL_MANIFEST_FILE,
        *DL_CHECKPOINT_FILES,
    ]
    for path in INPUT_FILES:
        require(
            path.is_file(), "GSE327167 upstream integrity check failed: path.is_file()"
        )
    require(
        len(OGM_PATIENTS) == 31,
        "GSE327167 upstream integrity check failed: len(OGM_PATIENTS) == 31",
    )
    require(
        len(set(OGM_PATIENTS)) == 31,
        "GSE327167 upstream integrity check failed: len(set(OGM_PATIENTS)) == 31",
    )
    require(
        OGM_DEEP_DELETION_PATIENTS.issubset(set(OGM_PATIENTS)),
        "GSE327167 upstream integrity check failed: OGM_DEEP_DELETION_PATIENTS.issubset(set(OGM_PATIENTS))",
    )
    SHEET_1A = "1A_SAMPLE_CHARACTERISTICS"
    SUPP_COLUMNS = [
        "sc_SeqDataID",
        "sc_SampleSetID",
        "TB_CaseID",
        "anatomic site",
        "case diagnosis",
    ]
    header_row = find_excel_header(
        SUPPLEMENT_FILE, SHEET_1A, ["sc_SeqDataID", "sc_SampleSetID", "TB_CaseID"]
    )
    sample_map = pd.read_excel(
        SUPPLEMENT_FILE, sheet_name=SHEET_1A, header=header_row, dtype=str
    )
    missing_columns = set(SUPP_COLUMNS) - set(sample_map.columns)
    require(
        not missing_columns,
        "GSE327167 upstream integrity check failed: not missing_columns",
    )
    sample_map = sample_map.loc[:, SUPP_COLUMNS].copy()
    sample_map["library_id"] = normalize_identifier(sample_map["sc_SeqDataID"])
    sample_map["sample_set_id"] = normalize_identifier(sample_map["sc_SampleSetID"])
    sample_map["patient_id"] = normalize_identifier(sample_map["TB_CaseID"])
    sample_map["anatomic_site"] = clean_text(sample_map["anatomic site"])
    sample_map["case_diagnosis"] = clean_text(sample_map["case diagnosis"])
    sample_map = sample_map.loc[sample_map["patient_id"].notna()].copy()
    for key in ["library_id", "sample_set_id"]:
        concordance = (
            sample_map.loc[sample_map[key].notna(), [key, "patient_id"]]
            .drop_duplicates()
            .groupby(key)["patient_id"]
            .nunique()
        )
        require(
            not concordance.gt(1).any(),
            "GSE327167 upstream integrity check failed: not concordance.gt(1).any()",
        )
    library_to_patient = (
        sample_map.dropna(subset=["library_id"])
        .drop_duplicates("library_id")
        .set_index("library_id")["patient_id"]
    )
    sampleset_to_patient = (
        sample_map.dropna(subset=["sample_set_id"])
        .drop_duplicates("sample_set_id")
        .set_index("sample_set_id")["patient_id"]
    )
    cell_metadata = pd.read_csv(
        CELL_METADATA_FILE, sep="\t", dtype=str, compression="gzip", low_memory=False
    )
    required_metadata_columns = {
        "cell_id",
        "library_id",
        "manuscript_id",
        "malignant_classification_clonal_analysis",
        "stringent_cell_type",
        "stringent_cell_class",
    }
    missing_metadata_columns = required_metadata_columns - set(cell_metadata.columns)
    require(
        not missing_metadata_columns,
        "GSE327167 upstream integrity check failed: not missing_metadata_columns",
    )
    cell_metadata["cell_id_clean"] = normalize_identifier(cell_metadata["cell_id"])
    cell_metadata["library_id_clean"] = normalize_identifier(
        cell_metadata["library_id"]
    )
    cell_metadata["sample_set_id_clean"] = normalize_identifier(
        cell_metadata["manuscript_id"]
    )
    require(
        cell_metadata["cell_id_clean"].notna().all(),
        "GSE327167 upstream integrity check failed: cell_metadata['cell_id_clean'].notna().all()",
    )
    require(
        cell_metadata["cell_id_clean"].is_unique,
        "GSE327167 upstream integrity check failed: cell_metadata['cell_id_clean'].is_unique",
    )
    matrix_cell_ids = read_one_column_file(CELL_FILE)
    require(
        len(matrix_cell_ids) == 93351,
        "GSE327167 upstream integrity check failed: len(matrix_cell_ids) == 93351",
    )
    require(
        matrix_cell_ids.is_unique,
        "GSE327167 upstream integrity check failed: matrix_cell_ids.is_unique",
    )
    cell_metadata = pd.DataFrame({"matrix_cell_id": matrix_cell_ids}).merge(
        cell_metadata,
        left_on="matrix_cell_id",
        right_on="cell_id_clean",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    require(
        cell_metadata["_merge"].eq("both").all(),
        "GSE327167 upstream integrity check failed: cell_metadata['_merge'].eq('both').all()",
    )
    cell_metadata = cell_metadata.drop(columns="_merge")
    cell_metadata["patient_by_library"] = cell_metadata["library_id_clean"].map(
        library_to_patient
    )
    cell_metadata["patient_by_sampleset"] = cell_metadata["sample_set_id_clean"].map(
        sampleset_to_patient
    )
    both_patient = (
        cell_metadata["patient_by_library"].notna()
        & cell_metadata["patient_by_sampleset"].notna()
    )
    require(
        cell_metadata.loc[both_patient, "patient_by_library"]
        .eq(cell_metadata.loc[both_patient, "patient_by_sampleset"])
        .all(),
        "GSE327167 upstream integrity check failed: cell_metadata.loc[both_patient, 'patient_by_library'].eq(cell_metadata.loc[both_patient, 'patient_by_sampleset']).all()",
    )
    cell_metadata["patient_id"] = cell_metadata["patient_by_library"].fillna(
        cell_metadata["patient_by_sampleset"]
    )
    require(
        cell_metadata["patient_id"].notna().all(),
        "GSE327167 upstream integrity check failed: cell_metadata['patient_id'].notna().all()",
    )
    cell_metadata["is_target_patient"] = cell_metadata["patient_id"].isin(OGM_PATIENTS)
    cell_metadata["author_malignant"] = (
        clean_text(cell_metadata["malignant_classification_clonal_analysis"])
        .str.lower()
        .eq("tumor")
        .fillna(False)
        .astype(bool)
    )
    covered_patients = set(
        cell_metadata.loc[cell_metadata["is_target_patient"], "patient_id"]
    )
    require(
        covered_patients == set(OGM_PATIENTS),
        "GSE327167 upstream integrity check failed: covered_patients == set(OGM_PATIENTS)",
    )
    feature_mapping = pd.read_csv(FEATURE_MAPPING_FILE, sep="\t", dtype=str)
    required_feature_columns = {"feature", "feature_order", "Ensembl_ID"}
    missing_feature_columns = required_feature_columns - set(feature_mapping.columns)
    require(
        not missing_feature_columns,
        "GSE327167 upstream integrity check failed: not missing_feature_columns",
    )
    feature_mapping["feature_order"] = pd.to_numeric(
        feature_mapping["feature_order"], errors="raise"
    )
    feature_mapping = feature_mapping.sort_values("feature_order").reset_index(
        drop=True
    )
    features = feature_mapping["feature"].astype(str).tolist()
    require(
        len(features) == 44,
        "GSE327167 upstream integrity check failed: len(features) == 44",
    )
    require(
        len(set(features)) == 44,
        "GSE327167 upstream integrity check failed: len(set(features)) == 44",
    )
    require(
        set(CORE_GENES).issubset(features),
        "GSE327167 upstream integrity check failed: set(CORE_GENES).issubset(features)",
    )
    gene_table = pd.read_csv(
        GENE_FILE,
        sep="\t",
        header=None,
        dtype=str,
        compression="gzip",
        keep_default_na=False,
    )
    gene_normalized = gene_table.apply(
        lambda column: column.astype(str)
        .str.strip()
        .str.replace("\\.\\d+$", "", regex=True)
    )
    GSE327167_RUN_DIR = OUTPUT_DIR

    # Record provenance locally; do not export OGM identifiers in notebook displays.
    with (OUTPUT_DIR / "00_local_OGM_reference_provenance.json").open("w") as handle:
        json.dump(
            {
                "reference_sha256": sha256_file(OGM_REFERENCE_FILE),
                "n_patients": len(OGM_PATIENTS),
                "n_OGM_detected_deep_deletion": len(OGM_DEEP_DELETION_PATIENTS),
                "upstream_source": "GSE327167 input setup; local verified OGM reference",
            },
            handle,
            indent=2,
        )


In [ ]:
@_nb9_registry.register("s74_0")
def _nb9_stage_s74_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import gzip
    import importlib.util
    import json
    import sys
    import warnings
    import joblib
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    from torch import nn
    from scipy import sparse
    from scipy.io import mmread
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        average_precision_score,
        balanced_accuracy_score,
        brier_score_loss,
        confusion_matrix,
        precision_recall_curve,
        roc_auc_score,
        roc_curve,
    )
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    required_objects = [
        "feature_mapping",
        "features",
        "gene_table",
        "gene_normalized",
        "cell_metadata",
        "sample_map",
        "matrix_cell_ids",
        "MATRIX_FILE",
        "OUTPUT_DIR",
        "MODEL_OUT",
        "FIGURE_DIR",
        "GBM_X_FILE",
        "GBM_META_FILE",
        "L2_MODEL_FILE",
        "DL_LOCKED_DIR",
        "DL_NETWORK_FILE",
        "DL_MANIFEST_FILE",
        "DL_CHECKPOINT_FILES",
        "OGM_PATIENTS",
        "OGM_DEEP_DELETION_PATIENTS",
        "CORE_GENES",
        "ELASTIC_C",
        "ELASTIC_L1_RATIO",
        "MODEL_THRESHOLD",
        "ADAPTER_EPOCHS",
        "ADAPTER_LR",
        "IDENTITY_PENALTY",
        "MAX_LOG_SCALE",
        "MAX_SHIFT",
        "MIN_MALIGNANT_CELLS_PRIMARY",
        "RANDOM_SEED",
        "DEVICE",
        "MODEL_ORDER",
        "MODEL_COLORS",
        "require",
        "sha256_file",
        "positive_probability",
        "calculate_metrics",
    ]
    missing_objects = [
        name for name in required_objects if name not in globals()
    ]
    require(
        not missing_objects,
        f"GSE327167 upstream setup is required. Missing objects: {missing_objects}. See REPRODUCIBILITY.md.",
    )
    MAX_ALLOWED_MISSING_COMPLEX_FEATURES = 5
    feature_to_gene_row = {}
    mapping_audit = []
    for record in feature_mapping.itertuples(index=False):
        feature = str(record.feature)
        ensembl_id = str(record.Ensembl_ID).split(".")[0]
        matched_rows = set()
        match_type = None
        for column in gene_normalized.columns:
            matched_rows.update(
                gene_normalized.index[
                    gene_normalized[column].eq(ensembl_id)
                ].tolist()
            )
        if matched_rows:
            match_type = "Ensembl_ID"
        else:
            for column in gene_normalized.columns:
                matched_rows.update(
                    gene_normalized.index[
                        gene_normalized[column].eq(feature)
                    ].tolist()
                )
            if matched_rows:
                match_type = "gene_symbol"
        require(
            len(matched_rows) <= 1,
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: len(matched_rows) <= 1.",
        )
        if len(matched_rows) == 1:
            row_index = next(iter(matched_rows))
            feature_to_gene_row[feature] = row_index
            gene_record = "|".join(gene_table.iloc[row_index].astype(str))
            status = "PRESENT"
        else:
            row_index = np.nan
            gene_record = ""
            status = "MISSING_FROM_GSE327167_GENE_AXIS"
        mapping_audit.append(
            {
                "feature_order": int(record.feature_order),
                "feature": feature,
                "Ensembl_ID": ensembl_id,
                "status": status,
                "GSE327167_gene_row_zero_based": row_index,
                "match_type": match_type if match_type else "NOT_MATCHED",
                "GSE327167_gene_record": gene_record,
            }
        )
    mapping_audit = (
        pd.DataFrame(mapping_audit)
        .sort_values("feature_order")
        .reset_index(drop=True)
    )
    missing_features = mapping_audit.loc[
        mapping_audit["status"].ne("PRESENT"), "feature"
    ].tolist()
    present_features = mapping_audit.loc[
        mapping_audit["status"].eq("PRESENT"), "feature"
    ].tolist()
    missing_core_genes = sorted(set(CORE_GENES).intersection(missing_features))
    require(
        not missing_core_genes,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: not missing_core_genes.",
    )
    require(
        len(missing_features) <= MAX_ALLOWED_MISSING_COMPLEX_FEATURES,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: len(missing_features) <= MAX_ALLOWED_MISSING_COMPLEX_FEATURES.",
    )
    mapping_audit["external_inference_handling"] = np.where(
        mapping_audit["status"].eq("PRESENT"),
        "Observed_GSE327167_expression",
        "EN_GBM16_median;DL_checkpoint_training_mean;L2_not_used_if_noncore",
    )
    mapping_audit.to_csv(
        OUTPUT_DIR / "01_GSE327167_44gene_mapping_with_missing_audit.tsv",
        sep="\t",
        index=False,
    )
    with Path(MATRIX_FILE).open("rb") as raw_handle:
        with gzip.GzipFile(fileobj=raw_handle) as gzip_handle:
            counts = mmread(gzip_handle)
    require(
        sparse.issparse(counts),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: sparse.issparse(counts).",
    )
    counts = counts.tocsr()
    require(
        counts.shape == (len(gene_table), len(matrix_cell_ids)),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: counts.shape == (len(gene_table), len(matrix_cell_ids)).",
    )
    require(
        np.isfinite(counts.data).all(),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(counts.data).all().",
    )
    require(
        (counts.data >= 0).all(),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: (counts.data >= 0).all().",
    )
    require(
        np.equal(counts.data, np.rint(counts.data)).all(),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.equal(counts.data, np.rint(counts.data)).all().",
    )
    counts = counts.astype(np.int64, copy=False)
    group_names = [f"Whole::{patient}" for patient in OGM_PATIENTS] + [
        f"Malignant::{patient}" for patient in OGM_PATIENTS
    ]
    group_index = {name: index for index, name in enumerate(group_names)}
    assignment_rows = []
    assignment_columns = []
    for cell_position, record in enumerate(
        cell_metadata.itertuples(index=False)
    ):
        if not bool(record.is_target_patient):
            continue
        patient = str(record.patient_id)
        assignment_rows.append(cell_position)
        assignment_columns.append(group_index[f"Whole::{patient}"])
        if bool(record.author_malignant):
            assignment_rows.append(cell_position)
            assignment_columns.append(group_index[f"Malignant::{patient}"])
    assignment = sparse.coo_matrix(
        (
            np.ones(len(assignment_rows), dtype=np.int8),
            (assignment_rows, assignment_columns),
        ),
        shape=(counts.shape[1], len(group_names)),
    ).tocsr()
    group_cell_counts = np.asarray(assignment.sum(axis=0)).ravel().astype(int)
    per_cell_total_umi = (
        np.asarray(counts.sum(axis=0)).ravel().astype(np.int64)
    )
    group_total_umi = (
        np.asarray(assignment.T.dot(per_cell_total_umi))
        .ravel()
        .astype(np.int64)
    )
    present_gene_rows = [
        feature_to_gene_row[gene] for gene in present_features
    ]
    present_group_counts = (
        counts[present_gene_rows, :].dot(assignment).toarray().T
    )
    all_group_logcpm = np.full(
        (len(group_names), len(features)), np.nan, dtype=float
    )
    feature_position = {gene: index for index, gene in enumerate(features)}
    for group_position in range(len(group_names)):
        total_umi = int(group_total_umi[group_position])
        if group_cell_counts[group_position] == 0:
            continue
        require(
            total_umi > 0,
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: total_umi > 0.",
        )
        values = np.log2(
            1.0
            + present_group_counts[group_position].astype(float)
            / total_umi
            * 1000000.0
        )
        for gene, value in zip(present_features, values):
            all_group_logcpm[group_position, feature_position[gene]] = value
    pseudobulk_rows = []
    for group_position, group_name in enumerate(group_names):
        scope, patient = group_name.split("::", 1)
        row = {
            "patient_id": patient,
            "input_scope": scope,
            "n_cells": int(group_cell_counts[group_position]),
            "total_gene_expression_UMI": int(group_total_umi[group_position]),
            "OGM_deep_deletion_label": int(
                patient in OGM_DEEP_DELETION_PATIENTS
            ),
            "OGM_label_text": (
                "OGM-detected MTAP deep deletion"
                if patient in OGM_DEEP_DELETION_PATIENTS
                else "No OGM-detected MTAP deep deletion"
            ),
            "n_missing_model_features": len(missing_features),
            "missing_model_features": ";".join(missing_features),
        }
        row.update(
            {
                gene: float(all_group_logcpm[group_position, feature_index])
                for feature_index, gene in enumerate(features)
            }
        )
        pseudobulk_rows.append(row)
    pseudobulk = pd.DataFrame(pseudobulk_rows)
    pseudobulk.to_csv(
        OUTPUT_DIR / "02_GSE327167_patient_pseudobulk_44_with_missing.tsv",
        sep="\t",
        index=False,
    )
    audit_rows = []
    for patient in OGM_PATIENTS:
        subset = cell_metadata.loc[
            cell_metadata["is_target_patient"]
            & cell_metadata["patient_id"].eq(patient)
        ]
        whole = pseudobulk.loc[
            pseudobulk["patient_id"].eq(patient)
            & pseudobulk["input_scope"].eq("Whole")
        ].iloc[0]
        malignant = pseudobulk.loc[
            pseudobulk["patient_id"].eq(patient)
            & pseudobulk["input_scope"].eq("Malignant")
        ].iloc[0]
        audit_rows.append(
            {
                "patient_id": patient,
                "OGM_deep_deletion_label": int(
                    patient in OGM_DEEP_DELETION_PATIENTS
                ),
                "OGM_label_text": whole["OGM_label_text"],
                "n_whole_stringent_cells": int(whole["n_cells"]),
                "n_author_malignant_cells": int(malignant["n_cells"]),
                "malignant_ge50_primary_eligible": int(
                    malignant["n_cells"] >= MIN_MALIGNANT_CELLS_PRIMARY
                ),
                "n_libraries": int(subset["library_id_clean"].nunique()),
                "n_sample_sets": int(subset["sample_set_id_clean"].nunique()),
                "missing_model_features": ";".join(missing_features),
            }
        )
    patient_audit = pd.DataFrame(audit_rows)
    patient_audit.to_csv(
        OUTPUT_DIR / "03_GSE327167_31patient_input_audit.tsv",
        sep="\t",
        index=False,
    )
    del counts
    del assignment
    del present_group_counts
    gbm_expression = pd.read_csv(GBM_X_FILE, sep="\t", index_col="patient_id")
    gbm_expression.index = gbm_expression.index.astype(str)
    gbm_metadata = pd.read_csv(
        GBM_META_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    require(
        gbm_expression.index.is_unique,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: gbm_expression.index.is_unique.",
    )
    require(
        gbm_metadata.index.is_unique,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: gbm_metadata.index.is_unique.",
    )
    require(
        set(gbm_expression.index) == set(gbm_metadata.index),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: set(gbm_expression.index) == set(gbm_metadata.index).",
    )
    gbm_expression = gbm_expression.sort_index().loc[:, features]
    gbm_metadata = gbm_metadata.reindex(gbm_expression.index)
    gbm_expression = gbm_expression.apply(pd.to_numeric, errors="raise")
    require(
        len(gbm_expression) == 16,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: len(gbm_expression) == 16.",
    )
    require(
        np.isfinite(gbm_expression.to_numpy()).all(),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(gbm_expression.to_numpy()).all().",
    )
    gbm_calls = pd.to_numeric(gbm_metadata["MTAP_GISTIC_call"], errors="raise")
    gbm_labels = gbm_calls.eq(-2).astype(int)
    require(
        int(gbm_labels.sum()) == 8,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: int(gbm_labels.sum()) == 8.",
    )
    require(
        int((gbm_labels == 0).sum()) == 8,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: int((gbm_labels == 0).sum()) == 8.",
    )
    l2_bundle = joblib.load(L2_MODEL_FILE)
    require(
        list(l2_bundle["feature_names"]) == CORE_GENES,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: list(l2_bundle['feature_names']) == CORE_GENES.",
    )
    require(
        l2_bundle["expression_unit"] == "log2(CPM+1)",
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: l2_bundle['expression_unit'] == 'log2(CPM+1)'.",
    )
    l2_pipeline = l2_bundle["pipeline"]
    en_imputation_values = gbm_expression.median(axis=0)
    require(
        np.isfinite(en_imputation_values.to_numpy()).all(),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(en_imputation_values.to_numpy()).all().",
    )
    elastic_pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    penalty="elasticnet",
                    solver="saga",
                    C=ELASTIC_C,
                    l1_ratio=ELASTIC_L1_RATIO,
                    class_weight=None,
                    max_iter=20000,
                    tol=1e-05,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        elastic_pipeline.fit(gbm_expression, gbm_labels)
    joblib.dump(
        {
            "pipeline": elastic_pipeline,
            "feature_names": features,
            "expression_unit": "log2(CPM+1)",
            "training_patients": gbm_expression.index.tolist(),
            "training_labels": gbm_labels.to_dict(),
            "external_missing_feature_imputation": {
                gene: float(en_imputation_values[gene])
                for gene in missing_features
            },
            "GSE327167_used_for_training": False,
        },
        MODEL_OUT / "GBM16_snRNA_ElasticNet_final_with_imputation.joblib",
    )
    with Path(DL_MANIFEST_FILE).open("r", encoding="utf-8") as handle:
        dl_manifest = json.load(handle)
    locked_hashes = dl_manifest["locked_file_hashes"]
    for path in [DL_NETWORK_FILE, *DL_CHECKPOINT_FILES]:
        relative = Path(path).relative_to(DL_LOCKED_DIR).as_posix()
        require(
            relative in locked_hashes,
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: relative in locked_hashes.",
        )
        require(
            sha256_file(path) == locked_hashes[relative],
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: sha256_file(path) == locked_hashes[relative].",
        )
    module_name = "mtap_locked_TPM44_GSE327167_missing_feature"
    spec = importlib.util.spec_from_file_location(module_name, DL_NETWORK_FILE)
    network_module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = network_module
    spec.loader.exec_module(network_module)
    Network = network_module.RegionalMultiTaskNet
    checkpoints = [
        torch.load(path, map_location="cpu", weights_only=False)
        for path in DL_CHECKPOINT_FILES
    ]
    reference = checkpoints[0]
    require(
        list(reference["feature_names"]) == features,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: list(reference['feature_names']) == features.",
    )
    regions = reference["region_indices"]
    require(
        len(regions) == 4,
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: len(regions) == 4.",
    )
    for checkpoint in checkpoints:
        require(
            checkpoint["feature_names"] == reference["feature_names"],
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: checkpoint['feature_names'] == reference['feature_names'].",
        )
        require(
            checkpoint["region_indices"] == reference["region_indices"],
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: checkpoint['region_indices'] == reference['region_indices'].",
        )
        require(
            "GBM" in checkpoint["cancer_to_index"],
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: 'GBM' in checkpoint['cancer_to_index'].",
        )
        require(
            "MESO" in checkpoint["cancer_to_index"],
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: 'MESO' in checkpoint['cancer_to_index'].",
        )

    class FrozenBulkEnsemble(nn.Module):

        def __init__(self, checkpoint_list):
            super().__init__()
            self.networks = nn.ModuleList()
            self.cancer_maps = []
            self.mtap_indices = []
            for model_index, checkpoint in enumerate(checkpoint_list):
                model = Network(checkpoint)
                model.load_state_dict(checkpoint["state_dict"], strict=True)
                model.to(DEVICE).eval()
                for parameter in model.parameters():
                    parameter.requires_grad_(False)
                mean = np.asarray(
                    checkpoint["preprocessing"]["mean"], dtype=np.float32
                )
                std = np.asarray(
                    checkpoint["preprocessing"]["std"], dtype=np.float32
                )
                require(
                    mean.shape == (44,) and std.shape == (44,),
                    "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: mean.shape == (44,) and std.shape == (44,).",
                )
                require(
                    np.isfinite(mean).all() and np.isfinite(std).all(),
                    "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(mean).all() and np.isfinite(std).all().",
                )
                require(
                    (std > 0).all(),
                    "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: (std > 0).all().",
                )
                self.register_buffer(
                    f"mean_{model_index}", torch.tensor(mean, device=DEVICE)
                )
                self.register_buffer(
                    f"std_{model_index}", torch.tensor(std, device=DEVICE)
                )
                self.networks.append(model)
                self.cancer_maps.append(checkpoint["cancer_to_index"])
                self.mtap_indices.append(int(checkpoint["MTAP_INDEX"]))

        def forward(
            self, expression, cancer_acronym, adapter=None, missing_mask=None
        ):
            positive_logs = []
            negative_logs = []
            for model_index, network in enumerate(self.networks):
                standardized = (
                    expression - getattr(self, f"mean_{model_index}")
                ) / getattr(self, f"std_{model_index}")
                if missing_mask is not None and bool(missing_mask.any()):
                    standardized = standardized.clone()
                    standardized[:, missing_mask] = 0.0
                if adapter is not None:
                    standardized = adapter(standardized)
                require(
                    cancer_acronym in self.cancer_maps[model_index],
                    "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: cancer_acronym in self.cancer_maps[model_index].",
                )
                cancer = torch.full(
                    (len(expression),),
                    int(self.cancer_maps[model_index][cancer_acronym]),
                    dtype=torch.long,
                    device=expression.device,
                )
                logits = network(standardized, cancer)
                mtap_logits = logits[:, self.mtap_indices[model_index], :]
                log_class = torch.log_softmax(mtap_logits, dim=1)
                positive_logs.append(log_class[:, 0])
                negative_logs.append(torch.logsumexp(log_class[:, 1:], dim=1))
            normalizer = np.log(len(self.networks))
            return (
                torch.logsumexp(torch.stack(positive_logs), dim=0)
                - normalizer,
                torch.logsumexp(torch.stack(negative_logs), dim=0)
                - normalizer,
            )

    class RegionAffineAdapter(nn.Module):

        def __init__(self, region_indices):
            super().__init__()
            membership = torch.empty(44, dtype=torch.long)
            for region_id, indices in enumerate(region_indices):
                membership[indices] = region_id
            self.register_buffer("membership", membership)
            self.scale_raw = nn.Parameter(torch.zeros(4))
            self.shift_raw = nn.Parameter(torch.zeros(4))

        def coefficients(self):
            scale = torch.exp(MAX_LOG_SCALE * torch.tanh(self.scale_raw))
            shift = MAX_SHIFT * torch.tanh(self.shift_raw)
            return (scale, shift)

        def forward(self, standardized):
            scale, shift = self.coefficients()
            return (
                standardized * scale[self.membership] + shift[self.membership]
            )

        def identity_penalty(self):
            scale, shift = self.coefficients()
            return (scale - 1.0).square().mean() + shift.square().mean()

    ensemble = FrozenBulkEnsemble(checkpoints).to(DEVICE).eval()
    require(
        not any(
            (parameter.requires_grad for parameter in ensemble.parameters())
        ),
        "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: not any((parameter.requires_grad for parameter in ensemble.parameters())).",
    )
    adapter = RegionAffineAdapter(regions).to(DEVICE)
    optimizer = torch.optim.Adam(adapter.parameters(), lr=ADAPTER_LR)
    gbm_tensor = torch.tensor(
        gbm_expression.to_numpy(dtype=np.float32),
        dtype=torch.float32,
        device=DEVICE,
    )
    gbm_label_tensor = torch.tensor(
        gbm_labels.to_numpy(dtype=np.float32),
        dtype=torch.float32,
        device=DEVICE,
    )
    adapter.train()
    adapter_history = []
    for epoch in range(1, ADAPTER_EPOCHS + 1):
        optimizer.zero_grad(set_to_none=True)
        log_p, log_q = ensemble(gbm_tensor, "GBM", adapter=adapter)
        bce = -(
            gbm_label_tensor * log_p + (1.0 - gbm_label_tensor) * log_q
        ).mean()
        penalty = adapter.identity_penalty()
        loss = bce + IDENTITY_PENALTY * penalty
        require(
            torch.isfinite(loss),
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: torch.isfinite(loss).",
        )
        loss.backward()
        for parameter in adapter.parameters():
            require(
                parameter.grad is not None
                and torch.isfinite(parameter.grad).all(),
                "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: parameter.grad is not None and torch.isfinite(parameter.grad).all().",
            )
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 5.0)
        optimizer.step()
        adapter_history.append(
            {
                "epoch": epoch,
                "BCE": float(bce.detach()),
                "identity_penalty": float(penalty.detach()),
                "total_loss": float(loss.detach()),
            }
        )
    adapter.eval()
    torch.save(
        {
            "adapter_state_dict": {
                key: value.detach().cpu().clone()
                for key, value in adapter.state_dict().items()
            },
            "feature_names": features,
            "training_patients": gbm_expression.index.tolist(),
            "training_labels": gbm_labels.to_dict(),
            "external_missing_features": missing_features,
            "missing_feature_handling": "checkpoint-specific training mean; standardized value zero",
            "GSE327167_used_for_training": False,
        },
        MODEL_OUT / "GBM16_snRNA_DL_adapter_final_missing_feature_aware.pt",
    )
    pd.DataFrame(adapter_history).to_csv(
        MODEL_OUT / "GBM16_DL_adapter_training_history.tsv",
        sep="\t",
        index=False,
    )
    missing_mask_numpy = np.array(
        [gene in missing_features for gene in features], dtype=bool
    )
    missing_mask_tensor = torch.tensor(
        missing_mask_numpy, dtype=torch.bool, device=DEVICE
    )
    prediction_frames = []
    for input_scope in ["Whole", "Malignant"]:
        scope_frame = (
            pseudobulk.loc[
                pseudobulk["input_scope"].eq(input_scope)
                & pseudobulk["n_cells"].gt(0)
            ]
            .copy()
            .sort_values("patient_id")
            .reset_index(drop=True)
        )
        raw_input = scope_frame.loc[:, features].apply(
            pd.to_numeric, errors="coerce"
        )
        unexpected_nan = [
            gene
            for gene in features
            if raw_input[gene].isna().any() and gene not in missing_features
        ]
        require(
            not unexpected_nan,
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: not unexpected_nan.",
        )
        l2_input = raw_input.loc[:, CORE_GENES]
        require(
            np.isfinite(l2_input.to_numpy()).all(),
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(l2_input.to_numpy()).all().",
        )
        l2_probability = positive_probability(l2_pipeline, l2_input)
        en_input = raw_input.copy()
        for gene in missing_features:
            en_input[gene] = en_input[gene].fillna(en_imputation_values[gene])
        require(
            np.isfinite(en_input.to_numpy()).all(),
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(en_input.to_numpy()).all().",
        )
        elastic_probability = positive_probability(elastic_pipeline, en_input)
        dl_input = raw_input.fillna(0.0)
        dl_tensor = torch.tensor(
            dl_input.to_numpy(dtype=np.float32),
            dtype=torch.float32,
            device=DEVICE,
        )
        with torch.no_grad():
            dl_logp, _ = ensemble(
                dl_tensor,
                "MESO",
                adapter=adapter,
                missing_mask=missing_mask_tensor,
            )
            dl_probability = dl_logp.exp().cpu().numpy()
        scored = scope_frame.loc[
            :,
            [
                "patient_id",
                "input_scope",
                "n_cells",
                "total_gene_expression_UMI",
                "OGM_deep_deletion_label",
                "OGM_label_text",
                "n_missing_model_features",
                "missing_model_features",
            ],
        ].copy()
        scored["Three-gene L2"] = l2_probability
        scored["44-gene Elastic Net"] = elastic_probability
        scored["Bulk DL + input adapter"] = dl_probability
        scored["malignant_ge50_primary_eligible"] = (
            scored["n_cells"].ge(MIN_MALIGNANT_CELLS_PRIMARY)
            if input_scope == "Malignant"
            else True
        )
        scored["limited_cell_number_flag"] = (
            scored["n_cells"].lt(MIN_MALIGNANT_CELLS_PRIMARY)
            if input_scope == "Malignant"
            else False
        )
        prediction_frames.append(scored)
    predictions = pd.concat(prediction_frames, ignore_index=True)
    for model in MODEL_ORDER:
        require(
            np.isfinite(predictions[model]).all(),
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: np.isfinite(predictions[model]).all().",
        )
        require(
            predictions[model].between(0, 1).all(),
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: predictions[model].between(0, 1).all().",
        )
    predictions.to_csv(
        OUTPUT_DIR
        / "04_GSE327167_three_model_predictions_missing_feature_aware.tsv",
        sep="\t",
        index=False,
    )


## 8. Evaluation and Performance Metrics

Assess nonmalignant-cell dilution, preserve the GBM threshold comparison, and carry the GBM-selected 0.75 rule into deployment summaries. This is not a claim that 0.75 is optimal in every external cohort. TCGA development performance is OOF, not apparent training-set performance. The unified summary retains threshold 0.50 results and adds prespecified threshold 0.75 comparisons without changing model scores.


### Final three-model performance


#### GBM snRNA three-model performance and confidence intervals


In [ ]:
@_nb9_registry.register("s62_105")
def _nb9_stage_s62_105():
    performance_rows = []
    for model in MODELS:
        for metric_index, metric in enumerate(METRICS):
            lower, upper = percentile_ci(
                bootstrap_results[model][:, metric_index]
            )
            performance_rows.append(
                {
                    "model": model,
                    "display_label": LABELS[model].replace("\n", " "),
                    "metric": metric,
                    "estimate": point_estimates[model][metric],
                    "CI95_low": lower,
                    "CI95_high": upper,
                    "n_patients": len(y),
                    "n_positive": int(y.sum()),
                    "n_negative": int((y == 0).sum()),
                    "descriptive_threshold": THRESHOLD,
                    "bootstrap_retraining": False,
                }
            )
    performance = pd.DataFrame(performance_rows)
    performance.to_csv(
        OUT / "snRNA_adapter_L2_ElasticNet_performance_CI.tsv",
        sep="\t",
        index=False,
    )
    confusion_rows = []
    for model in MODELS:
        result = point_estimates[model]
        confusion_rows.append(
            {
                "model": model,
                "display_label": LABELS[model].replace("\n", " "),
                "threshold": THRESHOLD,
                "TP": result["TP"],
                "FP": result["FP"],
                "FN": result["FN"],
                "TN": result["TN"],
            }
        )
    confusion_table = pd.DataFrame(confusion_rows)
    confusion_table.to_csv(
        OUT / "snRNA_adapter_L2_ElasticNet_confusion_matrix.tsv",
        sep="\t",
        index=False,
    )
    formatted = performance.copy()
    formatted["value"] = formatted.apply(
        lambda row: (
            f"{row['estimate']:.3f} ({row['CI95_low']:.3f}–{row['CI95_high']:.3f})"
            if np.isfinite(row["estimate"])
            else "Undefined"
        ),
        axis=1,
    )
    summary = formatted.pivot(index="model", columns="metric", values="value")
    summary = summary.reindex(index=MODELS, columns=METRICS)
    summary.to_csv(
        OUT / "snRNA_adapter_L2_ElasticNet_performance_summary.tsv", sep="\t"
    )
    FIGURE_METRICS = ["AUROC", "AP", "Balanced_accuracy"]
    FIGURE_TITLES = {
        "AUROC": "AUROC",
        "AP": "Average precision",
        "Balanced_accuracy": "Balanced accuracy",
    }


#### TCGA OOF and CPTAC final three-model performance


In [ ]:
@_nb9_registry.register("s64_0")
def _nb9_stage_s64_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from itertools import combinations
    import json
    import numpy as np
    import pandas as pd
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        roc_curve,
        precision_recall_curve,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    COMPARISON_DIR = (
        PROJECT
        / "data_processed"
        / "MTAP_RNA_prediction"
        / "15_region_multitask_DL"
        / "08_TPM_retraining"
        / "06_three_gene_L2_comparison"
    )
    TCGA_PATH = COMPARISON_DIR / "three_model_development_OOF.tsv"
    CPTAC_PATH = COMPARISON_DIR / "CPTAC_three_model_patient_predictions.tsv"
    OUT = COMPARISON_DIR / "TCGA_CPTAC_three_model_performance_figure"
    OUT.mkdir(parents=True, exist_ok=True)
    SEED = 20261018
    THRESHOLD = 0.5
    N_BOOTSTRAP = 1000
    MODEL_ORDER = ["Bulk_DL", "Three_gene_L2", "Elastic_Net"]
    PROBABILITY_COLUMNS = {
        "Bulk_DL": "prob_DL",
        "Three_gene_L2": "prob_L2",
        "Elastic_Net": "prob_ElasticNet",
    }
    MODEL_LABELS = {
        "Bulk_DL": "Bulk DL",
        "Three_gene_L2": "Three-gene L2",
        "Elastic_Net": "Elastic Net",
    }
    MODEL_COLORS = {
        "Bulk_DL": "#A5678E",
        "Three_gene_L2": "#33539E",
        "Elastic_Net": "#7FACD6",
    }
    COHORT_ORDER = ["TCGA_OOF", "CPTAC_external"]
    COHORT_LABELS = {
        "TCGA_OOF": "TCGA development cohort (OOF)",
        "CPTAC_external": "CPTAC external test cohort",
    }
    METRICS = [
        "AUROC",
        "AP",
        "Balanced_accuracy",
        "Sensitivity",
        "Specificity",
        "Accuracy",
        "Precision",
        "F1",
        "Brier",
    ]
    FIGURE_METRICS = ["AUROC", "AP", "Balanced_accuracy"]
    FIGURE_METRIC_LABELS = {
        "AUROC": "AUROC",
        "AP": "Average precision",
        "Balanced_accuracy": "Balanced accuracy",
    }
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.titlesize": 10,
            "axes.labelsize": 9,
            "legend.fontsize": 8,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.dpi": 400,
        }
    )
    for path in [TCGA_PATH, CPTAC_PATH]:
        if not path.is_file():
            raise FileNotFoundError(
                "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
            )
    tcga = pd.read_csv(TCGA_PATH, sep="\t", dtype={"patient_id": str})
    cptac = pd.read_csv(CPTAC_PATH, sep="\t", dtype={"patient_id": str})
    cohort_frames = {"TCGA_OOF": tcga, "CPTAC_external": cptac}
    REQUIRED_COLUMNS = {
        "patient_id",
        "label_MTAP_deep",
        "prob_DL",
        "prob_L2",
        "prob_ElasticNet",
    }
    audit_rows = []
    for cohort_name, frame in cohort_frames.items():
        missing_columns = REQUIRED_COLUMNS - set(frame.columns)
        if missing_columns:
            raise ValueError(
                "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
            )
        frame["patient_id"] = frame["patient_id"].astype(str).str.strip()
        if frame["patient_id"].isna().any():
            raise ValueError(
                "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
            )
        if frame["patient_id"].duplicated().any():
            duplicated = frame.loc[
                frame["patient_id"].duplicated(keep=False), "patient_id"
            ].tolist()
            raise ValueError(
                "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
            )
        frame["label_MTAP_deep"] = pd.to_numeric(
            frame["label_MTAP_deep"], errors="raise"
        )
        if not frame["label_MTAP_deep"].isin([0, 1]).all():
            invalid_labels = sorted(
                frame.loc[
                    ~frame["label_MTAP_deep"].isin([0, 1]), "label_MTAP_deep"
                ].unique()
            )
            raise ValueError(
                "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
            )
        for model_name, probability_column in PROBABILITY_COLUMNS.items():
            frame[probability_column] = pd.to_numeric(
                frame[probability_column], errors="raise"
            )
            probabilities = frame[probability_column].to_numpy(dtype=float)
            if not np.isfinite(probabilities).all():
                raise ValueError(
                    "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
                )
            if not ((probabilities >= 0) & (probabilities <= 1)).all():
                raise ValueError(
                    "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
                )
        y = frame["label_MTAP_deep"].astype(int).to_numpy()
        if len(np.unique(y)) != 2:
            raise ValueError(
                "TCGA OOF and CPTAC final three-model performance: integrity check failed; condition: input or artifact integrity check."
            )
        audit_rows.append(
            {
                "cohort": cohort_name,
                "cohort_label": COHORT_LABELS[cohort_name],
                "n_patients": len(frame),
                "n_positive": int(y.sum()),
                "n_negative": int((y == 0).sum()),
                "positive_fraction": float(y.mean()),
                "prediction_type": (
                    "Patient-level out-of-fold predictions"
                    if cohort_name == "TCGA_OOF"
                    else "Frozen-model external predictions"
                ),
                "n_cancers": (
                    int(frame["acronym"].nunique())
                    if "acronym" in frame.columns
                    else np.nan
                ),
            }
        )
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(OUT / "cohort_audit.tsv", sep="\t", index=False)

    def safe_ratio(numerator, denominator):
        """Return a missing value when a performance denominator is zero."""
        if denominator == 0:
            return np.nan
        return numerator / denominator

    def evaluate_model(y_true, probabilities, threshold=0.5):
        """Calculate patient-level metrics using the prespecified threshold."""
        predicted_positive = probabilities >= threshold
        tp = int(((y_true == 1) & predicted_positive).sum())
        fp = int(((y_true == 0) & predicted_positive).sum())
        fn = int(((y_true == 1) & ~predicted_positive).sum())
        tn = int(((y_true == 0) & ~predicted_positive).sum())
        sensitivity = safe_ratio(tp, tp + fn)
        specificity = safe_ratio(tn, tn + fp)
        return {
            "AUROC": roc_auc_score(y_true, probabilities),
            "AP": average_precision_score(y_true, probabilities),
            "Balanced_accuracy": (sensitivity + specificity) / 2,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Accuracy": safe_ratio(tp + tn, len(y_true)),
            "Precision": safe_ratio(tp, tp + fp),
            "F1": safe_ratio(2 * tp, 2 * tp + fp + fn),
            "Brier": float(np.mean((probabilities - y_true) ** 2)),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }

    def percentile_ci(values):
        """Calculate a percentile bootstrap confidence interval."""
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            return (np.nan, np.nan)
        lower, upper = np.quantile(values, [0.025, 0.975])
        return (float(lower), float(upper))

    point_estimates = {}
    bootstrap_estimates = {}
    bootstrap_indices = {}
    for cohort_index, cohort_name in enumerate(COHORT_ORDER):
        frame = cohort_frames[cohort_name]
        y = frame["label_MTAP_deep"].astype(int).to_numpy()
        positive_indices = np.flatnonzero(y == 1)
        negative_indices = np.flatnonzero(y == 0)
        rng = np.random.default_rng(SEED + cohort_index)
        sampled_indices_list = []
        for bootstrap_index in range(N_BOOTSTRAP):
            sampled_positive = rng.choice(
                positive_indices, size=len(positive_indices), replace=True
            )
            sampled_negative = rng.choice(
                negative_indices, size=len(negative_indices), replace=True
            )
            sampled_indices = np.concatenate(
                [sampled_positive, sampled_negative]
            )
            sampled_indices = rng.permutation(sampled_indices)
            sampled_indices_list.append(sampled_indices)
        bootstrap_indices[cohort_name] = sampled_indices_list
        for model_name in MODEL_ORDER:
            probability_column = PROBABILITY_COLUMNS[model_name]
            probabilities = frame[probability_column].to_numpy(dtype=float)
            point_estimates[cohort_name, model_name] = evaluate_model(
                y, probabilities, threshold=THRESHOLD
            )
            model_bootstrap = np.full((N_BOOTSTRAP, len(METRICS)), np.nan)
            for bootstrap_index, sampled_indices in enumerate(
                sampled_indices_list
            ):
                bootstrap_result = evaluate_model(
                    y[sampled_indices],
                    probabilities[sampled_indices],
                    threshold=THRESHOLD,
                )
                model_bootstrap[bootstrap_index] = [
                    bootstrap_result[metric] for metric in METRICS
                ]
            bootstrap_estimates[cohort_name, model_name] = model_bootstrap
    performance_rows = []
    for cohort_name in COHORT_ORDER:
        frame = cohort_frames[cohort_name]
        y = frame["label_MTAP_deep"].astype(int).to_numpy()
        for model_name in MODEL_ORDER:
            point_result = point_estimates[cohort_name, model_name]
            model_bootstrap = bootstrap_estimates[cohort_name, model_name]
            for metric_index, metric in enumerate(METRICS):
                lower, upper = percentile_ci(model_bootstrap[:, metric_index])
                performance_rows.append(
                    {
                        "cohort": cohort_name,
                        "cohort_label": COHORT_LABELS[cohort_name],
                        "model": model_name,
                        "model_label": MODEL_LABELS[model_name],
                        "metric": metric,
                        "estimate": point_result[metric],
                        "CI95_low": lower,
                        "CI95_high": upper,
                        "n_patients": len(y),
                        "n_positive": int(y.sum()),
                        "n_negative": int((y == 0).sum()),
                        "threshold": THRESHOLD,
                        "bootstrap_retraining": False,
                        "prediction_source": (
                            "OOF"
                            if cohort_name == "TCGA_OOF"
                            else "Frozen external"
                        ),
                    }
                )
    performance = pd.DataFrame(performance_rows)
    performance.to_csv(OUT / "performance_with_CI.tsv", sep="\t", index=False)
    formatted = performance.copy()
    formatted["estimate_CI95"] = formatted.apply(
        lambda row: f"{row['estimate']:.3f} ({row['CI95_low']:.3f}–{row['CI95_high']:.3f})",
        axis=1,
    )
    summary = formatted.pivot_table(
        index=["cohort", "model"],
        columns="metric",
        values="estimate_CI95",
        aggfunc="first",
    )
    summary = summary.reindex(
        pd.MultiIndex.from_product(
            [COHORT_ORDER, MODEL_ORDER], names=["cohort", "model"]
        )
    )
    summary.to_csv(OUT / "performance_summary.tsv", sep="\t")
    confusion_rows = []
    for cohort_name in COHORT_ORDER:
        frame = cohort_frames[cohort_name]
        y = frame["label_MTAP_deep"].astype(int).to_numpy()
        for model_name in MODEL_ORDER:
            result = point_estimates[cohort_name, model_name]
            confusion_rows.append(
                {
                    "cohort": cohort_name,
                    "model": model_name,
                    "model_label": MODEL_LABELS[model_name],
                    "threshold": THRESHOLD,
                    "n_patients": len(y),
                    "n_positive": int(y.sum()),
                    "TP": result["TP"],
                    "FP": result["FP"],
                    "FN": result["FN"],
                    "TN": result["TN"],
                }
            )
    confusion_table = pd.DataFrame(confusion_rows)
    confusion_table.to_csv(
        OUT / "confusion_matrix_at_0.5.tsv", sep="\t", index=False
    )
    PAIRWISE_METRICS = ["AUROC", "AP", "Balanced_accuracy", "Brier"]
    pairwise_rows = []
    model_pairs = [
        ("Bulk_DL", "Three_gene_L2"),
        ("Bulk_DL", "Elastic_Net"),
        ("Three_gene_L2", "Elastic_Net"),
    ]
    for cohort_name in COHORT_ORDER:
        for model_a, model_b in model_pairs:
            point_a = point_estimates[cohort_name, model_a]
            point_b = point_estimates[cohort_name, model_b]
            bootstrap_a = bootstrap_estimates[cohort_name, model_a]
            bootstrap_b = bootstrap_estimates[cohort_name, model_b]
            for metric in PAIRWISE_METRICS:
                metric_index = METRICS.index(metric)
                point_difference = point_a[metric] - point_b[metric]
                bootstrap_difference = (
                    bootstrap_a[:, metric_index] - bootstrap_b[:, metric_index]
                )
                lower, upper = percentile_ci(bootstrap_difference)
                if metric == "Brier":
                    favorable_interpretation = f"Negative difference favors {MODEL_LABELS[model_a]}; positive difference favors {MODEL_LABELS[model_b]}"
                else:
                    favorable_interpretation = f"Positive difference favors {MODEL_LABELS[model_a]}; negative difference favors {MODEL_LABELS[model_b]}"
                pairwise_rows.append(
                    {
                        "cohort": cohort_name,
                        "model_A": model_a,
                        "model_B": model_b,
                        "metric": metric,
                        "difference_A_minus_B": point_difference,
                        "CI95_low": lower,
                        "CI95_high": upper,
                        "CI_excludes_zero": bool(lower > 0 or upper < 0),
                        "interpretation": favorable_interpretation,
                        "interval_scope": "Patient-level stratified paired percentile bootstrap",
                    }
                )
    pairwise_comparison = pd.DataFrame(pairwise_rows)
    pairwise_comparison.to_csv(
        OUT / "paired_model_differences.tsv", sep="\t", index=False
    )
    GENERALIZATION_METRICS = ["AUROC", "AP", "Balanced_accuracy", "Brier"]
    generalization_rows = []
    for model_name in MODEL_ORDER:
        for metric in GENERALIZATION_METRICS:
            metric_index = METRICS.index(metric)
            tcga_point = point_estimates["TCGA_OOF", model_name][metric]
            cptac_point = point_estimates["CPTAC_external", model_name][metric]
            point_difference = cptac_point - tcga_point
            difference_distribution = (
                bootstrap_estimates["CPTAC_external", model_name][
                    :, metric_index
                ]
                - bootstrap_estimates["TCGA_OOF", model_name][:, metric_index]
            )
            lower, upper = percentile_ci(difference_distribution)
            generalization_rows.append(
                {
                    "model": model_name,
                    "model_label": MODEL_LABELS[model_name],
                    "metric": metric,
                    "TCGA_OOF_estimate": tcga_point,
                    "CPTAC_external_estimate": cptac_point,
                    "difference_CPTAC_minus_TCGA": point_difference,
                    "CI95_low": lower,
                    "CI95_high": upper,
                    "CI_excludes_zero": bool(lower > 0 or upper < 0),
                    "interval_scope": "Independent-cohort stratified bootstrap",
                }
            )
    generalization = pd.DataFrame(generalization_rows)
    generalization.to_csv(
        OUT / "generalization_gap_CPTAC_minus_TCGA.tsv", sep="\t", index=False
    )


#### TCGA holdout bootstrap intervals and model comparisons


In [ ]:
@_nb9_registry.register("s72_129")
def _nb9_stage_s72_129():
    MODEL_ORDER = ["Bulk_DL", "Three_gene_L2", "Elastic_Net"]
    COLORS = {
        "Three_gene_L2": "#33539E",
        "Elastic_Net": "#7FACD6",
        "Bulk_DL": "#A5678E",
    }

    def metrics(labels, probability):
        predicted = probability >= THRESHOLD
        tn, fp, fn, tp = confusion_matrix(
            labels, predicted, labels=[0, 1]
        ).ravel()
        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)
        return {
            "AUROC": roc_auc_score(labels, probability),
            "AP": average_precision_score(labels, probability),
            "Brier": brier_score_loss(labels, probability),
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Balanced_accuracy": (sensitivity + specificity) / 2,
            "PPV": tp / (tp + fp) if tp + fp else np.nan,
            "Accuracy": (tp + tn) / len(labels),
        }

    point = {name: metrics(y, probabilities[name]) for name in MODEL_ORDER}
    metric_names = list(point[MODEL_ORDER[0]])
    bootstrap = {
        name: np.empty((N_BOOTSTRAP, len(metric_names)))
        for name in MODEL_ORDER
    }
    positive_indices = np.flatnonzero(y == 1)
    negative_indices = np.flatnonzero(y == 0)
    rng = np.random.default_rng(SEED)
    for b in range(N_BOOTSTRAP):
        indices = np.concatenate(
            [
                rng.choice(
                    positive_indices, len(positive_indices), replace=True
                ),
                rng.choice(
                    negative_indices, len(negative_indices), replace=True
                ),
            ]
        )
        for name in MODEL_ORDER:
            result = metrics(y[indices], probabilities[name][indices])
            bootstrap[name][b] = [result[m] for m in metric_names]
        if (b + 1) % 250 == 0:
            pass
    records = []
    for name in MODEL_ORDER:
        for j, metric in enumerate(metric_names):
            lower, upper = np.nanpercentile(bootstrap[name][:, j], [2.5, 97.5])
            records.append(
                {
                    "cohort": "TCGA_original_holdout_TPM_matched",
                    "model": name,
                    "metric": metric,
                    "estimate": point[name][metric],
                    "CI95_low": lower,
                    "CI95_high": upper,
                    "n_patients": len(y),
                    "n_positive": int(y.sum()),
                    "classification_threshold": THRESHOLD,
                    "bootstrap_retraining": False,
                }
            )
    performance = pd.DataFrame(records)
    performance.to_csv(
        OUT / "06_performance_with_CI.tsv", sep="\t", index=False
    )
    confusion_records = []
    for name in MODEL_ORDER:
        tn, fp, fn, tp = confusion_matrix(
            y, probabilities[name] >= THRESHOLD, labels=[0, 1]
        ).ravel()
        confusion_records.append(
            {
                "model": name,
                "threshold": THRESHOLD,
                "TP": tp,
                "FP": fp,
                "FN": fn,
                "TN": tn,
            }
        )
    pd.DataFrame(confusion_records).to_csv(
        OUT / "07_confusion_counts.tsv", sep="\t", index=False
    )
    difference_records = []
    for comparator in ["Elastic_Net", "Bulk_DL"]:
        for j, metric in enumerate(metric_names):
            differences = (
                bootstrap["Three_gene_L2"][:, j] - bootstrap[comparator][:, j]
            )
            lower, upper = np.nanpercentile(differences, [2.5, 97.5])
            difference_records.append(
                {
                    "comparison": f"Three_gene_L2_minus_{comparator}",
                    "metric": metric,
                    "difference": point["Three_gene_L2"][metric]
                    - point[comparator][metric],
                    "CI95_low": lower,
                    "CI95_high": upper,
                }
            )
    pd.DataFrame(difference_records).to_csv(
        OUT / "08_paired_model_differences.tsv", sep="\t", index=False
    )
    cancer_records = []
    for cancer, group in predictions.groupby("acronym"):
        labels = group["label_MTAP_deep"].to_numpy(dtype=int)
        if len(np.unique(labels)) < 2:
            cancer_records.append(
                {
                    "acronym": cancer,
                    "model": "Not_evaluable_for_AUROC",
                    "n_patients": len(labels),
                    "n_positive": int(labels.sum()),
                }
            )
            continue
        for name in MODEL_ORDER:
            cancer_records.append(
                {
                    "acronym": cancer,
                    "model": name,
                    "n_patients": len(labels),
                    "n_positive": int(labels.sum()),
                    **metrics(labels, group[f"prob_{name}"].to_numpy()),
                }
            )
    pd.DataFrame(cancer_records).to_csv(
        OUT / "09_per_cancer_descriptive_performance.tsv",
        sep="\t",
        index=False,
    )


#### GSE327167 patient-level evaluation sets and metrics


In [ ]:
@_nb9_registry.register("s74_129")
def _nb9_stage_s74_129():
    evaluation_sets = {
        "Whole_31": predictions.loc[
            predictions["input_scope"].eq("Whole")
        ].copy(),
        "Malignant_any_author_tumor": predictions.loc[
            predictions["input_scope"].eq("Malignant")
        ].copy(),
        "Malignant_ge50_primary": predictions.loc[
            predictions["input_scope"].eq("Malignant")
            & predictions["malignant_ge50_primary_eligible"]
        ].copy(),
    }
    metric_rows = []
    for evaluation_name, frame in evaluation_sets.items():
        frame = frame.sort_values("patient_id")
        y = frame["OGM_deep_deletion_label"].to_numpy(dtype=int)
        require(
            len(np.unique(y)) == 2,
            "GSE327167 missing-feature handling, inference, and evaluation: required condition failed: len(np.unique(y)) == 2.",
        )
        for model in MODEL_ORDER:
            probability = frame[model].to_numpy(dtype=float)
            metric_rows.append(
                {
                    "evaluation_set": evaluation_name,
                    "model": model,
                    "n_patients": len(frame),
                    "n_OGM_detected_deep_deletion": int(y.sum()),
                    "n_no_OGM_detected_deep_deletion": int((y == 0).sum()),
                    "threshold": MODEL_THRESHOLD,
                    **calculate_metrics(y, probability, MODEL_THRESHOLD),
                }
            )
    performance = pd.DataFrame(metric_rows)
    performance.to_csv(
        OUTPUT_DIR
        / "05_GSE327167_three_model_performance_missing_feature_aware.tsv",
        sep="\t",
        index=False,
    )
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


#### GSE327167 three-model metrics and bootstrap intervals at 0.50


In [ ]:
@_nb9_registry.register("s75_0")
def _nb9_stage_s75_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import warnings
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, Normalize
    import numpy as np
    import pandas as pd
    from sklearn.metrics import (
        average_precision_score,
        balanced_accuracy_score,
        brier_score_loss,
        confusion_matrix,
        roc_auc_score,
    )

    OUTPUT_DIR = Path(globals().get(
        "GSE327167_RUN_DIR",
        Path(os.environ.get(
            "MTAP_GSE327167_RUN_DIR",
            str(PROJECT_ROOT / (
                "data_processed/MTAP_prediction_registry/external_validation/"
                "20261103_110248_573136_GSE327167_31patient_three_models"
            )),
        )),
    ))
    PREDICTION_FILE = (
        OUTPUT_DIR
        / "04_GSE327167_three_model_predictions_missing_feature_aware.tsv"
    )
    FIGURE_DIR = OUTPUT_DIR / "figures"
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    THRESHOLD = 0.5
    N_BOOTSTRAP = 5000
    RANDOM_SEED = 20261103
    MODELS = [
        "Three-gene L2",
        "44-gene Elastic Net",
        "Bulk DL + input adapter",
    ]
    MODEL_LABELS = {
        "Three-gene L2": "Three-gene\nL2",
        "44-gene Elastic Net": "44-gene\nElastic Net",
        "Bulk DL + input adapter": "Bulk DL +\ninput adapter",
    }
    EVALUATION_ORDER = [
        "Whole_31",
        "Malignant_any_author_tumor",
        "Malignant_ge50_primary",
    ]
    EVALUATION_SHORT_LABELS = {
        "Whole_31": "Whole tissue",
        "Malignant_any_author_tumor": "Malignant cells\n(any count)",
        "Malignant_ge50_primary": "Malignant cells\n(≥50 cells)",
    }
    METRICS = [
        "AUROC",
        "AP",
        "Brier",
        "Balanced_accuracy",
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
    ]
    METRIC_TITLES = {
        "AUROC": "AUROC ↑",
        "AP": "Average precision ↑",
        "Brier": "Brier score ↓",
        "Balanced_accuracy": "Balanced accuracy ↑",
        "Sensitivity": "Sensitivity ↑",
        "Specificity": "Specificity ↑",
        "PPV": "Positive predictive value ↑",
        "NPV": "Negative predictive value ↑",
    }
    FIVE_COLOR_PALETTE = [
        "#33539E",
        "#7FACD6",
        "#BFB8DA",
        "#E8B7D4",
        "#A5678E",
    ]
    PERFORMANCE_CMAP = LinearSegmentedColormap.from_list(
        "MTAP_five_color_performance", FIVE_COLOR_PALETTE, N=256
    )
    PERFORMANCE_CMAP.set_bad("#E8E8E8")

    def require(condition, message):
        if not bool(condition):
            raise RuntimeError(message)

    require(
        PREDICTION_FILE.is_file(),
        "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: PREDICTION_FILE.is_file().",
    )
    predictions = pd.read_csv(
        PREDICTION_FILE, sep="\t", dtype={"patient_id": str}, low_memory=False
    )
    required_columns = {
        "patient_id",
        "input_scope",
        "n_cells",
        "OGM_deep_deletion_label",
        "malignant_ge50_primary_eligible",
        *MODELS,
    }
    missing_columns = sorted(required_columns.difference(predictions.columns))
    require(
        not missing_columns,
        "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: not missing_columns.",
    )
    numeric_columns = [
        "n_cells",
        "OGM_deep_deletion_label",
        "malignant_ge50_primary_eligible",
        *MODELS,
    ]
    for column in numeric_columns:
        predictions[column] = pd.to_numeric(
            predictions[column], errors="raise"
        )
    require(
        predictions["patient_id"].notna().all(),
        "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: predictions['patient_id'].notna().all().",
    )
    require(
        predictions["OGM_deep_deletion_label"].isin([0, 1]).all(),
        "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: predictions['OGM_deep_deletion_label'].isin([0, 1]).all().",
    )
    for model in MODELS:
        require(
            np.isfinite(predictions[model]).all(),
            "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: np.isfinite(predictions[model]).all().",
        )
        require(
            predictions[model].between(0, 1).all(),
            "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: predictions[model].between(0, 1).all().",
        )
    evaluation_sets = {
        "Whole_31": predictions.loc[
            predictions["input_scope"].eq("Whole")
        ].copy(),
        "Malignant_any_author_tumor": predictions.loc[
            predictions["input_scope"].eq("Malignant")
            & predictions["n_cells"].gt(0)
        ].copy(),
        "Malignant_ge50_primary": predictions.loc[
            predictions["input_scope"].eq("Malignant")
            & predictions["malignant_ge50_primary_eligible"].eq(1)
        ].copy(),
    }
    for evaluation_name, frame in evaluation_sets.items():
        require(
            frame["patient_id"].is_unique,
            "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: frame['patient_id'].is_unique.",
        )
        require(
            frame["OGM_deep_deletion_label"].nunique() == 2,
            "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: frame['OGM_deep_deletion_label'].nunique() == 2.",
        )
        evaluation_sets[evaluation_name] = frame.sort_values(
            "patient_id"
        ).reset_index(drop=True)

    def safe_ratio(numerator, denominator):
        if denominator == 0:
            return np.nan
        return float(numerator / denominator)

    def calculate_all_metrics(y_true, probability, threshold=0.5):
        y_true = np.asarray(y_true, dtype=int)
        probability = np.asarray(probability, dtype=float)
        predicted = (probability >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            y_true, predicted, labels=[0, 1]
        ).ravel()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            balanced_accuracy = balanced_accuracy_score(y_true, predicted)
        return {
            "AUROC": float(roc_auc_score(y_true, probability)),
            "AP": float(average_precision_score(y_true, probability)),
            "Brier": float(brier_score_loss(y_true, probability)),
            "Balanced_accuracy": float(balanced_accuracy),
            "Sensitivity": safe_ratio(tp, tp + fn),
            "Specificity": safe_ratio(tn, tn + fp),
            "PPV": safe_ratio(tp, tp + fp),
            "NPV": safe_ratio(tn, tn + fn),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
        }

    def stratified_bootstrap_metrics(y_true, probability, n_bootstrap, seed):
        y_true = np.asarray(y_true, dtype=int)
        probability = np.asarray(probability, dtype=float)
        positive_index = np.flatnonzero(y_true == 1)
        negative_index = np.flatnonzero(y_true == 0)
        require(
            len(positive_index) > 0 and len(negative_index) > 0,
            "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: len(positive_index) > 0 and len(negative_index) > 0.",
        )
        rng = np.random.default_rng(seed)
        bootstrap_values = {metric: [] for metric in METRICS}
        for _ in range(n_bootstrap):
            sampled_positive = rng.choice(
                positive_index, size=len(positive_index), replace=True
            )
            sampled_negative = rng.choice(
                negative_index, size=len(negative_index), replace=True
            )
            sampled_index = np.concatenate(
                [sampled_positive, sampled_negative]
            )
            sampled_index = rng.permutation(sampled_index)
            values = calculate_all_metrics(
                y_true[sampled_index],
                probability[sampled_index],
                threshold=THRESHOLD,
            )
            for metric in METRICS:
                bootstrap_values[metric].append(values[metric])
        intervals = {}
        for metric, values in bootstrap_values.items():
            values = np.asarray(values, dtype=float)
            values = values[np.isfinite(values)]
            if len(values) == 0:
                intervals[metric] = (np.nan, np.nan)
            else:
                intervals[metric] = tuple(np.percentile(values, [2.5, 97.5]))
        return intervals

    summary_rows = []
    for evaluation_index, evaluation_name in enumerate(EVALUATION_ORDER):
        frame = evaluation_sets[evaluation_name]
        y_true = frame["OGM_deep_deletion_label"].to_numpy(dtype=int)
        for model_index, model in enumerate(MODELS):
            probability = frame[model].to_numpy(dtype=float)
            point = calculate_all_metrics(
                y_true, probability, threshold=THRESHOLD
            )
            intervals = stratified_bootstrap_metrics(
                y_true,
                probability,
                n_bootstrap=N_BOOTSTRAP,
                seed=RANDOM_SEED + evaluation_index * 100 + model_index,
            )
            row = {
                "evaluation_set": evaluation_name,
                "model": model,
                "n_patients": len(frame),
                "n_positive": int(y_true.sum()),
                "n_negative": int((y_true == 0).sum()),
                "threshold": THRESHOLD,
            }
            for metric in METRICS:
                row[metric] = point[metric]
                row[f"{metric}_CI95_low"] = intervals[metric][0]
                row[f"{metric}_CI95_high"] = intervals[metric][1]
            for metric in ["TP", "FP", "FN", "TN"]:
                row[metric] = point[metric]
            summary_rows.append(row)
    performance_ci = pd.DataFrame(summary_rows)
    bootstrap_tsv = (
        OUTPUT_DIR / "08_GSE327167_three_model_performance_bootstrap_CI.tsv"
    )
    performance_ci.to_csv(bootstrap_tsv, sep="\t", index=False)

    def metric_matrix(metric):
        matrix = np.full(
            (len(EVALUATION_ORDER), len(MODELS)), np.nan, dtype=float
        )
        lower = np.full_like(matrix, np.nan)
        upper = np.full_like(matrix, np.nan)
        for row_index, evaluation_name in enumerate(EVALUATION_ORDER):
            for column_index, model in enumerate(MODELS):
                record = performance_ci.loc[
                    performance_ci["evaluation_set"].eq(evaluation_name)
                    & performance_ci["model"].eq(model)
                ]
                require(
                    len(record) == 1,
                    "GSE327167 performance heatmaps and bootstrap intervals: required condition failed: len(record) == 1.",
                )
                record = record.iloc[0]
                matrix[row_index, column_index] = record[metric]
                lower[row_index, column_index] = record[f"{metric}_CI95_low"]
                upper[row_index, column_index] = record[f"{metric}_CI95_high"]
        return (matrix, lower, upper)

    def favourable_color_matrix(metric, raw_matrix):
        if metric == "Brier":
            return 1.0 - raw_matrix
        return raw_matrix.copy()

    def format_metric_annotation(value, low, high):
        if not np.isfinite(value):
            return "NA"
        if np.isfinite(low) and np.isfinite(high):
            return f"{value:.3f}\n({low:.3f}–{high:.3f})"
        return f"{value:.3f}"

    def contrast_text_color(rgba):
        red, green, blue, _ = rgba
        luminance = 0.2126 * red + 0.7152 * green + 0.0722 * blue
        if luminance < 0.54:
            return "white"
        return "#202020"

    evaluation_axis_labels = []
    for evaluation_name in EVALUATION_ORDER:
        frame = evaluation_sets[evaluation_name]
        n_patients = len(frame)
        n_positive = int(frame["OGM_deep_deletion_label"].sum())
        evaluation_axis_labels.append(
            f"{EVALUATION_SHORT_LABELS[evaluation_name]}\nn={n_patients}; OGM+={n_positive}"
        )
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "axes.linewidth": 0.8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.unicode_minus": False,
        }
    )


#### GSE327167 three-model metrics and bootstrap intervals at 0.75


In [ ]:
@_nb9_registry.register("s76_0")
def _nb9_stage_s76_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, Normalize
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
    )

    OUTPUT_DIR = Path(globals().get(
        "GSE327167_RUN_DIR",
        Path(os.environ.get(
            "MTAP_GSE327167_RUN_DIR",
            str(PROJECT_ROOT / (
                "data_processed/MTAP_prediction_registry/external_validation/"
                "20261103_110248_573136_GSE327167_31patient_three_models"
            )),
        )),
    ))
    PREDICTION_FILE = (
        OUTPUT_DIR
        / "04_GSE327167_three_model_predictions_missing_feature_aware.tsv"
    )
    PREVIOUS_CI_FILE = (
        OUTPUT_DIR / "08_GSE327167_three_model_performance_bootstrap_CI.tsv"
    )
    PLOT_DIR = OUTPUT_DIR / "figures_threshold_075"
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    THRESHOLD = 0.75
    N_BOOTSTRAP = 5000
    RANDOM_SEED = 20261103
    MODELS = [
        "Three-gene L2",
        "44-gene Elastic Net",
        "Bulk DL + input adapter",
    ]
    MODEL_LABELS = [
        "Three-gene\nL2",
        "44-gene\nElastic Net",
        "Bulk DL +\ninput adapter",
    ]
    SCOPES = [
        "Whole_31",
        "Malignant_any_author_tumor",
        "Malignant_ge50_primary",
    ]
    SCOPE_LABELS = {
        "Whole_31": "Whole tissue",
        "Malignant_any_author_tumor": "Malignant cells\n(any count)",
        "Malignant_ge50_primary": "Malignant cells\n(≥50 cells)",
    }
    METRICS = [
        "AUROC",
        "AP",
        "Brier",
        "Balanced_accuracy",
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
    ]
    TITLES = [
        "AUROC ↑",
        "Average precision ↑",
        "Brier score ↓",
        "Balanced accuracy ↑",
        "Sensitivity ↑",
        "Specificity ↑",
        "Positive predictive value ↑",
        "Negative predictive value ↑",
    ]
    PALETTE = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]
    CMAP = LinearSegmentedColormap.from_list(
        "MTAP_five_colors", PALETTE, N=256
    )
    CMAP.set_bad("#E8E8E8")
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.unicode_minus": False,
        }
    )
    if not PREDICTION_FILE.is_file():
        raise FileNotFoundError(PREDICTION_FILE)
    pred = pd.read_csv(PREDICTION_FILE, sep="\t", dtype={"patient_id": str})
    required = [
        "patient_id",
        "input_scope",
        "n_cells",
        "OGM_deep_deletion_label",
        *MODELS,
    ]
    missing = sorted(set(required) - set(pred.columns))
    if missing:
        raise RuntimeError(
            "GSE327167 three-model comparison at threshold 0.75: integrity check failed; condition: input or artifact integrity check."
        )
    for column in ["n_cells", "OGM_deep_deletion_label", *MODELS]:
        pred[column] = pd.to_numeric(pred[column], errors="raise")
    if not pred["OGM_deep_deletion_label"].isin([0, 1]).all():
        raise RuntimeError(
            "GSE327167 three-model comparison at threshold 0.75: integrity check failed; condition: input or artifact integrity check."
        )
    for model in MODELS:
        if (
            not np.isfinite(pred[model]).all()
            or not pred[model].between(0, 1).all()
        ):
            raise RuntimeError(
                "GSE327167 three-model comparison at threshold 0.75: integrity check failed; condition: input or artifact integrity check."
            )
    evaluation_sets = {
        "Whole_31": pred.loc[pred["input_scope"].eq("Whole")].copy(),
        "Malignant_any_author_tumor": pred.loc[
            pred["input_scope"].eq("Malignant") & pred["n_cells"].gt(0)
        ].copy(),
        "Malignant_ge50_primary": pred.loc[
            pred["input_scope"].eq("Malignant") & pred["n_cells"].ge(50)
        ].copy(),
    }
    for scope, frame in evaluation_sets.items():
        if not frame["patient_id"].is_unique:
            raise RuntimeError(
                "GSE327167 three-model comparison at threshold 0.75: integrity check failed; condition: input or artifact integrity check."
            )
        if frame["OGM_deep_deletion_label"].nunique() != 2:
            raise RuntimeError(
                "GSE327167 three-model comparison at threshold 0.75: integrity check failed; condition: input or artifact integrity check."
            )
        evaluation_sets[scope] = frame.sort_values("patient_id").reset_index(
            drop=True
        )
    previous_ci = None
    if PREVIOUS_CI_FILE.is_file():
        previous_ci = pd.read_csv(PREVIOUS_CI_FILE, sep="\t")

    def divide(a, b):
        a, b = np.broadcast_arrays(
            np.asarray(a, dtype=float), np.asarray(b, dtype=float)
        )
        result = np.full(a.shape, np.nan, dtype=float)
        np.divide(a, b, out=result, where=b != 0)
        return result

    def threshold_metrics(y, p):
        """Evaluate one cohort or batched bootstrap arrays along the patient axis."""
        call = p >= THRESHOLD
        positive = y == 1
        negative = y == 0
        tp = np.sum(call & positive, axis=-1)
        fp = np.sum(call & negative, axis=-1)
        fn = np.sum(~call & positive, axis=-1)
        tn = np.sum(~call & negative, axis=-1)
        sensitivity = divide(tp, tp + fn)
        specificity = divide(tn, tn + fp)
        return {
            "Balanced_accuracy": (sensitivity + specificity) / 2,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "PPV": divide(tp, tp + fp),
            "NPV": divide(tn, tn + fn),
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
        }

    def percentile_ci(values):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            return (np.nan, np.nan)
        return tuple(np.percentile(values, [2.5, 97.5]))

    rows = []
    patient_call_tables = []
    for scope_index, scope in enumerate(SCOPES):
        frame = evaluation_sets[scope]
        y = frame["OGM_deep_deletion_label"].to_numpy(dtype=int)
        positive_index = np.flatnonzero(y == 1)
        negative_index = np.flatnonzero(y == 0)
        rng = np.random.default_rng(RANDOM_SEED + scope_index)
        bootstrap_index = np.concatenate(
            [
                rng.choice(
                    positive_index,
                    size=(N_BOOTSTRAP, len(positive_index)),
                    replace=True,
                ),
                rng.choice(
                    negative_index,
                    size=(N_BOOTSTRAP, len(negative_index)),
                    replace=True,
                ),
            ],
            axis=1,
        )
        bootstrap_y = y[bootstrap_index]
        calls = frame[
            ["patient_id", "input_scope", "n_cells", "OGM_deep_deletion_label"]
        ].copy()
        calls["evaluation_set"] = scope
        calls["threshold"] = THRESHOLD
        for model in MODELS:
            p = frame[model].to_numpy(dtype=float)
            point = threshold_metrics(y, p)
            boot = threshold_metrics(bootstrap_y, p[bootstrap_index])
            row = {
                "evaluation_set": scope,
                "model": model,
                "n_patients": len(y),
                "n_positive": int(y.sum()),
                "n_negative": int((y == 0).sum()),
                "threshold": THRESHOLD,
                "AUROC": roc_auc_score(y, p),
                "AP": average_precision_score(y, p),
                "Brier": brier_score_loss(y, p),
            }
            for metric in [
                "Balanced_accuracy",
                "Sensitivity",
                "Specificity",
                "PPV",
                "NPV",
            ]:
                row[metric] = float(point[metric])
                low, high = percentile_ci(boot[metric])
                if not np.isfinite(row[metric]):
                    low, high = (np.nan, np.nan)
                row[f"{metric}_CI95_low"] = low
                row[f"{metric}_CI95_high"] = high
            for count in ["TP", "FP", "FN", "TN"]:
                row[count] = int(point[count])
            previous_record = None
            if previous_ci is not None:
                matched = previous_ci.loc[
                    previous_ci["evaluation_set"].eq(scope)
                    & previous_ci["model"].eq(model)
                ]
                if len(matched) == 1:
                    candidate = matched.iloc[0]
                    same_points = all(
                        (
                            metric in candidate.index
                            and np.isclose(
                                float(candidate[metric]),
                                row[metric],
                                atol=1e-08,
                            )
                            for metric in ["AUROC", "AP", "Brier"]
                        )
                    )
                    if same_points:
                        previous_record = candidate
            for metric in ["AUROC", "AP", "Brier"]:
                low_key = f"{metric}_CI95_low"
                high_key = f"{metric}_CI95_high"
                if (
                    previous_record is not None
                    and low_key in previous_record.index
                    and (high_key in previous_record.index)
                ):
                    low = previous_record[low_key]
                    high = previous_record[high_key]
                elif metric == "Brier":
                    low, high = percentile_ci(
                        np.mean(
                            (p[bootstrap_index] - bootstrap_y) ** 2, axis=1
                        )
                    )
                else:
                    function = (
                        roc_auc_score
                        if metric == "AUROC"
                        else average_precision_score
                    )
                    values = [
                        function(bootstrap_y[i], p[bootstrap_index[i]])
                        for i in range(N_BOOTSTRAP)
                    ]
                    low, high = percentile_ci(values)
                row[low_key] = low
                row[high_key] = high
            rows.append(row)
            calls[f"{model}_probability"] = p
            calls[f"{model}_call_075"] = (p >= THRESHOLD).astype(int)
        patient_call_tables.append(calls)
    result = pd.DataFrame(rows)
    result_file = (
        OUTPUT_DIR / "09_GSE327167_performance_threshold_075_with_CI.tsv"
    )
    result.to_csv(result_file, sep="\t", index=False)
    pd.concat(patient_call_tables, ignore_index=True).to_csv(
        OUTPUT_DIR / "10_GSE327167_patient_predictions_threshold_075.tsv",
        sep="\t",
        index=False,
    )
    lookup = result.set_index(["evaluation_set", "model"])
    row_labels = []
    for scope in SCOPES:
        frame = evaluation_sets[scope]
        row_labels.append(
            f"{SCOPE_LABELS[scope]}\nn={len(frame)}; OGM+={int(frame['OGM_deep_deletion_label'].sum())}"
        )

    def text_color(value, norm, cmap):
        r, g, b, _ = cmap(norm(value))
        luminance = 0.2126 * r + 0.7152 * g + 0.0722 * b
        return "white" if luminance < 0.54 else "#202020"

    def style_matrix_axis(axis):
        axis.set_xticks(range(len(MODELS)))
        axis.set_xticklabels(MODEL_LABELS, fontsize=8.5)
        axis.set_yticks(range(len(SCOPES)))
        axis.set_xticks(np.arange(-0.5, len(MODELS), 1), minor=True)
        axis.set_yticks(np.arange(-0.5, len(SCOPES), 1), minor=True)
        axis.grid(which="minor", color="white", linewidth=1.5)
        axis.tick_params(which="both", length=0, pad=6)
        for spine in axis.spines.values():
            spine.set_color("#777777")
            spine.set_linewidth(0.7)


### Development evaluation and sensitivity analyses


#### Baseline bootstrap intervals and calibration assessment


In [ ]:
@_nb9_registry.register("s06_0")
def _nb9_stage_s06_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    PREVIOUS_DIR = (
        PROJECT / "data_processed/MTAP_RNA_prediction/03_baseline_models"
    )
    PREDICTION_FILE = PREVIOUS_DIR / "development_OOF_predictions.tsv"
    OUTPUT_DIR = (
        PROJECT / "data_processed/MTAP_RNA_prediction/04_development_review"
    )
    FIGURE_DIR = OUTPUT_DIR / "figures"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    RANDOM_SEED = 20261016
    N_BOOTSTRAPS = 1000
    MAIN_MODEL = "Three_gene"
    COMPARATOR_MODELS = ["MTAP_only", "MTAP_CDKN2A"]
    REVIEW_MODELS = ["MTAP_only", "MTAP_CDKN2A", "Three_gene"]
    MIN_POSITIVE_FOR_STABLE_DISPLAY = 10
    MIN_NEGATIVE_FOR_STABLE_DISPLAY = 20
    DESCRIPTIVE_THRESHOLD = 0.5
    COLORS = {
        "MTAP_only": "#33539E",
        "MTAP_CDKN2A": "#7FACD6",
        "Three_gene": "#A5678E",
    }
    if not PREDICTION_FILE.is_file():
        raise FileNotFoundError(
            "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
        )
    data = pd.read_csv(
        PREDICTION_FILE,
        sep="\t",
        dtype={"patient_id": "string", "acronym": "string"},
    )
    required = {
        "patient_id",
        "acronym",
        "label_MTAP_deep",
        "CV_fold",
        *[f"probability_{name}" for name in REVIEW_MODELS],
    }
    missing = required.difference(data.columns)
    if missing:
        raise ValueError(
            "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
        )
    if data["patient_id"].isna().any():
        raise ValueError(
            "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
        )
    if data["patient_id"].duplicated().any():
        raise ValueError(
            "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
        )
    if data["acronym"].isna().any():
        raise ValueError(
            "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
        )
    if not data["label_MTAP_deep"].isin([0, 1]).all():
        raise ValueError(
            "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
        )
    for name in REVIEW_MODELS:
        column = f"probability_{name}"
        data[column] = pd.to_numeric(data[column], errors="coerce")
        values = data[column].to_numpy(dtype=float)
        if (
            not np.isfinite(values).all()
            or (values < 0).any()
            or (values > 1).any()
        ):
            raise ValueError(
                "Baseline bootstrap intervals and calibration assessment: integrity check failed; condition: input or artifact integrity check."
            )
    data = data.sort_values("patient_id").reset_index(drop=True)
    y = data["label_MTAP_deep"].astype(int).to_numpy()

    def calculate_metrics(labels, probabilities):
        labels = np.asarray(labels, dtype=int)
        probabilities = np.asarray(probabilities, dtype=float)
        has_both_classes = np.unique(labels).size == 2
        return {
            "AUROC": (
                roc_auc_score(labels, probabilities)
                if has_both_classes
                else np.nan
            ),
            "AUPRC_AP": (
                average_precision_score(labels, probabilities)
                if has_both_classes
                else np.nan
            ),
            "Brier_score": brier_score_loss(labels, probabilities),
        }

    rng = np.random.default_rng(RANDOM_SEED)
    strata_indices = [
        subset.index.to_numpy()
        for _, subset in data.groupby(
            ["acronym", "label_MTAP_deep"], observed=True, sort=True
        )
    ]
    probability_arrays = {
        name: data[f"probability_{name}"].to_numpy(dtype=float)
        for name in REVIEW_MODELS
    }
    point_metrics = {
        name: calculate_metrics(y, probability)
        for name, probability in probability_arrays.items()
    }
    bootstrap_metrics = {
        name: {metric: [] for metric in ["AUROC", "AUPRC_AP", "Brier_score"]}
        for name in REVIEW_MODELS
    }
    for iteration in range(N_BOOTSTRAPS):
        sampled_indices = np.concatenate(
            [
                rng.choice(indices, size=len(indices), replace=True)
                for indices in strata_indices
            ]
        )
        sampled_y = y[sampled_indices]
        for name in REVIEW_MODELS:
            result = calculate_metrics(
                sampled_y, probability_arrays[name][sampled_indices]
            )
            for metric, value in result.items():
                bootstrap_metrics[name][metric].append(value)
        if (iteration + 1) % 250 == 0:
            pass
    metric_records = []
    difference_records = []
    for name in REVIEW_MODELS:
        for metric in ["AUROC", "AUPRC_AP", "Brier_score"]:
            values = np.asarray(bootstrap_metrics[name][metric], dtype=float)
            low, high = np.quantile(values, [0.025, 0.975])
            metric_records.append(
                {
                    "model": name,
                    "metric": metric,
                    "estimate": point_metrics[name][metric],
                    "CI95_low": low,
                    "CI95_high": high,
                    "n_bootstraps": N_BOOTSTRAPS,
                    "interval_scope": "Conditional_on_fixed_OOF_predictions",
                }
            )
    for comparator in COMPARATOR_MODELS:
        for metric in ["AUROC", "AUPRC_AP", "Brier_score"]:
            main_values = np.asarray(
                bootstrap_metrics[MAIN_MODEL][metric], dtype=float
            )
            comparator_values = np.asarray(
                bootstrap_metrics[comparator][metric], dtype=float
            )
            differences = main_values - comparator_values
            low, high = np.quantile(differences, [0.025, 0.975])
            point_difference = (
                point_metrics[MAIN_MODEL][metric]
                - point_metrics[comparator][metric]
            )
            difference_records.append(
                {
                    "main_model": MAIN_MODEL,
                    "comparator": comparator,
                    "metric": metric,
                    "difference_main_minus_comparator": point_difference,
                    "CI95_low": low,
                    "CI95_high": high,
                    "favorable_direction": (
                        "negative" if metric == "Brier_score" else "positive"
                    ),
                    "CI_excludes_zero": bool(low > 0 or high < 0),
                    "n_bootstraps": N_BOOTSTRAPS,
                }
            )
    metric_table = pd.DataFrame(metric_records)
    difference_table = pd.DataFrame(difference_records)
    metric_table.to_csv(
        OUTPUT_DIR / "OOF_metrics_bootstrap_CI.tsv", sep="\t", index=False
    )
    difference_table.to_csv(
        OUTPUT_DIR / "paired_model_differences_bootstrap.tsv",
        sep="\t",
        index=False,
    )
    cancer_records = []
    for cancer, subset in data.groupby("acronym", observed=True):
        labels = subset["label_MTAP_deep"].to_numpy(dtype=int)
        n_positive = int(labels.sum())
        n_negative = int((labels == 0).sum())
        if n_positive == 0 or n_negative == 0:
            support_status = "Single_class_no_AUROC"
        elif (
            n_positive < MIN_POSITIVE_FOR_STABLE_DISPLAY
            or n_negative < MIN_NEGATIVE_FOR_STABLE_DISPLAY
        ):
            support_status = "Low_support"
        else:
            support_status = "Adequate_for_descriptive_display"
        for name in REVIEW_MODELS:
            probabilities = subset[f"probability_{name}"].to_numpy(dtype=float)
            result = calculate_metrics(labels, probabilities)
            predicted_positive = probabilities >= DESCRIPTIVE_THRESHOLD
            tp = int((predicted_positive & (labels == 1)).sum())
            fp = int((predicted_positive & (labels == 0)).sum())
            cancer_records.append(
                {
                    "acronym": cancer,
                    "model": name,
                    "n_patients": len(subset),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "prevalence": labels.mean(),
                    "support_status": support_status,
                    **result,
                    "sensitivity_at_05": (
                        tp / n_positive if n_positive > 0 else np.nan
                    ),
                    "false_positive_rate_at_05": (
                        fp / n_negative if n_negative > 0 else np.nan
                    ),
                }
            )
    cancer_table = pd.DataFrame(cancer_records)
    cancer_table.to_csv(
        OUTPUT_DIR / "OOF_performance_within_cancer.tsv", sep="\t", index=False
    )
    cancer_comparison = cancer_table.pivot(
        index="acronym", columns="model", values=["AUROC", "AUPRC_AP"]
    )
    cancer_comparison.columns = [
        f"{metric}_{model}" for metric, model in cancer_comparison.columns
    ]
    cancer_comparison = cancer_comparison.reset_index()
    for metric in ["AUROC", "AUPRC_AP"]:
        cancer_comparison[
            f"{metric}_difference_Three_gene_minus_MTAP_only"
        ] = (
            cancer_comparison[f"{metric}_Three_gene"]
            - cancer_comparison[f"{metric}_MTAP_only"]
        )
    cancer_comparison.to_csv(
        OUTPUT_DIR / "within_cancer_model_comparison.tsv",
        sep="\t",
        index=False,
    )
    probability_bins = [0.0, 0.01, 0.025, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
    calibration_records = []
    for name in REVIEW_MODELS:
        temporary = pd.DataFrame(
            {"label": y, "probability": probability_arrays[name]}
        )
        temporary["probability_bin"] = pd.cut(
            temporary["probability"],
            bins=probability_bins,
            include_lowest=True,
        )
        for interval, subset in temporary.groupby(
            "probability_bin", observed=True
        ):
            calibration_records.append(
                {
                    "model": name,
                    "probability_bin": str(interval),
                    "n_patients": len(subset),
                    "n_positive": int(subset["label"].sum()),
                    "mean_predicted_probability": subset["probability"].mean(),
                    "observed_positive_fraction": subset["label"].mean(),
                    "support_status": (
                        "Low_support" if len(subset) < 20 else "Descriptive"
                    ),
                }
            )
    calibration_table = pd.DataFrame(calibration_records)
    calibration_table.to_csv(
        OUTPUT_DIR / "OOF_probability_calibration_bins.tsv",
        sep="\t",
        index=False,
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


#### Regional Elastic Net model-comparison summaries


In [ ]:
@_nb9_registry.register("s15_0")
def _nb9_stage_s15_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from sklearn.metrics import roc_auc_score, average_precision_score

    PROJECT = Path(str(PROJECT_ROOT))
    PRED_DIR = PROJECT / "data_processed/MTAP_RNA_prediction"
    MODEL_DIR = PRED_DIR / "09_9p21_region_model/signedlog_v2"
    OOF_FILE = MODEL_DIR / "development_OOF_predictions.tsv"
    BASELINE_METADATA_FILE = (
        PRED_DIR / "03_baseline_models/cohort_with_three_gene_expression.tsv"
    )
    RAW_CACHE = (
        PRED_DIR / "09_9p21_region_model/development_region_raw_expression.tsv"
    )
    OUT = MODEL_DIR / "development_review"
    OUT.mkdir(parents=True, exist_ok=True)
    LABEL = "label_MTAP_deep"
    MODELS = {
        "Three_gene_L2": "probability_Three_gene_L2",
        "Region_ElasticNet": "probability_Region_ElasticNet",
    }
    SEED = 20261016
    N_BOOTSTRAPS = 2000
    for path in [OOF_FILE, BASELINE_METADATA_FILE, RAW_CACHE]:
        if not path.is_file():
            raise FileNotFoundError(
                "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
            )
    data = pd.read_csv(OOF_FILE, sep="\t", dtype={"patient_id": str})
    metadata = pd.read_csv(
        BASELINE_METADATA_FILE, sep="\t", dtype={"patient_id": str}
    )
    required_oof = {
        "patient_id",
        "acronym",
        LABEL,
        "CV_fold",
        *MODELS.values(),
    }
    required_metadata = {
        "patient_id",
        "dataset_split",
        "MTAP_GISTIC_call",
        "CDKN2A_GISTIC_call",
        "CDKN2B_GISTIC_call",
    }
    if not required_oof.issubset(data.columns):
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if not required_metadata.issubset(metadata.columns):
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if (
        data["patient_id"].duplicated().any()
        or metadata["patient_id"].duplicated().any()
    ):
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    dev_metadata = metadata.loc[
        metadata["dataset_split"].eq("development"), list(required_metadata)
    ].copy()
    del metadata
    if set(data["patient_id"]) != set(dev_metadata["patient_id"]):
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    data = data.merge(
        dev_metadata, on="patient_id", how="left", validate="one_to_one"
    )
    numeric_columns = [
        LABEL,
        "MTAP_GISTIC_call",
        "CDKN2A_GISTIC_call",
        "CDKN2B_GISTIC_call",
        *MODELS.values(),
    ]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="raise")
    if not data[LABEL].isin([0, 1]).all():
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if not data["MTAP_GISTIC_call"].isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        data[LABEL].to_numpy(),
        data["MTAP_GISTIC_call"].eq(-2).astype(int).to_numpy(),
    ):
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    for column in MODELS.values():
        if data[column].isna().any() or not data[column].between(0, 1).all():
            raise ValueError(
                "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
            )
    data = data.sort_values("patient_id").reset_index(drop=True)
    if len(data) != 7431 or int(data[LABEL].sum()) != 558:
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    y = data[LABEL].astype(int).to_numpy()
    strata = [
        np.asarray(indices, dtype=int)
        for indices in data.groupby(
            ["acronym", LABEL], sort=True
        ).indices.values()
    ]
    rng = np.random.default_rng(SEED)
    boot = {name: np.zeros((N_BOOTSTRAPS, 2)) for name in MODELS}
    for iteration in range(N_BOOTSTRAPS):
        sampled = np.concatenate(
            [
                rng.choice(indices, size=len(indices), replace=True)
                for indices in strata
            ]
        )
        sampled_y = y[sampled]
        for name, column in MODELS.items():
            probabilities = data[column].to_numpy()[sampled]
            boot[name][iteration, 0] = roc_auc_score(sampled_y, probabilities)
            boot[name][iteration, 1] = average_precision_score(
                sampled_y, probabilities
            )
        if (iteration + 1) % 500 == 0:
            pass
    metric_names = ["AUROC", "AP"]
    summary_rows = []
    difference_rows = []
    for metric_index, metric in enumerate(metric_names):
        for name, column in MODELS.items():
            if metric == "AUROC":
                estimate = roc_auc_score(y, data[column])
            else:
                estimate = average_precision_score(y, data[column])
            low, high = np.quantile(
                boot[name][:, metric_index], [0.025, 0.975]
            )
            summary_rows.append(
                {
                    "model": name,
                    "metric": metric,
                    "estimate": estimate,
                    "CI95_low": low,
                    "CI95_high": high,
                    "n_bootstraps": N_BOOTSTRAPS,
                    "interval_scope": "Conditional_on_fixed_OOF_predictions",
                }
            )
        difference = (
            boot["Region_ElasticNet"][:, metric_index]
            - boot["Three_gene_L2"][:, metric_index]
        )
        low, high = np.quantile(difference, [0.025, 0.975])
        region_values = data[MODELS["Region_ElasticNet"]]
        core_values = data[MODELS["Three_gene_L2"]]
        metric_function = (
            roc_auc_score if metric == "AUROC" else average_precision_score
        )
        difference_rows.append(
            {
                "metric": metric,
                "difference_region_minus_core": metric_function(
                    y, region_values
                )
                - metric_function(y, core_values),
                "CI95_low": low,
                "CI95_high": high,
                "CI_excludes_zero": bool(low > 0 or high < 0),
                "n_bootstraps": N_BOOTSTRAPS,
                "interval_scope": "Conditional_on_fixed_OOF_predictions",
            }
        )
    summary = pd.DataFrame(summary_rows)
    differences = pd.DataFrame(difference_rows)
    summary.to_csv(
        OUT / "paired_bootstrap_model_metrics.tsv", sep="\t", index=False
    )
    differences.to_csv(
        OUT / "paired_bootstrap_performance_differences.tsv",
        sep="\t",
        index=False,
    )
    cancer_rows = []
    for acronym, subset in data.groupby("acronym"):
        labels = subset[LABEL].astype(int)
        n_positive = int(labels.sum())
        n_negative = int((labels == 0).sum())
        row = {
            "acronym": acronym,
            "n_patients": len(subset),
            "n_positive": n_positive,
            "n_negative": n_negative,
            "adequate_for_macro_summary": n_positive >= 10
            and n_negative >= 10,
        }
        for name, column in MODELS.items():
            row[f"{name}_AUROC"] = (
                roc_auc_score(labels, subset[column])
                if labels.nunique() == 2
                else np.nan
            )
            row[f"{name}_AP"] = (
                average_precision_score(labels, subset[column])
                if labels.nunique() == 2
                else np.nan
            )
        for metric in metric_names:
            row[f"delta_{metric}"] = (
                row[f"Region_ElasticNet_{metric}"]
                - row[f"Three_gene_L2_{metric}"]
            )
        cancer_rows.append(row)
    cancers = pd.DataFrame(cancer_rows)
    cancers.to_csv(
        OUT / "per_cancer_paired_comparison.tsv", sep="\t", index=False
    )
    supported = cancers.loc[cancers["adequate_for_macro_summary"]]
    macro_rows = []
    for name in MODELS:
        macro_rows.append(
            {
                "model": name,
                "n_cancers": len(supported),
                "macro_AUROC": supported[f"{name}_AUROC"].mean(),
                "macro_AP": supported[f"{name}_AP"].mean(),
                "scope": "Descriptive_equal_cancer_weight",
            }
        )
    pd.DataFrame(macro_rows).to_csv(
        OUT / "descriptive_macro_performance.tsv", sep="\t", index=False
    )
    positive = data[LABEL].eq(1)
    negative = data[LABEL].eq(0)
    a_deep = data["CDKN2A_GISTIC_call"].eq(-2)
    b_deep = data["CDKN2B_GISTIC_call"].eq(-2)
    subgroups = {
        "All_MTAP_deep": positive,
        "All_MTAP_non_deep": negative,
        "CDKN2A_deep_MTAP_non_deep": a_deep & negative,
        "CDKN2B_deep_MTAP_non_deep": b_deep & negative,
        "CDKN2A_or_B_deep_MTAP_non_deep": (a_deep | b_deep) & negative,
        "MTAP_deep_without_CDKN2B_deep": positive
        & data["CDKN2B_GISTIC_call"].notna()
        & ~b_deep,
    }
    subgroup_rows = []
    for subgroup, mask in subgroups.items():
        subset = data.loc[mask]
        for name, column in MODELS.items():
            probabilities = subset[column]
            n = len(subset)
            subgroup_rows.append(
                {
                    "subgroup": subgroup,
                    "model": name,
                    "n_patients": n,
                    "n_DNA_positive": int(subset[LABEL].sum()),
                    "mean_probability": probabilities.mean(),
                    "median_probability": probabilities.median(),
                    "n_predicted_positive_at_05": int(
                        probabilities.ge(0.5).sum()
                    ),
                    "fraction_predicted_positive_at_05": (
                        probabilities.ge(0.5).mean() if n else np.nan
                    ),
                    "threshold_scope": "Descriptive_not_final",
                }
            )
    subgroup_results = pd.DataFrame(subgroup_rows)
    subgroup_results.to_csv(
        OUT / "DNA_subgroup_prediction_review.tsv", sep="\t", index=False
    )
    raw = pd.read_csv(
        RAW_CACHE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    if raw.index.duplicated().any() or set(raw.index) != set(
        data["patient_id"]
    ):
        raise ValueError(
            "Regional Elastic Net model-comparison summaries: integrity check failed; condition: input or artifact integrity check."
        )
    raw = raw.reindex(data["patient_id"])
    raw = raw.apply(pd.to_numeric, errors="raise")
    technical = pd.DataFrame(
        {
            "patient_id": data["patient_id"].to_numpy(),
            "acronym": data["acronym"].to_numpy(),
            "n_negative_features": raw.lt(0).sum(axis=1).to_numpy(),
            "n_missing_features": raw.isna().sum(axis=1).to_numpy(),
        }
    )
    technical["has_negative"] = technical["n_negative_features"] > 0
    technical["has_missing"] = technical["n_missing_features"] > 0
    technical_summary = technical.groupby("acronym", as_index=False).agg(
        n_patients=("patient_id", "size"),
        n_patients_with_negative=("has_negative", "sum"),
        n_patients_with_missing=("has_missing", "sum"),
        mean_negative_features=("n_negative_features", "mean"),
        mean_missing_features=("n_missing_features", "mean"),
    )
    technical_summary.to_csv(
        OUT / "technical_coverage_by_cancer.tsv", sep="\t", index=False
    )
    technical.to_csv(
        OUT / "technical_coverage_by_patient.tsv", sep="\t", index=False
    )
    _public_summary(
        globals(), "Regional Elastic Net model-comparison summaries"
    )


### Cell-number robustness


#### GBM L2 cell-number robustness and candidate model


In [ ]:
@_nb9_registry.register("s59_0")
def _nb9_stage_s59_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import hashlib
    import json
    import time
    import joblib
    import numpy as np
    import pandas as pd
    import sklearn
    import matplotlib.pyplot as plt
    from scipy import sparse
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    INPUT = BASE / "05_Tumor_UMI_and_pseudobulk"
    CELL_CACHE = INPUT / "patient_cell_caches"
    EXPRESSION_FILE = INPUT / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    METADATA_FILE = INPUT / "patient_input_audit.tsv"
    FEATURE_FILE = INPUT / "fixed_44_feature_mapping.tsv"
    PREVIOUS_L2 = (
        BASE
        / "06_LOPO_baseline_and_DL_interface"
        / "three_gene_L2_LOPO_predictions.tsv"
    )
    OUT = BASE / "08_L2_cell_number_stability_and_candidate"
    FIG = OUT / "figures"
    MODELS = OUT / "models"
    OUT.mkdir(parents=True, exist_ok=True)
    FIG.mkdir(parents=True, exist_ok=True)
    MODELS.mkdir(parents=True, exist_ok=True)
    COMPLETE = OUT / "analysis_manifest.json"
    if COMPLETE.exists():
        raise FileExistsError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    CELL_NUMBERS = [50, 100, 200, 500, 1000]
    N_REPEATS = 30
    RANDOM_SEED = 20261017
    for path in [EXPRESSION_FILE, METADATA_FILE, FEATURE_FILE, PREVIOUS_L2]:
        if not path.is_file():
            raise FileNotFoundError(path)
    start_time = time.time()
    X = pd.read_csv(
        EXPRESSION_FILE, sep="\t", index_col="patient_id"
    ).sort_index()
    meta = pd.read_csv(
        METADATA_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    previous = pd.read_csv(
        PREVIOUS_L2, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    feature_table = pd.read_csv(
        FEATURE_FILE, sep="\t", dtype={"feature": str, "Ensembl_ID": str}
    ).sort_values("feature_order")
    features = feature_table["feature"].tolist()
    for name, frame in [
        ("Expression", X),
        ("Patient metadata", meta),
        ("Existing L2 predictions", previous),
    ]:
        if not frame.index.is_unique:
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
    if not set(X.index) == set(meta.index) == set(previous.index):
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    meta = meta.reindex(X.index)
    previous = previous.reindex(X.index)
    if X.columns.tolist() != features or len(features) != 44:
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    if not set(CORE).issubset(features):
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.isfinite(X.to_numpy(dtype=float)).all():
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    calls = pd.to_numeric(meta["MTAP_GISTIC_call"], errors="raise")
    if calls.isna().any() or not calls.isin([-2, -1, 0, 1, 2]).all():
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    y = calls.eq(-2).astype(int)
    if not np.array_equal(
        previous["label_MTAP_deep"].to_numpy(), y.to_numpy()
    ):
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    if not np.array_equal(
        previous["MTAP_GISTIC_call"].to_numpy(), calls.to_numpy()
    ):
        raise ValueError(
            "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
        )
    core_positions = [features.index(gene) for gene in CORE]

    def make_model():
        return Pipeline(
            [
                ("scale", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        penalty="l2",
                        C=1.0,
                        solver="lbfgs",
                        class_weight=None,
                        max_iter=5000,
                        random_state=41,
                    ),
                ),
            ]
        )

    def positive_probability(model, values):
        index = list(model.named_steps["model"].classes_).index(1)
        return model.predict_proba(values)[:, index]

    def performance(labels, probabilities):
        labels = np.asarray(labels, dtype=int)
        probabilities = np.asarray(probabilities, dtype=float)
        tn, fp, fn, tp = confusion_matrix(
            labels, probabilities >= 0.5, labels=[0, 1]
        ).ravel()
        return {
            "n_patients": len(labels),
            "n_positive": int(labels.sum()),
            "AUROC": (
                roc_auc_score(labels, probabilities)
                if len(np.unique(labels)) == 2
                else np.nan
            ),
            "AP": (
                average_precision_score(labels, probabilities)
                if len(np.unique(labels)) == 2
                else np.nan
            ),
            "Brier_score": brier_score_loss(labels, probabilities),
            "TP_at_05": int(tp),
            "FP_at_05": int(fp),
            "FN_at_05": int(fn),
            "TN_at_05": int(tn),
            "sensitivity_at_05": tp / (tp + fn) if tp + fn else np.nan,
            "specificity_at_05": tn / (tn + fp) if tn + fp else np.nan,
        }

    sample_predictions = []
    full_predictions = []
    cache_audit = []
    coefficient_rows = []
    for fold, patient in enumerate(X.index, start=1):
        train_ids = X.index[X.index != patient]
        if y.loc[train_ids].nunique() != 2:
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        model = make_model()
        model.fit(X.loc[train_ids, CORE], y.loc[train_ids])
        full_probability = float(
            positive_probability(model, X.loc[[patient], CORE])[0]
        )
        old_probability = float(previous.loc[patient, "probability"])
        if abs(full_probability - old_probability) > 1e-08:
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        full_predictions.append(
            {
                "patient_id": patient,
                "MTAP_GISTIC_call": int(calls.loc[patient]),
                "label_MTAP_deep": int(y.loc[patient]),
                "full_pseudobulk_probability": full_probability,
            }
        )
        for gene, coefficient in zip(
            CORE, model.named_steps["model"].coef_[0]
        ):
            coefficient_rows.append(
                {
                    "held_out_patient": patient,
                    "gene": gene,
                    "standardized_coefficient": float(coefficient),
                }
            )
        joblib.dump(
            {
                "pipeline": model,
                "training_patients": train_ids.tolist(),
                "held_out_patient": patient,
                "feature_names": CORE,
            },
            MODELS / f"L2_held_out_{patient}.joblib",
        )
        count_file = CELL_CACHE / f"{patient}_Tumor_44_UMI.npz"
        cell_file = CELL_CACHE / f"{patient}_Tumor_metadata.tsv.gz"
        for path in [count_file, cell_file]:
            if not path.is_file():
                raise FileNotFoundError(path)
        counts = sparse.load_npz(count_file).tocsr()
        cells = pd.read_csv(
            cell_file,
            sep="\t",
            dtype={"patient_id": str, "cell_id": str, "cell_barcode": str},
        )
        if counts.shape != (len(cells), 44):
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        if not cells["patient_id"].eq(patient).all():
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        if (
            cells["cell_id"].isna().any()
            or cells["cell_id"].duplicated().any()
        ):
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        if (
            not np.isfinite(counts.data).all()
            or (counts.data < 0).any()
            or (not np.equal(counts.data, np.rint(counts.data)).all())
        ):
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        libraries = pd.to_numeric(
            cells["total_gene_expression_UMI"], errors="raise"
        ).to_numpy(dtype=float)
        if (
            not np.isfinite(libraries).all()
            or (libraries <= 0).any()
            or (not np.equal(libraries, np.rint(libraries)).all())
        ):
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        libraries = libraries.astype(np.int64)
        if len(cells) < max(CELL_NUMBERS):
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        total_gene_counts = (
            np.asarray(counts.sum(axis=0)).ravel().astype(np.float64)
        )
        reconstructed = np.log2(
            1.0 + total_gene_counts / libraries.sum() * 1000000.0
        )
        reconstruction_error = float(
            np.max(
                np.abs(
                    reconstructed
                    - X.loc[patient, features].to_numpy(dtype=float)
                )
            )
        )
        if reconstruction_error > 1e-08:
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        cache_audit.append(
            {
                "patient_id": patient,
                "n_tumor_nuclei": len(cells),
                "max_pseudobulk_reconstruction_error": reconstruction_error,
                "full_pseudobulk_probability": full_probability,
            }
        )
        core_counts = counts[:, core_positions].toarray().astype(np.int64)
        rng = np.random.default_rng(
            np.random.SeedSequence([RANDOM_SEED, fold])
        )
        for n_cells in CELL_NUMBERS:
            for repeat in range(1, N_REPEATS + 1):
                selected = rng.choice(len(cells), size=n_cells, replace=False)
                gene_umi = core_counts[selected].sum(axis=0)
                total_umi = int(libraries[selected].sum())
                logcpm = np.log2(
                    1.0 + gene_umi.astype(float) / total_umi * 1000000.0
                )
                values = pd.DataFrame([logcpm], columns=CORE)
                probability = float(positive_probability(model, values)[0])
                sample_predictions.append(
                    {
                        "patient_id": patient,
                        "MTAP_GISTIC_call": int(calls.loc[patient]),
                        "label_MTAP_deep": int(y.loc[patient]),
                        "n_cells": n_cells,
                        "repeat": repeat,
                        "sample_total_UMI": total_umi,
                        "all_three_genes_zero": bool((gene_umi == 0).all()),
                        "probability": probability,
                        "full_pseudobulk_probability": full_probability,
                        "absolute_probability_difference": abs(
                            probability - full_probability
                        ),
                        "correct_at_05": int(probability >= 0.5)
                        == int(y.loc[patient]),
                        "agrees_with_full_at_05": (probability >= 0.5)
                        == (full_probability >= 0.5),
                    }
                )
        pd.DataFrame(sample_predictions).to_csv(
            OUT / "cell_subsampling_predictions.tsv.gz",
            sep="\t",
            index=False,
            compression="gzip",
        )
        del counts, core_counts, cells
    sample = pd.DataFrame(sample_predictions)
    full = pd.DataFrame(full_predictions)
    patient_stability = (
        sample.groupby(
            ["patient_id", "MTAP_GISTIC_call", "label_MTAP_deep", "n_cells"]
        )
        .agg(
            probability_median=("probability", "median"),
            probability_Q1=("probability", lambda v: v.quantile(0.25)),
            probability_Q3=("probability", lambda v: v.quantile(0.75)),
            probability_min=("probability", "min"),
            probability_max=("probability", "max"),
            probability_SD=("probability", "std"),
            mean_absolute_probability_difference=(
                "absolute_probability_difference",
                "mean",
            ),
            fraction_correct_at_05=("correct_at_05", "mean"),
            fraction_agreeing_with_full_at_05=(
                "agrees_with_full_at_05",
                "mean",
            ),
            fraction_all_three_zero=("all_three_genes_zero", "mean"),
        )
        .reset_index()
    )
    patient_stability.to_csv(
        OUT / "patient_cell_number_stability.tsv", sep="\t", index=False
    )
    repeat_metrics = []
    for (n_cells, repeat), group in sample.groupby(["n_cells", "repeat"]):
        if len(group) != len(X) or group["patient_id"].nunique() != len(X):
            raise ValueError(
                "GBM L2 cell-number robustness and candidate model: integrity check failed; condition: input or artifact integrity check."
            )
        row = {"n_cells": n_cells, "repeat": repeat}
        row.update(performance(group["label_MTAP_deep"], group["probability"]))
        repeat_metrics.append(row)
    repeat_metrics = pd.DataFrame(repeat_metrics)
    repeat_metrics.to_csv(
        OUT / "performance_by_cell_number_and_repeat.tsv",
        sep="\t",
        index=False,
    )
    summary = (
        repeat_metrics.groupby("n_cells")
        .agg(
            AUROC_mean=("AUROC", "mean"),
            AUROC_min=("AUROC", "min"),
            AUROC_max=("AUROC", "max"),
            AP_mean=("AP", "mean"),
            AP_min=("AP", "min"),
            AP_max=("AP", "max"),
            Brier_mean=("Brier_score", "mean"),
            sensitivity_mean=("sensitivity_at_05", "mean"),
            specificity_mean=("specificity_at_05", "mean"),
        )
        .reset_index()
    )
    summary.to_csv(
        OUT / "cell_number_stability_summary.tsv", sep="\t", index=False
    )
    full.to_csv(
        OUT / "reproduced_full_pseudobulk_LOPO_predictions.tsv",
        sep="\t",
        index=False,
    )
    pd.DataFrame(cache_audit).to_csv(
        OUT / "cell_cache_consistency_audit.tsv", sep="\t", index=False
    )
    coefficients = pd.DataFrame(coefficient_rows)
    coefficients.to_csv(
        OUT / "LOPO_standardized_coefficients.tsv", sep="\t", index=False
    )
    candidate = make_model()
    candidate.fit(X.loc[:, CORE], y)
    bundle = {
        "pipeline": candidate,
        "model_status": "GBM_snRNA_patient_pseudobulk_candidate",
        "feature_names": CORE,
        "expression_unit": "log2(CPM+1)",
        "normalization": "First sum raw gene counts across selected Tumor nuclei; divide each gene sum by total Gene Expression UMI across those same nuclei; multiply by 1e6; log2(x+1).",
        "prediction_unit": "Patient_or_tumor_sample_pseudobulk",
        "target": "Patient_MTAP_GISTIC_minus2_vs_non_minus2",
        "training_patients": X.index.tolist(),
        "training_GISTIC_calls": {
            patient: int(calls.loc[patient]) for patient in X.index
        },
        "training_positive_fraction": float(y.mean()),
        "annotation_status": "Preliminary_Tumor_annotation",
        "single_cell_genotype_validated": False,
        "pan_cancer_transfer_validated": False,
        "deployment_threshold_locked": False,
        "parameters": {
            "C": 1.0,
            "penalty": "l2",
            "solver": "lbfgs",
            "class_weight": None,
        },
        "sklearn_version": sklearn.__version__,
    }
    candidate_path = MODELS / "GBM_snRNA_three_gene_L2_candidate.joblib"
    joblib.dump(bundle, candidate_path)
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


### GBM whole-tissue versus malignant-cell pseudobulk


#### GBM whole-tissue versus malignant-cell L2 comparison


In [ ]:
@_nb9_registry.register("s66_0")
def _nb9_stage_s66_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from datetime import datetime
    import gc
    import gzip
    import hashlib
    import json
    import re
    import zipfile
    import h5py
    import joblib
    import numpy as np
    import pandas as pd
    from scipy import sparse
    from scipy.io import mmread
    from scipy.stats import spearmanr
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
    )
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    PROJECT = Path(str(PROJECT_ROOT))
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    GBM = PROJECT / "data_processed/CPTAC_GBM_joint_analysis"
    EXPR_FILE = GBM / "CPTAC_GBM_16patients_SCT_merged.h5ad"
    ANNOT_FILE = (
        GBM
        / "scVI_exploratory/run_20261008_151430_065424"
        / "annotation/scVI_manual_annotated.h5ad"
    )
    MEX_FILE = (
        BASE
        / "04_UMI_download_and_barcode_matching"
        / "UMI_MEX_download_and_matching.tsv"
    )
    PATIENT_FILE = (
        BASE / "01_source_resolution/patient_DNA_and_cell_coverage.tsv"
    )
    TUMOR_FILE = BASE / "01_source_resolution/Tumor_nucleus_metadata.tsv.gz"
    OLD_EXPRESSION_FILE = (
        BASE
        / "05_Tumor_UMI_and_pseudobulk"
        / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    )
    MODEL_DIR = BASE / "08_L2_cell_number_stability_and_candidate/models"
    OLD_PREDICTION_FILE = (
        BASE
        / "08_L2_cell_number_stability_and_candidate"
        / "reproduced_full_pseudobulk_LOPO_predictions.tsv"
    )
    if not OLD_PREDICTION_FILE.is_file():
        OLD_PREDICTION_FILE = (
            BASE
            / "06_LOPO_baseline_and_DL_interface"
            / "three_gene_L2_LOPO_predictions.tsv"
        )
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    OUT = BASE / f"10_whole_tissue_L2_paired_test_{RUN_ID}"
    FIG = OUT / "figures"
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    ENSEMBL = {
        "MTAP": "ENSG00000099810",
        "CDKN2A": "ENSG00000147889",
        "CDKN2B": "ENSG00000147883",
    }
    THRESHOLD = 0.5
    SEED = 20261021
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    OUT.mkdir(parents=True, exist_ok=False)
    FIG.mkdir()

    def require(condition, message):
        if not bool(condition):
            raise RuntimeError(message)

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    def decode(value):
        return value.decode("utf-8") if isinstance(value, bytes) else value

    def read_vector(node):
        if isinstance(node, h5py.Dataset):
            values = node[:]
            if values.dtype.kind in {"O", "S", "U"}:
                return np.asarray([decode(x) for x in values], dtype=object)
            return values
        if "codes" in node and "categories" in node:
            codes = node["codes"][:]
            categories = read_vector(node["categories"])
            values = np.full(len(codes), pd.NA, dtype=object)
            valid = codes >= 0
            values[valid] = categories[codes[valid]]
            return values
        if "values" in node and "mask" in node:
            values = np.asarray(read_vector(node["values"]), dtype=object)
            values[node["mask"][:].astype(bool)] = pd.NA
            return values
        raise RuntimeError(
            "GBM whole-tissue versus malignant-cell L2 comparison: integrity check failed; condition: input or artifact integrity check."
        )

    def read_obs(path):
        with h5py.File(path, "r") as f:
            group = f["obs"]
            index_name = decode(group.attrs["_index"])
            index = pd.Index(
                read_vector(group[index_name]).astype(str), name="cell_id"
            )
            frame = pd.DataFrame(index=index)
            for name in group:
                if name != index_name:
                    frame[name] = read_vector(group[name])
        require(
            frame.index.is_unique,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: frame.index.is_unique.",
        )
        return frame

    def open_text(path):
        path = Path(path)
        if path.name.endswith(".gz"):
            return gzip.open(path, "rt", encoding="utf-8")
        return open(path, "r", encoding="utf-8")

    def positive_probability(pipeline, frame):
        classes = list(pipeline.classes_)
        require(
            1 in classes,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: 1 in classes.",
        )
        p = pipeline.predict_proba(frame)[:, classes.index(1)]
        require(
            np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all(),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all().",
        )
        return p

    def save_figure(fig, stem):
        fig.savefig(FIG / f"{stem}.pdf", dpi=400, bbox_inches="tight")
        fig.savefig(FIG / f"{stem}.png", dpi=250, bbox_inches="tight")
        plt.close(fig)

    for path in [
        EXPR_FILE,
        ANNOT_FILE,
        MEX_FILE,
        PATIENT_FILE,
        TUMOR_FILE,
        OLD_EXPRESSION_FILE,
        OLD_PREDICTION_FILE,
    ]:
        require(
            path.is_file(),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: path.is_file().",
        )
    obs = read_obs(EXPR_FILE)
    annot = read_obs(ANNOT_FILE)
    require(
        "patient_id" in obs,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: 'patient_id' in obs.",
    )
    require(
        "cell_type_manual" in annot,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: 'cell_type_manual' in annot.",
    )
    require(
        len(obs) == len(annot) and obs.index.isin(annot.index).all(),
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: len(obs) == len(annot) and obs.index.isin(annot.index).all().",
    )
    annot = annot.reindex(obs.index)
    obs["patient_id"] = obs["patient_id"].astype(str)
    obs["cell_type_manual"] = (
        annot["cell_type_manual"].fillna("UNANNOTATED").astype(str)
    )
    if "patient_id" in annot:
        require(
            np.array_equal(
                obs["patient_id"].to_numpy(),
                annot["patient_id"].astype(str).to_numpy(),
            ),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: np.array_equal(obs['patient_id'].to_numpy(), annot['patient_id'].astype(str).to_numpy()).",
        )
    patients = pd.read_csv(
        PATIENT_FILE, sep="\t", dtype={"patient_id": str}
    ).set_index("patient_id")
    tumor = pd.read_csv(TUMOR_FILE, sep="\t", dtype=str, keep_default_na=False)
    mex = pd.read_csv(MEX_FILE, sep="\t", dtype=str, keep_default_na=False)
    old_x = pd.read_csv(OLD_EXPRESSION_FILE, sep="\t", index_col=0)
    old_x.index = old_x.index.astype(str)
    old_p = pd.read_csv(OLD_PREDICTION_FILE, sep="\t", index_col=0)
    old_p.index = old_p.index.astype(str)
    require(
        len(patients) == 16,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: len(patients) == 16.",
    )
    require(
        patients.index.is_unique,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: patients.index.is_unique.",
    )
    require(
        not mex["patient_id"].duplicated().any(),
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: not mex['patient_id'].duplicated().any().",
    )
    expected = set(patients.index)
    for name, identifiers in [
        ("All cells", set(obs["patient_id"])),
        ("Tumor-cell list", set(tumor["patient_id"])),
        ("MEX manifest", set(mex["patient_id"])),
        ("Existing expression", set(old_x.index)),
        ("Existing predictions", set(old_p.index)),
    ]:
        require(
            identifiers == expected,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: identifiers == expected.",
        )
    require(
        {"cell_id", "cell_barcode"}.issubset(tumor.columns),
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: {'cell_id', 'cell_barcode'}.issubset(tumor.columns).",
    )
    require(
        not tumor.duplicated(["patient_id", "cell_barcode"]).any(),
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: not tumor.duplicated(['patient_id', 'cell_barcode']).any().",
    )
    require(
        tumor["cell_id"].is_unique,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: tumor['cell_id'].is_unique.",
    )
    tumor_ids = set(tumor["cell_id"])
    current_tumor_ids = set(obs.index[obs["cell_type_manual"].eq("Tumor")])
    require(
        tumor_ids == current_tumor_ids,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: tumor_ids == current_tumor_ids.",
    )
    calls = pd.to_numeric(patients["MTAP_GISTIC_call"], errors="raise").astype(
        int
    )
    require(
        calls.isin([-2, -1, 0, 1, 2]).all(),
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: calls.isin([-2, -1, 0, 1, 2]).all().",
    )
    y = calls.eq(-2).astype(int)
    immune_names = {
        "tam",
        "tams",
        "monocytes",
        "monocyte",
        "t_cells",
        "t_cell",
        "b_plasma",
        "b_cells",
        "b_cell",
        "plasma",
        "nk",
        "nk_cells",
        "myeloid",
        "macrophages",
        "macrophage",
        "dendritic",
        "dendritic_cells",
        "mast",
    }
    normalised_labels = (
        obs["cell_type_manual"]
        .str.strip()
        .str.lower()
        .str.replace("[\\s/-]+", "_", regex=True)
    )
    obs["is_immune_for_descriptive_analysis"] = normalised_labels.isin(
        immune_names
    )
    obs.groupby(
        ["patient_id", "cell_type_manual"], observed=True
    ).size().rename("n_cells").reset_index().to_csv(
        OUT / "all_nuclei_annotation_counts.tsv", sep="\t", index=False
    )

    def resolve_barcodes(cell_meta, mex_barcodes, tumor_patient):
        candidates = {
            "cell_id_exact": pd.Series(
                cell_meta.index.to_numpy(), index=cell_meta.index, dtype=str
            )
        }
        for col in [
            "cell_barcode",
            "barcode",
            "barcodes",
            "original_barcode",
            "source_cell_id",
        ]:
            if col in cell_meta:
                candidates[f"obs:{col}"] = (
                    cell_meta[col].fillna("").astype(str)
                )
        barcode_pattern = re.compile(
            "(?<![ACGTN])([ACGTN]{14,20}-\\d+)(?!\\d)"
        )
        extracted = []
        for cell_id in cell_meta.index:
            hits = barcode_pattern.findall(str(cell_id))
            extracted.append(hits[0] if len(hits) == 1 else "")
        candidates["unique_10x_barcode_in_cell_id"] = pd.Series(
            extracted, index=cell_meta.index, dtype=str
        )
        known = tumor_patient.set_index("cell_id")["cell_barcode"]
        valid = []
        for method, values in candidates.items():
            if (
                values.eq("").any()
                or values.duplicated().any()
                or (not values.isin(mex_barcodes).all())
            ):
                continue
            if not known.index.isin(values.index).all():
                continue
            if not np.array_equal(
                values.reindex(known.index).to_numpy(), known.to_numpy()
            ):
                continue
            valid.append((method, values))
        require(
            len(valid) > 0,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: len(valid) > 0.",
        )
        first_method, first_values = valid[0]
        for _, values in valid[1:]:
            require(
                np.array_equal(values.to_numpy(), first_values.to_numpy()),
                "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: np.array_equal(values.to_numpy(), first_values.to_numpy()).",
            )
        return (first_values, ";".join((method for method, _ in valid)))

    whole_expression = {}
    tumor_expression = {}
    raw_count_rows = []
    prediction_rows = []
    mapping_rows = []
    model_rows = []
    old_probability_column = next(
        (
            col
            for col in ["full_pseudobulk_probability", "probability"]
            if col in old_p
        ),
        None,
    )
    require(
        old_probability_column is not None,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: old_probability_column is not None.",
    )
    for record in mex.sort_values("patient_id").itertuples(index=False):
        patient = record.patient_id
        cell_meta = obs.loc[obs["patient_id"].eq(patient)].copy()
        tumor_patient = tumor.loc[tumor["patient_id"].eq(patient)].copy()
        expected_n = int(patients.loc[patient, "n_all_nuclei"])
        require(
            len(cell_meta) == expected_n,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: len(cell_meta) == expected_n.",
        )
        model_path = MODEL_DIR / f"L2_held_out_{patient}.joblib"
        require(
            model_path.is_file(),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: model_path.is_file().",
        )
        bundle = joblib.load(model_path)
        require(
            bundle.get("held_out_patient") == patient,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: bundle.get('held_out_patient') == patient.",
        )
        require(
            list(bundle.get("feature_names", [])) == CORE,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: list(bundle.get('feature_names', [])) == CORE.",
        )
        training_patients = set(bundle.get("training_patients", []))
        require(
            training_patients == expected - {patient},
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: training_patients == expected - {patient}.",
        )
        pipeline = bundle["pipeline"]
        model_rows.append(
            {
                "held_out_patient": patient,
                "model_file": str(model_path),
                "model_sha256": sha256_file(model_path),
                "n_training_patients": len(training_patients),
            }
        )
        matrix_path = Path(record.matrix_file)
        features_path = Path(record.features_file)
        barcodes_path = Path(record.barcodes_file)
        for path in [matrix_path, features_path, barcodes_path]:
            require(
                path.is_file(),
                "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: path.is_file().",
            )
        features = pd.read_csv(
            features_path,
            sep="\t",
            header=None,
            dtype=str,
            keep_default_na=False,
        )
        require(
            features.shape[1] in {2, 3},
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: features.shape[1] in {2, 3}.",
        )
        with open_text(barcodes_path) as handle:
            barcodes = pd.Index(
                [line.rstrip("\r\n").split("\t")[0] for line in handle]
            )
        require(
            barcodes.is_unique,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: barcodes.is_unique.",
        )
        gene_ids = features.iloc[:, 0].str.replace("\\.\\d+$", "", regex=True)
        if features.shape[1] == 3:
            gene_mask = features.iloc[:, 2].eq("Gene Expression").to_numpy()
        else:
            require(
                gene_ids.str.fullmatch("ENSG\\d+").all(),
                "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: gene_ids.str.fullmatch('ENSG\\\\d+').all().",
            )
            gene_mask = np.ones(len(features), dtype=bool)
        gene_index = pd.Index(gene_ids.loc[gene_mask])
        require(
            gene_mask.any(),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: gene_mask.any().",
        )
        require(
            gene_index.is_unique,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: gene_index.is_unique.",
        )
        core_positions = gene_index.get_indexer([ENSEMBL[g] for g in CORE])
        require(
            (core_positions >= 0).all(),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: (core_positions >= 0).all().",
        )
        cell_barcodes, barcode_method = resolve_barcodes(
            cell_meta, barcodes, tumor_patient
        )
        positions = barcodes.get_indexer(cell_barcodes)
        mapping_rows.append(
            {
                "patient_id": patient,
                "n_QC_nuclei": len(cell_meta),
                "n_MEX_barcodes": len(barcodes),
                "n_MEX_barcodes_not_used": len(barcodes) - len(cell_meta),
                "barcode_mapping_method": barcode_method,
                "all_QC_nuclei_matched": bool((positions >= 0).all()),
                "Tumor_crosswalk_reproduced": True,
            }
        )
        with open(matrix_path, "rb") as raw_handle:
            if matrix_path.name.endswith(".gz"):
                with gzip.GzipFile(fileobj=raw_handle) as handle:
                    raw = mmread(handle)
            else:
                raw = mmread(raw_handle)
        require(
            sparse.issparse(raw),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: sparse.issparse(raw).",
        )
        require(
            raw.shape == (len(features), len(barcodes)),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: raw.shape == (len(features), len(barcodes)).",
        )
        for start in range(0, len(raw.data), 1000000):
            values = raw.data[start : start + 1000000]
            require(
                np.isfinite(values).all()
                and (values >= 0).all()
                and np.equal(values, np.rint(values)).all(),
                "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: np.isfinite(values).all() and (values >= 0).all() and np.equal(values, np.rint(values)).all().",
            )
        raw = raw.astype(np.int64, copy=False).tocsc()
        counts = raw[gene_mask, :][:, positions].tocsc()
        del raw
        per_cell_total = (
            np.asarray(counts.sum(axis=0)).ravel().astype(np.int64)
        )
        require(
            (per_cell_total > 0).all(),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: (per_cell_total > 0).all().",
        )
        core_counts = counts[core_positions, :].toarray()
        tumor_mask = cell_meta.index.isin(tumor_ids)
        require(
            int(tumor_mask.sum()) == len(tumor_patient),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: int(tumor_mask.sum()) == len(tumor_patient).",
        )
        scope_inputs = {}
        scope_totals = {}
        for scope, mask in [
            ("Tumor", tumor_mask),
            ("Whole_tissue", np.ones(len(cell_meta), dtype=bool)),
        ]:
            gene_sum = core_counts[:, mask].sum(axis=1, dtype=np.int64)
            total_umi = int(per_cell_total[mask].sum())
            require(
                total_umi > 0,
                "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: total_umi > 0.",
            )
            logcpm = np.log2(
                1.0 + gene_sum.astype(float) / total_umi * 1000000.0
            )
            scope_inputs[scope] = pd.DataFrame(
                [logcpm], index=[patient], columns=CORE
            )
            scope_totals[scope] = total_umi
            for gene, value in zip(CORE, gene_sum):
                raw_count_rows.append(
                    {
                        "patient_id": patient,
                        "scope": scope,
                        "gene": gene,
                        "raw_UMI": int(value),
                        "total_gene_expression_UMI": total_umi,
                        "n_nuclei": int(mask.sum()),
                    }
                )
        reconstructed = scope_inputs["Tumor"].iloc[0].to_numpy()
        previous = old_x.loc[patient, CORE].to_numpy(dtype=float)
        require(
            np.allclose(reconstructed, previous, rtol=0, atol=1e-08),
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: np.allclose(reconstructed, previous, rtol=0, atol=1e-08).",
        )
        p_tumor = float(
            positive_probability(pipeline, scope_inputs["Tumor"])[0]
        )
        p_whole = float(
            positive_probability(pipeline, scope_inputs["Whole_tissue"])[0]
        )
        p_previous = float(old_p.loc[patient, old_probability_column])
        require(
            abs(p_tumor - p_previous) <= 1e-08,
            "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: abs(p_tumor - p_previous) <= 1e-08.",
        )
        whole_expression[patient] = (
            scope_inputs["Whole_tissue"].iloc[0].to_numpy()
        )
        tumor_expression[patient] = reconstructed
        fraction_tumor = float(tumor_mask.mean())
        fraction_immune = float(
            cell_meta["is_immune_for_descriptive_analysis"].mean()
        )
        prediction_rows.append(
            {
                "patient_id": patient,
                "MTAP_GISTIC_call": int(calls.loc[patient]),
                "label_MTAP_deep": int(y.loc[patient]),
                "n_all_nuclei": len(cell_meta),
                "n_Tumor_nuclei": int(tumor_mask.sum()),
                "Tumor_cell_fraction": fraction_tumor,
                "immune_cell_fraction": fraction_immune,
                "Tumor_UMI_fraction": scope_totals["Tumor"]
                / scope_totals["Whole_tissue"],
                "Tumor_LOPO_probability": p_tumor,
                "Whole_tissue_transfer_probability": p_whole,
                "delta_whole_minus_Tumor": p_whole - p_tumor,
                "absolute_probability_difference": abs(p_whole - p_tumor),
                "Tumor_call_at_05": int(p_tumor >= THRESHOLD),
                "Whole_tissue_call_at_05": int(p_whole >= THRESHOLD),
                "discordant_at_05": (p_tumor >= THRESHOLD)
                != (p_whole >= THRESHOLD),
                "Tumor_input_reproduced": True,
                "Tumor_prediction_reproduced": True,
                "held_out_patient_excluded_from_training": True,
            }
        )
        pd.DataFrame(prediction_rows).to_csv(
            OUT / "patient_paired_predictions.tsv", sep="\t", index=False
        )
        del counts, core_counts, cell_meta
        gc.collect()
    pred = pd.DataFrame(prediction_rows).set_index("patient_id")
    require(
        len(pred) == 16,
        "GBM whole-tissue versus malignant-cell L2 comparison: required condition failed: len(pred) == 16.",
    )
    for values, name in [
        (whole_expression, "Whole_tissue_three_gene_log2CPM_plus_1.tsv"),
        (tumor_expression, "Reproduced_Tumor_three_gene_log2CPM_plus_1.tsv"),
    ]:
        pd.DataFrame.from_dict(
            values, orient="index", columns=CORE
        ).rename_axis("patient_id").to_csv(OUT / name, sep="\t")
    pd.DataFrame(raw_count_rows).to_csv(
        OUT / "three_gene_raw_UMI_and_denominators.tsv", sep="\t", index=False
    )
    pd.DataFrame(mapping_rows).to_csv(
        OUT / "barcode_mapping_audit.tsv", sep="\t", index=False
    )
    pd.DataFrame(model_rows).to_csv(
        OUT / "held_out_model_audit.tsv", sep="\t", index=False
    )
    score_columns = {
        "Tumor_LOPO": "Tumor_LOPO_probability",
        "Whole_tissue_transfer": "Whole_tissue_transfer_probability",
    }
    truth = pred["label_MTAP_deep"].to_numpy(dtype=int)

    def metrics(truth, probability):
        tn, fp, fn, tp = confusion_matrix(
            truth, probability >= THRESHOLD, labels=[0, 1]
        ).ravel()
        return {
            "AUROC": roc_auc_score(truth, probability),
            "AP": average_precision_score(truth, probability),
            "Brier": brier_score_loss(truth, probability),
            "sensitivity_at_05": tp / (tp + fn),
            "specificity_at_05": tn / (tn + fp),
            "precision_at_05": tp / (tp + fp) if tp + fp else np.nan,
            "accuracy_at_05": (tp + tn) / len(truth),
            "TP_at_05": int(tp),
            "FP_at_05": int(fp),
            "FN_at_05": int(fn),
            "TN_at_05": int(tn),
        }

    metric_rows = []
    for name, column in score_columns.items():
        metric_rows.append(
            {
                "analysis": name,
                "n_patients": len(pred),
                "n_DNA_deep_deletion": int(truth.sum()),
                **metrics(truth, pred[column].to_numpy()),
            }
        )
    performance = pd.DataFrame(metric_rows)
    performance.to_csv(
        OUT / "patient_level_performance.tsv", sep="\t", index=False
    )
    rng = np.random.default_rng(SEED)
    positive = np.flatnonzero(truth == 1)
    negative = np.flatnonzero(truth == 0)
    bootstrap_rows = []
    for repeat in range(2000):
        selected = np.concatenate(
            [
                rng.choice(positive, len(positive), replace=True),
                rng.choice(negative, len(negative), replace=True),
            ]
        )
        a = metrics(
            truth[selected],
            pred["Tumor_LOPO_probability"].to_numpy()[selected],
        )
        b = metrics(
            truth[selected],
            pred["Whole_tissue_transfer_probability"].to_numpy()[selected],
        )
        bootstrap_rows.append(
            {
                metric: b[metric] - a[metric]
                for metric in ["AUROC", "AP", "Brier"]
            }
        )
    bootstrap = pd.DataFrame(bootstrap_rows)
    difference_rows = []
    for metric in ["AUROC", "AP", "Brier"]:
        difference_rows.append(
            {
                "metric": metric,
                "difference_whole_minus_Tumor": performance.loc[
                    performance["analysis"].eq("Whole_tissue_transfer"), metric
                ].iloc[0]
                - performance.loc[
                    performance["analysis"].eq("Tumor_LOPO"), metric
                ].iloc[0],
                "bootstrap_CI95_low": bootstrap[metric].quantile(0.025),
                "bootstrap_CI95_high": bootstrap[metric].quantile(0.975),
                "interval_scope": "Conditional_on_fixed_patient_predictions",
            }
        )
    pd.DataFrame(difference_rows).to_csv(
        OUT / "paired_performance_differences.tsv", sep="\t", index=False
    )
    correlation_rows = []
    for composition in [
        "Tumor_cell_fraction",
        "immune_cell_fraction",
        "Tumor_UMI_fraction",
    ]:
        for outcome in [
            "delta_whole_minus_Tumor",
            "absolute_probability_difference",
        ]:
            x = pred[composition].to_numpy()
            z = pred[outcome].to_numpy()
            if np.unique(x).size > 1 and np.unique(z).size > 1:
                rho, pvalue = spearmanr(x, z)
            else:
                rho, pvalue = (np.nan, np.nan)
            correlation_rows.append(
                {
                    "composition": composition,
                    "outcome": outcome,
                    "n_patients": len(pred),
                    "Spearman_rho": rho,
                    "p_value_exploratory_unadjusted": pvalue,
                }
            )
    pd.DataFrame(correlation_rows).to_csv(
        OUT / "composition_associations_exploratory.tsv", sep="\t", index=False
    )
    COLORS = {0: "#33539E", 1: "#C56B9A"}


#### Whole-tissue three-model comparison


In [ ]:
@_nb9_registry.register("s68_0")
def _nb9_stage_s68_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from datetime import datetime
    from itertools import combinations
    import gc
    import gzip
    import hashlib
    import importlib.util
    import json
    import re
    import sys
    import warnings
    import zipfile
    import anndata as ad
    import h5py
    import joblib
    import numpy as np
    import pandas as pd
    import torch
    from scipy import sparse
    from scipy.io import mmread
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
        roc_curve,
        precision_recall_curve,
    )
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    PROJECT = Path(str(PROJECT_ROOT))
    BASE = PROJECT / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
    GBM = PROJECT / "data_processed/CPTAC_GBM_joint_analysis"
    OBS_FILE = GBM / "CPTAC_GBM_16patients_SCT_merged.h5ad"
    MEX_FILE = (
        BASE
        / "04_UMI_download_and_barcode_matching"
        / "UMI_MEX_download_and_matching.tsv"
    )
    PATIENT_FILE = (
        BASE / "01_source_resolution/patient_DNA_and_cell_coverage.tsv"
    )
    TUMOR_FILE = BASE / "01_source_resolution/Tumor_nucleus_metadata.tsv.gz"
    INPUT_DIR = BASE / "05_Tumor_UMI_and_pseudobulk"
    TUMOR_X_FILE = INPUT_DIR / "Tumor_pseudobulk_44_log2CPM_plus_1.tsv"
    MAPPING_FILE = INPUT_DIR / "fixed_44_feature_mapping.tsv"
    L2_DIR = BASE / "08_L2_cell_number_stability_and_candidate/models"
    ADAPTER_DIR = BASE / "07_frozen_DL_and_region_adapter_v1/LOPO_adapters"
    PREVIOUS_PREDICTIONS = (
        BASE
        / "09_snRNA_adapter_L2_ElasticNet_comparison"
        / "snRNA_adapter_L2_ElasticNet_patient_predictions.tsv"
    )
    LOCKED = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction"
        / "15_region_multitask_DL/08_TPM_retraining"
        / "04_locked_models_and_thresholds"
    )
    NETWORK_FILE = LOCKED / "region_multitask_TPM44_network.py"
    LOCKED_MANIFEST = LOCKED / "locked_manifest.json"
    CHECKPOINT_FILES = [
        LOCKED / "models" / f"DL_seed{seed}.pt" for seed in [41, 42, 43]
    ]
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    OUT = BASE / f"11_whole_tissue_three_model_comparison_{RUN_ID}"
    FIG = OUT / "figures"
    EN_SEED = 20261017
    EN_C = 1.0
    EN_L1_RATIO = 0.5
    BOOTSTRAP_SEED = 20261021
    N_BOOTSTRAP = 2000
    THRESHOLD = 0.5
    CORE = ["MTAP", "CDKN2A", "CDKN2B"]
    MODEL_NAMES = ["DL_adapter", "L2", "ElasticNet"]
    DISPLAY = {
        "DL_adapter": "Bulk DL + input adapter",
        "L2": "Three-gene L2",
        "ElasticNet": "44-gene Elastic Net",
    }
    COLORS = {
        "DL_adapter": "#A5678E",
        "L2": "#33539E",
        "ElasticNet": "#7FACD6",
    }
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    OUT.mkdir(parents=True, exist_ok=False)
    FIG.mkdir()
    torch.set_num_threads(1)

    def require(condition, message):
        if not bool(condition):
            raise RuntimeError(message)

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    def read_indexed(path):
        frame = pd.read_csv(path, sep="\t", index_col=0)
        frame.index = frame.index.astype(str)
        require(
            frame.index.is_unique,
            "Whole-tissue three-model comparison: required condition failed: frame.index.is_unique.",
        )
        return frame

    def open_text(path):
        path = Path(path)
        if path.name.endswith(".gz"):
            return gzip.open(path, "rt", encoding="utf-8")
        return open(path, "r", encoding="utf-8")

    def positive_probability(model, values):
        classes = list(model.classes_)
        require(
            1 in classes,
            "Whole-tissue three-model comparison: required condition failed: 1 in classes.",
        )
        result = model.predict_proba(values)[:, classes.index(1)]
        require(
            np.isfinite(result).all()
            and ((result >= 0) & (result <= 1)).all(),
            "Whole-tissue three-model comparison: required condition failed: np.isfinite(result).all() and ((result >= 0) & (result <= 1)).all().",
        )
        return result

    def savefig(fig, name):
        for extension in ["pdf", "png"]:
            fig.savefig(
                FIG / f"{name}.{extension}",
                dpi=400 if extension == "pdf" else 250,
                bbox_inches="tight",
            )
        plt.close(fig)

    required_files = [
        OBS_FILE,
        MEX_FILE,
        PATIENT_FILE,
        TUMOR_FILE,
        TUMOR_X_FILE,
        MAPPING_FILE,
        PREVIOUS_PREDICTIONS,
        NETWORK_FILE,
        LOCKED_MANIFEST,
        *CHECKPOINT_FILES,
    ]
    for path in required_files:
        require(
            path.is_file(),
            "Whole-tissue three-model comparison: required condition failed: path.is_file().",
        )
    with h5py.File(OBS_FILE, "r") as handle:
        obs = ad.io.read_elem(handle["obs"])
    obs.index = obs.index.astype(str)
    require(
        obs.index.is_unique,
        "Whole-tissue three-model comparison: required condition failed: obs.index.is_unique.",
    )
    require(
        "patient_id" in obs,
        "Whole-tissue three-model comparison: required condition failed: 'patient_id' in obs.",
    )
    obs["patient_id"] = obs["patient_id"].astype(str)
    patients = read_indexed(PATIENT_FILE)
    tumor_x = read_indexed(TUMOR_X_FILE).sort_index()
    previous = read_indexed(PREVIOUS_PREDICTIONS)
    tumor = pd.read_csv(TUMOR_FILE, sep="\t", dtype=str, keep_default_na=False)
    mex = pd.read_csv(MEX_FILE, sep="\t", dtype=str, keep_default_na=False)
    mapping = pd.read_csv(
        MAPPING_FILE, sep="\t", dtype=str, keep_default_na=False
    )
    require(
        {"cell_id", "cell_barcode", "patient_id"}.issubset(tumor.columns),
        "Whole-tissue three-model comparison: required condition failed: {'cell_id', 'cell_barcode', 'patient_id'}.issubset(tumor.columns).",
    )
    require(
        tumor["cell_id"].is_unique,
        "Whole-tissue three-model comparison: required condition failed: tumor['cell_id'].is_unique.",
    )
    require(
        not tumor.duplicated(["patient_id", "cell_barcode"]).any(),
        "Whole-tissue three-model comparison: required condition failed: not tumor.duplicated(['patient_id', 'cell_barcode']).any().",
    )
    require(
        not mex["patient_id"].duplicated().any(),
        "Whole-tissue three-model comparison: required condition failed: not mex['patient_id'].duplicated().any().",
    )
    patient_ids = tumor_x.index
    expected = set(patient_ids)
    require(
        len(patient_ids) == 16,
        "Whole-tissue three-model comparison: required condition failed: len(patient_ids) == 16.",
    )
    for name, values in [
        ("Patient metadata", set(patients.index)),
        ("All cells", set(obs["patient_id"])),
        ("Tumor-cell list", set(tumor["patient_id"])),
        ("MEX manifest", set(mex["patient_id"])),
        ("Existing predictions", set(previous.index)),
    ]:
        require(
            values == expected,
            "Whole-tissue three-model comparison: required condition failed: values == expected.",
        )
    patients = patients.reindex(patient_ids)
    previous = previous.reindex(patient_ids)
    calls = pd.to_numeric(patients["MTAP_GISTIC_call"], errors="raise").astype(
        int
    )
    require(
        calls.isin([-2, -1, 0, 1, 2]).all(),
        "Whole-tissue three-model comparison: required condition failed: calls.isin([-2, -1, 0, 1, 2]).all().",
    )
    y = calls.eq(-2).astype(int).to_numpy()
    require(
        y.sum() == 8,
        "Whole-tissue three-model comparison: required condition failed: y.sum() == 8.",
    )
    require(
        np.array_equal(
            pd.to_numeric(
                previous["label_MTAP_deep"], errors="raise"
            ).to_numpy(),
            y,
        ),
        "Whole-tissue three-model comparison: required condition failed: np.array_equal(pd.to_numeric(previous['label_MTAP_deep'], errors='raise').to_numpy(), y).",
    )
    with open(LOCKED_MANIFEST, encoding="utf-8") as handle:
        locked_manifest = json.load(handle)
    hashes = locked_manifest["locked_file_hashes"]
    source_audit = []
    for path in [NETWORK_FILE, *CHECKPOINT_FILES]:
        relative = path.relative_to(LOCKED).as_posix()
        actual = sha256_file(path)
        require(
            hashes.get(relative) == actual,
            "Whole-tissue three-model comparison: required condition failed: hashes.get(relative) == actual.",
        )
        source_audit.append(
            {"file": str(path), "sha256": actual, "role": "Frozen_bulk_DL"}
        )
    module_name = "mtap_verified_network_whole_tissue_comparison"
    spec = importlib.util.spec_from_file_location(module_name, NETWORK_FILE)
    network_module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = network_module
    spec.loader.exec_module(network_module)
    Network = network_module.RegionalMultiTaskNet
    checkpoints = [
        torch.load(path, map_location="cpu", weights_only=False)
        for path in CHECKPOINT_FILES
    ]
    reference = checkpoints[0]
    features = list(reference["feature_names"])
    regions = reference["region_indices"]
    require(
        len(features) == 44 and len(set(features)) == 44,
        "Whole-tissue three-model comparison: required condition failed: len(features) == 44 and len(set(features)) == 44.",
    )
    require(
        sorted((i for region in regions for i in region)) == list(range(44)),
        "Whole-tissue three-model comparison: required condition failed: sorted((i for region in regions for i in region)) == list(range(44)).",
    )
    require(
        set(tumor_x.columns) == set(features),
        "Whole-tissue three-model comparison: required condition failed: set(tumor_x.columns) == set(features).",
    )
    tumor_x = tumor_x.loc[:, features]
    require(
        mapping["feature"].is_unique,
        "Whole-tissue three-model comparison: required condition failed: mapping['feature'].is_unique.",
    )
    target_ids = (
        mapping.set_index("feature")
        .loc[features, "Ensembl_ID"]
        .str.replace("\\.\\d+$", "", regex=True)
        .tolist()
    )
    require(
        len(set(target_ids)) == 44,
        "Whole-tissue three-model comparison: required condition failed: len(set(target_ids)) == 44.",
    )
    for cp in checkpoints:
        for key in [
            "feature_names",
            "region_indices",
            "core_indices",
            "target_Entrez_order",
            "target_symbols",
            "MTAP_INDEX",
            "cancer_to_index",
            "architecture",
        ]:
            require(
                cp[key] == reference[key],
                "Whole-tissue three-model comparison: required condition failed: cp[key] == reference[key].",
            )
        require(
            cp["expression_unit"] == "log2(TPM+1)",
            "Whole-tissue three-model comparison: required condition failed: cp['expression_unit'] == 'log2(TPM+1)'.",
        )
        mi = int(cp["MTAP_INDEX"])
        require(
            cp["target_symbols"][mi] == "MTAP",
            "Whole-tissue three-model comparison: required condition failed: cp['target_symbols'][mi] == 'MTAP'.",
        )
        require(
            str(cp["target_Entrez_order"][mi]) == "4507",
            "Whole-tissue three-model comparison: required condition failed: str(cp['target_Entrez_order'][mi]) == '4507'.",
        )

    def resolve_barcodes(meta, barcodes, known_tumor):
        """Match QC cell identities exactly to MEX barcodes and audit tumor-cell links."""

        def safe_strings(series):
            return series.astype("string").fillna("").astype(str)

        candidates = {
            "cell_id_exact": pd.Series(
                meta.index.astype(str), index=meta.index, dtype=str
            )
        }
        for column in [
            "cell_barcode",
            "barcode",
            "barcodes",
            "original_barcode",
            "source_cell_id",
        ]:
            if column in meta.columns:
                candidates[f"obs:{column}"] = safe_strings(meta[column])
        pattern = re.compile("(?<![ACGTN])([ACGTN]{14,20}-\\d+)(?!\\d)")
        extracted = []
        for cell_id in meta.index:
            hits = pattern.findall(str(cell_id))
            extracted.append(hits[0] if len(hits) == 1 else "")
        candidates["barcode_in_cell_id"] = pd.Series(
            extracted, index=meta.index, dtype=str
        )
        require(
            meta.index.is_unique,
            "Whole-tissue three-model comparison: required condition failed: meta.index.is_unique.",
        )
        require(
            barcodes.is_unique,
            "Whole-tissue three-model comparison: required condition failed: barcodes.is_unique.",
        )
        require(
            known_tumor["cell_id"].is_unique,
            "Whole-tissue three-model comparison: required condition failed: known_tumor['cell_id'].is_unique.",
        )
        known = known_tumor.set_index("cell_id")["cell_barcode"]
        known = safe_strings(known)
        require(
            not known.eq("").any(),
            "Whole-tissue three-model comparison: required condition failed: not known.eq('').any().",
        )
        valid = []
        for method, values in candidates.items():
            if (
                values.eq("").any()
                or values.duplicated().any()
                or (not values.isin(barcodes).all())
                or (not known.index.isin(values.index).all())
            ):
                continue
            if np.array_equal(
                values.reindex(known.index).to_numpy(), known.to_numpy()
            ):
                valid.append((method, values))
        require(
            len(valid) > 0,
            "Whole-tissue three-model comparison: required condition failed: len(valid) > 0.",
        )
        first_values = valid[0][1]
        for method, values in valid[1:]:
            require(
                np.array_equal(values.to_numpy(), first_values.to_numpy()),
                "Whole-tissue three-model comparison: required condition failed: np.array_equal(values.to_numpy(), first_values.to_numpy()).",
            )
        return (first_values, ";".join((method for method, _ in valid)))

    whole_rows = {}
    audit_rows = []
    whole_raw_rows = {}
    tumor_ids = set(tumor["cell_id"])
    for record in mex.sort_values("patient_id").itertuples(index=False):
        patient = record.patient_id
        cell_meta = obs.loc[obs["patient_id"].eq(patient)]
        known_tumor = tumor.loc[tumor["patient_id"].eq(patient)]
        require(
            len(cell_meta) == int(patients.loc[patient, "n_all_nuclei"]),
            "Whole-tissue three-model comparison: required condition failed: len(cell_meta) == int(patients.loc[patient, 'n_all_nuclei']).",
        )
        fp = Path(record.features_file)
        bp = Path(record.barcodes_file)
        mp = Path(record.matrix_file)
        for path in [fp, bp, mp]:
            require(
                path.is_file(),
                "Whole-tissue three-model comparison: required condition failed: path.is_file().",
            )
        ft = pd.read_csv(
            fp, sep="\t", header=None, dtype=str, keep_default_na=False
        )
        require(
            ft.shape[1] in {2, 3},
            "Whole-tissue three-model comparison: required condition failed: ft.shape[1] in {2, 3}.",
        )
        with open_text(bp) as handle:
            barcodes = pd.Index(
                [line.rstrip("\r\n").split("\t")[0] for line in handle]
            )
        require(
            barcodes.is_unique,
            "Whole-tissue three-model comparison: required condition failed: barcodes.is_unique.",
        )
        gene_ids = ft.iloc[:, 0].str.replace("\\.\\d+$", "", regex=True)
        if ft.shape[1] == 3:
            gene_mask = ft.iloc[:, 2].eq("Gene Expression").to_numpy()
        else:
            require(
                gene_ids.str.fullmatch("ENSG\\d+").all(),
                "Whole-tissue three-model comparison: required condition failed: gene_ids.str.fullmatch('ENSG\\\\d+').all().",
            )
            gene_mask = np.ones(len(ft), dtype=bool)
        gene_index = pd.Index(gene_ids.loc[gene_mask])
        require(
            gene_mask.any(),
            "Whole-tissue three-model comparison: required condition failed: gene_mask.any().",
        )
        require(
            gene_index.is_unique,
            "Whole-tissue three-model comparison: required condition failed: gene_index.is_unique.",
        )
        gene_positions = gene_index.get_indexer(target_ids)
        require(
            (gene_positions >= 0).all(),
            "Whole-tissue three-model comparison: required condition failed: (gene_positions >= 0).all().",
        )
        selected_barcodes, method = resolve_barcodes(
            cell_meta, barcodes, known_tumor
        )
        positions = barcodes.get_indexer(selected_barcodes)
        with open(mp, "rb") as raw_handle:
            if mp.name.endswith(".gz"):
                with gzip.GzipFile(fileobj=raw_handle) as handle:
                    raw = mmread(handle)
            else:
                raw = mmread(raw_handle)
        require(
            sparse.issparse(raw),
            "Whole-tissue three-model comparison: required condition failed: sparse.issparse(raw).",
        )
        require(
            raw.shape == (len(ft), len(barcodes)),
            "Whole-tissue three-model comparison: required condition failed: raw.shape == (len(ft), len(barcodes)).",
        )
        for start in range(0, len(raw.data), 1000000):
            values = raw.data[start : start + 1000000]
            require(
                np.isfinite(values).all()
                and (values >= 0).all()
                and np.equal(values, np.rint(values)).all(),
                "Whole-tissue three-model comparison: required condition failed: np.isfinite(values).all() and (values >= 0).all() and np.equal(values, np.rint(values)).all().",
            )
        raw = raw.astype(np.int64, copy=False).tocsc()
        counts = raw[gene_mask, :][:, positions].tocsc()
        del raw
        totals = np.asarray(counts.sum(axis=0)).ravel()
        require(
            (totals > 0).all(),
            "Whole-tissue three-model comparison: required condition failed: (totals > 0).all().",
        )
        model_counts = counts[gene_positions, :].toarray()
        tumor_mask = cell_meta.index.isin(tumor_ids)
        require(
            tumor_mask.sum() == len(known_tumor),
            "Whole-tissue three-model comparison: required condition failed: tumor_mask.sum() == len(known_tumor).",
        )
        tumor_logcpm = np.log2(
            1.0
            + model_counts[:, tumor_mask].sum(axis=1).astype(float)
            / float(totals[tumor_mask].sum())
            * 1000000.0
        )
        require(
            np.allclose(
                tumor_logcpm,
                tumor_x.loc[patient].to_numpy(dtype=float),
                rtol=0,
                atol=1e-08,
            ),
            "Whole-tissue three-model comparison: required condition failed: np.allclose(tumor_logcpm, tumor_x.loc[patient].to_numpy(dtype=float), rtol=0, atol=1e-08).",
        )
        whole_sum = model_counts.sum(axis=1, dtype=np.int64)
        whole_total = int(totals.sum())
        whole_rows[patient] = np.log2(
            1.0 + whole_sum.astype(float) / whole_total * 1000000.0
        )
        whole_raw_rows[patient] = whole_sum
        audit_rows.append(
            {
                "patient_id": patient,
                "n_all_nuclei": len(cell_meta),
                "n_Tumor_nuclei": int(tumor_mask.sum()),
                "Tumor_cell_fraction": float(tumor_mask.mean()),
                "Tumor_UMI_fraction": float(
                    totals[tumor_mask].sum() / whole_total
                ),
                "whole_tissue_total_UMI": whole_total,
                "barcode_mapping_method": method,
                "Tumor_44_gene_input_reproduced": True,
            }
        )
        del counts, model_counts
        gc.collect()
    whole_x = pd.DataFrame.from_dict(
        whole_rows, orient="index", columns=features
    ).reindex(patient_ids)
    whole_x.index.name = "patient_id"
    whole_x.to_csv(OUT / "Whole_tissue_44_log2CPM_plus_1.tsv", sep="\t")
    pd.DataFrame.from_dict(
        whole_raw_rows, orient="index", columns=features
    ).rename_axis("patient_id").to_csv(
        OUT / "Whole_tissue_44_raw_UMI.tsv", sep="\t"
    )
    audit = pd.DataFrame(audit_rows).set_index("patient_id")
    audit.to_csv(OUT / "pseudobulk_input_audit.tsv", sep="\t")
    networks = []
    for cp in checkpoints:
        network = Network(cp)
        network.load_state_dict(cp["state_dict"], strict=True)
        network.eval()
        for parameter in network.parameters():
            parameter.requires_grad_(False)
        networks.append(network)

    def dl_predict(values, adapter_bundle):
        require(
            adapter_bundle["feature_names"] == features,
            "Whole-tissue three-model comparison: required condition failed: adapter_bundle['feature_names'] == features.",
        )
        require(
            adapter_bundle["region_indices"] == regions,
            "Whole-tissue three-model comparison: required condition failed: adapter_bundle['region_indices'] == regions.",
        )
        require(
            adapter_bundle["adapter_input_space"]
            == "Frozen_TCGA_mean_std_standardized_log2CPM_plus_1",
            "Whole-tissue three-model comparison: required condition failed: adapter_bundle['adapter_input_space'] == 'Frozen_TCGA_mean_std_standardized_log2CPM_plus_1'.",
        )
        state = adapter_bundle["adapter_state_dict"]
        membership = torch.as_tensor(state["membership"], dtype=torch.long)
        expected_membership = torch.empty(44, dtype=torch.long)
        for region_id, indices in enumerate(regions):
            expected_membership[indices] = region_id
        require(
            torch.equal(membership.cpu(), expected_membership),
            "Whole-tissue three-model comparison: required condition failed: torch.equal(membership.cpu(), expected_membership).",
        )
        scale = torch.exp(
            float(adapter_bundle["max_log_scale"])
            * torch.tanh(state["scale_raw"])
        )
        shift = float(adapter_bundle["max_shift"]) * torch.tanh(
            state["shift_raw"]
        )
        expression = torch.tensor(values, dtype=torch.float32)
        probabilities = []
        with torch.inference_mode():
            for cp, network in zip(checkpoints, networks):
                mean = torch.tensor(
                    np.asarray(cp["preprocessing"]["mean"]),
                    dtype=torch.float32,
                )
                std = torch.tensor(
                    np.asarray(cp["preprocessing"]["std"]), dtype=torch.float32
                )
                require(
                    mean.shape == (44,)
                    and std.shape == (44,)
                    and torch.isfinite(mean).all()
                    and torch.isfinite(std).all()
                    and (std > 0).all(),
                    "Whole-tissue three-model comparison: required condition failed: mean.shape == (44,) and std.shape == (44,) and torch.isfinite(mean).all() and torch.isfinite(std).all() and (std > 0).all().",
                )
                z = (expression - mean) / std
                z = z * scale[membership] + shift[membership]
                cancer = torch.full(
                    (len(expression),),
                    int(cp["cancer_to_index"]["GBM"]),
                    dtype=torch.long,
                )
                logits = network(z, cancer)
                mtap_logits = logits[:, int(cp["MTAP_INDEX"]), :]
                probabilities.append(torch.softmax(mtap_logits, dim=1)[:, 0])
        return torch.stack(probabilities).mean(dim=0).numpy()

    predictions = pd.DataFrame(index=patient_ids)
    predictions["MTAP_GISTIC_call"] = calls
    predictions["label_MTAP_deep"] = y
    old_columns = {
        "DL_adapter": "Region_adapter_DL_LOPO",
        "L2": "L2_LOPO",
        "ElasticNet": "ElasticNet_LOPO",
    }
    for column in old_columns.values():
        require(
            column in previous,
            "Whole-tissue three-model comparison: required condition failed: column in previous.",
        )
    for patient in patient_ids:
        train_ids = patient_ids[patient_ids != patient]
        train_y = pd.Series(y, index=patient_ids).loc[train_ids]
        l2_path = L2_DIR / f"L2_held_out_{patient}.joblib"
        require(
            l2_path.is_file(),
            "Whole-tissue three-model comparison: required condition failed: l2_path.is_file().",
        )
        l2_bundle = joblib.load(l2_path)
        require(
            l2_bundle["held_out_patient"] == patient
            and set(l2_bundle["training_patients"]) == set(train_ids)
            and (list(l2_bundle["feature_names"]) == CORE),
            "Whole-tissue three-model comparison: required condition failed: l2_bundle['held_out_patient'] == patient and set(l2_bundle['training_patients']) == set(train_ids) and (list(l2_bundle['feature_names']) == CORE).",
        )
        l2 = l2_bundle["pipeline"]
        adapter_path = ADAPTER_DIR / f"held_out_{patient}_adapter.pt"
        require(
            adapter_path.is_file(),
            "Whole-tissue three-model comparison: required condition failed: adapter_path.is_file().",
        )
        adapter = torch.load(
            adapter_path, map_location="cpu", weights_only=False
        )
        require(
            adapter["held_out_patient"] == patient
            and set(adapter["training_patients"]) == set(train_ids),
            "Whole-tissue three-model comparison: required condition failed: adapter['held_out_patient'] == patient and set(adapter['training_patients']) == set(train_ids).",
        )
        require(
            [str(p) for p in CHECKPOINT_FILES]
            == list(adapter["source_checkpoints"]),
            "Whole-tissue three-model comparison: required condition failed: [str(p) for p in CHECKPOINT_FILES] == list(adapter['source_checkpoints']).",
        )
        en = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "classifier",
                    LogisticRegression(
                        penalty="elasticnet",
                        solver="saga",
                        C=EN_C,
                        l1_ratio=EN_L1_RATIO,
                        class_weight=None,
                        max_iter=20000,
                        tol=1e-05,
                        random_state=EN_SEED,
                    ),
                ),
            ]
        )
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            en.fit(
                tumor_x.loc[train_ids].to_numpy(dtype=float),
                train_y.to_numpy(),
            )
        for scope, frame in [("Tumor", tumor_x), ("Whole", whole_x)]:
            values = frame.loc[[patient]]
            predictions.loc[patient, f"L2_{scope}"] = positive_probability(
                l2, values[CORE]
            )[0]
            predictions.loc[patient, f"ElasticNet_{scope}"] = (
                positive_probability(en, values.to_numpy(dtype=float))[0]
            )
            predictions.loc[patient, f"DL_adapter_{scope}"] = dl_predict(
                values.to_numpy(dtype=np.float32), adapter
            )[0]
        for model, column in old_columns.items():
            difference = abs(
                predictions.loc[patient, f"{model}_Tumor"]
                - float(previous.loc[patient, column])
            )
            tolerance = 1e-05 if model == "DL_adapter" else 1e-07
            require(
                difference <= tolerance,
                "Whole-tissue three-model comparison: required condition failed: difference <= tolerance.",
            )
        for path, role in [
            (l2_path, "Held_out_L2"),
            (adapter_path, "Held_out_DL_adapter"),
        ]:
            source_audit.append(
                {"file": str(path), "sha256": sha256_file(path), "role": role}
            )
        predictions.to_csv(
            OUT / "three_model_patient_predictions.tsv", sep="\t"
        )
    predictions = predictions.join(audit)
    for model in MODEL_NAMES:
        predictions[f"{model}_delta_Whole_minus_Tumor"] = (
            predictions[f"{model}_Whole"] - predictions[f"{model}_Tumor"]
        )
    predictions.to_csv(OUT / "three_model_patient_predictions.tsv", sep="\t")
    pd.DataFrame(source_audit).to_csv(
        OUT / "model_source_audit.tsv", sep="\t", index=False
    )

    def evaluate(truth, score):
        tn, fp, fn, tp = confusion_matrix(
            truth, score >= THRESHOLD, labels=[0, 1]
        ).ravel()
        sensitivity = tp / (tp + fn)
        specificity = tn / (tn + fp)
        return {
            "AUROC": roc_auc_score(truth, score),
            "AP": average_precision_score(truth, score),
            "Brier": brier_score_loss(truth, score),
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "Balanced_accuracy": (sensitivity + specificity) / 2,
            "Accuracy": (tp + tn) / len(truth),
            "Precision": tp / (tp + fp) if tp + fp else np.nan,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
        }

    metric_names = [
        "AUROC",
        "AP",
        "Brier",
        "Sensitivity",
        "Specificity",
        "Balanced_accuracy",
        "Accuracy",
    ]
    score_arrays = {
        (scope, model): predictions[f"{model}_{scope}"].to_numpy()
        for scope in ["Tumor", "Whole"]
        for model in MODEL_NAMES
    }
    points = {key: evaluate(y, score) for key, score in score_arrays.items()}
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    boot = {
        key: {metric: [] for metric in metric_names} for key in score_arrays
    }
    for repeat in range(N_BOOTSTRAP):
        ix = np.concatenate(
            [
                rng.choice(pos, len(pos), replace=True),
                rng.choice(neg, len(neg), replace=True),
            ]
        )
        for key, score in score_arrays.items():
            values = evaluate(y[ix], score[ix])
            for metric in metric_names:
                boot[key][metric].append(values[metric])
    performance_rows = []
    for (scope, model), point in points.items():
        row = {
            "scope": scope,
            "model": model,
            "n_patients": len(y),
            "n_positive": int(y.sum()),
            **point,
        }
        for metric in metric_names:
            values = np.asarray(boot[scope, model][metric])
            row[f"{metric}_CI_low"] = np.quantile(values, 0.025)
            row[f"{metric}_CI_high"] = np.quantile(values, 0.975)
        performance_rows.append(row)
    performance = pd.DataFrame(performance_rows)
    performance.to_csv(
        OUT / "three_model_performance_with_CI.tsv", sep="\t", index=False
    )
    difference_rows = []
    for a, b in combinations(MODEL_NAMES, 2):
        for metric in ["AUROC", "AP", "Brier"]:
            delta = np.asarray(boot["Whole", a][metric]) - np.asarray(
                boot["Whole", b][metric]
            )
            difference_rows.append(
                {
                    "comparison": "Whole_tissue_between_models",
                    "first": a,
                    "second": b,
                    "metric": metric,
                    "difference_first_minus_second": points["Whole", a][metric]
                    - points["Whole", b][metric],
                    "CI_low": np.quantile(delta, 0.025),
                    "CI_high": np.quantile(delta, 0.975),
                }
            )
    for model in MODEL_NAMES:
        for metric in ["AUROC", "AP", "Brier"]:
            delta = np.asarray(boot["Whole", model][metric]) - np.asarray(
                boot["Tumor", model][metric]
            )
            difference_rows.append(
                {
                    "comparison": "Whole_minus_Tumor",
                    "first": model,
                    "second": model,
                    "metric": metric,
                    "difference_first_minus_second": points["Whole", model][
                        metric
                    ]
                    - points["Tumor", model][metric],
                    "CI_low": np.quantile(delta, 0.025),
                    "CI_high": np.quantile(delta, 0.975),
                }
            )
    pd.DataFrame(difference_rows).to_csv(
        OUT / "paired_performance_differences.tsv", sep="\t", index=False
    )
    whole_performance = (
        performance.loc[performance["scope"].eq("Whole")]
        .set_index("model")
        .reindex(MODEL_NAMES)
    )


### Threshold selection and the 0.75 deployment rule


#### GBM threshold sensitivity at 0.50, 0.75, and 0.80


In [ ]:
@_nb9_registry.register("s70_0")
def _nb9_stage_s70_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from datetime import datetime
    import hashlib
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    from scipy.stats import beta
    from sklearn.metrics import (
        confusion_matrix,
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        matthews_corrcoef,
    )

    PROJECT = Path(str(PROJECT_ROOT))
    MODEL_ROOT = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
        / "08_L2_cell_number_stability_and_candidate"
    )
    MODEL_FILE = MODEL_ROOT / "models/GBM_snRNA_three_gene_L2_candidate.joblib"
    VALIDATION_FILE = (
        MODEL_ROOT / "reproduced_full_pseudobulk_LOPO_predictions.tsv"
    )
    THRESHOLDS = [0.5, 0.75, 0.8]
    TARGET_SPECIFICITY = 0.95
    PATIENT_COLUMN = "patient_id"
    LABEL_COLUMN = "label_MTAP_deep"
    GISTIC_COLUMN = "MTAP_GISTIC_call"
    PROBABILITY_COLUMN = "full_pseudobulk_probability"
    OUT = (
        PROJECT
        / "data_processed/MTAP_prediction_registry/threshold_evaluation"
        / (
            datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            + "_GBM_LOPO_three_thresholds"
        )
    )
    FONT_FAMILY = "DejaVu Sans"
    PDF_DPI = 400
    mpl.rcParams.update(
        {
            "font.family": FONT_FAMILY,
            "font.size": 10,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )

    def require(condition, message):
        if not condition:
            raise RuntimeError(message)

    def sha256_file(path):
        h = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator else np.nan

    def exact_binomial_ci(successes, total, alpha=0.05):
        """Compute a Clopper-Pearson exact binomial confidence interval."""
        if total == 0:
            return (np.nan, np.nan)
        lower = (
            0.0
            if successes == 0
            else float(beta.ppf(alpha / 2, successes, total - successes + 1))
        )
        upper = (
            1.0
            if successes == total
            else float(
                beta.ppf(1 - alpha / 2, successes + 1, total - successes)
            )
        )
        return (lower, upper)

    require(
        MODEL_FILE.is_file(),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: MODEL_FILE.is_file().",
    )
    if not VALIDATION_FILE.is_file():
        candidates = sorted(MODEL_ROOT.parent.rglob("*LOPO*predictions*.tsv"))
        for path in candidates[:30]:
            pass
        raise FileNotFoundError(
            "GBM threshold sensitivity at 0.50, 0.75, and 0.80: integrity check failed; condition: input or artifact integrity check."
        )
    OUT.mkdir(parents=True, exist_ok=False)
    bundle = joblib.load(MODEL_FILE)
    require(
        isinstance(bundle, dict),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: isinstance(bundle, dict).",
    )
    for key in [
        "model_status",
        "feature_names",
        "expression_unit",
        "normalization",
        "prediction_unit",
        "target",
        "training_positive_fraction",
        "pan_cancer_transfer_validated",
        "deployment_threshold_locked",
        "parameters",
    ]:
        pass
    pipeline = bundle.get("pipeline")
    require(
        pipeline is not None,
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: pipeline is not None.",
    )
    for name, step in pipeline.steps:
        parameters = step.get_params(deep=False)
        selected_parameters = {
            key: parameters[key]
            for key in [
                "C",
                "penalty",
                "class_weight",
                "solver",
                "max_iter",
                "with_mean",
                "with_std",
            ]
            if key in parameters
        }
    data = pd.read_csv(
        VALIDATION_FILE, sep="\t", dtype=str, keep_default_na=False
    )
    required_columns = [
        PATIENT_COLUMN,
        LABEL_COLUMN,
        GISTIC_COLUMN,
        PROBABILITY_COLUMN,
    ]
    if not set(required_columns).issubset(data.columns):
        raise RuntimeError(
            "GBM threshold sensitivity at 0.50, 0.75, and 0.80: integrity check failed; condition: input or artifact integrity check."
        )
    data[PATIENT_COLUMN] = data[PATIENT_COLUMN].str.strip()
    require(
        data[PATIENT_COLUMN].ne("").all(),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: data[PATIENT_COLUMN].ne('').all().",
    )
    require(
        not data[PATIENT_COLUMN].duplicated().any(),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: not data[PATIENT_COLUMN].duplicated().any().",
    )
    for column in [LABEL_COLUMN, GISTIC_COLUMN, PROBABILITY_COLUMN]:
        data[column] = pd.to_numeric(data[column], errors="raise")
    require(
        data[LABEL_COLUMN].isin([0, 1]).all(),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: data[LABEL_COLUMN].isin([0, 1]).all().",
    )
    require(
        data[GISTIC_COLUMN].isin([-2, -1, 0, 1, 2]).all(),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: data[GISTIC_COLUMN].isin([-2, -1, 0, 1, 2]).all().",
    )
    require(
        np.array_equal(
            data[LABEL_COLUMN].to_numpy(dtype=int),
            data[GISTIC_COLUMN].eq(-2).to_numpy(dtype=int),
        ),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: np.array_equal(data[LABEL_COLUMN].to_numpy(dtype=int), data[GISTIC_COLUMN].eq(-2).to_numpy(dtype=int)).",
    )
    require(
        np.isfinite(data[PROBABILITY_COLUMN]).all()
        and data[PROBABILITY_COLUMN].between(0, 1).all(),
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: np.isfinite(data[PROBABILITY_COLUMN]).all() and data[PROBABILITY_COLUMN].between(0, 1).all().",
    )
    training_patients = bundle.get("training_patients")
    if isinstance(training_patients, (list, tuple, np.ndarray)):
        if all(
            (isinstance(x, (str, int, np.integer)) for x in training_patients)
        ):
            saved_patients = {str(x).strip() for x in training_patients}
            observed_patients = set(data[PATIENT_COLUMN])
            require(
                observed_patients == saved_patients,
                "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: observed_patients == saved_patients.",
            )
    training_calls = bundle.get("training_GISTIC_calls")
    if isinstance(training_calls, dict):
        calls = {str(k).strip(): v for k, v in training_calls.items()}
        if set(data[PATIENT_COLUMN]).issubset(calls):
            saved_calls = pd.to_numeric(
                data[PATIENT_COLUMN].map(calls), errors="coerce"
            )
            if saved_calls.notna().all():
                require(
                    np.array_equal(
                        saved_calls.to_numpy(), data[GISTIC_COLUMN].to_numpy()
                    ),
                    "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: np.array_equal(saved_calls.to_numpy(), data[GISTIC_COLUMN].to_numpy()).",
                )
    y = data[LABEL_COLUMN].to_numpy(dtype=int)
    p = data[PROBABILITY_COLUMN].to_numpy(dtype=float)
    require(
        len(np.unique(y)) == 2,
        "GBM threshold sensitivity at 0.50, 0.75, and 0.80: required condition failed: len(np.unique(y)) == 2.",
    )
    n_positive = int(y.sum())
    n_negative = int((y == 0).sum())
    scope_columns = [
        c
        for c in ["input_scope", "scope", "cell_type", "prediction_unit"]
        if c in data
    ]
    for column in scope_columns:
        pass
    metric_rows = []
    patient_results = data.copy()
    for threshold in THRESHOLDS:
        prediction = (p >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
        sensitivity = safe_divide(tp, tp + fn)
        specificity = safe_divide(tn, tn + fp)
        ppv = safe_divide(tp, tp + fp)
        npv = safe_divide(tn, tn + fn)
        sensitivity_ci = exact_binomial_ci(int(tp), int(tp + fn))
        specificity_ci = exact_binomial_ci(int(tn), int(tn + fp))
        metric_rows.append(
            {
                "threshold": threshold,
                "n_patients": len(y),
                "TP": int(tp),
                "FP": int(fp),
                "TN": int(tn),
                "FN": int(fn),
                "sensitivity": sensitivity,
                "sensitivity_CI_low": sensitivity_ci[0],
                "sensitivity_CI_high": sensitivity_ci[1],
                "specificity": specificity,
                "specificity_CI_low": specificity_ci[0],
                "specificity_CI_high": specificity_ci[1],
                "PPV_in_this_validation_set": ppv,
                "NPV_in_this_validation_set": npv,
                "accuracy": safe_divide(tp + tn, len(y)),
                "balanced_accuracy": (sensitivity + specificity) / 2,
                "Youden_J": sensitivity + specificity - 1,
                "F1": safe_divide(2 * tp, 2 * tp + fp + fn),
                "MCC": matthews_corrcoef(y, prediction),
                "n_predicted_positive": int(prediction.sum()),
                "predicted_positive_fraction": float(prediction.mean()),
                "meets_specificity_point_target": specificity
                >= TARGET_SPECIFICITY,
                "specificity_CI_low_meets_target": specificity_ci[0]
                >= TARGET_SPECIFICITY,
            }
        )
        suffix = f"{threshold:.2f}"
        patient_results[f"prediction_t{suffix}"] = prediction
        patient_results[f"correct_t{suffix}"] = prediction == y
    metrics = pd.DataFrame(metric_rows).sort_values("threshold")
    display_columns = [
        "threshold",
        "TP",
        "FP",
        "TN",
        "FN",
        "sensitivity",
        "specificity",
        "balanced_accuracy",
        "MCC",
        "n_predicted_positive",
    ]
    prediction_columns = [
        f"prediction_t{threshold:.2f}" for threshold in THRESHOLDS
    ]
    changed = patient_results[
        patient_results[prediction_columns].nunique(axis=1).gt(1)
    ].copy()
    if changed.empty:
        pass
    base_prediction = (p >= 0.5).astype(int)
    for threshold in THRESHOLDS:
        if threshold == 0.5:
            continue
        current = (p >= threshold).astype(int)
        positive_to_negative = (base_prediction == 1) & (current == 0)
        removed_fp = int((positive_to_negative & (y == 0)).sum())
        added_fn = int((positive_to_negative & (y == 1)).sum())
    best_ba = metrics["balanced_accuracy"].max()
    ba_best = metrics.loc[np.isclose(metrics["balanced_accuracy"], best_ba)]
    eligible = metrics.loc[metrics["specificity"] >= TARGET_SPECIFICITY].copy()
    if eligible.empty:
        high_specificity_candidates = []
    else:
        max_sensitivity = eligible["sensitivity"].max()
        high_specificity_best = eligible.loc[
            np.isclose(eligible["sensitivity"], max_sensitivity)
        ]
        high_specificity_candidates = high_specificity_best[
            "threshold"
        ].tolist()
        if not high_specificity_best["specificity_CI_low_meets_target"].all():
            pass
    colors = ["#4477AA", "#EE9977", "#228833"]
    x = np.arange(len(metrics))


### Unified three-model performance across all evaluation settings

Read source prediction files for TCGA OOF, TCGA holdout, CPTAC, GBM
snRNA, and GSE327167 scRNA. Compute all common source metrics plus
accuracy, F1, MCC, NPV, log loss, and the four confusion counts.
Every model within a cohort uses the same evaluable patients.
Missing cohorts are reported explicitly; no result is imputed.

Classification summaries are computed at 0.50 and 0.75. AUROC, AP,
Brier score, and log loss are threshold-independent. Existing source
bootstrap analyses remain in their original stages. This unified
point-estimate summary does not replace those confidence intervals.


In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    roc_auc_score,
)


def _nb9_latest_file(root, pattern, environment_key):
    """Resolve an explicit override or the latest timestamped source output."""
    override = os.environ.get(environment_key)
    if override:
        candidate = Path(override).expanduser()
        return candidate if candidate.is_absolute() else root / candidate
    candidates = list(root.glob(pattern))
    return (
        max(candidates, key=lambda p: (p.parent.name, p.stat().st_mtime_ns))
        if candidates
        else None
    )


def _nb9_prediction_specs(root):
    """Use prediction schemas and relative locations recorded in Model."""
    base = root / "data_processed/MTAP_RNA_prediction"
    tpm = base / "15_region_multitask_DL/08_TPM_retraining"
    comparison = tpm / "06_three_gene_L2_comparison"
    transfer = base / "16_scRNA_transfer"
    meso = Path(globals().get(
        "GSE327167_RUN_DIR",
        Path(os.environ.get(
            "MTAP_GSE327167_RUN_DIR",
            str(root / (
                "data_processed/MTAP_prediction_registry/external_validation/"
                "20261103_110248_573136_GSE327167_31patient_three_models"
            )),
        )),
    ))
    bulk_columns = {
        "L2": "prob_L2",
        "Elastic Net": "prob_ElasticNet",
        "DL": "prob_DL",
    }
    specs = [
        {
            "cohort": "TCGA development OOF",
            "file": comparison / "three_model_development_OOF.tsv",
            "label": "label_MTAP_deep",
            "columns": bulk_columns,
        },
        {
            "cohort": "TCGA holdout",
            "file": _nb9_latest_file(
                root,
                "data_processed/MTAP_RNA_prediction/15_region_multitask_DL/"
                "08_TPM_retraining/07_TCGA_original_holdout_evaluation/*/"
                "05_TCGA_original_holdout_three_model_predictions.tsv",
                "MTAP_HOLDOUT_PREDICTIONS",
            ),
            "label": "label_MTAP_deep",
            "columns": {
                "L2": "Three_gene_L2",
                "Elastic Net": "Elastic_Net",
                "DL": "Bulk_DL",
            },
        },
        {
            "cohort": "CPTAC external",
            "file": comparison / "CPTAC_three_model_patient_predictions.tsv",
            "label": "label_MTAP_deep",
            "columns": bulk_columns,
        },
        {
            "cohort": "GBM snRNA malignant",
            "file": transfer / "09_snRNA_adapter_L2_ElasticNet_comparison/"
            "snRNA_adapter_L2_ElasticNet_patient_predictions.tsv",
            "label": "label_MTAP_deep",
            "columns": {
                "L2": "L2_LOPO",
                "Elastic Net": "ElasticNet_LOPO",
                "DL": "Region_adapter_DL_LOPO",
            },
        },
    ]
    whole_file = _nb9_latest_file(
        root,
        "data_processed/MTAP_RNA_prediction/16_scRNA_transfer/"
        "11_whole_tissue_three_model_comparison_*/three_model_patient_predictions.tsv",
        "MTAP_GBM_WHOLE_PREDICTIONS",
    )
    specs.append(
        {
            "cohort": "GBM snRNA whole tissue",
            "file": whole_file,
            "label": "label_MTAP_deep",
            "columns": {
                "L2": "L2_Whole",
                "Elastic Net": "ElasticNet_Whole",
                "DL": "DL_adapter_Whole",
            },
        }
    )
    for cohort, scope, minimum in (
        ("GSE327167 malignant >=50", "Malignant", 50),
        ("GSE327167 malignant any", "Malignant", 1),
        ("GSE327167 whole tissue", "Whole", 1),
    ):
        specs.append(
            {
                "cohort": cohort,
                "file": meso
                / "04_GSE327167_three_model_predictions_missing_feature_aware.tsv",
                "label": "OGM_deep_deletion_label",
                "columns": {
                    "L2": "Three-gene L2",
                    "Elastic Net": "44-gene Elastic Net",
                    "DL": "Bulk DL + input adapter",
                },
                "scope": scope,
                "minimum_cells": minimum,
                "eligibility_column": (
                    "malignant_ge50_primary_eligible"
                    if minimum == 50
                    else None
                ),
            }
        )
    return specs


def _nb9_metric_values(labels, scores, threshold):
    """Keep continuous-score metrics separate from classification thresholds."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    prediction = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        labels, prediction, labels=[0, 1]
    ).ravel()

    def divide(numerator, denominator):
        return numerator / denominator if denominator else np.nan

    both_classes = len(np.unique(labels)) == 2
    return {
        "n_patients": len(labels),
        "n_positive": int(labels.sum()),
        "n_negative": int((labels == 0).sum()),
        "prevalence": float(labels.mean()),
        "AUROC": roc_auc_score(labels, scores) if both_classes else np.nan,
        "AUPRC_AP": (
            average_precision_score(labels, scores) if both_classes else np.nan
        ),
        "Brier": brier_score_loss(labels, scores),
        "Log_loss": log_loss(labels, scores, labels=[0, 1]),
        "Accuracy": accuracy_score(labels, prediction),
        "Balanced_accuracy": (
            balanced_accuracy_score(labels, prediction)
            if both_classes
            else np.nan
        ),
        "Sensitivity": divide(tp, tp + fn),
        "Specificity": divide(tn, tn + fp),
        "PPV": divide(tp, tp + fp),
        "NPV": divide(tn, tn + fn),
        "F1": f1_score(labels, prediction, zero_division=0),
        "MCC": matthews_corrcoef(labels, prediction),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
    }


def _nb9_collect_metrics(root):
    """Compare three models on identical patients within each source cohort."""
    rows, audit, inputs = [], [], []
    for specification in _nb9_prediction_specs(root):
        cohort = specification["cohort"]
        path = specification["file"]
        if path is None or not path.is_file():
            audit.append(
                {
                    "cohort": cohort,
                    "status": "PREDICTION_FILE_UNAVAILABLE",
                    "n_patients": 0,
                }
            )
            continue
        frame = pd.read_csv(path, sep="\t", dtype={"patient_id": str})
        required = {
            "patient_id",
            specification["label"],
            *specification["columns"].values(),
        }
        if "scope" in specification:
            required.update({"input_scope", "n_cells"})
            if specification.get("eligibility_column"):
                required.add(specification["eligibility_column"])
        if not required.issubset(frame.columns):
            audit.append(
                {
                    "cohort": cohort,
                    "status": "SOURCE_SCHEMA_MISMATCH",
                    "n_patients": 0,
                }
            )
            continue
        if "scope" in specification:
            frame = frame.loc[
                frame["input_scope"].eq(specification["scope"])
                & pd.to_numeric(frame["n_cells"], errors="coerce").ge(
                    specification["minimum_cells"]
                )
            ].copy()
            if specification.get("eligibility_column"):
                eligibility = (
                    frame[specification["eligibility_column"]]
                    .astype(str)
                    .str.lower()
                )
                frame = frame.loc[
                    eligibility.isin(["true", "1", "1.0"])
                ].copy()
        if frame.empty:
            audit.append(
                {
                    "cohort": cohort,
                    "status": "NO_ELIGIBLE_PATIENTS",
                    "n_patients": 0,
                }
            )
            continue
        if (
            frame["patient_id"].isna().any()
            or frame["patient_id"].duplicated().any()
        ):
            raise ValueError(
                f"{cohort}: patient identities must be present and unique."
            )
        labels = pd.to_numeric(frame[specification["label"]], errors="coerce")
        if (frame[specification["label"]].notna() & labels.isna()).any():
            raise ValueError(
                f"{cohort}: nonmissing outcome labels are not numeric."
            )
        n_without_label = int(labels.isna().sum())
        frame = frame.loc[labels.notna()].copy()
        labels = labels.loc[frame.index]
        if frame.empty or not labels.isin([0, 1]).all():
            raise ValueError(
                f"{cohort}: no complete valid binary outcome labels."
            )
        scores = frame[list(specification["columns"].values())].apply(
            pd.to_numeric, errors="coerce"
        )
        values = scores.to_numpy(dtype=float)
        if (
            not np.isfinite(values).all()
            or ((values < 0) | (values > 1)).any()
        ):
            raise ValueError(
                f"{cohort}: all three models need finite probabilities in [0, 1]."
            )
        for threshold in (0.50, 0.75):
            for model, column in specification["columns"].items():
                rows.append(
                    {
                        "cohort": cohort,
                        "model": model,
                        "threshold": threshold,
                        **_nb9_metric_values(
                            labels.to_numpy(dtype=int),
                            scores[column].to_numpy(),
                            threshold,
                        ),
                    }
                )
        audit.append(
            {
                "cohort": cohort,
                "status": "EVALUATED",
                "n_patients": len(frame),
                "n_without_DNA_label": n_without_label,
            }
        )
        try:
            relative_file = str(path.relative_to(root))
        except ValueError:
            relative_file = "ENVIRONMENT_OVERRIDE_OUTSIDE_PROJECT"
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        inputs.append(
            {
                "cohort": cohort,
                "relative_file": relative_file,
                "sha256": digest.hexdigest(),
            }
        )
    performance, availability = pd.DataFrame(rows), pd.DataFrame(audit)
    if not performance.empty:
        output = root / "outputs/nine_part_model_comparison"
        output.mkdir(parents=True, exist_ok=True)
        performance.to_csv(
            output / "all_cohort_three_model_metrics.tsv",
            sep="\t",
            index=False,
        )
        availability.to_csv(
            output / "prediction_availability.tsv", sep="\t", index=False
        )
        (output / "source_prediction_manifest.json").write_text(
            json.dumps(
                {
                    "classification_thresholds": [0.50, 0.75],
                    "inputs": inputs,
                    "timestamp_selection": "Latest timestamped source output unless an environment override is supplied.",
                    "uncertainty": "Point estimates; original source bootstrap analyses are retained separately.",
                    "patient_level_exports": False,
                },
                indent=2,
            )
            + "\n"
        )
    return performance, availability


def _nb9_display_metrics(performance, availability):
    """Display aggregate cohort/model metrics without individual predictions."""
    display(availability.fillna(0))
    if performance.empty:
        print(
            "No cached prediction files were available. No performance values were generated."
        )
        return
    ranking = [
        "cohort",
        "model",
        "n_patients",
        "n_positive",
        "AUROC",
        "AUPRC_AP",
        "Brier",
        "Log_loss",
    ]
    classification = [
        "cohort",
        "model",
        "Accuracy",
        "Balanced_accuracy",
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
        "F1",
        "MCC",
        "TP",
        "FP",
        "FN",
        "TN",
    ]
    for threshold in (0.50, 0.75):
        subset = performance.loc[performance["threshold"].eq(threshold)]
        print(
            f"Three-model comparison: classification threshold = {threshold:.2f}"
        )
        display(subset[ranking].round(4).reset_index(drop=True))
        display(subset[classification].round(4).reset_index(drop=True))


## 9. Performance Visualization

Retain source ROC/PR curves, bootstrap comparisons, dilution figures, threshold figures, heatmaps, and coefficient plots. Cross-cohort figures use the same five-color palette and display numeric estimates.


### Source figures


#### Fixed L2 baselines and development out-of-fold predictions - figures and completion records


In [ ]:
@_nb9_registry.register("s05_95")
def _nb9_stage_s05_95():
    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.7), facecolor="white")
    for color, model_name in zip(MTAP_PALETTE, models.keys()):
        probability = prediction_table[f"probability_{model_name}"].to_numpy()
        fpr, tpr, _ = roc_curve(y, probability)
        precision, recall, _ = precision_recall_curve(y, probability)
        auc = roc_auc_score(y, probability)
        ap = average_precision_score(y, probability)
        linestyle = "--" if model_name == "Cancer_only" else "-"
        axes[0].plot(
            fpr,
            tpr,
            color=color,
            linestyle=linestyle,
            linewidth=1.8,
            label=f"{model_name}: {auc:.3f}",
        )
        axes[1].step(
            recall,
            precision,
            where="post",
            color=color,
            linestyle=linestyle,
            linewidth=1.8,
            label=f"{model_name}: {ap:.3f}",
        )
    axes[0].plot([0, 1], [0, 1], color="#999999", linewidth=1, linestyle=":")
    axes[1].axhline(
        y.mean(),
        color="#999999",
        linewidth=1,
        linestyle=":",
        label=f"Prevalence: {y.mean():.3f}",
    )
    axes[0].set(
        xlabel="False positive rate",
        ylabel="True positive rate",
        title="Development OOF ROC",
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="Development OOF precision-recall",
    )
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(
            frameon=False,
            fontsize=8,
            loc="lower right" if ax is axes[0] else "upper right",
        )
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURE_DIR / f"baseline_development_OOF_ROC_PR.{extension}",
            dpi=350,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    configuration = {
        "target": "DNA-defined MTAP GISTIC -2 versus non--2",
        "evaluation": "Development-only stratified 5-fold OOF",
        "random_seed": RANDOM_SEED,
        "n_folds": N_FOLDS,
        "classifier": "L2-penalized logistic regression",
        "C": REGULARIZATION_C,
        "class_weight": None,
        "hyperparameter_search": False,
        "feature_selection": False,
        "expression_transform": "log2(input RNA value + 1)",
        "scaling": "StandardScaler fitted within each training fold",
        "descriptive_threshold": DESCRIPTIVE_THRESHOLD,
        "test_predictions_computed": False,
        "model_names": list(models),
        "note": "Saved development models are not final selected models. OOF metrics are model-development evidence, not independent external validation.",
    }
    with open(
        OUTPUT_DIR / "baseline_configuration.json", "w", encoding="utf-8"
    ) as handle:
        json.dump(configuration, handle, ensure_ascii=False, indent=2)
    _public_summary(
        globals(), "Fixed L2 baselines and development out-of-fold predictions"
    )


#### Baseline bootstrap intervals and calibration assessment - figures and completion records


In [ ]:
@_nb9_registry.register("s06_65")
def _nb9_stage_s06_65():
    fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.3), facecolor="white")
    for ax, metric in zip(axes, ["AUROC", "AUPRC_AP", "Brier_score"]):
        row = difference_table.loc[
            difference_table["comparator"].eq("MTAP_only")
            & difference_table["metric"].eq(metric)
        ].iloc[0]
        estimate = row["difference_main_minus_comparator"]
        ax.hlines(
            0,
            row["CI95_low"],
            row["CI95_high"],
            color=COLORS["Three_gene"],
            linewidth=2,
        )
        ax.scatter(estimate, 0, color=COLORS["Three_gene"], s=42, zorder=3)
        ax.axvline(0, color="#888888", linestyle="--", linewidth=0.9)
        ax.set_yticks([])
        ax.set_ylim(-0.7, 0.7)
        direction = (
            "Negative favors three genes"
            if metric == "Brier_score"
            else "Positive favors three genes"
        )
        ax.set_title(metric)
        ax.set_xlabel(f"Three genes minus MTAP only\n{direction}", fontsize=8)
        for side in ["top", "right", "left"]:
            ax.spines[side].set_visible(False)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURE_DIR / f"paired_model_difference_CI.{extension}",
            dpi=350,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    display_cancers = (
        cancer_table.loc[
            cancer_table["model"].eq("Three_gene")
            & cancer_table["support_status"].eq(
                "Adequate_for_descriptive_display"
            )
        ]
        .sort_values(["n_positive", "acronym"], ascending=[False, True])[
            "acronym"
        ]
        .tolist()
    )
    if display_cancers:
        fig, ax = plt.subplots(
            figsize=(8.0, max(3.5, 0.38 * len(display_cancers) + 1.2)),
            facecolor="white",
        )
        y_positions = np.arange(len(display_cancers))
        for name, offset in [("MTAP_only", -0.12), ("Three_gene", 0.12)]:
            subset = (
                cancer_table.loc[cancer_table["model"].eq(name)]
                .set_index("acronym")
                .loc[display_cancers]
            )
            ax.scatter(
                subset["AUROC"],
                y_positions + offset,
                color=COLORS[name],
                label=name,
                s=35,
            )
        ax.set_yticks(y_positions, display_cancers)
        ax.invert_yaxis()
        ax.set_xlim(0.45, 1.01)
        ax.axvline(0.5, color="#999999", linestyle=":", linewidth=1)
        ax.set_xlabel("Within-cancer development OOF AUROC")
        ax.legend(frameon=False, fontsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.tight_layout()
        for extension in ["pdf", "png"]:
            fig.savefig(
                FIGURE_DIR / f"within_cancer_AUROC_comparison.{extension}",
                dpi=350,
                bbox_inches="tight",
                facecolor="white",
            )
        plt.show()
    fig, ax = plt.subplots(figsize=(5.5, 5.0), facecolor="white")
    ax.plot(
        [0, 1],
        [0, 1],
        color="#999999",
        linestyle="--",
        linewidth=1,
        label="Perfect calibration",
    )
    for name in ["MTAP_only", "Three_gene"]:
        subset = calibration_table.loc[
            calibration_table["model"].eq(name)
        ].sort_values("mean_predicted_probability")
        ax.plot(
            subset["mean_predicted_probability"],
            subset["observed_positive_fraction"],
            color=COLORS[name],
            linewidth=1.5,
            label=name,
        )
        adequate = subset["n_patients"].ge(20)
        ax.scatter(
            subset.loc[adequate, "mean_predicted_probability"],
            subset.loc[adequate, "observed_positive_fraction"],
            color=COLORS[name],
            s=35,
        )
        ax.scatter(
            subset.loc[~adequate, "mean_predicted_probability"],
            subset.loc[~adequate, "observed_positive_fraction"],
            facecolors="white",
            edgecolors=COLORS[name],
            s=35,
        )
    ax.set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="Mean predicted probability",
        ylabel="Observed DNA-positive fraction",
    )
    ax.legend(frameon=False, fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURE_DIR / f"development_OOF_calibration.{extension}",
            dpi=350,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    config = {
        "input": str(PREDICTION_FILE),
        "n_patients": int(len(data)),
        "random_seed": RANDOM_SEED,
        "n_bootstraps": N_BOOTSTRAPS,
        "bootstrap_unit": "patient",
        "bootstrap_strata": ["acronym", "label_MTAP_deep"],
        "paired_comparison": True,
        "refit_models_in_bootstrap": False,
        "main_model": MAIN_MODEL,
        "comparators": COMPARATOR_MODELS,
        "test_used": False,
        "new_model_training": False,
        "calibration_refitting": False,
        "probability_bins": probability_bins,
        "notes": [
            "Intervals are conditional on existing OOF predictions.",
            "Rare strata cannot provide reliable stratum-specific uncertainty.",
            "Development comparisons do not replace held-out testing.",
            "No bootstrap-derived hypothesis-test P values were calculated.",
        ],
    }
    with open(
        OUTPUT_DIR / "development_review_config.json", "w", encoding="utf-8"
    ) as handle:
        json.dump(config, handle, ensure_ascii=False, indent=2)
    _public_summary(
        globals(), "Baseline bootstrap intervals and calibration assessment"
    )


#### Development-only probability-calibration sensitivity analysis - figures and completion records


In [ ]:
@_nb9_registry.register("s07_73")
def _nb9_stage_s07_73():
    fig, ax = plt.subplots(figsize=(6.2, 5.3), facecolor="white")
    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="#999999",
        linewidth=1,
        label="Perfect calibration",
    )
    for method in METHODS:
        subset = calibration_table.loc[
            calibration_table["method"].eq(method)
        ].sort_values("mean_predicted_probability")
        ax.plot(
            subset["mean_predicted_probability"],
            subset["observed_positive_fraction"],
            color=COLORS[method],
            linewidth=1.6,
            label=method,
        )
        adequate = subset["n_patients"].ge(20)
        ax.scatter(
            subset.loc[adequate, "mean_predicted_probability"],
            subset.loc[adequate, "observed_positive_fraction"],
            color=COLORS[method],
            s=30,
        )
        ax.scatter(
            subset.loc[~adequate, "mean_predicted_probability"],
            subset.loc[~adequate, "observed_positive_fraction"],
            facecolors="white",
            edgecolors=COLORS[method],
            s=30,
        )
    ax.set(
        xlim=(0, 1),
        ylim=(0, 1),
        xlabel="Mean predicted probability",
        ylabel="Observed DNA-positive fraction",
        title="Development OOF probability calibration",
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURE_DIR / f"calibration_comparison.{extension}",
            dpi=350,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    joblib.dump(
        clone(base_template).fit(X, y),
        MODEL_DIR / "Three_gene_Uncalibrated_NOT_FINAL.joblib",
    )
    for display_name, calibration_method in [
        ("Sigmoid", "sigmoid"),
        ("Isotonic", "isotonic"),
    ]:
        candidate = CalibratedClassifierCV(
            estimator=clone(base_template),
            method=calibration_method,
            cv=StratifiedKFold(
                n_splits=INNER_FOLDS, shuffle=True, random_state=RANDOM_SEED
            ),
            ensemble=False,
            n_jobs=1,
        )
        candidate.fit(X, y)
        joblib.dump(
            candidate,
            MODEL_DIR / f"Three_gene_{display_name}_NOT_FINAL.joblib",
        )
    configuration = {
        "target": "MTAP GISTIC -2 versus non--2",
        "feature_genes": ["MTAP", "CDKN2A", "CDKN2B"],
        "base_model": str(BASE_MODEL_FILE),
        "outer_folds": "Reused previous development CV folds",
        "inner_calibration_folds": INNER_FOLDS,
        "calibration_methods": METHODS,
        "ensemble": False,
        "primary_comparison_metric": "Brier_score",
        "secondary_probability_metric": "log_loss",
        "test_used": False,
        "external_cohort_used": False,
        "threshold_selected": False,
        "final_method_selected": False,
    }
    with open(
        OUTPUT_DIR / "calibration_configuration.json", "w", encoding="utf-8"
    ) as handle:
        json.dump(configuration, handle, ensure_ascii=False, indent=2)
    _public_summary(
        globals(),
        "Development-only probability-calibration sensitivity analysis",
    )


#### Three-gene L2 and Elastic Net comparison - figures and completion records


In [ ]:
@_nb9_registry.register("s12_97")
def _nb9_stage_s12_97():
    fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.5), facecolor="white")
    colors = [PALETTE[1], PALETTE[4], PALETTE[0]]
    display_names = {
        "MTAP_only_L2": "MTAP only: L2",
        "Three_gene_L2": "Three genes: L2",
        "Three_gene_SGD_ElasticNet": "Three genes: SGD Elastic Net",
    }
    for (name, metrics), color in zip(curve_results.items(), colors):
        roc = metrics["roc_df"]
        pr = metrics["pr_df"]
        axes[0].plot(
            roc["fpr"],
            roc["tpr"],
            color=color,
            linewidth=1.8,
            label=f"{display_names[name]}\nAUROC = {metrics['auroc']:.3f}",
        )
        axes[1].plot(
            pr["recall"],
            pr["precision"],
            color=color,
            linewidth=1.8,
            label=f"{display_names[name]}\nAP = {metrics['aupr']:.3f}",
        )
    axes[0].plot(
        [0, 1], [0, 1], linestyle="--", color="#AAAAAA", linewidth=0.8
    )
    axes[1].axhline(
        y.mean(),
        linestyle="--",
        color="#AAAAAA",
        linewidth=0.8,
        label=f"Positive fraction = {y.mean():.3f}",
    )
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].set_title("Development OOF ROC")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Development OOF precision-recall")
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(
            frameon=False,
            fontsize=7.5,
            loc="lower left",
            bbox_to_anchor=(0, 1.12),
        )
    fig.subplots_adjust(
        left=0.08, right=0.98, bottom=0.14, top=0.69, wspace=0.28
    )
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURE_DIR / f"development_ROC_PR.{extension}",
            dpi=350,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    final_search = fit_search(
        X, y, cv=outer_splits, stage="full_development_candidate"
    )
    pd.DataFrame(final_search.cv_results_).to_csv(
        CV_DIR / "full_development_grid_results.tsv", sep="\t", index=False
    )
    candidate_pipeline = final_search.best_estimator_
    coefficient_table(
        candidate_pipeline, stage="full_development_candidate"
    ).to_csv(
        OUTPUT_DIR / "development_candidate_coefficients.tsv",
        sep="\t",
        index=False,
    )
    joblib.dump(
        {
            "pipeline": candidate_pipeline,
            "features": FEATURES,
            "label": LABEL,
            "label_definition": "MTAP GISTIC == -2 versus all non--2",
            "training_scope": "development_only",
            "n_training_patients": len(y),
            "class_weight": "balanced",
            "output_type": "uncalibrated_decision_score",
            "author_util_sha256": author_hash,
            "sklearn_version": sklearn.__version__,
            "best_parameters": final_search.best_params_,
        },
        MODEL_DIR / "SGD_ElasticNet_development_candidate_NOT_FINAL.joblib",
    )
    run_record = {
        "seed": SEED,
        "features": FEATURES,
        "label": LABEL,
        "n_development": len(y),
        "n_positive": int(y.sum()),
        "outer_folds": "Reused existing CV_fold",
        "inner_folds": 5,
        "selection_metric": PRIMARY_SELECTION_METRIC,
        "parameter_grid": PARAM_GRID,
        "best_development_parameters": final_search.best_params_,
        "author_util": str(AUTHOR_UTIL),
        "author_util_sha256": author_hash,
        "sklearn_version": sklearn.__version__,
        "test_evaluated": False,
        "threshold_selected": False,
        "probability_calibrated": False,
        "adaptations": [
            "Three RNA features rather than original transcriptome-wide features",
            "Fixed audited MTAP GISTIC label",
            "No mutation-burden or cancer covariates",
            "Standardization fitted inside each training fold",
            "Nested cross-validation",
            "AP used for inner model selection",
            "Modern log_loss name and increased iteration limit",
        ],
    }
    with open(OUTPUT_DIR / "run_record.json", "w", encoding="utf-8") as handle:
        json.dump(run_record, handle, ensure_ascii=False, indent=2)
    _public_summary(globals(), "Three-gene L2 and Elastic Net comparison")


#### Regional Elastic Net search with nested evaluation - figures and completion records


In [ ]:
@_nb9_registry.register("s14_115")
def _nb9_stage_s14_115():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), facecolor="white")
    for name, probabilities, color in [
        ("Three-gene L2", oof_core, PALETTE[4]),
        ("9p21.3 Elastic Net", oof_region, PALETTE[0]),
    ]:
        fpr, tpr, _ = roc_curve(y, probabilities)
        precision, recall, _ = precision_recall_curve(y, probabilities)
        axes[0].plot(
            fpr,
            tpr,
            color=color,
            linewidth=1.7,
            label=f"{name}: AUROC={roc_auc_score(y, probabilities):.3f}",
        )
        axes[1].plot(
            recall,
            precision,
            color=color,
            linewidth=1.7,
            label=f"{name}: AP={average_precision_score(y, probabilities):.3f}",
        )
    axes[0].plot([0, 1], [0, 1], "--", color="#AAAAAA", linewidth=0.8)
    axes[1].axhline(y.mean(), linestyle="--", color="#AAAAAA", linewidth=0.8)
    axes[0].set(
        xlabel="False positive rate",
        ylabel="True positive rate",
        title="Development OOF ROC",
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="Development OOF precision-recall",
    )
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.legend(
            loc="upper center",
            bbox_to_anchor=(0.5, -0.2),
            frameon=False,
            fontsize=8,
        )
    fig.subplots_adjust(
        left=0.08, right=0.98, bottom=0.3, top=0.9, wspace=0.28
    )
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIG / f"region_vs_three_gene_ROC_PR.{extension}",
            dpi=350,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    final_search = checked_fit(
        make_search(outer_splits), X_region, y, stage="full_development_region"
    )
    pd.DataFrame(final_search.cv_results_).to_csv(
        CV_DIR / "full_development_search.tsv", sep="\t", index=False
    )
    final_core = checked_fit(
        clone(core_pipeline), X_core, y, stage="full_development_core"
    )
    transform_description = "sign(x) * log2(1 + abs(x)); x = patient mean of source RNA values; preserve NaN"
    for name, fitted, features in [
        ("Region_ElasticNet", final_search.best_estimator_, feature_ids),
        ("Three_gene_L2", final_core, CORE_IDS),
    ]:
        joblib.dump(
            {
                "pipeline": fitted,
                "feature_Entrez_order": features,
                "pipeline_input": "already_signedlog_transformed",
                "preprocessing_before_pipeline": transform_description,
                "label_definition": "MTAP GISTIC -2 versus non--2",
                "training_scope": "development_only",
                "status": "candidate_not_final",
                "sklearn_version": sklearn.__version__,
            },
            MODELS / f"{name}_development_candidate.joblib",
        )
    get_coefficients(
        final_search.best_estimator_, feature_ids, stage="full_development"
    ).merge(
        candidate[["Entrez_ID", "DNA_symbol"]],
        on="Entrez_ID",
        how="left",
        validate="many_to_one",
    ).to_csv(
        OUT / "development_candidate_region_coefficients.tsv",
        sep="\t",
        index=False,
    )
    with open(OUT / "run_record.json", "w", encoding="utf-8") as handle:
        json.dump(
            {
                "seed": SEED,
                "candidate_features": feature_ids,
                "source_raw_cache": str(RAW_CACHE),
                "expression_transform": transform_description,
                "transform_is_author_EBpp_protocol": False,
                "negative_values_clipped": False,
                "selection_metric": "average_precision",
                "parameter_grid": REGION_GRID,
                "final_region_parameters": final_search.best_params_,
                "class_weight": None,
                "test_evaluated": False,
                "threshold_selected": False,
                "sklearn_version": sklearn.__version__,
            },
            handle,
            ensure_ascii=False,
            indent=2,
        )
    _public_summary(
        globals(), "Regional Elastic Net search with nested evaluation"
    )


#### Regional multitask DL training and seed analysis - figures and completion records


In [ ]:
@_nb9_registry.register("s24_140")
def _nb9_stage_s24_140():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    fpr, tpr, _ = roc_curve(y_binary, mtap_oof)
    axes[0].plot(
        fpr,
        tpr,
        color="#33539E",
        linewidth=2,
        label=f"AUROC = {performance.iloc[0]['AUROC']:.3f}",
    )
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="0.7")
    axes[0].set(
        xlabel="False positive rate",
        ylabel="Sensitivity",
        title="Development OOF ROC",
    )
    axes[0].legend(frameon=False)
    precision, recall, _ = precision_recall_curve(y_binary, mtap_oof)
    axes[1].plot(
        recall,
        precision,
        color="#A5678E",
        linewidth=2,
        label=f"AP = {performance.iloc[0]['AP']:.3f}",
    )
    axes[1].axhline(
        y_binary.mean(),
        linestyle="--",
        color="0.7",
        label="Positive prevalence",
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="Development OOF precision–recall",
    )
    axes[1].legend(frameon=False)
    for axis in axes:
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "development_MTAP_OOF_ROC_PR.pdf", bbox_inches="tight"
    )
    fig.savefig(
        FIGURE_DIR / "development_MTAP_OOF_ROC_PR.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)
    final_epochs = max(1, int(np.median(selected_epochs)))
    full_preprocessing = fit_preprocessor(X)
    full_x = transform_values(X, full_preprocessing)
    for seed in SEEDS:
        final_model, _, _, final_history = fit_network(
            full_x,
            Y,
            cancer_codes,
            seed=seed,
            epochs=final_epochs,
            description=f"Full development/seed{seed}",
        )
        save_checkpoint(
            final_model,
            full_preprocessing,
            MODEL_DIR / f"development_candidate_seed{seed}.pt",
            extra={
                "seed": seed,
                "epochs": final_epochs,
                "status": "development_candidate",
            },
        )
        pd.DataFrame(final_history).to_csv(
            OUT / f"full_development_history_seed{seed}.tsv",
            sep="\t",
            index=False,
        )
        del final_model
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    manifest = {
        "model": "RCANE-inspired regional multitask classifier",
        "not_original_RCANE_reproduction": True,
        "n_patients": n_patients,
        "n_MTAP_positive": int(y_binary.sum()),
        "n_expression_features": X.shape[1],
        "n_targets": n_targets,
        "feature_names": feature_names,
        "target_Entrez_order": target_ids,
        "MTAP_target_index": MTAP_INDEX,
        "outer_fold_source": str(FOLD_FILE),
        "seeds": SEEDS,
        "region_hidden": REGION_HIDDEN,
        "shared_hidden": SHARED_HIDDEN,
        "cancer_embedding": CANCER_EMBEDDING,
        "dropout": DROPOUT,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "auxiliary_weight": AUXILIARY_WEIGHT,
        "final_epochs": final_epochs,
        "class_weights_used": False,
        "test_evaluated": False,
        "final_threshold_selected": False,
        "evaluation_scope": "Exploratory development OOF; independent external validation required",
    }
    (OUT / "training_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _public_summary(
        globals(), "Regional multitask DL training and seed analysis"
    )


#### Forty-five-feature Elastic Net development - figures and completion records


In [ ]:
@_nb9_registry.register("s26_104")
def _nb9_stage_s26_104():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for name, probability in model_probabilities.items():
        fpr, tpr, _ = roc_curve(y, probability)
        precision, recall, _ = precision_recall_curve(y, probability)
        auc = roc_auc_score(y, probability)
        ap = average_precision_score(y, probability)
        axes[0].plot(
            fpr,
            tpr,
            color=colors[name],
            linewidth=2,
            label=f"{labels[name]}: {auc:.3f}",
        )
        axes[1].plot(
            recall,
            precision,
            color=colors[name],
            linewidth=2,
            label=f"{labels[name]}: {ap:.3f}",
        )
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="0.75")
    axes[1].axhline(y.mean(), linestyle="--", color="0.75")
    axes[0].set(
        xlabel="False positive rate",
        ylabel="Sensitivity",
        title="Development OOF ROC",
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="Development OOF precision–recall",
    )
    for axis in axes:
        axis.legend(frameon=False, fontsize=8)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURE_DIR / f"same_input_ROC_PR.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    full_search = make_search(SEED)
    full_search.fit(X, y)
    full_model = full_search.best_estimator_
    if full_model.named_steps["model"].n_iter_.max() >= MAX_ITER:
        raise RuntimeError(
            "Forty-five-feature Elastic Net development: integrity check failed; condition: input or artifact integrity check."
        )
    joblib.dump(
        {
            "pipeline": full_model,
            "expression_feature_names": feature_names,
            "input_columns": X.columns.tolist(),
            "outcome": "MTAP GISTIC -2 versus non--2",
            "best_params": full_search.best_params_,
        },
        MODEL_DIR / "same_input_ElasticNet_development_candidate.joblib",
    )
    transformed_feature_names = full_model.named_steps[
        "preprocessing"
    ].get_feature_names_out()
    coefficients = pd.DataFrame(
        {
            "feature": transformed_feature_names,
            "coefficient": full_model.named_steps["model"].coef_[0],
        }
    )
    coefficients.to_csv(
        OUT / "full_development_coefficients.tsv", sep="\t", index=False
    )
    manifest = {
        "comparison": "Same expression features and cancer information; MTAP binary endpoint",
        "n_patients": int(len(y)),
        "n_positive": int(y.sum()),
        "n_expression_features": len(feature_names),
        "feature_names": feature_names,
        "outer_fold_source": str(DL_FILE),
        "inner_cv_folds": INNER_FOLDS,
        "parameter_grid": PARAM_GRID,
        "full_development_best_params": full_search.best_params_,
        "DL_retrained": False,
        "test_evaluated": False,
        "threshold_selected": False,
        "bootstrap_iterations": N_BOOTSTRAPS,
        "bootstrap_scope": "Paired, class-stratified, conditional on fixed OOF",
        "interpretation_limit": "Compares complete prediction approaches; does not isolate multitask or ensemble effects",
    }
    (OUT / "comparison_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _public_summary(globals(), "Forty-five-feature Elastic Net development")


#### CPTAC Elastic Net and DL figures


In [ ]:
@_nb9_registry.register("s37_144")
def _nb9_stage_s37_144():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    yy = evaluation["label_MTAP_deep"].to_numpy(dtype=int)
    if len(np.unique(yy)) == 2:
        for name in MODEL_NAMES:
            pp = evaluation[probability_columns[name]].to_numpy()
            fpr, tpr, _ = roc_curve(yy, pp)
            precision, recall, _ = precision_recall_curve(yy, pp)
            auc = roc_auc_score(yy, pp)
            ap = average_precision_score(yy, pp)
            axes[0].plot(
                fpr,
                tpr,
                color=COLORS[name],
                linewidth=2,
                label=f"{name}: AUC={auc:.3f}",
            )
            axes[1].plot(
                recall,
                precision,
                color=COLORS[name],
                linewidth=2,
                label=f"{name}: AP={ap:.3f}",
            )
    axes[0].plot([0, 1], [0, 1], "--", color="#BFB8DA", linewidth=1)
    axes[1].axhline(
        yy.mean(),
        linestyle="--",
        color="#BFB8DA",
        linewidth=1,
        label=f"Positive fraction={yy.mean():.3f}",
    )
    axes[0].set(
        xlabel="False-positive rate",
        ylabel="Sensitivity",
        title="CPTAC external ROC",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="CPTAC external precision–recall",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    for ax in axes:
        ax.legend(frameon=False, fontsize=7, loc="lower left")
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"CPTAC_overall_ROC_PR.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    cancers = list(CANCER_MAP)
    fig, ax = plt.subplots(figsize=(8, 6))
    labels_for_axis = []
    for position, cohort in enumerate(cancers):
        cohort_data = evaluation.loc[evaluation["database_cancer"].eq(cohort)]
        n_pos = int(cohort_data["label_MTAP_deep"].sum())
        n_total = len(cohort_data)
        labels_for_axis.append(f"{cohort}  ({n_pos}/{n_total} positive)")
        for name, offset in [(DL_NAME, -0.12), (EN_NAME, 0.12)]:
            row = metrics_table.loc[
                metrics_table["cohort"].eq(cohort)
                & metrics_table["model"].eq(name)
            ]
            if row.empty:
                continue
            row = row.iloc[0]
            if not np.isfinite(row["AUROC"]):
                continue
            low = row["AUROC_CI95_low"]
            high = row["AUROC_CI95_high"]
            if np.isfinite(low) and np.isfinite(high):
                ax.hlines(
                    position + offset,
                    low,
                    high,
                    color=COLORS[name],
                    linewidth=1.4,
                )
            ax.plot(
                row["AUROC"],
                position + offset,
                "o",
                color=COLORS[name],
                markersize=4,
            )
    ax.axvline(0.5, linestyle="--", color="#BFB8DA", linewidth=1)
    ax.set_yticks(range(len(cancers)))
    ax.set_yticklabels(labels_for_axis)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.02)
    ax.set_xlabel("AUROC with conditional bootstrap 95% CI")
    ax.set_title("CPTAC external validation by cancer")
    for name in MODEL_NAMES:
        ax.plot([], [], "o", color=COLORS[name], label=name)
    ax.legend(frameon=False, loc="lower left", fontsize=8)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"CPTAC_per_cancer_AUROC.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    bins = [0, 0.01, 0.025, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1]
    calibration_rows = []
    fig, ax = plt.subplots(figsize=(5, 4.5))
    for name in MODEL_NAMES:
        calibration_data = pd.DataFrame(
            {
                "probability": evaluation[
                    probability_columns[name]
                ].to_numpy(),
                "label": yy,
            }
        )
        calibration_data["bin"] = pd.cut(
            calibration_data["probability"], bins=bins, include_lowest=True
        )
        model_rows = []
        for interval, group in calibration_data.groupby("bin", observed=True):
            n = len(group)
            n_pos = int(group["label"].sum())
            low, high = wilson_interval(n_pos, n)
            row = {
                "model": name,
                "probability_bin": str(interval),
                "n_patients": n,
                "n_positive": n_pos,
                "mean_probability": group["probability"].mean(),
                "observed_positive_fraction": n_pos / n,
                "Wilson_CI95_low": low,
                "Wilson_CI95_high": high,
            }
            calibration_rows.append(row)
            model_rows.append(row)
        curve = pd.DataFrame(model_rows)
        ax.plot(
            curve["mean_probability"],
            curve["observed_positive_fraction"],
            "-o",
            color=COLORS[name],
            markersize=4,
            label=name,
        )
    ax.plot([0, 1], [0, 1], "--", color="#BFB8DA")
    ax.set(
        xlabel="Mean predicted probability",
        ylabel="Observed DNA-positive fraction",
        title="CPTAC external calibration",
        xlim=(0, 1),
        ylim=(0, 1),
    )
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"CPTAC_external_calibration.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    pd.DataFrame(calibration_rows).to_csv(
        EXTERNAL_OUT / "CPTAC_calibration_bins.tsv", sep="\t", index=False
    )
    save_json(
        {
            "locked_manifest_SHA256": sha256_file(LOCK_MANIFEST),
            "external_input_hashes": external_file_hashes,
            "n_patients_predicted": len(external_predictions),
            "n_patients_evaluated": len(evaluation),
            "n_DNA_positive": int(evaluation["label_MTAP_deep"].sum()),
            "n_bootstraps": N_BOOTSTRAPS,
            "bootstrap_strata": "database_cancer × DNA label",
            "external_preprocessing_fitted": False,
            "thresholds_changed_after_external_evaluation": False,
            "interpretation": "Predicted non-deep is not confirmed DNA-neutral. Bulk validation does not establish scRNA validity.",
        },
        EXTERNAL_OUT / "external_validation_manifest.json",
    )
    _public_summary(
        globals(),
        "Forty-four-member Elastic Net training and frozen CPTAC validation",
    )


#### CPTAC final three-model figures and records


In [ ]:
@_nb9_registry.register("s39_120")
def _nb9_stage_s39_120():
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))
    if len(np.unique(yy)) == 2:
        for name in MODEL_NAMES:
            probability = evaluation[PROBABILITY_COLUMNS[name]].to_numpy(
                dtype=float
            )
            fpr, tpr, _ = roc_curve(yy, probability)
            precision, recall, _ = precision_recall_curve(yy, probability)
            axes[0].plot(
                fpr,
                tpr,
                color=COLORS[name],
                linewidth=2,
                label=f"{name}: AUC={roc_auc_score(yy, probability):.3f}",
            )
            axes[1].plot(
                recall,
                precision,
                color=COLORS[name],
                linewidth=2,
                label=f"{name}: AP={average_precision_score(yy, probability):.3f}",
            )
    axes[0].plot([0, 1], [0, 1], "--", color="#BFB8DA", linewidth=1)
    axes[1].axhline(
        yy.mean(),
        linestyle="--",
        color="#BFB8DA",
        linewidth=1,
        label=f"Positive fraction={yy.mean():.3f}",
    )
    axes[0].set(
        xlabel="False-positive rate",
        ylabel="Sensitivity",
        title="CPTAC external ROC",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="CPTAC external precision–recall",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    for ax in axes:
        ax.legend(frameon=False, fontsize=7, loc="lower left")
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"CPTAC_three_model_ROC_PR.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    save_json(
        {
            "baseline_locked_manifest_SHA256": sha256_file(
                baseline_manifest_file
            ),
            "previous_external_predictions_SHA256": sha256_file(
                EXTERNAL_PREDICTIONS_FILE
            ),
            "external_expression_hashes": external_input_hashes,
            "n_external_evaluated": len(evaluation),
            "n_external_positive": int(evaluation["label_MTAP_deep"].sum()),
            "n_bootstraps": N_BOOTSTRAPS,
            "L2_hyperparameters_selected_using_CPTAC": False,
            "thresholds_selected_using_CPTAC": False,
            "previous_DL_EN_predictions_changed": False,
            "analysis_status": "Additional baseline comparison; previous CPTAC results had already been examined",
        },
        OUT / "comparison_manifest.json",
    )
    _public_summary(
        globals(), "Final TPM three-model comparison and L2 locking"
    )


#### Frozen-model transfer-error diagnostics - figures and completion records


In [ ]:
@_nb9_registry.register("s40_104")
def _nb9_stage_s40_104():
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
    for ax, short_name in zip(axes, ["DL", "L2"]):
        for label, color, name in [
            (0, "#7FACD6", "DNA non-deep"),
            (1, "#A5678E", "DNA deep"),
        ]:
            group = development.loc[development["label_MTAP_deep"].eq(label)]
            ax.scatter(
                group[f"{short_name}_OOF"],
                group[f"{short_name}_final_fit"],
                s=5,
                alpha=0.25,
                color=color,
                label=name,
                rasterized=True,
            )
        ax.plot([0, 1], [0, 1], "--", color="#BFB8DA", linewidth=1)
        ax.set(
            xlabel="OOF probability",
            ylabel="Final-model fitted probability",
            title=f"{short_name}: diagnostic only",
            xlim=(0, 1),
            ylim=(0, 1),
        )
        ax.legend(frameon=False, fontsize=7, markerscale=2)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"development_OOF_vs_final_fit.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 4.3))
    calls = [-2, -1, 0, 1, 2]
    for model, color, offset in [
        (DL_NAME, "#33539E", -0.1),
        (L2_NAME, "#A5678E", 0.1),
    ]:
        table = gistic_errors.loc[
            gistic_errors["cohort"].eq("ALL_CPTAC")
            & gistic_errors["model"].eq(model)
        ].set_index("MTAP_GISTIC_call")
        for position, call in enumerate(calls):
            if call not in table.index:
                continue
            row = table.loc[call]
            ax.vlines(
                position + offset,
                row["Wilson_CI95_low"],
                row["Wilson_CI95_high"],
                color=color,
                linewidth=1.4,
            )
            ax.plot(
                position + offset,
                row["predicted_positive_fraction_at_05"],
                "o",
                color=color,
                markersize=5,
            )
        ax.plot([], [], "o", color=color, label=model)
    ax.set_xticks(range(len(calls)))
    ax.set_xticklabels(
        [
            "-2\nSensitivity",
            "-1\nFalse-positive rate",
            "0\nFalse-positive rate",
            "+1\nFalse-positive rate",
            "+2\nFalse-positive rate",
        ]
    )
    ax.set(
        ylabel="Fraction predicted positive at 0.5",
        title="CPTAC predictions by DNA GISTIC call",
        ylim=(0, 1.02),
    )
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"CPTAC_prediction_rate_by_GISTIC.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    manifest = {
        "analysis": "Frozen model transfer error diagnosis",
        "models": MODEL_NAMES,
        "n_development": len(development),
        "n_external": len(patient_diagnostics),
        "retrained": False,
        "thresholds_changed": False,
        "external_distribution_adjusted": False,
        "final_fit_predictions": "Technical diagnostics only; not independent validation",
        "expression_z": "Diagnostic relative to same-cancer TCGA development distribution; not used as replacement model input",
        "purity": "Not analysed in this step; no purity values inferred or substituted",
        "causal_interpretation": False,
    }
    with (OUT / "diagnostic_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    _public_summary(globals(), "Frozen-model transfer-error diagnostics")


#### CPTAC DNA and prediction review - figures and completion records


In [ ]:
@_nb9_registry.register("s41_96")
def _nb9_stage_s41_96():
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for row_position, (comparison, frame) in enumerate(comparisons.items()):
        y = frame["label_MTAP_deep"].to_numpy(dtype=int)
        if len(np.unique(y)) != 2:
            continue
        for model, column in MODEL_COLUMNS.items():
            p = frame[column].to_numpy()
            fpr, tpr, _ = roc_curve(y, p)
            precision, recall, _ = precision_recall_curve(y, p)
            axes[row_position, 0].plot(
                fpr,
                tpr,
                color=colors[model],
                linewidth=2,
                label=f"{model}: AUC={roc_auc_score(y, p):.3f}",
            )
            axes[row_position, 1].plot(
                recall,
                precision,
                color=colors[model],
                linewidth=2,
                label=f"{model}: AP={average_precision_score(y, p):.3f}",
            )
        axes[row_position, 0].plot(
            [0, 1], [0, 1], "--", color="#BFB8DA", linewidth=1
        )
        axes[row_position, 1].axhline(
            y.mean(),
            linestyle="--",
            color="#BFB8DA",
            label=f"Positive fraction={y.mean():.3f}",
        )
        axes[row_position, 0].set(
            title=f"{comparison}: ROC",
            xlabel="False-positive rate",
            ylabel="Sensitivity",
            xlim=(0, 1),
            ylim=(0, 1.02),
        )
        axes[row_position, 1].set(
            title=f"{comparison}: precision–recall",
            xlabel="Recall",
            ylabel="Precision",
            xlim=(0, 1),
            ylim=(0, 1.02),
        )
        for ax in axes[row_position]:
            ax.legend(frameon=False, fontsize=7)
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            FIGURES / f"fixed_prediction_sensitivity.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.close(fig)
    if review["MTAP_DNA_log2ratio"].notna().any():
        fig, axes = plt.subplots(2, 5, figsize=(15, 6), squeeze=False)
        palette = {
            -2: "#33539E",
            -1: "#7FACD6",
            0: "#BFB8DA",
            1: "#E8B7D4",
            2: "#A5678E",
        }
        for ax, cancer in zip(axes.ravel(), CANCERS):
            frame = review.loc[
                review["database_cancer"].eq(cancer)
                & review["MTAP_DNA_log2ratio"].notna()
            ]
            for call in [-2, -1, 0, 1, 2]:
                group = frame.loc[frame["MTAP_GISTIC_call"].eq(call)]
                if group.empty:
                    continue
                ax.scatter(
                    group["MTAP_DNA_log2ratio"],
                    group["prob_DL"],
                    color=palette[call],
                    s=14,
                    alpha=0.75,
                    label=f"GISTIC {call}",
                )
            ax.axhline(0.5, linestyle="--", color="grey", linewidth=0.8)
            ax.set_title(f"{cancer}, n={len(frame)}")
            ax.set_ylim(-0.02, 1.02)
            ax.set_xlabel("MTAP DNA log2 ratio")
            ax.set_ylabel("DL probability")
        handles, labels = ([], [])
        for ax in axes.ravel():
            h, l = ax.get_legend_handles_labels()
            for handle, label in zip(h, l):
                if label not in labels:
                    handles.append(handle)
                    labels.append(label)
        if handles:
            fig.legend(
                handles,
                labels,
                loc="upper center",
                ncol=5,
                frameon=False,
                fontsize=8,
            )
        fig.tight_layout(rect=(0, 0, 1, 0.93))
        for extension in ["pdf", "png"]:
            fig.savefig(
                FIGURES / f"MTAP_continuous_DNA_vs_DL_probability.{extension}",
                dpi=300,
                bbox_inches="tight",
            )
        plt.close(fig)
    manifest = {
        "fixed_prediction_file": str(PREDICTION_FILE),
        "fixed_prediction_SHA256": sha256_file(PREDICTION_FILE),
        "additional_DNA_source_hashes": source_hashes,
        "models_retrained": False,
        "thresholds_changed": False,
        "comparisons": ["Deep_vs_non_deep", "Deep_vs_neutral"],
        "n_bootstraps_overall": N_BOOTSTRAPS,
        "purity_status": purity_status,
        "continuous_DNA_interpretation": "Retained original log2 ratio; no conversion to absolute copy number or replacement of GISTIC labels",
        "region_review_scope": "MTAP/CDKN2A/CDKN2B only; not a deletion-boundary analysis",
        "evaluation_status": "Post-hoc sensitivity and error diagnosis using fixed predictions",
    }
    with (OUT / "analysis_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    missing_continuous = file_review.loc[
        file_review["continuous_status"].ne("Read_successfully"),
        "database_cancer",
    ].tolist()
    if missing_continuous:
        pass
    _public_summary(globals(), "CPTAC DNA and prediction review")


#### CPTAC fixed comparison summaries - figures and completion records


In [ ]:
@_nb9_registry.register("s42_61")
def _nb9_stage_s42_61():
    for model in MODELS:
        for call in [-2, -1, 0]:
            if call == -2:
                error_name, correct_name = ("FN", "TP")
            else:
                error_name, correct_name = ("FP", "TN")
            fig, axes = plt.subplots(2, 5, figsize=(15, 7), squeeze=False)
            for ax, cancer in zip(axes.ravel(), CANCERS):
                subset = long_data.loc[
                    long_data["model"].eq(model)
                    & long_data["database_cancer"].eq(cancer)
                    & long_data["MTAP_GISTIC_call"].eq(call)
                ]
                for position, group_name, color in [
                    (0, correct_name, "#33539E"),
                    (1, error_name, "#A5678E"),
                ]:
                    values = (
                        subset.loc[
                            subset["prediction_error"].eq(group_name),
                            "MTAP_DNA_log2ratio",
                        ]
                        .dropna()
                        .to_numpy(dtype=float)
                    )
                    if len(values) == 0:
                        continue
                    box = ax.boxplot(
                        [values],
                        positions=[position],
                        widths=0.5,
                        patch_artist=True,
                        showfliers=False,
                        manage_ticks=False,
                    )
                    box["boxes"][0].set_facecolor(color)
                    box["boxes"][0].set_alpha(0.25)
                    box["medians"][0].set_color(color)
                    jitter = plot_rng.uniform(-0.13, 0.13, size=len(values))
                    ax.scatter(
                        position + jitter,
                        values,
                        s=10,
                        color=color,
                        alpha=0.65,
                        rasterized=True,
                    )
                n_correct = int(
                    subset.loc[
                        subset["prediction_error"].eq(correct_name),
                        "MTAP_DNA_log2ratio",
                    ]
                    .notna()
                    .sum()
                )
                n_error = int(
                    subset.loc[
                        subset["prediction_error"].eq(error_name),
                        "MTAP_DNA_log2ratio",
                    ]
                    .notna()
                    .sum()
                )
                row = mtap_results.loc[
                    mtap_results["model"].eq(model)
                    & mtap_results["database_cancer"].eq(cancer)
                    & mtap_results["MTAP_GISTIC_call"].eq(call)
                ]
                if not row.empty and np.isfinite(
                    row.iloc[0]["q_BH_all_new_tests"]
                ):
                    annotation = (
                        f"BH q={row.iloc[0]['q_BH_all_new_tests']:.3g}"
                    )
                else:
                    annotation = "Descriptive"
                ax.set_title(f"{cancer}\n{annotation}", fontsize=10)
                ax.set_xticks([0, 1])
                ax.set_xticklabels(
                    [
                        f"{correct_name}\nn={n_correct}",
                        f"{error_name}\nn={n_error}",
                    ]
                )
                ax.set_xlim(-0.5, 1.5)
                ax.set_ylabel("MTAP DNA log2 ratio")
                if n_correct + n_error == 0:
                    ax.text(
                        0.5,
                        0.5,
                        "No samples",
                        transform=ax.transAxes,
                        ha="center",
                        va="center",
                        color="grey",
                    )
            fig.suptitle(
                f"{model}: MTAP GISTIC {call}; {error_name} versus {correct_name}",
                fontsize=13,
            )
            fig.tight_layout(rect=(0, 0, 1, 0.94))
            filename = (
                f"{model}_MTAP_GISTIC_{call}_{error_name}_vs_{correct_name}"
            )
            for extension in ["pdf", "png"]:
                fig.savefig(
                    FIGURES / f"{filename}.{extension}",
                    dpi=300,
                    bbox_inches="tight",
                )
            plt.close(fig)
    digest = hashlib.sha256()
    with INPUT_FILE.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    manifest = {
        "input_file": str(INPUT_FILE),
        "input_SHA256": digest.hexdigest(),
        "models": list(MODELS),
        "genes_examined": GENES,
        "diagnostic_probability_threshold": DIAGNOSTIC_THRESHOLD,
        "comparison_direction": "Error minus correct",
        "minimum_observed_patients_per_group_for_test": MIN_N_PER_GROUP,
        "test": "Two-sided Mann-Whitney U",
        "multiple_testing": "BH across all valid new tests over models, cancers, GISTIC states and genes",
        "bootstrap": "Within-group patient bootstrap for median differences; conditional on fixed predictions",
        "n_bootstraps": N_BOOTSTRAPS,
        "model_retrained": False,
        "classification_threshold_changed": False,
        "DNA_labels_changed": False,
        "purity_adjusted": False,
    }
    with (OUT / "analysis_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    display_columns = [
        "model",
        "database_cancer",
        "MTAP_GISTIC_call",
        "comparison",
        "n_error_continuous",
        "n_correct_continuous",
        "error_median",
        "correct_median",
        "median_difference_error_minus_correct",
        "median_difference_CI95_low",
        "median_difference_CI95_high",
        "p_value",
        "q_BH_all_new_tests",
        "status",
    ]
    display_rows = mtap_results.loc[
        mtap_results["MTAP_GISTIC_call"].isin([-2, -1, 0])
        & mtap_results["n_error_continuous"].gt(0)
        & mtap_results["n_correct_continuous"].gt(0)
    ]
    significant = results.loc[results["q_BH_all_new_tests"].lt(0.05)]
    if significant.empty:
        pass
    _public_summary(globals(), "CPTAC fixed comparison summaries")


#### GBM frozen bulk DL with a bounded input adapter - figures and completion records


In [ ]:
@_nb9_registry.register("s57_116")
def _nb9_stage_s57_116():
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
    for name in model_names:
        probability = predictions[name].to_numpy()
        fpr, tpr, _ = roc_curve(y, probability)
        precision, recall, _ = precision_recall_curve(y, probability)
        auc = roc_auc_score(y, probability)
        ap = average_precision_score(y, probability)
        axes[0].plot(
            fpr,
            tpr,
            color=colors[name],
            linewidth=1.8,
            label=f"{labels[name]}: {auc:.3f}",
        )
        axes[1].step(
            recall,
            precision,
            where="post",
            color=colors[name],
            linewidth=1.8,
            label=f"{labels[name]}: {ap:.3f}",
        )
    axes[0].plot([0, 1], [0, 1], "--", color="#999999", linewidth=1)
    axes[1].axhline(y.mean(), linestyle="--", color="#999999", linewidth=1)
    axes[0].set(
        xlabel="False positive rate", ylabel="True positive rate", title="ROC"
    )
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–recall")
    for ax in axes:
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.04)
        ax.legend(fontsize=7, loc="lower left")
    fig.tight_layout()
    fig.savefig(FIG_OUT / "three_model_ROC_PR.pdf", bbox_inches="tight")
    fig.savefig(
        FIG_OUT / "three_model_ROC_PR.png", dpi=300, bbox_inches="tight"
    )
    plt.close(fig)
    display = predictions.sort_values(
        ["MTAP_GISTIC_call", "patient_id"]
    ).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(11.5, 4.7))
    positions = np.arange(len(display))
    offsets = [-0.18, 0.0, 0.18]
    for name, offset in zip(model_names, offsets):
        ax.scatter(
            positions + offset,
            display[name],
            color=colors[name],
            label=labels[name],
            s=32,
            alpha=0.9,
        )
    ax.axhline(0.5, linestyle="--", color="#999999", linewidth=1)
    ax.set_xticks(positions)
    ax.set_xticklabels(
        [
            f"{patient}\nGISTIC {call}"
            for patient, call in zip(
                display["patient_id"], display["MTAP_GISTIC_call"]
            )
        ],
        rotation=55,
        ha="right",
        fontsize=7,
    )
    ax.set_ylim(-0.03, 1.03)
    ax.set_ylabel("Model output")
    ax.set_title("Patient-level predictions; threshold 0.5 is descriptive")
    ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, 1.2), ncol=3, fontsize=8
    )
    fig.tight_layout()
    fig.savefig(FIG_OUT / "patient_level_predictions.pdf", bbox_inches="tight")
    fig.savefig(
        FIG_OUT / "patient_level_predictions.png", dpi=300, bbox_inches="tight"
    )
    plt.close(fig)
    for path in [EXPRESSION_FILE, METADATA_FILE, L2_FILE]:
        source_hashes[str(path)] = sha256_file(path)
    manifest = {
        "version": "GBM_snRNA_frozen_bulk_and_8parameter_adapter_v1",
        "n_patients": len(X),
        "n_positive": int(y.sum()),
        "input_unit": "log2(CPM+1)",
        "target": "Patient_MTAP_GISTIC_minus2_vs_non_minus2",
        "feature_order": features,
        "MTAP_target_index": mtap_index,
        "deep_deletion_class_index": 0,
        "frozen_model_seeds": SEEDS,
        "frozen_model_preprocessing": "Checkpoint_TCGA_mean_std",
        "adapter": {
            "type": "Region_affine_on_standardized_input",
            "n_parameters": n_adapter_parameters,
            "shared_across_three_frozen_networks": True,
            "epochs": ADAPTER_EPOCHS,
            "learning_rate": ADAPTER_LR,
            "identity_penalty_weight": IDENTITY_PENALTY,
            "max_log_scale": MAX_LOG_SCALE,
            "max_shift": MAX_SHIFT,
        },
        "evaluation": "Leave_one_patient_out",
        "held_out_patient_used_for_early_stopping": False,
        "hyperparameter_search_performed": False,
        "threshold_search_performed": False,
        "whole_network_fine_tuning_performed": False,
        "final_all_patient_adapter_trained": False,
        "cell_level_genotype_validation_performed": False,
        "annotation_status": "Existing preliminary Tumor labels retained",
        "source_hashes": source_hashes,
    }
    with open(completion_file, "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    coefficient_table = pd.DataFrame(coefficient_rows)
    _public_summary(
        globals(), "GBM frozen bulk DL with a bounded input adapter"
    )


#### GBM L2 cell-number robustness and candidate model - figures and completion records


In [ ]:
@_nb9_registry.register("s59_83")
def _nb9_stage_s59_83():
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8))
    for ax, metric, color in [
        (axes[0], "AUROC", "#33539E"),
        (axes[1], "AP", "#A5678E"),
    ]:
        mean = summary[f"{metric}_mean"].to_numpy()
        lower = summary[f"{metric}_min"].to_numpy()
        upper = summary[f"{metric}_max"].to_numpy()
        ax.errorbar(
            summary["n_cells"],
            mean,
            yerr=np.vstack(
                [np.maximum(0, mean - lower), np.maximum(0, upper - mean)]
            ),
            marker="o",
            color=color,
            capsize=3,
            linewidth=1.5,
        )
        ax.set_xscale("log")
        ax.set_xticks(CELL_NUMBERS)
        ax.set_xticklabels(CELL_NUMBERS)
        ax.set_xlabel("Tumor nuclei per patient")
        ax.set_ylabel(metric)
        ax.set_ylim(-0.02, 1.04)
    fig.suptitle(
        f"Cell-number stability: mean and range across {N_REPEATS} resamples\nRanges are not confidence intervals",
        fontsize=9,
    )
    fig.tight_layout()
    fig.savefig(
        FIG / "cell_number_performance_stability.pdf", bbox_inches="tight"
    )
    fig.savefig(
        FIG / "cell_number_performance_stability.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    def sha256_file(path):
        h = hashlib.sha256()
        with open(path, "rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()

    manifest = {
        "status": "completed",
        "n_patients": len(X),
        "n_positive": int(y.sum()),
        "core_genes": CORE,
        "cell_numbers": CELL_NUMBERS,
        "n_repeats": N_REPEATS,
        "random_seed": RANDOM_SEED,
        "training": "Other_patients_full_pseudobulk",
        "evaluation": "Held_out_patient_subsampled_pseudobulk",
        "sampling": "Cells_without_replacement",
        "UMI_thinning_performed": False,
        "hyperparameter_search": False,
        "threshold_search": False,
        "independent_external_validation": False,
        "final_candidate": str(candidate_path),
        "candidate_SHA256": sha256_file(candidate_path),
        "input_hashes": {
            str(p): sha256_file(p)
            for p in [
                EXPRESSION_FILE,
                METADATA_FILE,
                FEATURE_FILE,
                PREVIOUS_L2,
            ]
        },
    }
    with open(COMPLETE, "w", encoding="utf-8") as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    _public_summary(
        globals(), "GBM L2 cell-number robustness and candidate model"
    )


#### GBM snRNA three-model metric and ROC/PR figures


In [ ]:
@_nb9_registry.register("s62_120")
def _nb9_stage_s62_120():
    fig, axes = plt.subplots(
        1, 3, figsize=(11.5, 4.3), constrained_layout=True
    )
    for ax, metric in zip(axes, FIGURE_METRICS):
        metric_table = (
            performance.loc[performance["metric"].eq(metric)]
            .set_index("model")
            .reindex(MODELS)
        )
        for model_index, model in enumerate(MODELS):
            row = metric_table.loc[model]
            ax.vlines(
                x=model_index,
                ymin=row["CI95_low"],
                ymax=row["CI95_high"],
                color=COLORS[model],
                linewidth=2,
                zorder=2,
            )
            ax.hlines(
                y=[row["CI95_low"], row["CI95_high"]],
                xmin=model_index - 0.07,
                xmax=model_index + 0.07,
                color=COLORS[model],
                linewidth=1.5,
                zorder=2,
            )
            ax.scatter(
                model_index,
                row["estimate"],
                color=COLORS[model],
                edgecolor="white",
                linewidth=0.6,
                s=65,
                zorder=3,
            )
            ax.annotate(
                f"{row['estimate']:.3f}",
                (model_index, row["estimate"]),
                xytext=(0, 9),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
            )
        if metric == "AP":
            baseline = float(y.mean())
        else:
            baseline = 0.5
        ax.axhline(
            baseline, color="#AAAAAA", linestyle="--", linewidth=1, zorder=1
        )
        ax.set_xticks(range(len(MODELS)))
        ax.set_xticklabels([LABELS[model] for model in MODELS], fontsize=8)
        ax.set_xlim(-0.5, len(MODELS) - 0.5)
        ax.set_ylim(0, 1.07)
        ax.set_title(FIGURE_TITLES[metric], pad=10)
        ax.grid(axis="y", alpha=0.15, linewidth=0.7)
    axes[0].set_ylabel("Patient-level performance")
    fig.suptitle(
        f"GBM snRNA-seq patient-level MTAP deep-deletion prediction\nN={len(y)}; MTAP GISTIC −2={int(y.sum())}; non−2={int((y == 0).sum())}",
        fontsize=11,
    )
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUT / f"snRNA_adapter_L2_ElasticNet_performance.{extension}",
            dpi=400,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
    for model in MODELS:
        probability = predictions[model].to_numpy(dtype=float)
        display_label = LABELS[model].replace("\n", " ")
        false_positive_rate, true_positive_rate, _ = roc_curve(y, probability)
        precision, recall, _ = precision_recall_curve(y, probability)
        axes[0].plot(
            false_positive_rate,
            true_positive_rate,
            color=COLORS[model],
            linewidth=2,
            label=f"{display_label}: AUROC={point_estimates[model]['AUROC']:.3f}",
        )
        axes[1].step(
            recall,
            precision,
            where="post",
            color=COLORS[model],
            linewidth=2,
            label=f"{display_label}: AP={point_estimates[model]['AP']:.3f}",
        )
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="#AAAAAA", linewidth=1)
    axes[1].axhline(
        float(y.mean()),
        linestyle="--",
        color="#AAAAAA",
        linewidth=1,
        label=f"Prevalence={y.mean():.3f}",
    )
    axes[0].set(
        title="Receiver operating characteristic",
        xlabel="False-positive rate",
        ylabel="Sensitivity",
    )
    axes[1].set(
        title="Precision–recall curve", xlabel="Recall", ylabel="Precision"
    )
    for ax in axes:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.03)
        ax.grid(alpha=0.15, linewidth=0.7)
        ax.legend(loc="best", frameon=False, fontsize=7.5)
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUT / f"snRNA_adapter_L2_ElasticNet_ROC_PR.{extension}",
            dpi=400,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    manifest = {
        "analysis_name": "GBM_snRNA_adapter_L2_ElasticNet_comparison_v1",
        "expression_file": str(X_PATH),
        "existing_prediction_file": str(PRED_PATH),
        "positive_label": "Patient MTAP GISTIC -2",
        "negative_label": "Patient MTAP GISTIC non--2",
        "analysis_unit": "Patient-level malignant-cell pseudobulk",
        "input_scale": "log2(CPM+1)",
        "model_order_in_figure": [
            "Bulk DL + Input Adapter",
            "Three-gene L2",
            "44-gene Elastic Net",
        ],
        "Bulk_DL_Input_Adapter": {
            "prediction_column": "Region_adapter_DL_LOPO",
            "evaluation": "Leave-one-patient-out",
            "bulk_DL_status": "Frozen",
            "MTAP_target_index": DL_MTAP_TARGET_INDEX,
            "deep_deletion_class_index": DL_DEEP_DELETION_CLASS_INDEX,
            "n_frozen_DL_seeds": DL_FROZEN_SEEDS,
            "adapter_type": "Four-region affine input adapter",
            "n_adapter_regions": N_ADAPTER_REGIONS,
            "n_adapter_parameters": N_ADAPTER_PARAMETERS,
            "adapter_epochs": ADAPTER_EPOCHS,
            "adapter_learning_rate": ADAPTER_LR,
            "identity_penalty": IDENTITY_PENALTY,
            "max_log_scale": MAX_LOG_SCALE,
            "max_shift": MAX_SHIFT,
            "source_manifest_version": "GBM_snRNA_frozen_bulk_and_8parameter_adapter_v1",
        },
        "Three_gene_L2": {
            "prediction_column": "Three_gene_L2_LOPO",
            "evaluation": "Leave-one-patient-out",
            "features": ["MTAP", "CDKN2A", "CDKN2B"],
            "input_scale": "log2(CPM+1)",
        },
        "ElasticNet": {
            "evaluation": "Leave-one-patient-out",
            "n_gene_features": 44,
            "C": C_FIXED,
            "l1_ratio": L1_RATIO_FIXED,
            "penalty": "elasticnet",
            "solver": "saga",
            "class_weight": None,
            "scaler": "StandardScaler fitted only on training patients within each fold",
            "hyperparameter_search": False,
            "max_iter": 20000,
            "tolerance": 1e-05,
        },
        "binary_threshold": THRESHOLD,
        "bootstrap": "Patient-level stratified paired percentile bootstrap",
        "bootstrap_retraining": False,
        "n_bootstrap": N_BOOTSTRAP,
        "seed": SEED,
    }
    with (OUT / "analysis_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
    _public_summary(
        globals(), "GBM three-model comparison including Elastic Net LOPO"
    )


#### TCGA OOF and CPTAC final three-model performance - figures and completion records


In [ ]:
@_nb9_registry.register("s64_68")
def _nb9_stage_s64_68():
    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(11.5, 7.2),
        sharey=True,
        constrained_layout=True,
    )
    for cohort_index, cohort_name in enumerate(COHORT_ORDER):
        cohort_frame = cohort_frames[cohort_name]
        cohort_y = cohort_frame["label_MTAP_deep"].astype(int).to_numpy()
        for metric_index, metric in enumerate(FIGURE_METRICS):
            ax = axes[cohort_index, metric_index]
            metric_table = (
                performance.loc[
                    performance["cohort"].eq(cohort_name)
                    & performance["metric"].eq(metric)
                ]
                .set_index("model")
                .reindex(MODEL_ORDER)
            )
            for model_index, model_name in enumerate(MODEL_ORDER):
                row = metric_table.loc[model_name]
                ax.vlines(
                    x=model_index,
                    ymin=row["CI95_low"],
                    ymax=row["CI95_high"],
                    color=MODEL_COLORS[model_name],
                    linewidth=2,
                    zorder=2,
                )
                ax.hlines(
                    y=[row["CI95_low"], row["CI95_high"]],
                    xmin=model_index - 0.07,
                    xmax=model_index + 0.07,
                    color=MODEL_COLORS[model_name],
                    linewidth=1.4,
                    zorder=2,
                )
                ax.scatter(
                    model_index,
                    row["estimate"],
                    color=MODEL_COLORS[model_name],
                    edgecolor="white",
                    linewidth=0.6,
                    s=65,
                    zorder=3,
                )
                ax.annotate(
                    f"{row['estimate']:.3f}",
                    (model_index, row["estimate"]),
                    xytext=(0, 8),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=7.5,
                )
            if metric == "AP":
                baseline = float(cohort_y.mean())
            else:
                baseline = 0.5
            ax.axhline(
                baseline,
                color="#AAAAAA",
                linestyle="--",
                linewidth=1,
                zorder=1,
            )
            ax.set_xticks(range(len(MODEL_ORDER)))
            ax.set_xticklabels(
                [MODEL_LABELS[model_name] for model_name in MODEL_ORDER],
                fontsize=8,
                rotation=0,
            )
            ax.set_xlim(-0.5, len(MODEL_ORDER) - 0.5)
            ax.set_ylim(0, 1.08)
            ax.grid(axis="y", alpha=0.15, linewidth=0.7)
            if cohort_index == 0:
                ax.set_title(FIGURE_METRIC_LABELS[metric], pad=10)
            if metric_index == 0:
                n_total = len(cohort_y)
                n_positive = int(cohort_y.sum())
                ax.set_ylabel(
                    f"{COHORT_LABELS[cohort_name]}\nN={n_total}; MTAP GISTIC −2={n_positive}\nPerformance",
                    fontsize=8.5,
                )
    fig.suptitle(
        "Bulk RNA-based prediction of MTAP deep deletion", fontsize=12
    )
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUT / f"TCGA_CPTAC_three_model_performance.{extension}",
            dpi=400,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(
        nrows=2, ncols=2, figsize=(9.5, 8), constrained_layout=True
    )
    for cohort_index, cohort_name in enumerate(COHORT_ORDER):
        frame = cohort_frames[cohort_name]
        y = frame["label_MTAP_deep"].astype(int).to_numpy()
        roc_ax = axes[cohort_index, 0]
        pr_ax = axes[cohort_index, 1]
        for model_name in MODEL_ORDER:
            probability_column = PROBABILITY_COLUMNS[model_name]
            probabilities = frame[probability_column].to_numpy(dtype=float)
            fpr, tpr, _ = roc_curve(y, probabilities)
            precision, recall, _ = precision_recall_curve(y, probabilities)
            auroc = point_estimates[cohort_name, model_name]["AUROC"]
            average_precision = point_estimates[cohort_name, model_name]["AP"]
            roc_ax.plot(
                fpr,
                tpr,
                color=MODEL_COLORS[model_name],
                linewidth=2,
                label=f"{MODEL_LABELS[model_name]}: AUROC={auroc:.3f}",
            )
            pr_ax.step(
                recall,
                precision,
                where="post",
                color=MODEL_COLORS[model_name],
                linewidth=2,
                label=f"{MODEL_LABELS[model_name]}: AP={average_precision:.3f}",
            )
        roc_ax.plot(
            [0, 1], [0, 1], color="#AAAAAA", linestyle="--", linewidth=1
        )
        pr_ax.axhline(
            float(y.mean()),
            color="#AAAAAA",
            linestyle="--",
            linewidth=1,
            label=f"Prevalence={y.mean():.3f}",
        )
        roc_ax.set_title(
            f"{COHORT_LABELS[cohort_name]}\nReceiver operating characteristic"
        )
        pr_ax.set_title(
            f"{COHORT_LABELS[cohort_name]}\nPrecision–recall curve"
        )
        roc_ax.set_xlabel("False-positive rate")
        roc_ax.set_ylabel("Sensitivity")
        pr_ax.set_xlabel("Recall")
        pr_ax.set_ylabel("Precision")
        for ax in [roc_ax, pr_ax]:
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1.03)
            ax.grid(alpha=0.15, linewidth=0.7)
            ax.legend(loc="best", frameon=False, fontsize=7.5)
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUT / f"TCGA_CPTAC_three_model_ROC_PR.{extension}",
            dpi=400,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    manifest = {
        "analysis": "Bulk DL versus three-gene L2 versus Elastic Net",
        "target": "MTAP GISTIC -2 versus non--2",
        "TCGA_prediction_file": str(TCGA_PATH),
        "TCGA_prediction_type": "Patient-level out-of-fold predictions",
        "CPTAC_prediction_file": str(CPTAC_PATH),
        "CPTAC_prediction_type": "Frozen-model external predictions",
        "models": {
            "Bulk_DL": {
                "probability_column": "prob_DL",
                "description": "44-input regional multitask deep-learning model; three frozen seeds",
                "color": MODEL_COLORS["Bulk_DL"],
            },
            "Three_gene_L2": {
                "probability_column": "prob_L2",
                "features": ["MTAP", "CDKN2A", "CDKN2B"],
                "color": MODEL_COLORS["Three_gene_L2"],
            },
            "Elastic_Net": {
                "probability_column": "prob_ElasticNet",
                "description": "44-input Elastic Net",
                "color": MODEL_COLORS["Elastic_Net"],
            },
        },
        "classification_threshold": THRESHOLD,
        "bootstrap": {
            "method": "Patient-level stratified percentile bootstrap",
            "paired_between_models": True,
            "paired_between_cohorts": False,
            "model_retraining": False,
            "n_bootstrap": N_BOOTSTRAP,
        },
        "seed": SEED,
        "interpretation": "TCGA results are based on OOF predictions rather than apparent full-training-set predictions. CPTAC results represent frozen-model external evaluation.",
    }
    with (OUT / "analysis_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
    for model_name in MODEL_ORDER:
        pass
    _public_summary(
        globals(), "TCGA OOF and CPTAC final three-model performance"
    )


#### GBM whole-tissue versus malignant-cell L2 comparison - figures and completion records


In [ ]:
@_nb9_registry.register("s66_121")
def _nb9_stage_s66_121():
    fig, ax = plt.subplots(figsize=(6, 5.5))
    for state in [0, 1]:
        sub = pred.loc[pred["label_MTAP_deep"].eq(state)]
        ax.scatter(
            sub["Tumor_LOPO_probability"],
            sub["Whole_tissue_transfer_probability"],
            s=55,
            color=COLORS[state],
            label="DNA deep deletion" if state else "DNA non-deep deletion",
        )
    for patient, row in pred.iterrows():
        ax.annotate(
            patient.replace("C3L-", "L-").replace("C3N-", "N-"),
            (
                row["Tumor_LOPO_probability"],
                row["Whole_tissue_transfer_probability"],
            ),
            xytext=(3, 3),
            textcoords="offset points",
            fontsize=7,
        )
    ax.plot([0, 1], [0, 1], "--", color="#777777", linewidth=1)
    ax.axvline(0.5, color="#BBBBBB", linewidth=0.8)
    ax.axhline(0.5, color="#BBBBBB", linewidth=0.8)
    ax.set(
        xlim=(-0.03, 1.03),
        ylim=(-0.03, 1.03),
        xlabel="Tumor-subset LOPO score",
        ylabel="Whole-tissue transfer score",
        title="Same held-out L2 model; different cell scope",
    )
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figure1_paired_prediction_scores")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, variable, xlabel in [
        (axes[0], "Tumor_cell_fraction", "Tumor cell fraction"),
        (axes[1], "immune_cell_fraction", "Immune cell fraction"),
    ]:
        for state in [0, 1]:
            sub = pred.loc[pred["label_MTAP_deep"].eq(state)]
            ax.scatter(
                sub[variable],
                sub["delta_whole_minus_Tumor"],
                color=COLORS[state],
                s=50,
            )
        ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
        ax.set(xlabel=xlabel, ylabel="Whole-tissue score − Tumor score")
    fig.tight_layout()
    save_figure(fig, "Figure2_prediction_difference_vs_composition")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    for name, column, color in [
        ("Tumor subset", "Tumor_LOPO_probability", "#33539E"),
        ("Whole tissue", "Whole_tissue_transfer_probability", "#C56B9A"),
    ]:
        probability = pred[column].to_numpy()
        fpr, tpr, _ = roc_curve(truth, probability)
        precision, recall, _ = precision_recall_curve(truth, probability)
        axes[0].plot(
            fpr,
            tpr,
            color=color,
            linewidth=2,
            label=f"{name}: AUC={roc_auc_score(truth, probability):.3f}",
        )
        axes[1].plot(
            recall,
            precision,
            color=color,
            linewidth=2,
            label=f"{name}: AP={average_precision_score(truth, probability):.3f}",
        )
    axes[0].plot([0, 1], [0, 1], "--", color="#AAAAAA")
    axes[1].axhline(truth.mean(), linestyle="--", color="#AAAAAA")
    axes[0].set(
        xlabel="False-positive rate", ylabel="Sensitivity", title="ROC"
    )
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision–recall")
    for ax in axes:
        ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save_figure(fig, "Figure3_ROC_PR")
    completion = {
        "status": "WHOLE_TISSUE_TRANSFER_TEST_COMPLETED",
        "n_patients": len(pred),
        "n_DNA_deep_deletion": int(truth.sum()),
        "n_DNA_non_deep_deletion": int((truth == 0).sum()),
        "n_all_QC_nuclei": int(pred["n_all_nuclei"].sum()),
        "n_Tumor_nuclei": int(pred["n_Tumor_nuclei"].sum()),
        "model": "Existing GBM Tumor-subset L2 held-out-patient models",
        "prediction_input": "Whole-tissue QC-filtered raw-UMI pseudobulk",
        "input_scale": "log2(1 + summed_gene_UMI / summed_all_gene_UMI * 1e6)",
        "all_Tumor_inputs_and_predictions_reproduced": True,
        "test_patient_excluded_from_classifier_training": True,
        "n_discordant_at_descriptive_threshold_05": int(
            pred["discordant_at_05"].sum()
        ),
        "threshold_05_is_deployment_validated": False,
        "model_retrained": False,
        "adapter_fitted": False,
        "original_files_modified": False,
        "pan_cancer_transfer_validated": False,
        "sequencing_modality_comparison_possible": False,
        "note": "Patient-level GBM snRNA input-scope comparison. Whole-tissue fractions describe captured QC nuclei, not unbiased tissue composition. Non-deep deletion includes GISTIC -1 and is not synonymous with copy-neutral. Bootstrap intervals condition on fixed predictions and do not include model retraining uncertainty.",
    }
    with open(OUT / "completion.json", "w", encoding="utf-8") as handle:
        json.dump(completion, handle, ensure_ascii=False, indent=2)
    bundle_path = OUT / "GBM_whole_tissue_L2_review.zip"
    with zipfile.ZipFile(
        bundle_path, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in sorted(OUT.rglob("*")):
            if path.is_file() and path != bundle_path:
                archive.write(path, arcname=str(path.relative_to(OUT)))
    _public_summary(
        globals(), "GBM whole-tissue versus malignant-cell L2 comparison"
    )


#### Paired whole-tissue and malignant-cell metric visualization


In [ ]:
@_nb9_registry.register("s67_0")
def _nb9_stage_s67_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        roc_auc_score,
        average_precision_score,
        brier_score_loss,
        confusion_matrix,
    )

    comparison_base = Path(
        str(
            PROJECT_ROOT
            / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer"
        )
    )
    comparison_out = None
    if "OUT" in globals():
        candidate = Path(OUT)
        if (candidate / "patient_paired_predictions.tsv").is_file():
            comparison_out = candidate
    if comparison_out is None:
        candidates = sorted(
            [
                path
                for path in comparison_base.glob(
                    "10_whole_tissue_L2_paired_test_*"
                )
                if (path / "patient_paired_predictions.tsv").is_file()
                and (path / "completion.json").is_file()
            ],
            key=lambda path: path.name,
        )
        if not candidates:
            raise RuntimeError(
                "Paired whole-tissue and malignant-cell metric visualization: integrity check failed; condition: input or artifact integrity check."
            )
        comparison_out = candidates[-1]
    prediction_file = comparison_out / "patient_paired_predictions.tsv"
    comparison_fig_dir = comparison_out / "figures"
    comparison_fig_dir.mkdir(parents=True, exist_ok=True)
    paired = pd.read_csv(prediction_file, sep="\t", dtype={"patient_id": str})
    required_columns = [
        "patient_id",
        "label_MTAP_deep",
        "Tumor_LOPO_probability",
        "Whole_tissue_transfer_probability",
    ]
    missing_columns = sorted(set(required_columns) - set(paired.columns))
    if missing_columns:
        raise RuntimeError(
            "Paired whole-tissue and malignant-cell metric visualization: integrity check failed; condition: input or artifact integrity check."
        )
    if (
        len(paired) != 16
        or paired["patient_id"].isna().any()
        or (not paired["patient_id"].is_unique)
    ):
        raise RuntimeError(
            "Paired whole-tissue and malignant-cell metric visualization: integrity check failed; condition: input or artifact integrity check."
        )
    for column in required_columns[1:]:
        paired[column] = pd.to_numeric(paired[column], errors="raise")
    truth = paired["label_MTAP_deep"].to_numpy(dtype=int)
    if (
        not paired["label_MTAP_deep"].isin([0, 1]).all()
        or int(truth.sum()) != 8
    ):
        raise RuntimeError(
            "Paired whole-tissue and malignant-cell metric visualization: integrity check failed; condition: input or artifact integrity check."
        )
    PLOT_THRESHOLD = 0.5
    scope_columns = {
        "Whole tissue": "Whole_tissue_transfer_probability",
        "Tumour cell": "Tumor_LOPO_probability",
    }
    scope_colors = {"Whole tissue": "#33539E", "Tumour cell": "#A5678E"}

    def safe_divide(a, b):
        return float(a / b) if b else np.nan

    rows = []
    for scope, column in scope_columns.items():
        probability = paired[column].to_numpy(dtype=float)
        if (
            not np.isfinite(probability).all()
            or np.any(probability < 0)
            or np.any(probability > 1)
        ):
            raise RuntimeError(
                "Paired whole-tissue and malignant-cell metric visualization: integrity check failed; condition: input or artifact integrity check."
            )
        predicted = (probability >= PLOT_THRESHOLD).astype(int)
        tn, fp, fn, tp = confusion_matrix(
            truth, predicted, labels=[0, 1]
        ).ravel()
        sensitivity = safe_divide(tp, tp + fn)
        specificity = safe_divide(tn, tn + fp)
        rows.append(
            {
                "Input": scope,
                "n_patients": len(truth),
                "threshold": PLOT_THRESHOLD,
                "AUROC": roc_auc_score(truth, probability),
                "AP": average_precision_score(truth, probability),
                "Balanced accuracy": (sensitivity + specificity) / 2,
                "Sensitivity": sensitivity,
                "Specificity": specificity,
                "PPV": safe_divide(tp, tp + fp),
                "NPV": safe_divide(tn, tn + fn),
                "Accuracy": safe_divide(tp + tn, len(truth)),
                "Brier": brier_score_loss(truth, probability),
                "TP": int(tp),
                "FP": int(fp),
                "FN": int(fn),
                "TN": int(tn),
            }
        )
    bar_performance = pd.DataFrame(rows).set_index("Input")
    bar_performance.to_csv(
        comparison_out / "Whole_vs_Tumour_L2_barplot_source_data.tsv",
        sep="\t",
        index=True,
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.linewidth": 0.9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.unicode_minus": False,
        }
    )
    metrics_to_plot = [
        "AUROC",
        "AP",
        "Balanced accuracy",
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
        "Accuracy",
    ]
    metric_labels = [
        "AUROC",
        "AP",
        "Balanced\naccuracy",
        "Sensitivity",
        "Specificity",
        "PPV",
        "NPV",
        "Accuracy",
    ]


#### Paired whole-tissue and malignant-cell metric visualization - figures and completion records


In [ ]:
@_nb9_registry.register("s67_32")
def _nb9_stage_s67_32():
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(13.6, 4.6),
        gridspec_kw={"width_ratios": [6.5, 1.35], "wspace": 0.28},
        facecolor="white",
    )
    bar_width = 0.34
    x_positions = np.arange(len(metrics_to_plot))
    for scope_index, scope in enumerate(["Whole tissue", "Tumour cell"]):
        offset = -bar_width / 2 if scope_index == 0 else bar_width / 2
        values = bar_performance.loc[scope, metrics_to_plot].to_numpy(
            dtype=float
        )
        bars = axes[0].bar(
            x_positions + offset,
            values,
            width=bar_width,
            color=scope_colors[scope],
            edgecolor="white",
            linewidth=0.7,
            label=scope,
            zorder=3,
        )
        for bar, value in zip(bars, values):
            if np.isfinite(value):
                axes[0].annotate(
                    f"{value:.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, value),
                    xytext=(0, 5),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=8.1,
                    color="#202020",
                    clip_on=False,
                )
            else:
                axes[0].text(
                    bar.get_x() + bar.get_width() / 2,
                    0.025,
                    "NA",
                    ha="center",
                    va="bottom",
                    fontsize=8.1,
                    color="#666666",
                )
    axes[0].set_xticks(x_positions)
    axes[0].set_xticklabels(metric_labels, fontsize=9)
    axes[0].set_ylim(0, 1.16)
    axes[0].set_yticks(np.arange(0, 1.01, 0.2))
    axes[0].set_ylabel("Performance")
    axes[0].set_title(
        "A  Discrimination and classification performance",
        loc="left",
        fontsize=11,
        fontweight="bold",
        pad=15,
    )
    brier_values = bar_performance.loc[
        ["Whole tissue", "Tumour cell"], "Brier"
    ].to_numpy(dtype=float)
    bars = axes[1].bar(
        [0, 1],
        brier_values,
        width=0.58,
        color=[scope_colors["Whole tissue"], scope_colors["Tumour cell"]],
        edgecolor="white",
        linewidth=0.7,
        zorder=3,
    )
    for bar, value in zip(bars, brier_values):
        axes[1].annotate(
            f"{value:.3f}",
            xy=(bar.get_x() + bar.get_width() / 2, value),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
            color="#202020",
        )
    brier_top = max(0.2, float(np.max(brier_values)) * 1.55)
    axes[1].set_ylim(0, brier_top)
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels(["Whole\ntissue", "Tumour\ncell"], fontsize=9)
    axes[1].set_ylabel("Brier score (lower is better)")
    axes[1].set_title(
        "B  Probability error",
        loc="left",
        fontsize=11,
        fontweight="bold",
        pad=15,
    )
    for axis in axes:
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        axis.spines["left"].set_color("#444444")
        axis.spines["bottom"].set_color("#444444")
        axis.tick_params(axis="x", length=0, pad=8)
        axis.tick_params(axis="y", direction="out", length=3, width=0.8)
        axis.yaxis.grid(True, color="#E4E6EA", linewidth=0.7, zorder=0)
        axis.set_axisbelow(True)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.53, 0.91),
        ncol=2,
        frameon=False,
        fontsize=10,
        handlelength=1.6,
        columnspacing=2.5,
    )
    fig.suptitle(
        "L2 prediction: whole-tissue versus tumour-cell pseudobulk",
        fontsize=13,
        fontweight="bold",
        y=0.99,
    )
    fig.text(
        0.08,
        0.025,
        "Same 16 patients (8 DNA deep deletion; 8 non-deep deletion). The same held-out model was applied to both inputs for each patient.\nClassification threshold = 0.50. Bars show point estimates; AUROC, AP and Brier are threshold-independent. Panel B uses a separate y-axis scale.",
        ha="left",
        va="bottom",
        fontsize=8.2,
        color="#505050",
        linespacing=1.5,
    )
    fig.subplots_adjust(left=0.065, right=0.985, bottom=0.24, top=0.73)
    figure_stem = "Figure4_L2_Whole_vs_Tumour_grouped_bars_threshold_050"
    for extension in ["pdf", "png"]:
        fig.savefig(
            comparison_fig_dir / f"{figure_stem}.{extension}",
            dpi=400,
            bbox_inches="tight",
            facecolor="white",
        )
    from IPython.display import display

    plt.show()
    plt.close(fig)
    _public_summary(
        globals(),
        "Paired whole-tissue and malignant-cell metric visualization",
    )


#### Whole-tissue three-model comparison - figures and completion records


In [ ]:
@_nb9_registry.register("s68_162")
def _nb9_stage_s68_162():
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
    for ax, metric in zip(axes, ["AUROC", "AP", "Brier"]):
        for i, model in enumerate(MODEL_NAMES):
            row = whole_performance.loc[model]
            point = float(row[metric])
            low = float(row[f"{metric}_CI_low"])
            high = float(row[f"{metric}_CI_high"])
            ax.vlines(i, low, high, color=COLORS[model], linewidth=2)
            ax.hlines(
                [low, high],
                i - 0.07,
                i + 0.07,
                color=COLORS[model],
                linewidth=1.5,
            )
            ax.scatter(i, point, s=65, color=COLORS[model], zorder=3)
            ax.annotate(
                f"{point:.3f}",
                (i, high),
                xytext=(0, 7),
                textcoords="offset points",
                ha="center",
                fontsize=9,
            )
        ax.set_xticks(range(3))
        ax.set_xticklabels(
            ["DL + adapter", "L2", "Elastic Net"], rotation=20, ha="right"
        )
        ax.set_xlim(-0.5, 2.5)
        ax.set_title(metric, fontweight="bold")
        if metric in {"AUROC", "AP"}:
            ax.set_ylim(0, 1.12)
            ax.set_yticks(np.linspace(0, 1, 6))
        else:
            upper = max(
                0.15, float(whole_performance["Brier_CI_high"].max()) * 1.3
            )
            ax.set_ylim(0, upper)
    fig.suptitle(
        "Whole-tissue GBM snRNA pseudobulk\nPatient-held-out prediction; n = 16",
        fontsize=13,
        fontweight="bold",
    )
    fig.tight_layout(rect=[0, 0, 1, 0.88])
    savefig(fig, "Figure1_Whole_tissue_three_model_performance")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for model in MODEL_NAMES:
        score = score_arrays["Whole", model]
        fpr, tpr, _ = roc_curve(y, score)
        precision, recall, _ = precision_recall_curve(y, score)
        axes[0].plot(
            fpr,
            tpr,
            color=COLORS[model],
            linewidth=2,
            label=f"{DISPLAY[model]}: {points['Whole', model]['AUROC']:.3f}",
        )
        axes[1].plot(
            recall,
            precision,
            color=COLORS[model],
            linewidth=2,
            label=f"{DISPLAY[model]}: {points['Whole', model]['AP']:.3f}",
        )
    axes[0].plot([0, 1], [0, 1], "--", color="#AAAAAA")
    axes[1].axhline(y.mean(), linestyle="--", color="#AAAAAA")
    axes[0].set(
        xlabel="False-positive rate",
        ylabel="Sensitivity",
        title="Whole-tissue ROC",
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="Whole-tissue precision–recall",
    )
    for ax in axes:
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.04)
        ax.legend(frameon=False, fontsize=8, loc="lower left")
    fig.tight_layout()
    savefig(fig, "Figure2_Whole_tissue_three_model_ROC_PR")
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, model in zip(axes, MODEL_NAMES):
        for dna, color, label in [
            (0, "#33539E", "DNA non-deep deletion"),
            (1, "#C56B9A", "DNA deep deletion"),
        ]:
            mask = y == dna
            ax.scatter(
                predictions.loc[mask, f"{model}_Tumor"],
                predictions.loc[mask, f"{model}_Whole"],
                s=45,
                color=color,
                label=label,
            )
        ax.plot([0, 1], [0, 1], "--", color="#AAAAAA")
        ax.axvline(0.5, color="#DDDDDD", linewidth=0.8)
        ax.axhline(0.5, color="#DDDDDD", linewidth=0.8)
        ax.set(
            xlim=(-0.03, 1.03),
            ylim=(-0.03, 1.03),
            xlabel="Tumor-subset score",
            ylabel="Whole-tissue score",
            title=DISPLAY[model],
        )
    axes[0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    savefig(fig, "Figure3_Tumor_vs_Whole_paired_scores")
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
    for ax, model in zip(axes, MODEL_NAMES):
        cm = confusion_matrix(
            y, score_arrays["Whole", model] >= THRESHOLD, labels=[0, 1]
        )
        ax.imshow(cm, cmap="Blues", vmin=0, vmax=8)
        for row in range(2):
            for col in range(2):
                ax.text(
                    col,
                    row,
                    str(cm[row, col]),
                    ha="center",
                    va="center",
                    fontsize=15,
                    color="white" if cm[row, col] >= 5 else "black",
                )
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Non-deep", "Deep"])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["Non-deep", "Deep"])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("DNA reference")
        ax.set_title(DISPLAY[model], fontsize=10)
    fig.suptitle(
        "Whole-tissue predictions at descriptive threshold 0.5", fontsize=11
    )
    fig.tight_layout(rect=[0, 0, 1, 0.9])
    savefig(fig, "Figure4_Whole_tissue_confusion_matrices")
    completion = {
        "status": "WHOLE_TISSUE_THREE_MODEL_COMPARISON_COMPLETED",
        "n_patients": len(patient_ids),
        "n_DNA_deep_deletion": int(y.sum()),
        "n_DNA_non_deep_deletion": int((y == 0).sum()),
        "n_all_QC_nuclei": int(audit["n_all_nuclei"].sum()),
        "n_Tumor_nuclei": int(audit["n_Tumor_nuclei"].sum()),
        "all_previous_Tumor_predictions_reproduced": True,
        "models": MODEL_NAMES,
        "L2": "Existing patient-held-out Tumor-trained models",
        "DL": "Frozen bulk ensemble plus existing patient-held-out Tumor-trained adapters",
        "ElasticNet": "Reproduced original Tumor-trained LOPO; C=1, l1_ratio=0.5",
        "whole_tissue_used_for_training": False,
        "held_out_patient_used_for_training": False,
        "hyperparameter_search_performed": False,
        "threshold": THRESHOLD,
        "threshold_is_validated_for_deployment": False,
        "bootstrap_repeats": N_BOOTSTRAP,
        "bootstrap_scope": "Paired stratified patient resampling conditional on fixed predictions; does not include model retraining uncertainty",
        "original_files_modified": False,
        "pan_cancer_validation_completed": False,
    }
    with open(OUT / "completion.json", "w", encoding="utf-8") as handle:
        json.dump(completion, handle, ensure_ascii=False, indent=2)
    bundle = OUT / "GBM_whole_tissue_three_model_review.zip"
    with zipfile.ZipFile(
        bundle, "w", compression=zipfile.ZIP_DEFLATED
    ) as archive:
        for path in sorted(OUT.rglob("*")):
            if path.is_file() and path != bundle:
                archive.write(path, arcname=str(path.relative_to(OUT)))
    _public_summary(globals(), "Whole-tissue three-model comparison")


#### GBM threshold sensitivity at 0.50, 0.75, and 0.80 - figures and completion records


In [ ]:
@_nb9_registry.register("s70_77")
def _nb9_stage_s70_77():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for offset, metric, label, color in [
        (-0.08, "sensitivity", "Sensitivity", "#4477AA"),
        (0.08, "specificity", "Specificity", "#CC6677"),
    ]:
        center = metrics[metric].to_numpy()
        lower = metrics[f"{metric}_CI_low"].to_numpy()
        upper = metrics[f"{metric}_CI_high"].to_numpy()
        axes[0].errorbar(
            x + offset,
            center,
            yerr=np.vstack([center - lower, upper - center]),
            fmt="o",
            capsize=4,
            color=color,
            label=label,
        )
    axes[0].set_xticks(x, [f"{t:.2f}" for t in metrics.threshold])
    axes[0].set_ylim(-0.03, 1.05)
    axes[0].set_xlabel("Threshold")
    axes[0].set_ylabel("Estimate with exact 95% CI")
    axes[0].legend(frameon=False)
    axes[0].set_title("Sensitivity / specificity")
    width = 0.35
    axes[1].bar(
        x - width / 2,
        metrics["FP"],
        width,
        label="False positive",
        color="#CC6677",
    )
    axes[1].bar(
        x + width / 2,
        metrics["FN"],
        width,
        label="False negative",
        color="#4477AA",
    )
    axes[1].set_xticks(x, [f"{t:.2f}" for t in metrics.threshold])
    axes[1].set_xlabel("Threshold")
    axes[1].set_ylabel("Patients")
    axes[1].set_title("Classification errors")
    axes[1].legend(frameon=False)
    axes[1].yaxis.get_major_locator().set_params(integer=True)
    rng = np.random.default_rng(42)
    for label, color, name in [
        (0, "#4477AA", "DNA non-deep deletion"),
        (1, "#CC6677", "DNA deep deletion"),
    ]:
        selected = y == label
        jitter = rng.uniform(-0.07, 0.07, selected.sum())
        axes[2].scatter(
            np.full(selected.sum(), label) + jitter,
            p[selected],
            s=40,
            color=color,
            label=name,
        )
    for threshold, color in zip(THRESHOLDS, colors):
        axes[2].axhline(
            threshold,
            linestyle="--",
            linewidth=1,
            color=color,
            label=f"Threshold {threshold:.2f}",
        )
    axes[2].set_xticks([0, 1], ["DNA non-deep", "DNA deep"])
    axes[2].set_ylim(-0.03, 1.05)
    axes[2].set_ylabel("LOPO model score")
    axes[2].set_title("Patient-level held-out scores")
    axes[2].legend(frameon=False, fontsize=7)
    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    fig.tight_layout()
    fig.savefig(
        OUT / "04_three_threshold_comparison.pdf",
        dpi=PDF_DPI,
        bbox_inches="tight",
    )
    fig.savefig(
        OUT / "04_three_threshold_comparison.png",
        dpi=PDF_DPI,
        bbox_inches="tight",
    )
    plt.show()
    metrics.to_csv(
        OUT / "01_threshold_metrics.tsv", sep="\t", index=False, na_rep=""
    )
    patient_results.to_csv(
        OUT / "02_patient_predictions_at_three_thresholds.tsv",
        sep="\t",
        index=False,
    )
    changed.to_csv(
        OUT / "03_patients_with_changed_classification.tsv",
        sep="\t",
        index=False,
    )
    completion = {
        "status": "EXPLORATORY_LOPO_THRESHOLD_COMPARISON_COMPLETED",
        "validation_file": str(VALIDATION_FILE),
        "validation_file_sha256": sha256_file(VALIDATION_FILE),
        "model_file": str(MODEL_FILE),
        "model_sha256": sha256_file(MODEL_FILE),
        "n_patients": len(y),
        "n_DNA_deep_deletion": n_positive,
        "n_DNA_non_deep_deletion": n_negative,
        "thresholds_compared": THRESHOLDS,
        "best_balanced_accuracy_thresholds": ba_best.threshold.tolist(),
        "candidate_specificity_target": TARGET_SPECIFICITY,
        "high_specificity_candidate_thresholds": high_specificity_candidates,
        "deployment_threshold_locked_by_this_analysis": False,
        "BLCA_transfer_validated_by_this_analysis": False,
        "model_retrained": False,
        "prediction_master_modified": False,
    }
    (OUT / "completion.json").write_text(
        json.dumps(completion, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    _public_summary(
        globals(), "GBM threshold sensitivity at 0.50, 0.75, and 0.80"
    )


#### TCGA holdout ROC, precision-recall, and metric figures


In [ ]:
@_nb9_registry.register("s72_152")
def _nb9_stage_s72_152():
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    for name in MODEL_ORDER:
        probability = probabilities[name]
        fpr, tpr, _ = roc_curve(y, probability)
        precision, recall, _ = precision_recall_curve(y, probability)
        axes[0].plot(
            fpr,
            tpr,
            color=COLORS[name],
            lw=2,
            label=f"{name}: AUROC={point[name]['AUROC']:.3f}",
        )
        axes[1].plot(
            recall,
            precision,
            color=COLORS[name],
            lw=2,
            label=f"{name}: AP={point[name]['AP']:.3f}",
        )
    axes[0].plot([0, 1], [0, 1], "--", color="0.65", lw=1)
    axes[1].axhline(y.mean(), ls="--", color="0.65", lw=1)
    axes[0].set(
        xlabel="False-positive rate",
        ylabel="True-positive rate",
        title="TCGA original holdout: ROC",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    axes[1].set(
        xlabel="Recall",
        ylabel="Precision",
        title="TCGA original holdout: precision-recall",
        xlim=(0, 1),
        ylim=(0, 1.02),
    )
    for ax in axes:
        ax.legend(frameon=False, fontsize=8)
    fig.suptitle(
        f"Frozen TPM models | n={len(y):,}; MTAP deep deletion={int(y.sum())}"
    )
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUT / f"10_TCGA_holdout_three_model_ROC_PR.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    plot_metrics = ["AUROC", "AP", "Balanced_accuracy", "Brier"]
    fig, axes = plt.subplots(1, 4, figsize=(13.5, 4))
    for ax, metric in zip(axes, plot_metrics):
        rows = (
            performance.loc[performance["metric"].eq(metric)]
            .set_index("model")
            .loc[MODEL_ORDER]
        )
        for i, name in enumerate(MODEL_ORDER):
            row = rows.loc[name]
            ax.errorbar(
                i,
                row["estimate"],
                yerr=np.array(
                    [
                        [max(0, row["estimate"] - row["CI95_low"])],
                        [max(0, row["CI95_high"] - row["estimate"])],
                    ]
                ),
                fmt="o",
                color=COLORS[name],
                markersize=7,
                capsize=4,
                lw=1.5,
            )
        ax.set_xticks(range(3))
        ax.set_xticklabels(["L2", "Elastic Net", "DL"], rotation=25)
        ax.set_title(metric.replace("_", " "))
        if metric == "Brier":
            upper_limit = min(
                1.0, max(0.15, float(rows["CI95_high"].max()) * 1.25)
            )
            ax.set_ylim(0, upper_limit)
        else:
            ax.set_ylim(0, 1.03)
        ax.grid(axis="y", alpha=0.18)
    fig.suptitle(
        "TCGA original holdout: frozen-model performance\n95% stratified patient-bootstrap intervals; classification threshold=0.50"
    )
    fig.tight_layout()
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUT / f"11_TCGA_holdout_three_model_performance.{extension}",
            dpi=300,
            bbox_inches="tight",
        )
    plt.show()
    plt.close(fig)
    model_hashes_after = {str(p): sha256_file(p) for p in model_files}
    if model_hashes_before != model_hashes_after:
        raise RuntimeError(
            "Final three-model evaluation on the original TCGA holdout: integrity check failed; condition: input or artifact integrity check."
        )
    manifest = {
        "analysis": "Final_TPM_models_on_original_TCGA_20pct_holdout",
        "n_original_test": len(test),
        "n_original_positive": int(test["label_MTAP_deep"].sum()),
        "n_evaluable": len(y),
        "n_evaluable_positive": int(y.sum()),
        "n_excluded": len(test) - len(y),
        "development_test_overlap": 0,
        "positive_definition": "MTAP_GISTIC_call == -2",
        "negative_definition": "MTAP_GISTIC_call in [-1,0,1,2]",
        "input_unit": "log2(TPM+1)",
        "source_unit": "log2(TPM+0.001)",
        "multiple_primary_samples": "Mean TPM before log2(TPM+1)",
        "model_refitting": False,
        "threshold_selection": False,
        "classification_threshold": THRESHOLD,
        "n_bootstrap": N_BOOTSTRAP,
        "bootstrap": "Outcome-stratified paired patient bootstrap",
        "bootstrap_retraining": False,
        "source_row_identity": "Unique match to saved development-source expression profiles; no test labels used",
        "n_development_reference_samples": len(reference_ids),
        "model_file_sha256": model_hashes_before,
        "source_file": str(TOIL_FILE),
        "source_file_size_bytes": TOIL_FILE.stat().st_size,
        "source_file_mtime_ns": TOIL_FILE.stat().st_mtime_ns,
        "metadata_sha256": sha256_file(META_FILE),
        "feature_spec_sha256": sha256_file(SPEC_FILE),
        "interpretation": "Original holdout previously evaluated for earlier model versions; not described as a never-inspected test set",
    }
    with (OUT / "12_analysis_manifest.json").open("w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    formatted = performance.copy()
    formatted["result"] = formatted.apply(
        lambda r: f"{r['estimate']:.4f} ({r['CI95_low']:.4f}–{r['CI95_high']:.4f})",
        axis=1,
    )
    _public_summary(
        globals(), "Final three-model evaluation on the original TCGA holdout"
    )


#### GSE327167 ROC/PR and missing-feature records


In [ ]:
@_nb9_registry.register("s74_135")
def _nb9_stage_s74_135():
    fig, axes = plt.subplots(2, 2, figsize=(10.2, 8.2))
    plot_evaluation_sets = ["Whole_31", "Malignant_ge50_primary"]
    for row_index, evaluation_name in enumerate(plot_evaluation_sets):
        frame = evaluation_sets[evaluation_name].sort_values("patient_id")
        y = frame["OGM_deep_deletion_label"].to_numpy(dtype=int)
        for model in MODEL_ORDER:
            probability = frame[model].to_numpy(dtype=float)
            fpr, tpr, _ = roc_curve(y, probability)
            precision, recall, _ = precision_recall_curve(y, probability)
            axes[row_index, 0].plot(
                fpr,
                tpr,
                color=MODEL_COLORS[model],
                linewidth=2,
                label=f"{model} (AUC={roc_auc_score(y, probability):.3f})",
            )
            axes[row_index, 1].plot(
                recall,
                precision,
                color=MODEL_COLORS[model],
                linewidth=2,
                label=f"{model} (AP={average_precision_score(y, probability):.3f})",
            )
        axes[row_index, 0].plot([0, 1], [0, 1], "--", color="#999999")
        axes[row_index, 1].axhline(y.mean(), linestyle="--", color="#999999")
        axes[row_index, 0].set(
            xlim=(0, 1),
            ylim=(0, 1.02),
            xlabel="False positive rate",
            ylabel="True positive rate",
            title=f"{evaluation_name}: ROC (n={len(frame)})",
        )
        axes[row_index, 1].set(
            xlim=(0, 1),
            ylim=(0, 1.02),
            xlabel="Recall",
            ylabel="Precision",
            title=f"{evaluation_name}: precision–recall (n={len(frame)})",
        )
        axes[row_index, 0].legend(frameon=False, loc="lower right", fontsize=8)
        axes[row_index, 1].legend(frameon=False, loc="lower left", fontsize=8)
    fig.tight_layout()
    fig.savefig(
        FIGURE_DIR / "GSE327167_three_model_ROC_PR_missing_feature_aware.pdf",
        bbox_inches="tight",
        facecolor="white",
    )
    fig.savefig(
        FIGURE_DIR / "GSE327167_three_model_ROC_PR_missing_feature_aware.png",
        dpi=400,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.show()
    plt.close(fig)
    imputation_rows = []
    for gene in missing_features:
        imputation_rows.append(
            {
                "feature": gene,
                "L2_handling": "Not_used_noncore_feature",
                "ElasticNet_handling": "GBM16_training_median_log2CPM_plus_1",
                "ElasticNet_imputed_value": float(en_imputation_values[gene]),
                "DL_handling": "Checkpoint_specific_training_mean",
                "DL_standardized_value": 0.0,
            }
        )
    pd.DataFrame(imputation_rows).to_csv(
        OUTPUT_DIR / "06_missing_feature_imputation_audit.tsv",
        sep="\t",
        index=False,
    )
    manifest = {
        "analysis": "GSE327167_31patient_missing_feature_aware_external_validation",
        "missing_features": missing_features,
        "n_missing_features": len(missing_features),
        "L2_missing_handling": "No effect if missing features are noncore",
        "ElasticNet_missing_handling": "GBM16 training median on log2(CPM+1) scale",
        "DL_missing_handling": "Each checkpoint training mean; standardized value zero",
        "GSE327167_used_for_training_or_tuning": False,
        "case_27524_excluded": True,
        "whole_cohort": "31 patients; 4 positive and 27 negative",
    }
    with (OUTPUT_DIR / "07_missing_feature_aware_manifest.json").open(
        "w", encoding="utf-8"
    ) as handle:
        json.dump(manifest, handle, ensure_ascii=False, indent=2)
    for name, frame in evaluation_sets.items():
        pass
    _public_summary(
        globals(),
        "GSE327167 missing-feature handling, inference, and evaluation",
    )


#### GSE327167 performance and confusion-count heatmaps at 0.50


In [ ]:
@_nb9_registry.register("s75_53")
def _nb9_stage_s75_53():
    figure = plt.figure(figsize=(17.2, 9.4), facecolor="white")
    grid = figure.add_gridspec(
        nrows=2,
        ncols=4,
        left=0.12,
        right=0.93,
        bottom=0.12,
        top=0.9,
        wspace=0.18,
        hspace=0.3,
    )
    normalizer = Normalize(vmin=0.0, vmax=1.0)
    panel_letters = list("ABCDEFGH")
    for panel_index, metric in enumerate(METRICS):
        axis = figure.add_subplot(grid[panel_index // 4, panel_index % 4])
        raw_matrix, low_matrix, high_matrix = metric_matrix(metric)
        color_matrix = favourable_color_matrix(metric, raw_matrix)
        masked_color_matrix = np.ma.masked_invalid(color_matrix)
        axis.imshow(
            masked_color_matrix,
            cmap=PERFORMANCE_CMAP,
            norm=normalizer,
            aspect="auto",
            interpolation="nearest",
        )
        for row_index in range(len(EVALUATION_ORDER)):
            for column_index in range(len(MODELS)):
                value = raw_matrix[row_index, column_index]
                low = low_matrix[row_index, column_index]
                high = high_matrix[row_index, column_index]
                color_value = color_matrix[row_index, column_index]
                if np.isfinite(color_value):
                    rgba = PERFORMANCE_CMAP(normalizer(color_value))
                    text_color = contrast_text_color(rgba)
                else:
                    text_color = "#505050"
                axis.text(
                    column_index,
                    row_index,
                    format_metric_annotation(value, low, high),
                    ha="center",
                    va="center",
                    fontsize=8.1,
                    color=text_color,
                    linespacing=1.25,
                )
        axis.set_xticks(np.arange(len(MODELS)))
        axis.set_xticklabels(
            [MODEL_LABELS[model] for model in MODELS], fontsize=8.5
        )
        axis.set_yticks(np.arange(len(EVALUATION_ORDER)))
        if panel_index % 4 == 0:
            axis.set_yticklabels(evaluation_axis_labels, fontsize=8.5)
        else:
            axis.set_yticklabels([])
        axis.set_title(
            METRIC_TITLES[metric], fontsize=11, fontweight="bold", pad=9
        )
        axis.text(
            -0.12,
            1.08,
            panel_letters[panel_index],
            transform=axis.transAxes,
            ha="left",
            va="bottom",
            fontsize=16,
            fontweight="bold",
        )
        axis.set_xticks(np.arange(-0.5, len(MODELS), 1), minor=True)
        axis.set_yticks(np.arange(-0.5, len(EVALUATION_ORDER), 1), minor=True)
        axis.grid(which="minor", color="white", linewidth=1.5)
        axis.tick_params(which="minor", bottom=False, left=False)
        axis.tick_params(axis="both", which="major", length=0, pad=6)
        for spine in axis.spines.values():
            spine.set_color("#5A5A5A")
            spine.set_linewidth(0.8)
    figure.suptitle(
        "External validation of MTAP deep-deletion prediction in GSE327167",
        fontsize=14,
        fontweight="bold",
        y=0.965,
    )
    colorbar_axis = figure.add_axes([0.945, 0.27, 0.014, 0.45])
    scalar_mappable = mpl.cm.ScalarMappable(
        norm=normalizer, cmap=PERFORMANCE_CMAP
    )
    colorbar = figure.colorbar(scalar_mappable, cax=colorbar_axis)
    colorbar.set_label(
        "Favourable performance\n(higher is better)", fontsize=9, labelpad=9
    )
    colorbar.ax.tick_params(labelsize=8)
    figure.text(
        0.12,
        0.035,
        f"Values are point estimates with stratified-bootstrap 95% CIs ({N_BOOTSTRAP:,} resamples). For Brier score, the displayed value is the raw score, whereas cell colour represents 1 − Brier so that all panels share the same favourable direction. NA indicates an undefined metric because no sample was predicted positive at the fixed 0.50 threshold.",
        ha="left",
        va="bottom",
        fontsize=8.4,
        color="#303030",
        wrap=True,
    )
    main_pdf = FIGURE_DIR / "GSE327167_three_model_all_metrics_heatmap.pdf"
    main_png = FIGURE_DIR / "GSE327167_three_model_all_metrics_heatmap.png"
    figure.savefig(main_pdf, bbox_inches="tight", facecolor="white")
    figure.savefig(main_png, dpi=400, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(figure)
    count_rows = []
    for evaluation_name in EVALUATION_ORDER:
        for model in MODELS:
            record = performance_ci.loc[
                performance_ci["evaluation_set"].eq(evaluation_name)
                & performance_ci["model"].eq(model)
            ].iloc[0]
            model_label = MODEL_LABELS[model].replace("\n", " ")
            count_rows.append(
                {
                    "evaluation_set": evaluation_name,
                    "model": model,
                    "row_label": f"{EVALUATION_SHORT_LABELS[evaluation_name].replace(chr(10), ' ')} | {model_label}",
                    "TP": int(record["TP"]),
                    "FP": int(record["FP"]),
                    "FN": int(record["FN"]),
                    "TN": int(record["TN"]),
                }
            )
    count_frame = pd.DataFrame(count_rows)
    count_columns = ["TP", "FP", "FN", "TN"]
    count_matrix = count_frame[count_columns].to_numpy(dtype=float)
    count_cmap = LinearSegmentedColormap.from_list(
        "MTAP_five_color_counts", FIVE_COLOR_PALETTE, N=256
    )
    count_figure, count_axis = plt.subplots(
        figsize=(8.2, 7.2), facecolor="white"
    )
    count_maximum = max(1, float(np.nanmax(count_matrix)))
    count_image = count_axis.imshow(
        count_matrix,
        cmap=count_cmap,
        aspect="auto",
        interpolation="nearest",
        vmin=0,
        vmax=count_maximum,
    )
    count_norm = Normalize(vmin=0, vmax=count_maximum)
    for row_index in range(count_matrix.shape[0]):
        for column_index in range(count_matrix.shape[1]):
            value = count_matrix[row_index, column_index]
            rgba = count_cmap(count_norm(value))
            count_axis.text(
                column_index,
                row_index,
                f"{int(value)}",
                ha="center",
                va="center",
                fontsize=9,
                color=contrast_text_color(rgba),
                fontweight="bold",
            )
    count_axis.set_xticks(np.arange(len(count_columns)))
    count_axis.set_xticklabels(count_columns, fontsize=10, fontweight="bold")
    count_axis.set_yticks(np.arange(len(count_frame)))
    count_axis.set_yticklabels(count_frame["row_label"], fontsize=8.2)
    count_axis.set_xticks(np.arange(-0.5, len(count_columns), 1), minor=True)
    count_axis.set_yticks(np.arange(-0.5, len(count_frame), 1), minor=True)
    count_axis.grid(which="minor", color="white", linewidth=1.5)
    count_axis.tick_params(which="minor", bottom=False, left=False)
    count_axis.tick_params(axis="both", which="major", length=0, pad=7)
    count_axis.set_title(
        "Classification counts at the fixed 0.50 threshold",
        fontsize=13,
        fontweight="bold",
        pad=13,
    )
    for spine in count_axis.spines.values():
        spine.set_color("#5A5A5A")
        spine.set_linewidth(0.8)
    count_colorbar = count_figure.colorbar(
        count_image, ax=count_axis, fraction=0.04, pad=0.03
    )
    count_colorbar.set_label("Number of patients", fontsize=9)
    count_colorbar.ax.tick_params(labelsize=8)
    count_figure.tight_layout()
    count_pdf = (
        FIGURE_DIR / "GSE327167_three_model_confusion_counts_heatmap.pdf"
    )
    count_png = (
        FIGURE_DIR / "GSE327167_three_model_confusion_counts_heatmap.png"
    )
    count_figure.savefig(count_pdf, bbox_inches="tight", facecolor="white")
    count_figure.savefig(
        count_png, dpi=400, bbox_inches="tight", facecolor="white"
    )
    plt.show()
    plt.close(count_figure)
    display_columns = ["evaluation_set", "model", "n_patients", "n_positive"]
    for metric in METRICS:
        display_columns.extend(
            [metric, f"{metric}_CI95_low", f"{metric}_CI95_high"]
        )
    _public_summary(
        globals(), "GSE327167 performance heatmaps and bootstrap intervals"
    )


#### GSE327167 performance and confusion-count heatmaps at 0.75


In [ ]:
@_nb9_registry.register("s76_53")
def _nb9_stage_s76_53():
    fig, axes = plt.subplots(2, 4, figsize=(17.4, 8.8))
    fig.subplots_adjust(
        left=0.12, right=0.93, bottom=0.16, top=0.88, wspace=0.16, hspace=0.32
    )
    norm = Normalize(0, 1)
    for panel, (metric, title) in enumerate(zip(METRICS, TITLES)):
        axis = axes.flat[panel]
        matrix = np.array(
            [
                [lookup.loc[(scope, model), metric] for model in MODELS]
                for scope in SCOPES
            ]
        )
        color_matrix = 1 - matrix if metric == "Brier" else matrix
        axis.imshow(
            np.ma.masked_invalid(color_matrix),
            cmap=CMAP,
            norm=norm,
            aspect="auto",
            interpolation="nearest",
        )
        for i, scope in enumerate(SCOPES):
            for j, model in enumerate(MODELS):
                record = lookup.loc[scope, model]
                value = record[metric]
                low = record[f"{metric}_CI95_low"]
                high = record[f"{metric}_CI95_high"]
                if np.isfinite(value):
                    annotation = f"{value:.3f}\n({low:.3f}–{high:.3f})"
                    color = text_color(color_matrix[i, j], norm, CMAP)
                else:
                    annotation = "NA"
                    color = "#505050"
                axis.text(
                    j,
                    i,
                    annotation,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color,
                    linespacing=1.35,
                )
        style_matrix_axis(axis)
        axis.set_yticklabels(row_labels if panel % 4 == 0 else [])
        axis.set_title(title, fontsize=11, fontweight="bold", pad=10)
        axis.text(
            -0.08,
            1.08,
            "ABCDEFGH"[panel],
            transform=axis.transAxes,
            fontsize=15,
            fontweight="bold",
        )
    fig.suptitle(
        "GSE327167 external validation | Classification threshold = 0.75",
        fontsize=14,
        fontweight="bold",
        y=0.96,
    )
    cax = fig.add_axes([0.945, 0.28, 0.014, 0.42])
    colorbar = fig.colorbar(
        mpl.cm.ScalarMappable(norm=norm, cmap=CMAP), cax=cax
    )
    colorbar.set_label(
        "Favourable performance\n(higher is better)", fontsize=9
    )
    fig.text(
        0.12,
        0.055,
        "Values: point estimate (stratified-bootstrap 95% CI). NA: undefined PPV when no patient is predicted positive.\nBrier annotations show raw scores; colours represent 1 − Brier. AUROC, average precision and Brier do not depend on the classification threshold.",
        fontsize=8.4,
        ha="left",
        va="bottom",
        linespacing=1.6,
    )
    for extension in ["pdf", "png"]:
        fig.savefig(
            PLOT_DIR / ("GSE327167_all_metrics_threshold_075." + extension),
            dpi=400,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, 3, figsize=(13.4, 4.4))
    fig.subplots_adjust(
        left=0.14, right=0.92, bottom=0.18, top=0.78, wspace=0.22
    )
    count_columns = ["TP", "FP", "FN", "TN"]
    count_max = max(1, int(result[count_columns].to_numpy().max()))
    count_norm = Normalize(vmin=0, vmax=count_max)
    for index, scope in enumerate(SCOPES):
        axis = axes[index]
        matrix = np.array(
            [
                [lookup.loc[(scope, model), count] for count in count_columns]
                for model in MODELS
            ],
            dtype=int,
        )
        image = axis.imshow(
            matrix,
            cmap=CMAP,
            norm=count_norm,
            aspect="auto",
            interpolation="nearest",
        )
        for i in range(len(MODELS)):
            for j in range(len(count_columns)):
                axis.text(
                    j,
                    i,
                    str(matrix[i, j]),
                    ha="center",
                    va="center",
                    fontsize=11,
                    fontweight="bold",
                    color=text_color(matrix[i, j], count_norm, CMAP),
                )
        axis.set_xticks(range(4))
        axis.set_xticklabels(count_columns, fontweight="bold")
        axis.set_yticks(range(3))
        axis.set_yticklabels(MODEL_LABELS if index == 0 else [])
        axis.set_xticks(np.arange(-0.5, 4, 1), minor=True)
        axis.set_yticks(np.arange(-0.5, 3, 1), minor=True)
        axis.grid(which="minor", color="white", linewidth=1.5)
        axis.tick_params(which="both", length=0, pad=7)
        frame = evaluation_sets[scope]
        axis.set_title(
            f"{SCOPE_LABELS[scope]}\nn={len(frame)}; OGM+={int(frame['OGM_deep_deletion_label'].sum())}",
            fontsize=10,
            fontweight="bold",
            pad=10,
        )
    fig.suptitle(
        "Classification counts at threshold 0.75",
        fontsize=13,
        fontweight="bold",
        y=0.97,
    )
    cax = fig.add_axes([0.94, 0.22, 0.013, 0.48])
    cb = fig.colorbar(image, cax=cax)
    cb.set_label("Number of patients")
    for extension in ["pdf", "png"]:
        fig.savefig(
            PLOT_DIR
            / ("GSE327167_confusion_counts_threshold_075." + extension),
            dpi=400,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    plt.close(fig)
    _public_summary(
        globals(), "GSE327167 three-model comparison at threshold 0.75"
    )


#### Final single-cell L2 coefficient extraction and visualization - figures and completion records


In [ ]:
@_nb9_registry.register("s78_70")
def _nb9_stage_s78_70():
    fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0))
    signed = table.copy()
    ranked = table.sort_values(
        "absolute_standardized_coefficient", ascending=False
    ).reset_index(drop=True)
    panels = [
        (
            axes[0],
            signed,
            "standardized_coefficient",
            "A  Signed coefficients",
            "Standardized coefficient",
        ),
        (
            axes[1],
            ranked,
            "absolute_standardized_coefficient",
            "B  Coefficient magnitude",
            "Absolute standardized coefficient",
        ),
    ]
    for axis, frame, column, title, xlabel in panels:
        values = frame[column].to_numpy()
        y = np.arange(len(frame))
        axis.barh(
            y,
            values,
            height=0.52,
            color=[COLORS[gene] for gene in frame["gene"]],
            edgecolor="white",
            linewidth=0.8,
        )
        extent = max(float(np.max(np.abs(values))), 0.1)
        if column == "standardized_coefficient":
            axis.set_xlim(-1.35 * extent, 1.35 * extent)
            axis.axvline(0, color="#505050", linewidth=0.9)
        else:
            axis.set_xlim(0, 1.35 * extent)
        for position, value in enumerate(values):
            offset = extent * 0.04
            label = (
                f"{value:+.3f}"
                if column == "standardized_coefficient"
                else f"{value:.3f}"
            )
            axis.text(
                value + (offset if value >= 0 else -offset),
                position,
                label,
                ha="left" if value >= 0 else "right",
                va="center",
                fontsize=10,
            )
        axis.set_yticks(y)
        axis.set_yticklabels(frame["gene"], fontstyle="italic")
        axis.invert_yaxis()
        axis.set_title(title, loc="left", fontweight="bold", pad=12)
        axis.set_xlabel(xlabel)
        axis.spines["top"].set_visible(False)
        axis.spines["right"].set_visible(False)
        axis.tick_params(axis="y", length=0)
        axis.xaxis.grid(True, color="#E2E2E2", linewidth=0.7)
        axis.set_axisbelow(True)
    fig.suptitle(
        "Three-gene L2 model used for GSE327167 validation",
        fontsize=12,
        fontweight="bold",
    )
    fig.tight_layout(rect=[0, 0.08, 1, 0.92])
    fig.text(
        0.5,
        0.025,
        "Frozen penalized coefficients; coefficient magnitude is not a causal effect or a contribution percentage.",
        ha="center",
        fontsize=8,
        color="#505050",
    )
    for extension in ["pdf", "png"]:
        fig.savefig(
            OUTPUT_DIR / ("three_gene_L2_coefficient_comparison." + extension),
            dpi=400,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.show()
    plt.close(fig)
    _public_summary(
        globals(),
        "Final single-cell L2 coefficient extraction and visualization",
    )


### L2 model interpretation


#### Final single-cell L2 coefficient extraction and visualization


In [ ]:
@_nb9_registry.register("s78_0")
def _nb9_stage_s78_0():
    _publication_previous_tables = {
        name: id(value)
        for name, value in list(globals().items())
        if isinstance(value, pd.DataFrame)
    }
    from pathlib import Path
    from datetime import datetime
    import hashlib
    import json
    import joblib
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from scipy.special import expit

    PROJECT = Path(str(PROJECT_ROOT))
    MODEL_FILE = (
        PROJECT
        / "data_processed/MTAP_RNA_prediction/16_scRNA_transfer/08_L2_cell_number_stability_and_candidate/models/GBM_snRNA_three_gene_L2_candidate.joblib"
    )
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    OUTPUT_DIR = (
        PROJECT
        / "data_processed/MTAP_prediction_registry/model_interpretation"
        / f"{RUN_ID}_GBM_snRNA_three_gene_L2"
    )
    GENES = ["MTAP", "CDKN2A", "CDKN2B"]
    COLORS = {"MTAP": "#33539E", "CDKN2A": "#BFB8DA", "CDKN2B": "#A5678E"}

    def require(condition, message):
        if not bool(condition):
            raise RuntimeError(message)

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                digest.update(block)
        return digest.hexdigest()

    require(
        MODEL_FILE.is_file(),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: MODEL_FILE.is_file().",
    )
    bundle = joblib.load(MODEL_FILE)
    require(
        isinstance(bundle, dict),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: isinstance(bundle, dict).",
    )
    require(
        "pipeline" in bundle and "feature_names" in bundle,
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: 'pipeline' in bundle and 'feature_names' in bundle.",
    )
    pipeline = bundle["pipeline"]
    feature_names = list(bundle["feature_names"])
    require(
        feature_names == GENES,
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: feature_names == GENES.",
    )
    require(
        bundle.get("expression_unit") == "log2(CPM+1)",
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: bundle.get('expression_unit') == 'log2(CPM+1)'.",
    )
    require(
        isinstance(pipeline, Pipeline),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: isinstance(pipeline, Pipeline).",
    )
    classifier = pipeline.steps[-1][1]
    require(
        isinstance(classifier, LogisticRegression),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: isinstance(classifier, LogisticRegression).",
    )
    preprocessors = []
    for name, step in pipeline.steps[:-1]:
        if step is None:
            continue
        if isinstance(step, str) and step == "passthrough":
            continue
        preprocessors.append((name, step))
    require(
        len(preprocessors) == 1
        and isinstance(preprocessors[0][1], StandardScaler),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: len(preprocessors) == 1 and isinstance(preprocessors[0][1], StandardScaler).",
    )
    scaler = preprocessors[0][1]
    require(
        scaler.with_std,
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: scaler.with_std.",
    )
    require(
        classifier.coef_.shape == (1, 3),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: classifier.coef_.shape == (1, 3).",
    )
    require(
        np.array_equal(classifier.classes_, np.array([0, 1])),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: np.array_equal(classifier.classes_, np.array([0, 1])).",
    )
    for name, estimator in [
        ("pipeline", pipeline),
        ("scaler", scaler),
        ("classifier", classifier),
    ]:
        if hasattr(estimator, "feature_names_in_"):
            require(
                list(estimator.feature_names_in_) == feature_names,
                "Final single-cell L2 coefficient extraction and visualization: required condition failed: list(estimator.feature_names_in_) == feature_names.",
            )
    params = classifier.get_params()
    is_l2 = params.get("penalty") == "l2" or (
        params.get("penalty") == "elasticnet" and params.get("l1_ratio") == 0
    )
    require(
        is_l2,
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: is_l2.",
    )
    for key in ["penalty", "C", "solver", "class_weight", "fit_intercept"]:
        pass
    beta = np.asarray(classifier.coef_[0], dtype=float)
    intercept = float(classifier.intercept_[0])
    scale = np.asarray(scaler.scale_, dtype=float)
    center = (
        np.asarray(scaler.mean_, dtype=float)
        if scaler.with_mean
        else np.zeros(3)
    )
    require(
        scale.shape == (3,)
        and center.shape == (3,)
        and np.isfinite(scale).all()
        and np.isfinite(center).all()
        and np.isfinite(beta).all()
        and np.isfinite(intercept)
        and (scale > 0).all(),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: scale.shape == (3,) and center.shape == (3,) and np.isfinite(scale).all() and np.isfinite(center).all() and np.isfinite(beta).all() and np.isfinite(intercept) and (scale > 0).all().",
    )
    beta_expression = beta / scale
    intercept_expression = float(intercept - np.sum(beta * center / scale))
    with np.errstate(over="ignore"):
        model_or = np.exp(beta)
    table = pd.DataFrame(
        {
            "gene": feature_names,
            "standardization_center": center,
            "standardization_scale": scale,
            "standardized_coefficient": beta,
            "absolute_standardized_coefficient": np.abs(beta),
            "model_OR_per_scaled_unit": model_or,
            "expression_scale_coefficient": beta_expression,
        }
    )
    table["direction"] = [
        (
            "Higher expression increases predicted deletion probability"
            if value > 0
            else (
                "Higher expression decreases predicted deletion probability"
                if value < 0
                else "Zero coefficient"
            )
        )
        for value in beta
    ]
    rng = np.random.default_rng(20261106)
    test_x = center + rng.normal(size=(20, 3)) * scale
    test_input = (
        pd.DataFrame(test_x, columns=feature_names)
        if hasattr(pipeline, "feature_names_in_")
        else test_x
    )
    model_logit = pipeline.decision_function(test_input)
    formula_logit = intercept_expression + test_x @ beta_expression
    require(
        np.allclose(model_logit, formula_logit, rtol=1e-08, atol=1e-08),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: np.allclose(model_logit, formula_logit, rtol=1e-08, atol=1e-08).",
    )
    require(
        np.allclose(
            pipeline.predict_proba(test_input)[:, 1],
            expit(formula_logit),
            rtol=1e-08,
            atol=1e-08,
        ),
        "Final single-cell L2 coefficient extraction and visualization: required condition failed: np.allclose(pipeline.predict_proba(test_input)[:, 1], expit(formula_logit), rtol=1e-08, atol=1e-08).",
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    standardized_terms = " ".join(
        (
            f"{value:+.10g} * z_{gene}"
            for gene, value in zip(feature_names, beta)
        )
    )
    expression_terms = " ".join(
        (
            f"{value:+.10g} * x_{gene}"
            for gene, value in zip(feature_names, beta_expression)
        )
    )
    formula = f"Model: {MODEL_FILE}\nSHA256: {sha256_file(MODEL_FILE)}\nPositive class: 1 = MTAP deep deletion\nExpression unit: log2(CPM+1)\n\nStandardized input:\nz_j = (x_j - center_j) / scale_j\nS = {intercept:.10g} {standardized_terms}\n\nEquivalent expression-scale equation:\nS = {intercept_expression:.10g} {expression_terms}\n\nP(deletion) = 1 / (1 + exp(-S))\nV5 classification rule described in this project: P(deletion) >= 0.75\nThe classification threshold does not alter coefficients.\n"
    table.to_csv(
        OUTPUT_DIR / "three_gene_L2_coefficients.tsv", sep="\t", index=False
    )
    (OUTPUT_DIR / "three_gene_L2_formula.txt").write_text(
        formula, encoding="utf-8"
    )
    audit = {
        "model_file": str(MODEL_FILE),
        "model_sha256": sha256_file(MODEL_FILE),
        "feature_order": feature_names,
        "expression_unit": bundle["expression_unit"],
        "standardized_intercept": intercept,
        "expression_scale_intercept": intercept_expression,
        "penalty": params.get("penalty"),
        "C": float(params["C"]),
        "formula_verified": True,
        "model_refitted": False,
    }
    (OUTPUT_DIR / "model_coefficient_audit.json").write_text(
        json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    plt.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 10,
            "axes.linewidth": 0.9,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "axes.unicode_minus": False,
        }
    )


### Cross-cohort metric heatmaps


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize


def _nb9_plot_metrics(performance, project_root):
    """Show comparable metric panels with raw numeric annotations."""
    if performance.empty:
        return
    palette = ["#33539E", "#7FACD6", "#BFB8DA", "#E8B7D4", "#A5678E"]
    cmap = LinearSegmentedColormap.from_list("MTAP_nine_part", palette)
    cmap.set_bad("#ECECEC")
    models = ["L2", "Elastic Net", "DL"]
    cohorts = performance["cohort"].drop_duplicates().tolist()
    metrics = [
        ("AUROC", "AUROC", True),
        ("AUPRC_AP", "Average precision", True),
        ("Brier", "Brier score", False),
        ("Log_loss", "Log loss", False),
        ("Accuracy", "Accuracy", True),
        ("Balanced_accuracy", "Balanced accuracy", True),
        ("Sensitivity", "Sensitivity", True),
        ("Specificity", "Specificity", True),
        ("PPV", "PPV", True),
        ("NPV", "NPV", True),
        ("F1", "F1", True),
        ("MCC", "MCC", True),
    ]
    output = project_root / "outputs/nine_part_model_comparison/figures"
    output.mkdir(parents=True, exist_ok=True)
    for threshold in (0.50, 0.75):
        subset = performance.loc[performance["threshold"].eq(threshold)]
        figure, axes = plt.subplots(
            3, 4, figsize=(18, max(10, len(cohorts) * 1.25))
        )
        figure.subplots_adjust(
            left=0.18,
            right=0.98,
            bottom=0.08,
            top=0.91,
            hspace=0.5,
            wspace=0.42,
        )
        for panel, (metric, title, higher_better) in enumerate(metrics):
            axis = axes.flat[panel]
            matrix = subset.pivot(
                index="cohort", columns="model", values=metric
            ).reindex(index=cohorts, columns=models)
            raw = matrix.to_numpy(dtype=float)
            if metric == "MCC":
                normalizer = Normalize(-1, 1)
            elif metric == "Log_loss":
                normalizer = Normalize(0, max(1.0, float(np.nanmax(raw))))
            else:
                normalizer = Normalize(0, 1)
            panel_cmap = cmap if higher_better else cmap.reversed()
            panel_cmap.set_bad("#ECECEC")
            image = axis.imshow(
                np.ma.masked_invalid(raw),
                cmap=panel_cmap,
                norm=normalizer,
                aspect="auto",
            )
            for row in range(len(cohorts)):
                for column in range(len(models)):
                    value = raw[row, column]
                    annotation = (
                        "NA" if not np.isfinite(value) else f"{value:.3f}"
                    )
                    rgba = (
                        panel_cmap(normalizer(value))
                        if np.isfinite(value)
                        else (0.9, 0.9, 0.9, 1)
                    )
                    luminance = (
                        0.2126 * rgba[0] + 0.7152 * rgba[1] + 0.0722 * rgba[2]
                    )
                    axis.text(
                        column,
                        row,
                        annotation,
                        ha="center",
                        va="center",
                        fontsize=7.5,
                        color="white" if luminance < 0.5 else "#202020",
                    )
            axis.set_xticks(range(3), models, fontsize=8)
            axis.set_yticks(range(len(cohorts)))
            axis.set_yticklabels(
                cohorts if panel % 4 == 0 else [""] * len(cohorts),
                fontsize=7.7,
            )
            axis.set_xticks(np.arange(-0.5, 3, 1), minor=True)
            axis.set_yticks(np.arange(-0.5, len(cohorts), 1), minor=True)
            axis.grid(which="minor", color="white", linewidth=1)
            axis.tick_params(which="both", length=0)
            direction = "higher" if higher_better else "lower"
            axis.set_title(
                f"{chr(65 + panel)}  {title}\n({direction} is better)",
                fontsize=9,
            )
            figure.colorbar(
                image, ax=axis, fraction=0.035, pad=0.025
            ).ax.tick_params(labelsize=6)
        figure.suptitle(
            f"Patient-level MTAP prediction across cohorts | threshold = {threshold:.2f}",
            fontsize=14,
            fontweight="bold",
        )
        figure.text(
            0.18,
            0.025,
            "Source-specific patient sets; TCGA development uses OOF scores. Raw point estimates are shown. NA denotes an undefined metric.",
            fontsize=8,
        )
        stem = f"all_cohort_three_model_metrics_threshold_{int(threshold * 100):03d}"
        figure.savefig(output / f"{stem}.pdf", bbox_inches="tight")
        figure.savefig(output / f"{stem}.png", dpi=300, bbox_inches="tight")
        plt.show()
        if "inline" not in plt.get_backend().lower():
            display(figure)
        plt.close(figure)


### Execute the declared workflow and summarize available predictions

The default registers all scientific code and reads existing result
files only. Set `MTAP_RUN_WORKFLOW=1` before launching Jupyter to
execute source analyses in their original dependency order.
An interrupted workflow resumes after completed source segments in
the same kernel. Scientific checkpoints retain their source behavior.


In [ ]:
_nb9_registry.order = [
    "s02_0",
    "s03a_0",
    "s03b_0",
    "s05_0",
    "s05_95",
    "s06_0",
    "s06_65",
    "s07_0",
    "s07_73",
    "s09_0",
    "s10_0",
    "s12_0",
    "s12_97",
    "s14_0",
    "s14_115",
    "s15_0",
    "s17_0",
    "s19_0",
    "s21_0",
    "s22_0",
    "s23_0",
    "s24_0",
    "s24_140",
    "s26_0",
    "s26_104",
    "s27_0",
    "s29_0",
    "s30_0",
    "s31_0",
    "s32_0",
    "s34_0",
    "s35_0",
    "s37_0",
    "s37_113",
    "s37_144",
    "s39_0",
    "s39_95",
    "s39_120",
    "s40_0",
    "s40_104",
    "s41_0",
    "s41_96",
    "s42_0",
    "s42_61",
    "s45_0",
    "s46_0",
    "s47_0",
    "s48_0",
    "s49_0",
    "s50_0",
    "s51_0",
    "s53_0",
    "s55_0",
    "s57_0",
    "s57_116",
    "s59_0",
    "s59_83",
    "s60_0",
    "s62_0",
    "s62_105",
    "s62_120",
    "s64_0",
    "s64_68",
    "s66_0",
    "s66_121",
    "s67_0",
    "s67_32",
    "s68_0",
    "s68_162",
    "s70_0",
    "s70_77",
    "s72_0",
    "s72_117_l2",
    "s72_117_en",
    "s72_118",
    "s72_125",
    "s72_129",
    "s72_152",
    "s73_gse327167_setup",
    "s74_0",
    "s74_129",
    "s74_135",
    "s75_0",
    "s75_53",
    "s76_0",
    "s76_53",
    "s78_0",
    "s78_70",
]

if RUN_FULL_WORKFLOW:
    _nb9_registry.run_all()
unified_performance, prediction_availability = _nb9_collect_metrics(
    PROJECT_ROOT
)
_nb9_display_metrics(unified_performance, prediction_availability)
_nb9_plot_metrics(unified_performance, PROJECT_ROOT)
